<a href="https://colab.research.google.com/github/MWANIKID/Signal-or-Redundancy-The-Incremental-Value-of-Technical-Indicators-/blob/main/Forecasting_Sectoral_Crash_Risk_and_Contagion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# =============================================================================
# PYTHON PHASE 1 — DIRECTLY FROM THE FINAL R HANDOFF
#
# Forecasting Sectoral Crash Risk and Contagion:
# Integrating Dynamic Volatility, Extreme-Value Modelling and Graph Deep Learning
#
# INPUT
# -----
# 04_Python_Handoff.zip generated by the final corrected R script.
#
# This is intentionally the FIRST Python script. It does not depend on any
# earlier Python ZIP. It starts directly from the frozen R econometric outputs.
#
# WHAT THIS SCRIPT DOES
# ---------------------
#  1. Validates and imports the final R handoff by CONTENT.
#  2. Audits the corrected R GJR-GARCH / EVT / sector-eligibility outputs.
#  3. Uses the frozen R Crash_Main = 2.5% EVT crash definition.
#  4. Constructs censored time-to-next-crash targets at 1, 5, 10 and 22 days.
#  5. Fits nested discrete-time Hawkes models on TRAIN only:
#       - baseline intensity,
#       - self-exciting Hawkes,
#       - full multivariate Hawkes.
#  6. Selects Hawkes half-life by TRAIN BIC.
#  7. Tunes cross-sector L1 regularization using temporal CV WITHIN TRAIN only.
#  8. Performs block stability selection of directed cross-sector edges.
#  9. Re-fits the stability-selected support without L1 shrinkage.
# 10. Evaluates Validation using TRAIN-fitted coefficients.
# 11. Re-fits coefficients on TRAIN+VALIDATION with all hyperparameters/support
#     frozen, then evaluates the untouched TEST sample.
# 12. Creates split-aware dynamic Hawkes graphs for the later graph DL models.
# 13. Exports baseline forecasts, tables, figures, Excel, model objects and ZIP.
# 14. Automatically downloads the final ZIP in Google Colab.
#
# IMPORTANT CORRECTIONS INCORPORATED
# ----------------------------------
# * No Python-to-Python input dependency: starts from 04_Python_Handoff.zip.
# * R EVT labels are checked for parameter variation and threshold ordering.
# * No arbitrary re-estimation of R crash labels in Python.
# * Time-to-crash outcomes are censored when future sector-day labels are absent.
# * Historical baselines are strictly past-only; no overlapping-horizon leakage.
# * Hawkes multi-horizon survival probabilities use the correct t+1 state:
#       k=1 uses the post-event state at forecast origin t with NO extra decay.
# * Hawkes structure/penalty/support decisions never use Test outcomes.
# * Test coefficients are refitted on Train+Validation only AFTER all structural
#   decisions have been frozen.
# * If stable cross-sector edges do not survive, the script records that result
#   instead of manufacturing a graph.
#
# VERSION: 1.2
# DATE: 2026-09-01
# =============================================================================

from __future__ import annotations

import os
import sys
import json
import random
import shutil
import pickle
import zipfile
import warnings
import platform
import importlib.util
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from scipy.optimize import minimize

from sklearn.metrics import average_precision_score, roc_auc_score, mean_pinball_loss
from sklearn.linear_model import QuantileRegressor

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =============================================================================
# 0. REPRODUCIBILITY AND CONFIGURATION
# =============================================================================

SEED = 20260901
np.random.seed(SEED)
random.seed(SEED)

INPUT_ZIP = os.getenv(
    "NSE_R_HANDOFF_ZIP",
    "/content/04_Python_Handoff.zip"
)

OUTPUT_ROOT = Path(os.getenv(
    "NSE_PYTHON_PHASE1_OUTPUT_DIR",
    "/content/Sectoral_Crash_Risk_Contagion_Python_Phase1_v1_2"
))

HORIZONS = [1, 5, 10, 22]
MAX_H = max(HORIZONS)

# Hawkes memory candidates used for TRAIN-only BIC selection.
HALF_LIFE_CANDIDATES = [1, 2, 5, 10, 22]

# L1 grid: mean NLL + lambda * sum(cross-sector alpha).
LAMBDA_GRID = [0.0, 0.005, 0.01, 0.02, 0.05, 0.10, 0.20, 0.50]

# Primary Hawkes stability selection now uses lambda_min (the CV-loss minimizer).
# The more conservative 1-SE penalty is retained as a robustness check only.
STABILITY_REPS_CONSERVATIVE = int(
    os.getenv("NSE_STABILITY_REPS_CONSERVATIVE", "50")
)

# Fallback lower-tail quantile graph. It is activated only when no Hawkes
# cross-sector edge survives the pre-specified 60% stability threshold.
TAIL_QUANTILE = 0.05
TAIL_QUANTILE_ALPHA_GRID = [0.0, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05]
TAIL_GRAPH_STABILITY_REPS = int(
    os.getenv("NSE_TAIL_GRAPH_STABILITY_REPS", "100")
)
TAIL_GRAPH_STABILITY_THRESHOLD = 0.60
TAIL_BETA_TOL = 1e-5

# Temporal CV inside TRAIN only.
N_TEMPORAL_FOLDS = 3
INITIAL_TRAIN_FRACTION = 0.55
VALIDATION_BLOCK_FRACTION = 0.15

# Stability selection defaults. Environment variables allow a quick diagnostic
# run without modifying the submitted code.
STABILITY_REPS_PRIMARY = int(
    os.getenv("NSE_STABILITY_REPS_PRIMARY", "100")
)
STABILITY_REPS_SENSITIVITY = int(
    os.getenv("NSE_STABILITY_REPS_SENSITIVITY", "50")
)
STABILITY_SUBSAMPLE_FRACTION = 0.70
STABILITY_BLOCK_LENGTH = 60
STABILITY_THRESHOLD = 0.60
EDGE_NUMERIC_TOL = 1e-5

# Optimization.
EPS = 1e-10
MU_BOUNDS = (1e-8, 1.0)
ALPHA_BOUNDS = (0.0, 10.0)
FINAL_MULTISTARTS = int(
    os.getenv("NSE_HAWKES_MULTISTARTS", "5")
)
MAXITER = 3000

# Useful discrete-kernel stability guard:
# B = A / (1 - decay)
MAX_BRANCHING_SPECTRAL_RADIUS = 0.98

# Validation evidence is reported separately from edge-existence.
MIN_VALIDATION_SKILL_HORIZONS = 2

# =============================================================================
# 1. OUTPUT FOLDERS
# =============================================================================

TABLE_DIR = OUTPUT_ROOT / "01_Tables"
FIG_DIR = OUTPUT_ROOT / "02_Figures"
EXCEL_DIR = OUTPUT_ROOT / "03_Excel"
MODEL_DIR = OUTPUT_ROOT / "04_Model_Objects"
DATA_DIR = OUTPUT_ROOT / "05_Processed_Data"
LOG_DIR = OUTPUT_ROOT / "06_Logs"
ZIP_DIR = OUTPUT_ROOT / "07_Zip"
EXTRACT_DIR = OUTPUT_ROOT / "_R_Handoff_Extracted"

for d in [
    OUTPUT_ROOT, TABLE_DIR, FIG_DIR, EXCEL_DIR, MODEL_DIR,
    DATA_DIR, LOG_DIR, ZIP_DIR, EXTRACT_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / "Python_Phase1_run_log.txt"

def log(message: str):
    stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{stamp}] {message}"
    print(line)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")

log("=" * 96)
log("PYTHON PHASE 1 START — DIRECT R HANDOFF")
log(f"Python: {sys.version.split()[0]}")
log(f"Platform: {platform.platform()}")
log(f"Seed: {SEED}")

# =============================================================================
# 2. VALIDATE / UPLOAD FINAL R HANDOFF ZIP
# =============================================================================

REQUIRED_R_HANDOFF_MEMBERS = {
    "sector_forecasting_master.csv.gz",
    "evt_parameter_history.csv.gz",
    "study_metadata.csv.gz",
    "stock_clean_internal_features.csv.gz",
}

def zip_basenames(path: Path) -> set[str]:
    try:
        with zipfile.ZipFile(path, "r") as zf:
            return {
                Path(name).name
                for name in zf.namelist()
                if not name.endswith("/")
            }
    except Exception:
        return set()

def is_valid_r_handoff(path: Path) -> bool:
    if not path.exists() or path.suffix.lower() != ".zip":
        return False
    return REQUIRED_R_HANDOFF_MEMBERS.issubset(zip_basenames(path))

def explain_rejected_zip(path: Path):
    names = zip_basenames(path)
    missing = sorted(REQUIRED_R_HANDOFF_MEMBERS - names)
    log(
        f"Rejected ZIP '{path.name}'. It is not the final R Python handoff. "
        f"Missing required files: {missing}"
    )

def resolve_r_handoff(configured: str) -> Path:
    """
    In Google Colab, ALWAYS ask the user to upload the final R handoff ZIP.
    The script will not silently use any ZIP already present in /content.

    Outside Colab, it falls back to the explicitly configured path.
    """

    try:
        from google.colab import files

        while True:
            print(
                "\nPlease upload the FINAL R handoff file:\n"
                "    04_Python_Handoff.zip\n"
            )

            uploaded = files.upload()

            zip_names = [
                name for name in uploaded
                if name.lower().endswith(".zip")
            ]

            if not zip_names:
                print(
                    "\nNo ZIP file was selected. "
                    "Please choose 04_Python_Handoff.zip.\n"
                )
                continue

            valid_uploaded = []

            for name in zip_names:
                candidate = Path("/content") / name

                if is_valid_r_handoff(candidate):
                    valid_uploaded.append(candidate)
                else:
                    explain_rejected_zip(candidate)

            if len(valid_uploaded) == 1:
                chosen = valid_uploaded[0]
                log(
                    f"Validated uploaded R handoff: {chosen}"
                )
                return chosen

            if len(valid_uploaded) > 1:
                print(
                    "\nMore than one valid R handoff ZIP was uploaded. "
                    "Please upload only the single file "
                    "'04_Python_Handoff.zip'.\n"
                )
                continue

            print(
                "\nThe selected ZIP is not the final R handoff.\n"
                "Please choose the file generated by the final corrected R run:\n"
                "    04_Python_Handoff.zip\n"
            )

    except ImportError:
        # Non-Colab execution: use only the explicitly configured path.
        configured_path = Path(configured)

        if configured_path.exists() and is_valid_r_handoff(configured_path):
            log(
                f"Non-Colab execution: using configured R handoff: "
                f"{configured_path}"
            )
            return configured_path

        if configured_path.exists():
            explain_rejected_zip(configured_path)

        raise FileNotFoundError(
            "Not running in Google Colab and the configured R handoff "
            "could not be validated. Set NSE_R_HANDOFF_ZIP to the exact "
            "path of 04_Python_Handoff.zip."
        )

INPUT_ZIP_PATH = resolve_r_handoff(INPUT_ZIP)
log(f"Accepted R handoff: {INPUT_ZIP_PATH}")

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(INPUT_ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

def find_one(filename: str) -> Path:
    matches = list(EXTRACT_DIR.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one '{filename}' after extraction; "
            f"found {len(matches)}."
        )
    return matches[0]

MASTER_FILE = find_one("sector_forecasting_master.csv.gz")
EVT_HISTORY_FILE = find_one("evt_parameter_history.csv.gz")
META_FILE = find_one("study_metadata.csv.gz")

# =============================================================================
# 3. LOAD FINAL R OUTPUTS
# =============================================================================

master = pd.read_csv(MASTER_FILE, parse_dates=["Date"])
evt_history = pd.read_csv(EVT_HISTORY_FILE, parse_dates=["RefitDate"])
metadata_df = pd.read_csv(META_FILE)

metadata = dict(zip(
    metadata_df["Key"].astype(str),
    metadata_df["Value"].astype(str)
))

master = master.sort_values(["Date", "Sector"]).reset_index(drop=True)

def to_bool(s: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False)
    return (
        s.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "t", "1", "yes", "y"])
    )

master["PrimarySector"] = to_bool(master["PrimarySector"])
master["SectorDayEligible"] = to_bool(master["SectorDayEligible"])

required_master_cols = [
    "Date", "Sector", "Split", "PrimarySector",
    "SectorDayEligible", "SectorReturn_Model",
    "GARCH_Sigma", "StdInnovation",
    "EVT_u", "EVT_scale", "EVT_shape",
    "CrashThreshold_001", "CrashThreshold_0025", "CrashThreshold_005",
    "Crash_001", "Crash_0025", "Crash_005", "Crash_Main",
]
missing_cols = [
    c for c in required_master_cols
    if c not in master.columns
]
if missing_cols:
    raise ValueError(
        f"R handoff is missing required columns: {missing_cols}"
    )

primary_sectors = sorted(
    master.loc[master["PrimarySector"], "Sector"]
    .dropna()
    .unique()
    .tolist()
)

if len(primary_sectors) < 3:
    raise RuntimeError(
        f"Only {len(primary_sectors)} primary sectors found."
    )

log(f"Primary sectors ({len(primary_sectors)}): {primary_sectors}")
log(
    f"R master: {len(master):,} rows, "
    f"{master['Date'].nunique():,} dates, "
    f"{master['Sector'].nunique()} total sectors."
)

# =============================================================================
# 4. AUDIT THE CORRECTED R EVT OUTPUT BEFORE USING CRASH LABELS
# =============================================================================

primary = (
    master[master["PrimarySector"]]
    .copy()
    .sort_values(["Date", "Sector"])
    .reset_index(drop=True)
)

# 4.1 Crash_Main must be exactly the 2.5% EVT label wherever both exist.
both = primary["Crash_Main"].notna() & primary["Crash_0025"].notna()
main_mismatch = int(
    (
        primary.loc[both, "Crash_Main"].astype(float).to_numpy()
        != primary.loc[both, "Crash_0025"].astype(float).to_numpy()
    ).sum()
)
if main_mismatch:
    raise RuntimeError(
        f"Crash_Main differs from Crash_0025 in {main_mismatch} rows."
    )

# 4.2 Threshold ordering.
thr = primary[
    ["CrashThreshold_001", "CrashThreshold_0025", "CrashThreshold_005"]
].dropna()

threshold_order_ok = bool(
    (
        (thr["CrashThreshold_001"] <= thr["CrashThreshold_0025"])
        & (thr["CrashThreshold_0025"] <= thr["CrashThreshold_005"])
    ).all()
)

if not threshold_order_ok:
    raise RuntimeError(
        "EVT threshold ordering 1% <= 2.5% <= 5% failed."
    )

# 4.3 Corrected EVT must show real parameter variation.
shape_nonmissing = primary["EVT_shape"].dropna()
scale_nonmissing = primary["EVT_scale"].dropna()

if len(shape_nonmissing) == 0 or len(scale_nonmissing) == 0:
    raise RuntimeError("No non-missing EVT parameters found.")

shape_sd = float(shape_nonmissing.std())
scale_sd = float(scale_nonmissing.std())

if shape_sd < 1e-6 or scale_sd < 1e-6:
    raise RuntimeError(
        "EVT parameter-variation guard failed. "
        "The handoff may be from the defective pre-correction R run."
    )

# 4.4 All observed crash labels binary.
for c in ["Crash_001", "Crash_0025", "Crash_005", "Crash_Main"]:
    values = primary[c].dropna()
    if not values.isin([0, 1]).all():
        raise RuntimeError(f"{c} is not binary.")

r_audit = pd.DataFrame({
    "Check": [
        "Crash_Main equals Crash_0025",
        "EVT thresholds ordered 1% <= 2.5% <= 5%",
        "EVT shape parameter varies",
        "EVT scale parameter varies",
        "Primary sectors >= 3",
    ],
    "Passed": [
        main_mismatch == 0,
        threshold_order_ok,
        shape_sd >= 1e-6,
        scale_sd >= 1e-6,
        len(primary_sectors) >= 3,
    ],
    "Value": [
        main_mismatch,
        int(threshold_order_ok),
        shape_sd,
        scale_sd,
        len(primary_sectors),
    ],
})
r_audit.to_csv(
    TABLE_DIR / "Table_P01_R_Handoff_Integrity.csv",
    index=False
)

# 4.5 Sector audit.
sector_audit = (
    primary.groupby("Sector", as_index=False)
    .agg(
        Rows=("Date", "size"),
        EligibleDays=("SectorDayEligible", "sum"),
        ObservedCrashLabels=("Crash_Main", lambda x: x.notna().sum()),
        CrashEvents=("Crash_Main", lambda x: np.nansum(x)),
        MedianStocksReturn=("NStocksReturn", "median"),
        MedianConstituentShare=("ConstituentReturnShare", "median"),
        EVTShapeMean=("EVT_shape", "mean"),
        EVTShapeSD=("EVT_shape", "std"),
    )
)
sector_audit["EligibleCoverage"] = (
    sector_audit["EligibleDays"] / sector_audit["Rows"]
)
sector_audit["CrashRate"] = (
    sector_audit["CrashEvents"]
    / sector_audit["ObservedCrashLabels"].replace(0, np.nan)
)
sector_audit.to_csv(
    TABLE_DIR / "Table_P02_Primary_Sector_Audit.csv",
    index=False
)

# =============================================================================
# 5. ALIGN PRIMARY-SECTOR EVENT PANEL
# =============================================================================

sectors = primary_sectors
S = len(sectors)

dates = pd.DatetimeIndex(sorted(primary["Date"].unique()))
T = len(dates)

sector_to_idx = {sector: i for i, sector in enumerate(sectors)}
date_to_idx = {pd.Timestamp(date): i for i, date in enumerate(dates)}

if primary.duplicated(["Date", "Sector"]).any():
    raise RuntimeError("Duplicate Date-Sector rows found.")

C = np.zeros((T, S), dtype=float)
M = np.zeros((T, S), dtype=bool)

split_map = (
    primary[["Date", "Split"]]
    .drop_duplicates()
    .set_index("Date")["Split"]
    .to_dict()
)
splits = np.array(
    [str(split_map[pd.Timestamp(d)]) for d in dates],
    dtype=object
)

for row in primary[
    ["Date", "Sector", "Crash_Main"]
].itertuples(index=False):
    t = date_to_idx[pd.Timestamp(row.Date)]
    s = sector_to_idx[row.Sector]

    if pd.notna(row.Crash_Main):
        C[t, s] = float(row.Crash_Main)
        M[t, s] = True

train_date_mask = splits == "Train"
validation_date_mask = splits == "Validation"
test_date_mask = splits == "Test"
trainval_date_mask = train_date_mask | validation_date_mask

log(
    f"Event panel: {T} dates x {S} sectors. "
    f"Train crashes={int(C[train_date_mask].sum())}; "
    f"Validation crashes={int(C[validation_date_mask].sum())}; "
    f"Test crashes={int(C[test_date_mask].sum())}."
)

# =============================================================================
# 6. CENSORED TIME-TO-NEXT-CRASH TARGETS
# =============================================================================

event_observed = np.full((T, S), np.nan)
event_time = np.full((T, S), np.nan)
censor_time = np.full((T, S), np.nan)

target_within = {
    h: np.full((T, S), np.nan, dtype=float)
    for h in HORIZONS
}

for t in range(T):
    for s in range(S):

        # Forecast origin is usable only when current crash state is observed.
        if not M[t, s]:
            continue

        max_available = min(MAX_H, T - 1 - t)
        if max_available <= 0:
            continue

        observed_until = 0
        found_event = False
        event_k = None

        for k in range(1, max_available + 1):
            if not M[t + k, s]:
                break

            observed_until = k

            if C[t + k, s] == 1:
                found_event = True
                event_k = k
                break

        if found_event:
            event_observed[t, s] = 1.0
            event_time[t, s] = float(event_k)
            censor_time[t, s] = float(event_k)
        else:
            event_observed[t, s] = 0.0
            censor_time[t, s] = float(observed_until)

        for h in HORIZONS:
            if found_event and event_k <= h:
                target_within[h][t, s] = 1.0
            elif observed_until >= h:
                target_within[h][t, s] = 0.0
            else:
                target_within[h][t, s] = np.nan

target_rows = []

for t, date in enumerate(dates):
    for s, sector in enumerate(sectors):
        row = {
            "Date": date,
            "Sector": sector,
            "Split": splits[t],
            "CurrentCrashObserved": int(M[t, s]),
            "CurrentCrash": C[t, s] if M[t, s] else np.nan,
            "EventObservedWithin22": event_observed[t, s],
            "EventTime": event_time[t, s],
            "CensorTime": censor_time[t, s],
        }

        for h in HORIZONS:
            row[f"CrashWithin_{h}"] = target_within[h][t, s]

        target_rows.append(row)

targets_long = pd.DataFrame(target_rows)

target_summary_rows = []
for split_name in ["Train", "Validation", "Test"]:
    for sector in sectors:
        sub = targets_long[
            (targets_long["Split"] == split_name)
            & (targets_long["Sector"] == sector)
        ]
        for h in HORIZONS:
            y = sub[f"CrashWithin_{h}"].dropna()
            target_summary_rows.append({
                "Split": split_name,
                "Sector": sector,
                "Horizon": h,
                "ValidTargets": len(y),
                "PositiveTargets": int(y.sum()) if len(y) else 0,
                "PositiveRate": float(y.mean()) if len(y) else np.nan,
            })

target_summary = pd.DataFrame(target_summary_rows)
target_summary.to_csv(
    TABLE_DIR / "Table_P03_Time_To_Crash_Target_Summary.csv",
    index=False
)

# =============================================================================
# 7. DISCRETE-TIME HAWKES FUNCTIONS
# =============================================================================
#
# Predictive process:
#
#   H_j,t = d H_j,t-1 + C_j,t-1
#   lambda_i,t = mu_i + sum_j alpha_i,j H_j,t
#   P(C_i,t=1 | F_t-1) = 1 - exp(-lambda_i,t)
#
# alpha[i,j] is directed predictive excitation FROM source j TO receiver i.

def decay_from_half_life(half_life: float) -> float:
    return float(np.exp(-np.log(2.0) / float(half_life)))

def build_pre_event_state(
    events: np.ndarray,
    decay: float,
) -> np.ndarray:
    T_, S_ = events.shape
    H = np.zeros((T_, S_), dtype=float)
    state = np.zeros(S_, dtype=float)

    for t in range(T_):
        H[t] = state
        state = decay * state + events[t]

    return H

def build_post_event_state(
    events: np.ndarray,
    decay: float,
) -> np.ndarray:
    T_, S_ = events.shape
    G = np.zeros((T_, S_), dtype=float)
    state = np.zeros(S_, dtype=float)

    for t in range(T_):
        state = decay * state + events[t]
        G[t] = state

    return G

def receiver_objective(
    theta: np.ndarray,
    X: np.ndarray,
    y: np.ndarray,
    allowed_sources: np.ndarray,
    receiver: int,
    l1_lambda: float,
) -> float:
    mu = float(theta[0])

    alpha = np.zeros(S, dtype=float)
    alpha[allowed_sources] = theta[1:]

    intensity = np.clip(
        mu + X @ alpha,
        EPS,
        50.0,
    )

    probability = np.clip(
        -np.expm1(-intensity),
        EPS,
        1.0 - EPS,
    )

    mean_nll = -np.mean(
        y * np.log(probability)
        + (1.0 - y) * np.log1p(-probability)
    )

    cross_mask = np.arange(S) != receiver
    penalty = (
        float(l1_lambda)
        * alpha[cross_mask].sum()
    )

    return float(mean_nll + penalty)

def fit_receiver(
    X: np.ndarray,
    y: np.ndarray,
    receiver: int,
    l1_lambda: float = 0.0,
    allowed_mask: np.ndarray | None = None,
    n_starts: int = 1,
) -> dict:

    if allowed_mask is None:
        allowed_mask = np.ones(S, dtype=bool)
    else:
        allowed_mask = np.asarray(allowed_mask, dtype=bool).copy()

    allowed_sources = np.where(allowed_mask)[0]

    event_rate = float(
        np.clip(y.mean(), 1e-6, 0.50)
    )
    mu0 = float(
        -np.log(1.0 - event_rate)
    )

    if len(allowed_sources) == 0:
        return {
            "mu": mu0,
            "alpha": np.zeros(S, dtype=float),
            "success": True,
            "objective": np.nan,
            "message": "Baseline-only closed-form MLE",
        }

    rng = np.random.default_rng(
        SEED + 1009 * (receiver + 1) + int(y.sum())
    )

    starts = [
        np.r_[mu0, np.repeat(0.01, len(allowed_sources))]
    ]

    while len(starts) < n_starts:
        starts.append(
            np.r_[
                max(mu0 * rng.uniform(0.60, 1.40), 1e-6),
                rng.uniform(0.0, 0.08, len(allowed_sources)),
            ]
        )

    bounds = (
        [MU_BOUNDS]
        + [ALPHA_BOUNDS] * len(allowed_sources)
    )

    best = None

    for start in starts:
        result = minimize(
            receiver_objective,
            x0=start,
            args=(
                X,
                y,
                allowed_sources,
                receiver,
                l1_lambda,
            ),
            method="L-BFGS-B",
            bounds=bounds,
            options={
                "maxiter": MAXITER,
                "ftol": 1e-12,
                "gtol": 1e-8,
            },
        )

        if (
            best is None
            or (
                np.isfinite(result.fun)
                and result.fun < best.fun
            )
        ):
            best = result

    if best is None or not np.isfinite(best.fun):
        raise RuntimeError(
            f"Hawkes optimization failed for receiver "
            f"{sectors[receiver]}."
        )

    alpha = np.zeros(S, dtype=float)
    alpha[allowed_sources] = best.x[1:]

    return {
        "mu": float(best.x[0]),
        "alpha": alpha,
        "success": bool(best.success),
        "objective": float(best.fun),
        "message": str(best.message),
    }

def fit_network(
    fit_date_mask: np.ndarray,
    half_life: float,
    structure: str,
    l1_lambda: float = 0.0,
    support_mask: np.ndarray | None = None,
    n_starts: int = 1,
) -> dict:

    decay = decay_from_half_life(half_life)
    H_pre = build_pre_event_state(C, decay)

    mu = np.zeros(S, dtype=float)
    A = np.zeros((S, S), dtype=float)
    details = []

    for receiver in range(S):

        valid = (
            fit_date_mask
            & M[:, receiver]
        )

        X = H_pre[valid]
        y = C[valid, receiver].astype(float)

        if len(y) == 0 or y.sum() == 0:
            raise RuntimeError(
                f"No fitting crash events for "
                f"{sectors[receiver]}."
            )

        if structure == "baseline":
            allowed = np.zeros(S, dtype=bool)

        elif structure == "self":
            allowed = np.zeros(S, dtype=bool)
            allowed[receiver] = True

        elif structure in ("full", "sparse"):
            allowed = np.ones(S, dtype=bool)

        elif structure == "support":
            if support_mask is None:
                raise ValueError(
                    "support_mask required for support model."
                )
            allowed = support_mask[receiver].copy()
            # Own-sector self-excitation remains permissible.
            allowed[receiver] = True

        else:
            raise ValueError(
                f"Unknown Hawkes structure: {structure}"
            )

        fit = fit_receiver(
            X=X,
            y=y,
            receiver=receiver,
            l1_lambda=(
                l1_lambda
                if structure == "sparse"
                else 0.0
            ),
            allowed_mask=allowed,
            n_starts=n_starts,
        )

        mu[receiver] = fit["mu"]
        A[receiver] = fit["alpha"]

        details.append({
            "Receiver": sectors[receiver],
            "Structure": structure,
            "HalfLife": half_life,
            "L1Lambda": (
                l1_lambda
                if structure == "sparse"
                else 0.0
            ),
            "N": len(y),
            "Events": int(y.sum()),
            "Mu": fit["mu"],
            "Converged": fit["success"],
            "Message": fit["message"],
        })

    return {
        "structure": structure,
        "half_life": float(half_life),
        "decay": decay,
        "l1_lambda": float(l1_lambda),
        "mu": mu,
        "alpha": A,
        "details": pd.DataFrame(details),
    }

def network_loglik(
    fit: dict,
    date_mask: np.ndarray,
) -> tuple[float, int]:

    H_pre = build_pre_event_state(
        C,
        fit["decay"],
    )

    total_ll = 0.0
    nobs = 0

    for receiver in range(S):
        valid = date_mask & M[:, receiver]

        X = H_pre[valid]
        y = C[valid, receiver].astype(float)

        intensity = np.clip(
            fit["mu"][receiver]
            + X @ fit["alpha"][receiver],
            EPS,
            50.0,
        )

        p = np.clip(
            -np.expm1(-intensity),
            EPS,
            1.0 - EPS,
        )

        total_ll += float(
            np.sum(
                y * np.log(p)
                + (1.0 - y) * np.log1p(-p)
            )
        )
        nobs += len(y)

    return total_ll, nobs

# =============================================================================
# 8. TRAIN-ONLY HALF-LIFE SELECTION BY BIC
# =============================================================================

half_life_rows = []
half_life_fits = {}

log("Selecting Hawkes half-life by TRAIN-only BIC...")

for half_life in HALF_LIFE_CANDIDATES:
    log(f"  Fitting full Hawkes at half-life={half_life} days.")

    fit = fit_network(
        fit_date_mask=train_date_mask,
        half_life=half_life,
        structure="full",
        n_starts=FINAL_MULTISTARTS,
    )

    ll, nobs = network_loglik(
        fit,
        train_date_mask,
    )

    # Full model: S baseline intensities + S*S excitation coefficients.
    k = S + S * S

    aic = -2.0 * ll + 2.0 * k
    bic = -2.0 * ll + k * np.log(max(nobs, 2))

    half_life_fits[float(half_life)] = fit

    half_life_rows.append({
        "HalfLife": half_life,
        "Decay": fit["decay"],
        "TrainLogLik": ll,
        "NObs": nobs,
        "NParameters": k,
        "AIC": aic,
        "BIC": bic,
    })

half_life_selection = pd.DataFrame(
    half_life_rows
).sort_values("BIC")

primary_half_life = float(
    half_life_selection.iloc[0]["HalfLife"]
)

half_life_selection["DeltaBIC"] = (
    half_life_selection["BIC"]
    - half_life_selection["BIC"].min()
)

half_life_selection.to_csv(
    TABLE_DIR / "Table_P04_Hawkes_HalfLife_Selection.csv",
    index=False
)

log(
    f"TRAIN-BIC selected Hawkes half-life: "
    f"{primary_half_life:g} trading days."
)

# =============================================================================
# 9. STRICT MULTI-HORIZON HAWKES SURVIVAL PROBABILITIES
# =============================================================================

def network_probabilities(
    fit: dict,
) -> dict[int, np.ndarray]:
    """
    P(at least one crash within H days | information through t).

    IMPORTANT:
    If G_t is the post-event excitation state after observing date t,
    then the t+1 intensity uses G_t directly. Therefore k=1 uses
    decay**0, not decay**1.
    """

    mu = fit["mu"]
    A = fit["alpha"]
    decay = fit["decay"]

    H_post = build_post_event_state(
        C,
        decay,
    )

    probabilities = {
        h: np.full((T, S), np.nan, dtype=float)
        for h in HORIZONS
    }

    for t in range(T):
        state_t = H_post[t]
        cumulative_intensity = np.zeros(S, dtype=float)

        for k in range(1, MAX_H + 1):

            # No-event survival path.
            future_state = (
                (decay ** (k - 1))
                * state_t
            )

            lambda_k = np.clip(
                mu + A @ future_state,
                EPS,
                50.0,
            )

            cumulative_intensity += lambda_k

            if k in HORIZONS:
                probabilities[k][t] = (
                    1.0
                    - np.exp(-cumulative_intensity)
                )

    # Coherence guard.
    for t in range(T):
        for s in range(S):
            values = [
                probabilities[h][t, s]
                for h in HORIZONS
            ]
            if any(
                values[i] > values[i + 1] + 1e-12
                for i in range(len(values) - 1)
            ):
                raise RuntimeError(
                    "Hawkes horizon-probability coherence failed."
                )

    return probabilities

# =============================================================================
# 10. STRICT HISTORICAL BASELINES — NO OVERLAPPING-HORIZON LEAKAGE
# =============================================================================

def make_historical_baselines():
    """
    FixedTrainHistorical:
        Sector-specific one-day crash rate estimated on TRAIN, then converted:
        P_H = 1 - (1-p)^H.

    ExpandingHistorical:
        At forecast origin t, use same-day crash labels observed through t.
        This is permissible because C_t is known after market close when
        forecasting t+1 onward. It never uses CrashWithin_H labels, which would
        overlap future observations and cause leakage.
    """

    fixed = {
        h: np.full((T, S), np.nan, dtype=float)
        for h in HORIZONS
    }
    expanding = {
        h: np.full((T, S), np.nan, dtype=float)
        for h in HORIZONS
    }

    # Fixed TRAIN event rates.
    train_rates = np.zeros(S, dtype=float)

    for s in range(S):
        valid = train_date_mask & M[:, s]
        train_rates[s] = (
            C[valid, s].mean()
            if valid.sum()
            else 0.025
        )

    for h in HORIZONS:
        fixed[h][:] = (
            1.0
            - (1.0 - train_rates[np.newaxis, :]) ** h
        )

    # Expanding same-day event rates.
    event_sum = np.zeros(S, dtype=float)
    event_n = np.zeros(S, dtype=float)

    for t in range(T):
        # Current date is observed before forecasting t+1 onward.
        for s in range(S):
            if M[t, s]:
                event_sum[s] += C[t, s]
                event_n[s] += 1.0

        one_day_p = np.where(
            event_n > 0,
            event_sum / np.maximum(event_n, 1.0),
            0.025,
        )

        for h in HORIZONS:
            expanding[h][t] = (
                1.0
                - (1.0 - one_day_p) ** h
            )

    return fixed, expanding

fixed_historical, expanding_historical = make_historical_baselines()

# =============================================================================
# 11. FORECAST METRICS
# =============================================================================

def log_score(
    y: np.ndarray,
    p: np.ndarray,
) -> float:
    p = np.clip(p, EPS, 1.0 - EPS)

    return float(
        -np.mean(
            y * np.log(p)
            + (1.0 - y) * np.log1p(-p)
        )
    )

def probability_metrics(
    y_true: np.ndarray,
    probability: np.ndarray,
) -> dict:

    y_true = np.asarray(y_true, dtype=float)
    probability = np.asarray(probability, dtype=float)

    ok = (
        np.isfinite(y_true)
        & np.isfinite(probability)
    )

    y = y_true[ok]
    p = np.clip(
        probability[ok],
        EPS,
        1.0 - EPS,
    )

    result = {
        "N": len(y),
        "Events": int(y.sum()) if len(y) else 0,
        "EventRate": float(y.mean()) if len(y) else np.nan,
        "Brier": np.nan,
        "LogScore": np.nan,
        "PR_AUC": np.nan,
        "ROC_AUC": np.nan,
    }

    if len(y) == 0:
        return result

    result["Brier"] = float(
        np.mean((p - y) ** 2)
    )
    result["LogScore"] = log_score(y, p)

    if len(np.unique(y)) == 2:
        result["PR_AUC"] = float(
            average_precision_score(y, p)
        )
        result["ROC_AUC"] = float(
            roc_auc_score(y, p)
        )

    return result

def pooled_multihorizon_logscore(
    probabilities: dict[int, np.ndarray],
    date_mask: np.ndarray,
) -> float:

    values = []

    for h in HORIZONS:
        y = target_within[h][date_mask].reshape(-1)
        p = probabilities[h][date_mask].reshape(-1)

        ok = np.isfinite(y) & np.isfinite(p)

        if ok.sum():
            values.append(
                log_score(y[ok], p[ok])
            )

    return (
        float(np.mean(values))
        if values
        else np.nan
    )

# =============================================================================
# 12. TEMPORAL CV FOLDS WITHIN TRAIN ONLY
# =============================================================================

train_indices = np.where(train_date_mask)[0]
effective_train_indices = train_indices[
    M[train_indices].any(axis=1)
]

if len(effective_train_indices) < 300:
    raise RuntimeError(
        "Too few effective TRAIN dates for temporal CV."
    )

n_effective = len(effective_train_indices)
initial_n = int(
    np.floor(
        INITIAL_TRAIN_FRACTION
        * n_effective
    )
)
validation_n = int(
    np.floor(
        VALIDATION_BLOCK_FRACTION
        * n_effective
    )
)

folds = []

for fold_id in range(N_TEMPORAL_FOLDS):

    fit_end_position = (
        initial_n
        + fold_id * validation_n
    )

    validation_start_position = fit_end_position

    validation_end_position = (
        n_effective
        if fold_id == N_TEMPORAL_FOLDS - 1
        else min(
            n_effective,
            validation_start_position + validation_n,
        )
    )

    if validation_start_position >= n_effective:
        break

    fit_end_index = effective_train_indices[
        fit_end_position - 1
    ]
    validation_start_index = effective_train_indices[
        validation_start_position
    ]
    validation_end_index = effective_train_indices[
        validation_end_position - 1
    ]

    fit_mask = (
        train_date_mask
        & (np.arange(T) <= fit_end_index)
    )

    validation_mask = (
        train_date_mask
        & (np.arange(T) >= validation_start_index)
        & (np.arange(T) <= validation_end_index)
    )

    folds.append({
        "Fold": len(folds) + 1,
        "FitMask": fit_mask,
        "ValidationMask": validation_mask,
        "FitEnd": dates[fit_end_index],
        "ValidationStart": dates[validation_start_index],
        "ValidationEnd": dates[validation_end_index],
    })

if len(folds) < 2:
    raise RuntimeError(
        "Could not construct at least two TRAIN-only temporal CV folds."
    )

fold_table = pd.DataFrame([
    {
        "Fold": f["Fold"],
        "FitEnd": f["FitEnd"],
        "ValidationStart": f["ValidationStart"],
        "ValidationEnd": f["ValidationEnd"],
        "FitDates": int(f["FitMask"].sum()),
        "ValidationDates": int(f["ValidationMask"].sum()),
    }
    for f in folds
])

fold_table.to_csv(
    TABLE_DIR / "Table_P05_Temporal_CV_Folds.csv",
    index=False
)

# =============================================================================
# 13. L1 PENALTY TUNING WITHIN TRAIN
# =============================================================================
#
# IMPORTANT REVISION:
#   * lambda_min (minimum temporal-CV log score) is the PRIMARY penalty used
#     for stability selection.
#   * lambda_1SE is retained as a conservative robustness specification.
#
# This avoids "double sparsification" from first selecting an aggressively
# sparse 1-SE model and then applying a second 60% stability filter.

sensitivity_half_lives = sorted(
    set([2.0, 5.0, 10.0, primary_half_life])
)

cv_rows = []

for half_life in sensitivity_half_lives:

    log(
        f"Temporal-CV sparse Hawkes tuning: "
        f"half-life={half_life:g} days."
    )

    for l1_lambda in LAMBDA_GRID:

        for fold in folds:

            fit = fit_network(
                fit_date_mask=fold["FitMask"],
                half_life=half_life,
                structure="sparse",
                l1_lambda=l1_lambda,
                n_starts=1,
            )

            probabilities = network_probabilities(fit)

            loss = pooled_multihorizon_logscore(
                probabilities,
                fold["ValidationMask"],
            )

            cross_mask = ~np.eye(S, dtype=bool)
            n_cross = int(
                (
                    (fit["alpha"] > EDGE_NUMERIC_TOL)
                    & cross_mask
                ).sum()
            )

            cv_rows.append({
                "HalfLife": half_life,
                "Lambda": l1_lambda,
                "Fold": fold["Fold"],
                "ValidationLogScore": loss,
                "NonzeroCrossEdges": n_cross,
            })

cv_results = pd.DataFrame(cv_rows)
cv_results.to_csv(
    TABLE_DIR / "Table_P06_L1_Temporal_CV_All_Folds.csv",
    index=False
)

cv_summary = (
    cv_results
    .groupby(["HalfLife", "Lambda"], as_index=False)
    .agg(
        MeanValidationLogScore=("ValidationLogScore", "mean"),
        SDValidationLogScore=("ValidationLogScore", "std"),
        MeanCrossEdges=("NonzeroCrossEdges", "mean"),
        Folds=("Fold", "nunique"),
    )
)

cv_summary["SEValidationLogScore"] = (
    cv_summary["SDValidationLogScore"]
    / np.sqrt(cv_summary["Folds"])
)

lambda_selection_rows = []
lambda_min_by_half_life = {}
lambda_1se_by_half_life = {}

for half_life in sensitivity_half_lives:

    g = cv_summary[
        cv_summary["HalfLife"] == half_life
    ].copy()

    best_index = g["MeanValidationLogScore"].idxmin()
    best_row = g.loc[best_index]

    lambda_min = float(best_row["Lambda"])
    best_mean = float(best_row["MeanValidationLogScore"])
    best_se = float(best_row["SEValidationLogScore"])

    if not np.isfinite(best_se):
        best_se = 0.0

    one_se_limit = best_mean + best_se

    acceptable = g[
        g["MeanValidationLogScore"] <= one_se_limit
    ].copy()

    chosen_1se = acceptable.sort_values(
        ["Lambda", "MeanValidationLogScore"],
        ascending=[False, True],
    ).iloc[0]

    lambda_1se = float(chosen_1se["Lambda"])

    lambda_min_by_half_life[float(half_life)] = lambda_min
    lambda_1se_by_half_life[float(half_life)] = lambda_1se

    lambda_selection_rows.append({
        "HalfLife": half_life,
        "Lambda_Min": lambda_min,
        "MinimumCVLogScore": best_mean,
        "SEAtMinimum": best_se,
        "OneSELimit": one_se_limit,
        "Lambda_1SE": lambda_1se,
        "PrimaryPenaltyForStability": lambda_min,
        "RobustnessPenalty": lambda_1se,
        "MeanCrossEdgesAtLambdaMin": float(best_row["MeanCrossEdges"]),
        "MeanCrossEdgesAtLambda1SE": float(chosen_1se["MeanCrossEdges"]),
    })

lambda_selection = pd.DataFrame(lambda_selection_rows)

cv_summary.to_csv(
    TABLE_DIR / "Table_P07_L1_Temporal_CV_Summary.csv",
    index=False
)

lambda_selection.to_csv(
    TABLE_DIR / "Table_P08_L1_LambdaMin_and_1SE_Selection.csv",
    index=False
)

primary_lambda = lambda_min_by_half_life[float(primary_half_life)]
primary_lambda_1se = lambda_1se_by_half_life[float(primary_half_life)]

log(
    f"Primary half-life={primary_half_life:g}; "
    f"lambda_min={primary_lambda:g} (PRIMARY), "
    f"lambda_1SE={primary_lambda_1se:g} (ROBUSTNESS)."
)

# =============================================================================
# 14. BLOCK STABILITY SELECTION
# =============================================================================

def make_block_subsample_mask(
    eligible_indices: np.ndarray,
    fraction: float,
    block_length: int,
    rng: np.random.Generator,
) -> np.ndarray:

    eligible_indices = np.asarray(eligible_indices, dtype=int)
    n = len(eligible_indices)
    target_n = max(1, int(np.ceil(fraction * n)))
    selected_positions = np.zeros(n, dtype=bool)

    while selected_positions.sum() < target_n:
        start = int(
            rng.integers(
                0,
                max(1, n - block_length + 1),
            )
        )
        end = min(n, start + block_length)
        selected_positions[start:end] = True

    mask = np.zeros(T, dtype=bool)
    mask[eligible_indices[selected_positions]] = True
    return mask


def run_stability_selection(
    half_life: float,
    l1_lambda: float,
    repetitions: int,
    specification_label: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:

    eligible_indices = np.where(
        train_date_mask & M.any(axis=1)
    )[0]

    selection_count = np.zeros((S, S), dtype=int)
    coefficient_sum = np.zeros((S, S), dtype=float)
    replicate_rows = []

    for repetition in range(repetitions):

        rng = np.random.default_rng(
            SEED
            + int(half_life * 1000)
            + int(l1_lambda * 100000)
            + 7919 * (repetition + 1)
        )

        subsample_mask = make_block_subsample_mask(
            eligible_indices=eligible_indices,
            fraction=STABILITY_SUBSAMPLE_FRACTION,
            block_length=STABILITY_BLOCK_LENGTH,
            rng=rng,
        )

        fit = fit_network(
            fit_date_mask=subsample_mask,
            half_life=half_life,
            structure="sparse",
            l1_lambda=l1_lambda,
            n_starts=1,
        )

        selected = fit["alpha"] > EDGE_NUMERIC_TOL

        selection_count += selected.astype(int)
        coefficient_sum += fit["alpha"]

        replicate_rows.append({
            "Specification": specification_label,
            "HalfLife": half_life,
            "Lambda": l1_lambda,
            "Replicate": repetition + 1,
            "SubsampleDates": int(subsample_mask.sum()),
            "NonzeroAllEdges": int(selected.sum()),
            "NonzeroCrossEdges": int(
                (
                    selected
                    & ~np.eye(S, dtype=bool)
                ).sum()
            ),
        })

        if (repetition + 1) % 20 == 0:
            log(
                f"  Stability {specification_label}, half-life={half_life:g}: "
                f"{repetition + 1}/{repetitions} complete."
            )

    frequency = selection_count / float(repetitions)
    mean_coefficient = coefficient_sum / float(repetitions)

    edge_rows = []

    for receiver in range(S):
        for sender in range(S):
            edge_rows.append({
                "Specification": specification_label,
                "HalfLife": half_life,
                "Lambda": l1_lambda,
                "FromSector": sectors[sender],
                "ToSector": sectors[receiver],
                "SelfExcitation": int(sender == receiver),
                "SelectionFrequency": frequency[receiver, sender],
                "MeanCoefficientAcrossReps": mean_coefficient[receiver, sender],
                "StableAtThreshold": int(
                    frequency[receiver, sender] >= STABILITY_THRESHOLD
                ),
            })

    return pd.DataFrame(edge_rows), pd.DataFrame(replicate_rows)


# PRIMARY: lambda_min for each half-life.
stability_tables = []
stability_replicates = []

for half_life in sensitivity_half_lives:

    repetitions = (
        STABILITY_REPS_PRIMARY
        if float(half_life) == float(primary_half_life)
        else STABILITY_REPS_SENSITIVITY
    )

    selected_lambda = lambda_min_by_half_life[float(half_life)]

    log(
        f"PRIMARY block stability selection: half-life={half_life:g}, "
        f"lambda_min={selected_lambda:g}, reps={repetitions}."
    )

    edge_table, replicate_table = run_stability_selection(
        half_life=half_life,
        l1_lambda=selected_lambda,
        repetitions=repetitions,
        specification_label="LambdaMin_Primary",
    )

    stability_tables.append(edge_table)
    stability_replicates.append(replicate_table)


# CONSERVATIVE ROBUSTNESS: primary half-life with lambda_1SE.
log(
    f"ROBUSTNESS block stability selection: "
    f"half-life={primary_half_life:g}, "
    f"lambda_1SE={primary_lambda_1se:g}, "
    f"reps={STABILITY_REPS_CONSERVATIVE}."
)

edge_1se, reps_1se = run_stability_selection(
    half_life=primary_half_life,
    l1_lambda=primary_lambda_1se,
    repetitions=STABILITY_REPS_CONSERVATIVE,
    specification_label="Lambda1SE_Robustness",
)

stability_tables.append(edge_1se)
stability_replicates.append(reps_1se)

stability_all = pd.concat(stability_tables, ignore_index=True)
stability_reps_all = pd.concat(stability_replicates, ignore_index=True)

stability_all.to_csv(
    TABLE_DIR / "Table_P09_Edge_Stability_All_Specifications.csv",
    index=False
)

stability_reps_all.to_csv(
    TABLE_DIR / "Table_P10_Stability_Replicate_Summary.csv",
    index=False
)

primary_stability = stability_all[
    (stability_all["Specification"] == "LambdaMin_Primary")
    & (
        stability_all["HalfLife"].astype(float)
        == float(primary_half_life)
    )
].copy()

robustness_stability_1se = stability_all[
    (stability_all["Specification"] == "Lambda1SE_Robustness")
    & (
        stability_all["HalfLife"].astype(float)
        == float(primary_half_life)
    )
].copy()

stable_cross_support = np.zeros((S, S), dtype=bool)

for row in primary_stability.itertuples(index=False):
    sender = sector_to_idx[row.FromSector]
    receiver = sector_to_idx[row.ToSector]

    if (
        sender != receiver
        and row.SelectionFrequency >= STABILITY_THRESHOLD
    ):
        stable_cross_support[receiver, sender] = True

n_stable_cross_edges = int(stable_cross_support.sum())

robust_1se_cross_edges = int(
    (
        (robustness_stability_1se["SelfExcitation"] == 0)
        & (
            robustness_stability_1se["SelectionFrequency"]
            >= STABILITY_THRESHOLD
        )
    ).sum()
)

log(
    f"PRIMARY lambda_min stable cross-sector edges: "
    f"{n_stable_cross_edges}."
)
log(
    f"ROBUSTNESS lambda_1SE stable cross-sector edges: "
    f"{robust_1se_cross_edges}."
)

# =============================================================================
# 15. FIT NESTED MODELS ON TRAIN FOR VALIDATION
# =============================================================================

train_fits = {
    "BaselineIntensity": fit_network(
        train_date_mask,
        primary_half_life,
        structure="baseline",
        n_starts=1,
    ),
    "SelfHawkes": fit_network(
        train_date_mask,
        primary_half_life,
        structure="self",
        n_starts=FINAL_MULTISTARTS,
    ),
    "FullHawkes": fit_network(
        train_date_mask,
        primary_half_life,
        structure="full",
        n_starts=FINAL_MULTISTARTS,
    ),
    "SparsePenalizedHawkes": fit_network(
        train_date_mask,
        primary_half_life,
        structure="sparse",
        l1_lambda=primary_lambda,
        n_starts=FINAL_MULTISTARTS,
    ),
    "Sparse1SEHawkes_Robustness": fit_network(
        train_date_mask,
        primary_half_life,
        structure="sparse",
        l1_lambda=primary_lambda_1se,
        n_starts=FINAL_MULTISTARTS,
    ),
    "StablePostSelectionHawkes": fit_network(
        train_date_mask,
        primary_half_life,
        structure="support",
        support_mask=stable_cross_support,
        n_starts=FINAL_MULTISTARTS,
    ),
}

# =============================================================================
# 16. FREEZE STRUCTURE, THEN REFIT COEFFICIENTS ON TRAIN+VALIDATION FOR TEST
# =============================================================================

# Hyperparameters and stable support remain fixed from TRAIN-only procedures.
final_fits = {
    "BaselineIntensity": fit_network(
        trainval_date_mask,
        primary_half_life,
        structure="baseline",
        n_starts=1,
    ),
    "SelfHawkes": fit_network(
        trainval_date_mask,
        primary_half_life,
        structure="self",
        n_starts=FINAL_MULTISTARTS,
    ),
    "FullHawkes": fit_network(
        trainval_date_mask,
        primary_half_life,
        structure="full",
        n_starts=FINAL_MULTISTARTS,
    ),
    "SparsePenalizedHawkes": fit_network(
        trainval_date_mask,
        primary_half_life,
        structure="sparse",
        l1_lambda=primary_lambda,
        n_starts=FINAL_MULTISTARTS,
    ),
    "Sparse1SEHawkes_Robustness": fit_network(
        trainval_date_mask,
        primary_half_life,
        structure="sparse",
        l1_lambda=primary_lambda_1se,
        n_starts=FINAL_MULTISTARTS,
    ),
    "StablePostSelectionHawkes": fit_network(
        trainval_date_mask,
        primary_half_life,
        structure="support",
        support_mask=stable_cross_support,
        n_starts=FINAL_MULTISTARTS,
    ),
}

# =============================================================================
# 17. BRANCHING / STABILITY DIAGNOSTICS
# =============================================================================

def branching_spectral_radius(
    A: np.ndarray,
    decay: float,
) -> float:

    branching = (
        A
        / max(1.0 - decay, EPS)
    )

    eigenvalues = np.linalg.eigvals(branching)

    return float(
        np.max(np.abs(eigenvalues))
    )

network_diagnostic_rows = []

for fit_stage, fit_collection in [
    ("TrainFit", train_fits),
    ("TrainValidationRefit", final_fits),
]:
    for model_name, fit in fit_collection.items():

        cross_mask = ~np.eye(S, dtype=bool)

        spectral_radius = branching_spectral_radius(
            fit["alpha"],
            fit["decay"],
        )

        network_diagnostic_rows.append({
            "FitStage": fit_stage,
            "Model": model_name,
            "HalfLife": fit["half_life"],
            "Lambda": fit["l1_lambda"],
            "NonzeroAllEdges": int(
                (
                    fit["alpha"] > EDGE_NUMERIC_TOL
                ).sum()
            ),
            "NonzeroCrossEdges": int(
                (
                    (fit["alpha"] > EDGE_NUMERIC_TOL)
                    & cross_mask
                ).sum()
            ),
            "MaxAlpha": float(
                fit["alpha"].max()
            ),
            "BranchingSpectralRadiusApprox": spectral_radius,
            "BelowStabilityGuard": int(
                spectral_radius
                < MAX_BRANCHING_SPECTRAL_RADIUS
            ),
        })

network_diagnostics = pd.DataFrame(
    network_diagnostic_rows
)

network_diagnostics.to_csv(
    TABLE_DIR / "Table_P11_Network_Stability_Diagnostics.csv",
    index=False
)

final_stable_radius = float(
    network_diagnostics.loc[
        (network_diagnostics["FitStage"] == "TrainValidationRefit")
        & (
            network_diagnostics["Model"]
            == "StablePostSelectionHawkes"
        ),
        "BranchingSpectralRadiusApprox",
    ].iloc[0]
)

if final_stable_radius >= MAX_BRANCHING_SPECTRAL_RADIUS:
    raise RuntimeError(
        "Final stability-selected Hawkes network exceeds "
        "the pre-specified branching spectral-radius guard."
    )

# =============================================================================
# 18. STABLE EDGE TABLES AND HALF-LIFE ROBUSTNESS
# =============================================================================

train_stable_fit = train_fits[
    "StablePostSelectionHawkes"
]
final_stable_fit = final_fits[
    "StablePostSelectionHawkes"
]

stable_edge_rows = []

for receiver in range(S):
    for sender in range(S):

        match = primary_stability[
            (primary_stability["FromSector"] == sectors[sender])
            & (primary_stability["ToSector"] == sectors[receiver])
        ]

        stability_frequency = (
            float(match["SelectionFrequency"].iloc[0])
            if len(match)
            else np.nan
        )

        stable_edge_rows.append({
            "FromSector": sectors[sender],
            "ToSector": sectors[receiver],
            "SelfExcitation": int(sender == receiver),
            "SelectionFrequency": stability_frequency,
            "SelectedCrossEdge": int(
                stable_cross_support[receiver, sender]
            ),
            "TrainPostSelectionAlpha": float(
                train_stable_fit["alpha"][receiver, sender]
            ),
            "TrainValidationRefitAlpha": float(
                final_stable_fit["alpha"][receiver, sender]
            ),
            "HalfLife": primary_half_life,
        })

stable_edges = pd.DataFrame(
    stable_edge_rows
)

stable_edges.to_csv(
    TABLE_DIR / "Table_P12_Final_Stable_Edge_Parameters.csv",
    index=False
)

half_life_summary_rows = []
support_by_half_life = {}

for half_life in sensitivity_half_lives:

    g = stability_all[
        stability_all["HalfLife"].astype(float)
        == float(half_life)
    ].copy()

    cross = g[
        g["SelfExcitation"] == 0
    ]

    stable_cross = cross[
        cross["SelectionFrequency"]
        >= STABILITY_THRESHOLD
    ]

    support_by_half_life[
        float(half_life)
    ] = set(
        zip(
            stable_cross["FromSector"],
            stable_cross["ToSector"],
        )
    )

    half_life_summary_rows.append({
        "HalfLife": half_life,
        "SelectedLambda": lambda_min_by_half_life[
            float(half_life)
        ],
        "StableCrossEdges": len(stable_cross),
        "MeanCrossEdgeStability": float(
            cross["SelectionFrequency"].mean()
        ),
        "MaxCrossEdgeStability": float(
            cross["SelectionFrequency"].max()
        ),
    })

half_life_summary = pd.DataFrame(
    half_life_summary_rows
)

half_life_summary.to_csv(
    TABLE_DIR / "Table_P13_HalfLife_Stability_Summary.csv",
    index=False
)

overlap_rows = []

for i, h1 in enumerate(sensitivity_half_lives):
    for h2 in sensitivity_half_lives[i + 1:]:

        a = support_by_half_life[float(h1)]
        b = support_by_half_life[float(h2)]

        union = a | b

        overlap_rows.append({
            "HalfLife1": h1,
            "HalfLife2": h2,
            "Edges1": len(a),
            "Edges2": len(b),
            "CommonEdges": len(a & b),
            "Jaccard": (
                len(a & b) / len(union)
                if union
                else 1.0
            ),
        })

half_life_overlap = pd.DataFrame(
    overlap_rows
)

half_life_overlap.to_csv(
    TABLE_DIR / "Table_P14_HalfLife_Edge_Overlap.csv",
    index=False
)

# =============================================================================
# 19. VALIDATION AND TEST FORECASTS
# =============================================================================

# Validation: use TRAIN-fitted coefficients.
train_probabilities = {
    model_name: network_probabilities(fit)
    for model_name, fit in train_fits.items()
}

# Test: use TRAIN+VALIDATION refitted coefficients with structure frozen.
final_probabilities = {
    model_name: network_probabilities(fit)
    for model_name, fit in final_fits.items()
}

forecast_metric_rows = []

def add_metrics_for_split(
    split_name: str,
    split_mask: np.ndarray,
    model_probabilities: dict[str, dict[int, np.ndarray]],
):

    # Strict historical baselines.
    baseline_collection = {
        "FixedTrainHistorical": fixed_historical,
        "ExpandingHistorical": expanding_historical,
    }

    combined = {
        **baseline_collection,
        **model_probabilities,
    }

    for model_name, probabilities in combined.items():

        for h in HORIZONS:

            pooled = probability_metrics(
                target_within[h][split_mask].reshape(-1),
                probabilities[h][split_mask].reshape(-1),
            )

            forecast_metric_rows.append({
                "Split": split_name,
                "Sector": "POOLED",
                "Horizon": h,
                "Model": model_name,
                **pooled,
            })

            for s, sector in enumerate(sectors):

                sector_metrics = probability_metrics(
                    target_within[h][split_mask, s],
                    probabilities[h][split_mask, s],
                )

                forecast_metric_rows.append({
                    "Split": split_name,
                    "Sector": sector,
                    "Horizon": h,
                    "Model": model_name,
                    **sector_metrics,
                })

add_metrics_for_split(
    "Validation",
    validation_date_mask,
    train_probabilities,
)

add_metrics_for_split(
    "Test",
    test_date_mask,
    final_probabilities,
)

forecast_metrics = pd.DataFrame(
    forecast_metric_rows
)

forecast_metrics.to_csv(
    TABLE_DIR / "Table_P15_Nested_Hawkes_Forecast_Metrics.csv",
    index=False
)

# Skill is referenced to the stricter expanding historical probability.
pooled = forecast_metrics[
    forecast_metrics["Sector"] == "POOLED"
].copy()

reference = (
    pooled[
        pooled["Model"] == "ExpandingHistorical"
    ][
        ["Split", "Horizon", "Brier", "LogScore"]
    ]
    .rename(columns={
        "Brier": "ReferenceBrier",
        "LogScore": "ReferenceLogScore",
    })
)

forecast_skill = pooled.merge(
    reference,
    on=["Split", "Horizon"],
    how="left",
)

forecast_skill["BrierSkill_vs_ExpandingHistorical"] = (
    1.0
    - forecast_skill["Brier"]
    / forecast_skill["ReferenceBrier"]
)

forecast_skill["LogScoreImprovement_vs_ExpandingHistorical"] = (
    forecast_skill["ReferenceLogScore"]
    - forecast_skill["LogScore"]
)

forecast_skill.to_csv(
    TABLE_DIR / "Table_P16_Pooled_Forecast_Skill.csv",
    index=False
)

# =============================================================================
# 20. GRAPH DECISION — VALIDATION ONLY, NEVER TEST
# =============================================================================

validation_stable = forecast_skill[
    (forecast_skill["Split"] == "Validation")
    & (
        forecast_skill["Model"]
        == "StablePostSelectionHawkes"
    )
].copy()

positive_validation_skill_horizons = int(
    (
        validation_stable[
            "BrierSkill_vs_ExpandingHistorical"
        ] > 0
    ).sum()
)

if n_stable_cross_edges >= 1:

    if (
        positive_validation_skill_horizons
        >= MIN_VALIDATION_SKILL_HORIZONS
    ):
        hawkes_graph_decision = (
            "HAWKES_GRAPH_SUPPORTED_AS_SOFT_PRIOR"
        )
    else:
        hawkes_graph_decision = (
            "HAWKES_GRAPH_STRUCTURALLY_STABLE_BUT_WEAK_VALIDATION_"
            "USE_AS_SOFT_PRIOR_WITH_ABLATION"
        )

else:
    hawkes_graph_decision = (
        "NO_STABLE_CROSS_SECTOR_HAWKES_EDGES_"
        "ACTIVATE_LOWER_TAIL_QUANTILE_FALLBACK"
    )

# =============================================================================
# 20A. LOWER-TAIL QUANTILE GRAPH FALLBACK
# =============================================================================
#
# Activated only if Hawkes lambda_min + 60% stability selection yields no
# cross-sector edges.
#
# For each receiving sector i:
#
#   Q_tau(z_i,t+1 | z_1,t,...,z_S,t)
#       = a_i + sum_j beta_i,j z_j,t
#
# with tau = 0.05.
#
# A positive beta_i,j means a negative shock in source j lowers the conditional
# lower quantile of receiver i and is therefore consistent with downside
# predictive transmission. Stable positive CROSS-sector coefficients define
# the fallback graph. The dynamic edge weight is:
#
#   w_i,j,t = beta_i,j * max(-z_j,t, 0)
#
# so edges activate when the source sector experiences downside standardized
# shocks.

tail_graph_used = False
tail_graph_available = False
tail_graph_alpha = np.nan
tail_stable_cross_edges = 0
tail_support = np.zeros((S, S), dtype=bool)
tail_beta_train = np.zeros((S, S), dtype=float)
tail_beta_final = np.zeros((S, S), dtype=float)
tail_intercept_train = np.zeros(S, dtype=float)
tail_intercept_final = np.zeros(S, dtype=float)
tail_cv_summary = pd.DataFrame()
tail_stability_table = pd.DataFrame()

# Standardized innovations aligned to date x sector.
Z = np.full((T, S), np.nan, dtype=float)

for row in primary[
    ["Date", "Sector", "StdInnovation"]
].itertuples(index=False):
    t = date_to_idx[pd.Timestamp(row.Date)]
    s = sector_to_idx[row.Sector]
    if pd.notna(row.StdInnovation):
        Z[t, s] = float(row.StdInnovation)


def tail_row_mask_from_date_mask(date_mask: np.ndarray) -> np.ndarray:
    """
    Row t uses predictors Z_t and response Z_(t+1).
    Both predictor date t and response date t+1 must lie inside the supplied
    fitting/validation date region.
    """
    out = np.zeros(T - 1, dtype=bool)
    out[:] = date_mask[:-1] & date_mask[1:]
    return out


def tail_complete_case_rows(date_mask: np.ndarray) -> np.ndarray:
    base = tail_row_mask_from_date_mask(date_mask)
    complete_x = np.all(np.isfinite(Z[:-1]), axis=1)
    complete_y = np.all(np.isfinite(Z[1:]), axis=1)
    return base & complete_x & complete_y


def fit_quantile_network(
    row_mask: np.ndarray,
    alpha_penalty: float,
    support_mask: np.ndarray | None = None,
) -> dict:

    X = Z[:-1][row_mask]
    Y = Z[1:][row_mask]

    if len(X) < 100:
        raise RuntimeError(
            "Too few complete observations for lower-tail quantile graph."
        )

    intercept = np.zeros(S, dtype=float)
    beta = np.zeros((S, S), dtype=float)

    for receiver in range(S):

        if support_mask is None:
            allowed = np.ones(S, dtype=bool)
        else:
            allowed = support_mask[receiver].copy()
            # Own lag is always included as a control, but is not treated as
            # a contagion edge.
            allowed[receiver] = True

        allowed_idx = np.where(allowed)[0]

        model = QuantileRegressor(
            quantile=TAIL_QUANTILE,
            alpha=float(alpha_penalty),
            fit_intercept=True,
            solver="highs",
        )

        model.fit(
            X[:, allowed_idx],
            Y[:, receiver],
        )

        intercept[receiver] = float(model.intercept_)
        beta[receiver, allowed_idx] = np.asarray(
            model.coef_,
            dtype=float,
        )

    return {
        "intercept": intercept,
        "beta": beta,
        "n": int(row_mask.sum()),
        "alpha_penalty": float(alpha_penalty),
    }


def quantile_network_pinball(
    fit: dict,
    row_mask: np.ndarray,
) -> float:

    X = Z[:-1][row_mask]
    Y = Z[1:][row_mask]

    losses = []

    for receiver in range(S):
        pred = (
            fit["intercept"][receiver]
            + X @ fit["beta"][receiver]
        )

        losses.append(
            mean_pinball_loss(
                Y[:, receiver],
                pred,
                alpha=TAIL_QUANTILE,
            )
        )

    return float(np.mean(losses))


def make_tail_block_subsample_mask(
    eligible_rows: np.ndarray,
    fraction: float,
    block_length: int,
    rng: np.random.Generator,
) -> np.ndarray:

    eligible_rows = np.asarray(eligible_rows, dtype=int)
    n = len(eligible_rows)
    target_n = max(1, int(np.ceil(fraction * n)))
    selected_positions = np.zeros(n, dtype=bool)

    while selected_positions.sum() < target_n:
        start = int(
            rng.integers(
                0,
                max(1, n - block_length + 1),
            )
        )
        end = min(n, start + block_length)
        selected_positions[start:end] = True

    mask = np.zeros(T - 1, dtype=bool)
    mask[eligible_rows[selected_positions]] = True
    return mask


def build_dynamic_tail_graph(
    beta_matrix: np.ndarray,
) -> np.ndarray:

    graph = np.zeros((T, S, S), dtype=np.float32)

    for t in range(T):
        downside_state = np.where(
            np.isfinite(Z[t]),
            np.maximum(-Z[t], 0.0),
            0.0,
        )

        # Receiver x Source.
        graph[t] = (
            beta_matrix
            * downside_state[np.newaxis, :]
        ).astype(np.float32)

    return graph


if n_stable_cross_edges == 0:

    log(
        "No stable Hawkes cross-sector edges survived. "
        "Estimating lower-tail quantile graph fallback."
    )

    # ----- Tail graph temporal CV within TRAIN only -----
    tail_cv_rows = []

    for alpha_penalty in TAIL_QUANTILE_ALPHA_GRID:

        fold_losses = []

        for fold in folds:

            fit_rows = tail_complete_case_rows(
                fold["FitMask"]
            )
            val_rows = tail_complete_case_rows(
                fold["ValidationMask"]
            )

            if fit_rows.sum() < 100 or val_rows.sum() < 30:
                continue

            fit_tail = fit_quantile_network(
                row_mask=fit_rows,
                alpha_penalty=alpha_penalty,
            )

            loss = quantile_network_pinball(
                fit_tail,
                val_rows,
            )

            cross_mask = ~np.eye(S, dtype=bool)
            positive_cross_edges = int(
                (
                    (fit_tail["beta"] > TAIL_BETA_TOL)
                    & cross_mask
                ).sum()
            )

            fold_losses.append(loss)

            tail_cv_rows.append({
                "Alpha": alpha_penalty,
                "Fold": fold["Fold"],
                "PinballLoss": loss,
                "PositiveCrossEdges": positive_cross_edges,
            })

    tail_cv_all = pd.DataFrame(tail_cv_rows)

    if len(tail_cv_all) == 0:
        raise RuntimeError(
            "Lower-tail quantile fallback could not construct temporal CV folds."
        )

    tail_cv_all.to_csv(
        TABLE_DIR / "Table_P17A_TailQuantile_CV_All_Folds.csv",
        index=False
    )

    tail_cv_summary = (
        tail_cv_all
        .groupby("Alpha", as_index=False)
        .agg(
            MeanPinballLoss=("PinballLoss", "mean"),
            SDPinballLoss=("PinballLoss", "std"),
            MeanPositiveCrossEdges=("PositiveCrossEdges", "mean"),
            Folds=("Fold", "nunique"),
        )
        .sort_values("MeanPinballLoss")
    )

    tail_graph_alpha = float(
        tail_cv_summary.iloc[0]["Alpha"]
    )

    tail_cv_summary.to_csv(
        TABLE_DIR / "Table_P17B_TailQuantile_CV_Summary.csv",
        index=False
    )

    log(
        f"Lower-tail quantile graph selected alpha_min="
        f"{tail_graph_alpha:g} at tau={TAIL_QUANTILE:g}."
    )

    # ----- Tail graph block stability selection on TRAIN only -----
    tail_train_rows = np.where(
        tail_complete_case_rows(train_date_mask)
    )[0]

    selection_count = np.zeros((S, S), dtype=int)
    coefficient_sum = np.zeros((S, S), dtype=float)
    tail_rep_rows = []

    for repetition in range(TAIL_GRAPH_STABILITY_REPS):

        rng = np.random.default_rng(
            SEED + 17713 * (repetition + 1)
        )

        subsample_rows = make_tail_block_subsample_mask(
            eligible_rows=tail_train_rows,
            fraction=STABILITY_SUBSAMPLE_FRACTION,
            block_length=STABILITY_BLOCK_LENGTH,
            rng=rng,
        )

        fit_tail = fit_quantile_network(
            row_mask=subsample_rows,
            alpha_penalty=tail_graph_alpha,
        )

        # Only positive coefficients represent downside transmission under
        # this sign convention. Diagonal coefficients are controls.
        selected = fit_tail["beta"] > TAIL_BETA_TOL

        selection_count += selected.astype(int)
        coefficient_sum += fit_tail["beta"]

        tail_rep_rows.append({
            "Replicate": repetition + 1,
            "Rows": int(subsample_rows.sum()),
            "PositiveAllEdges": int(selected.sum()),
            "PositiveCrossEdges": int(
                (
                    selected
                    & ~np.eye(S, dtype=bool)
                ).sum()
            ),
        })

        if (repetition + 1) % 20 == 0:
            log(
                f"  Tail-graph stability: "
                f"{repetition + 1}/{TAIL_GRAPH_STABILITY_REPS} complete."
            )

    tail_frequency = selection_count / float(
        TAIL_GRAPH_STABILITY_REPS
    )
    tail_mean_beta = coefficient_sum / float(
        TAIL_GRAPH_STABILITY_REPS
    )

    tail_edge_rows = []

    for receiver in range(S):
        for sender in range(S):

            stable = (
                sender != receiver
                and tail_frequency[receiver, sender]
                >= TAIL_GRAPH_STABILITY_THRESHOLD
            )

            if stable:
                tail_support[receiver, sender] = True

            tail_edge_rows.append({
                "FromSector": sectors[sender],
                "ToSector": sectors[receiver],
                "SelfLagControl": int(sender == receiver),
                "SelectionFrequency": tail_frequency[receiver, sender],
                "MeanBetaAcrossReps": tail_mean_beta[receiver, sender],
                "StableCrossEdge": int(stable),
            })

    tail_stability_table = pd.DataFrame(tail_edge_rows)
    tail_stability_table.to_csv(
        TABLE_DIR / "Table_P17C_TailQuantile_Edge_Stability.csv",
        index=False
    )

    pd.DataFrame(tail_rep_rows).to_csv(
        TABLE_DIR / "Table_P17D_TailQuantile_Stability_Repetitions.csv",
        index=False
    )

    tail_stable_cross_edges = int(tail_support.sum())

    if tail_stable_cross_edges >= 1:

        tail_graph_available = True
        tail_graph_used = True

        # Post-selection refit on TRAIN for Train/Validation graph.
        train_rows_tail = tail_complete_case_rows(
            train_date_mask
        )
        fit_tail_train = fit_quantile_network(
            row_mask=train_rows_tail,
            alpha_penalty=0.0,
            support_mask=tail_support,
        )

        # Refit coefficients on TRAIN+VALIDATION after support is frozen.
        trainval_rows_tail = tail_complete_case_rows(
            trainval_date_mask
        )
        fit_tail_final = fit_quantile_network(
            row_mask=trainval_rows_tail,
            alpha_penalty=0.0,
            support_mask=tail_support,
        )

        tail_beta_train = fit_tail_train["beta"]
        tail_beta_final = fit_tail_final["beta"]
        tail_intercept_train = fit_tail_train["intercept"]
        tail_intercept_final = fit_tail_final["intercept"]

        # Remove own-lag diagonal from graph; own lag remains only as a control
        # in the quantile regression.
        np.fill_diagonal(tail_beta_train, 0.0)
        np.fill_diagonal(tail_beta_final, 0.0)

        # Keep only stable cross-sector support and positive direction.
        tail_beta_train = np.where(
            tail_support,
            np.maximum(tail_beta_train, 0.0),
            0.0,
        )
        tail_beta_final = np.where(
            tail_support,
            np.maximum(tail_beta_final, 0.0),
            0.0,
        )

        log(
            f"Lower-tail quantile graph retained "
            f"{tail_stable_cross_edges} stable cross-sector edges."
        )

    else:
        log(
            "No stable lower-tail quantile cross-sector edges survived "
            "the 60% threshold."
        )


# Final graph-source decision.
if n_stable_cross_edges >= 1:
    final_graph_source = "HAWKES_LAMBDA_MIN_STABILITY"
    graph_decision = hawkes_graph_decision
elif tail_graph_available:
    final_graph_source = "LOWER_TAIL_QUANTILE_STABILITY"
    graph_decision = (
        "HAWKES_UNSTABLE_QUANTILE_TAIL_GRAPH_USED_AS_SOFT_PRIOR"
    )
else:
    final_graph_source = "NO_STABLE_ECONOMETRIC_GRAPH"
    graph_decision = (
        "NO_STABLE_ECONOMETRIC_GRAPH_"
        "DEEP_MODEL_MUST_INCLUDE_NO_GRAPH_OR_LEARNED_GRAPH_ONLY"
    )

graph_decision_table = pd.DataFrame({
    "Criterion": [
        "Hawkes primary penalty rule",
        "Hawkes lambda_min",
        "Hawkes lambda_1SE robustness",
        "Stable Hawkes cross-sector edges",
        "Hawkes stability threshold",
        "Positive Validation Brier-skill horizons",
        "Tail fallback activated",
        "Tail quantile",
        "Tail selected alpha",
        "Stable tail cross-sector edges",
        "Final graph source",
        "Test used in graph-structure decision",
        "Decision",
    ],
    "Value": [
        "lambda_min + separate 60% stability selection",
        primary_lambda,
        primary_lambda_1se,
        n_stable_cross_edges,
        STABILITY_THRESHOLD,
        positive_validation_skill_horizons,
        int(n_stable_cross_edges == 0),
        TAIL_QUANTILE,
        tail_graph_alpha,
        tail_stable_cross_edges,
        final_graph_source,
        "NO",
        graph_decision,
    ],
})

graph_decision_table.to_csv(
    TABLE_DIR / "Table_P17_Graph_Decision.csv",
    index=False
)

log(f"Final graph source: {final_graph_source}")
log(f"Graph decision: {graph_decision}")

# =============================================================================
# 21. SPLIT-AWARE DYNAMIC GRAPH FOR LATER GRAPH DL
# =============================================================================

def dynamic_hawkes_graph_from_fit(
    fit: dict,
) -> np.ndarray:

    post_state = build_post_event_state(
        C,
        fit["decay"],
    )

    graph = np.zeros(
        (T, S, S),
        dtype=np.float32,
    )

    for t in range(T):
        graph[t] = (
            fit["alpha"]
            * post_state[t][np.newaxis, :]
        ).astype(np.float32)

    return graph


if final_graph_source == "HAWKES_LAMBDA_MIN_STABILITY":

    train_graph = dynamic_hawkes_graph_from_fit(
        train_stable_fit
    )
    final_graph = dynamic_hawkes_graph_from_fit(
        final_stable_fit
    )

elif final_graph_source == "LOWER_TAIL_QUANTILE_STABILITY":

    train_graph = build_dynamic_tail_graph(
        tail_beta_train
    )
    final_graph = build_dynamic_tail_graph(
        tail_beta_final
    )

else:

    train_graph = np.zeros(
        (T, S, S),
        dtype=np.float32,
    )
    final_graph = train_graph.copy()


# Split-aware:
#   * Train and Validation use TRAIN-estimated graph coefficients/support.
#   * Test uses TRAIN+VALIDATION coefficient refit with support/hyperparameters
#     frozen before Test.
split_aware_graph = train_graph.copy()
split_aware_graph[test_date_mask] = final_graph[test_date_mask]

graph_rows = []

for t, date in enumerate(dates):
    for receiver in range(S):
        for sender in range(S):

            if final_graph_source == "HAWKES_LAMBDA_MIN_STABILITY":
                stable_edge_indicator = int(
                    stable_cross_support[receiver, sender]
                )
            elif final_graph_source == "LOWER_TAIL_QUANTILE_STABILITY":
                stable_edge_indicator = int(
                    tail_support[receiver, sender]
                )
            else:
                stable_edge_indicator = 0

            graph_rows.append({
                "Date": date,
                "Split": splits[t],
                "GraphSource": final_graph_source,
                "FromSector": sectors[sender],
                "ToSector": sectors[receiver],
                "Weight": float(
                    split_aware_graph[
                        t,
                        receiver,
                        sender,
                    ]
                ),
                "StableCrossEdge": stable_edge_indicator,
                "SelfEdge": int(receiver == sender),
            })

dynamic_graph_long = pd.DataFrame(graph_rows)

dynamic_graph_long.to_csv(
    DATA_DIR / "econometric_dynamic_graph.csv.gz",
    index=False,
    compression="gzip",
)

# Compatibility alias for later DL code.
dynamic_graph_long.to_csv(
    DATA_DIR / "stable_hawkes_dynamic_graph.csv.gz",
    index=False,
    compression="gzip",
)

np.savez_compressed(
    DATA_DIR / "econometric_dynamic_graph_arrays.npz",
    dates=dates.astype(str).to_numpy(),
    sectors=np.array(sectors, dtype=object),
    graph_source=np.array([final_graph_source], dtype=object),
    adjacency=split_aware_graph,
    train_fit_adjacency=train_graph,
    final_fit_adjacency=final_graph,
    hawkes_stable_cross_support=stable_cross_support.astype(np.uint8),
    tail_stable_cross_support=tail_support.astype(np.uint8),
    hawkes_train_alpha=train_stable_fit["alpha"].astype(np.float32),
    hawkes_final_alpha=final_stable_fit["alpha"].astype(np.float32),
    tail_train_beta=tail_beta_train.astype(np.float32),
    tail_final_beta=tail_beta_final.astype(np.float32),
)

# Compatibility alias expected by later graph code.
np.savez_compressed(
    DATA_DIR / "stable_hawkes_dynamic_graph_arrays.npz",
    dates=dates.astype(str).to_numpy(),
    sectors=np.array(sectors, dtype=object),
    graph_source=np.array([final_graph_source], dtype=object),
    adjacency=split_aware_graph,
)

# =============================================================================
# 22. FINAL MODELLING MASTER FOR DEEP-LEARNING PHASE
# =============================================================================

model_master = primary.merge(
    targets_long,
    on=["Date", "Sector", "Split"],
    how="left",
    validate="one_to_one",
)

# Add strictly constructed baseline and Hawkes probabilities.
probability_rows = []

for t, date in enumerate(dates):
    for s, sector in enumerate(sectors):

        row = {
            "Date": date,
            "Sector": sector,
        }

        for h in HORIZONS:

            row[f"FixedHistoricalProb_{h}"] = (
                fixed_historical[h][t, s]
            )

            row[f"ExpandingHistoricalProb_{h}"] = (
                expanding_historical[h][t, s]
            )

            # Split-aware stable Hawkes probability:
            # validation uses train fit; test uses train+validation fit.
            if splits[t] == "Test":
                row[f"StableHawkesProb_{h}"] = (
                    final_probabilities[
                        "StablePostSelectionHawkes"
                    ][h][t, s]
                )
            else:
                row[f"StableHawkesProb_{h}"] = (
                    train_probabilities[
                        "StablePostSelectionHawkes"
                    ][h][t, s]
                )

        probability_rows.append(row)

probability_features = pd.DataFrame(
    probability_rows
)

model_master = model_master.merge(
    probability_features,
    on=["Date", "Sector"],
    how="left",
    validate="one_to_one",
)

model_master["EconometricGraphSource"] = final_graph_source

model_master.to_csv(
    DATA_DIR / "phase2_deep_learning_master.csv.gz",
    index=False,
    compression="gzip",
)

targets_long.to_csv(
    DATA_DIR / "time_to_crash_targets.csv.gz",
    index=False,
    compression="gzip",
)

# =============================================================================
# 23. SAVE MODEL BUNDLE
# =============================================================================

model_bundle = {
    "version": "PythonPhase1_1.2",
    "seed": SEED,
    "sectors": sectors,
    "horizons": HORIZONS,
    "primary_half_life": primary_half_life,
    "selected_lambda_min": primary_lambda,
    "selected_lambda_1se_robustness": primary_lambda_1se,
    "stability_threshold": STABILITY_THRESHOLD,
    "stable_cross_support": stable_cross_support,
    "graph_decision": graph_decision,
    "final_graph_source": final_graph_source,
    "tail_graph_selected_alpha": tail_graph_alpha,
    "tail_graph_stable_cross_edges": tail_stable_cross_edges,
    "tail_graph_support": tail_support,
    "tail_graph_train_beta": tail_beta_train,
    "tail_graph_final_beta": tail_beta_final,
    "train_fit": {
        "mu": train_stable_fit["mu"],
        "alpha": train_stable_fit["alpha"],
        "decay": train_stable_fit["decay"],
    },
    "train_validation_refit": {
        "mu": final_stable_fit["mu"],
        "alpha": final_stable_fit["alpha"],
        "decay": final_stable_fit["decay"],
    },
}

with open(
    MODEL_DIR / "sparse_stable_hawkes_bundle.pkl",
    "wb",
) as f:
    pickle.dump(model_bundle, f)

# =============================================================================
# 24. FIGURES
# =============================================================================

# Figure 1: Crash counts by sector and split.
crash_plot = (
    primary.dropna(subset=["Crash_Main"])
    .groupby(
        ["Sector", "Split"],
        as_index=False,
    )["Crash_Main"]
    .sum()
)

figure_data = crash_plot.pivot(
    index="Sector",
    columns="Split",
    values="Crash_Main",
).fillna(0)

fig, ax = plt.subplots(figsize=(11, 6))
figure_data.plot(
    kind="bar",
    ax=ax,
)
ax.set_title(
    "Observed 2.5% EVT Crash Events by Sector and Sample Split"
)
ax.set_xlabel("Sector")
ax.set_ylabel("Crash events")
ax.tick_params(
    axis="x",
    rotation=45,
)
ax.legend(title="Split")
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_P01_Crash_Counts.png",
    dpi=300,
)
plt.close(fig)

# Figure 2: BIC half-life selection.
figure_data = half_life_selection.sort_values(
    "HalfLife"
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    figure_data["HalfLife"],
    figure_data["BIC"],
    marker="o",
)
ax.set_xlabel("Hawkes half-life (trading days)")
ax.set_ylabel("TRAIN BIC")
ax.set_title("Hawkes Memory Selection")
ax.set_xticks(HALF_LIFE_CANDIDATES)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_P02_Hawkes_HalfLife_BIC.png",
    dpi=300,
)
plt.close(fig)

# Figure 3: L1 CV.
fig, ax = plt.subplots(figsize=(9, 6))
for half_life, g in cv_summary.groupby("HalfLife"):
    g = g.sort_values("Lambda")
    ax.plot(
        g["Lambda"],
        g["MeanValidationLogScore"],
        marker="o",
        label=f"{half_life:g}-day half-life",
    )
ax.set_xscale(
    "symlog",
    linthresh=0.005,
)
ax.set_xlabel("L1 penalty")
ax.set_ylabel(
    "Mean TRAIN-temporal-CV log score"
)
ax.set_title(
    "Sparse Hawkes Regularization Selection"
)
ax.legend()
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_P03_L1_Temporal_CV.png",
    dpi=300,
)
plt.close(fig)

# Figure 4: Edge stability heatmap.
stability_matrix = np.zeros(
    (S, S),
    dtype=float,
)

for row in primary_stability.itertuples(index=False):
    receiver = sector_to_idx[row.ToSector]
    sender = sector_to_idx[row.FromSector]
    stability_matrix[
        receiver,
        sender,
    ] = row.SelectionFrequency

fig, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(
    stability_matrix,
    vmin=0,
    vmax=1,
    aspect="auto",
)
ax.set_xticks(range(S))
ax.set_yticks(range(S))
ax.set_xticklabels(
    sectors,
    rotation=45,
    ha="right",
)
ax.set_yticklabels(sectors)
ax.set_xlabel("Source sector")
ax.set_ylabel("Receiving sector")
ax.set_title(
    f"Lambda-Min Block-Stability Selection Frequencies "
    f"({primary_half_life:g}-day half-life)"
)
fig.colorbar(
    image,
    ax=ax,
    label="Selection frequency",
)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_P04_Edge_Stability_Heatmap.png",
    dpi=300,
)
plt.close(fig)

# Figure 4B: Tail-quantile edge stability if fallback was activated.
if n_stable_cross_edges == 0 and len(tail_stability_table):

    tail_stability_matrix = np.zeros((S, S), dtype=float)

    for row in tail_stability_table.itertuples(index=False):
        receiver = sector_to_idx[row.ToSector]
        sender = sector_to_idx[row.FromSector]
        tail_stability_matrix[receiver, sender] = row.SelectionFrequency

    fig, ax = plt.subplots(figsize=(9, 8))
    image = ax.imshow(
        tail_stability_matrix,
        vmin=0,
        vmax=1,
        aspect="auto",
    )
    ax.set_xticks(range(S))
    ax.set_yticks(range(S))
    ax.set_xticklabels(
        sectors,
        rotation=45,
        ha="right",
    )
    ax.set_yticklabels(sectors)
    ax.set_xlabel("Source sector")
    ax.set_ylabel("Receiving sector")
    ax.set_title(
        f"Lower-Tail Quantile Edge Stability "
        f"(tau={TAIL_QUANTILE:g})"
    )
    fig.colorbar(
        image,
        ax=ax,
        label="Selection frequency",
    )
    fig.tight_layout()
    fig.savefig(
        FIG_DIR / "Figure_P04B_Tail_Quantile_Edge_Stability.png",
        dpi=300,
    )
    plt.close(fig)

# Figure 5: Final TEST Brier.
test_pooled = forecast_metrics[
    (forecast_metrics["Split"] == "Test")
    & (forecast_metrics["Sector"] == "POOLED")
].copy()

fig, ax = plt.subplots(figsize=(10, 6))

for model_name, g in test_pooled.groupby("Model"):
    g = g.sort_values("Horizon")
    ax.plot(
        g["Horizon"],
        g["Brier"],
        marker="o",
        label=model_name,
    )

ax.set_xlabel(
    "Forecast horizon (trading days)"
)
ax.set_ylabel(
    "Brier score (lower is better)"
)
ax.set_xticks(HORIZONS)
ax.set_title(
    "Final Test Probability Forecast Performance"
)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_P05_Test_Brier.png",
    dpi=300,
)
plt.close(fig)

# =============================================================================
# 25. EXCEL WORKBOOK WITH ENGINE FALLBACK
# =============================================================================

excel_path = (
    EXCEL_DIR
    / "Python_Phase1_Sparse_Stable_Hawkes.xlsx"
)

if importlib.util.find_spec("xlsxwriter") is not None:
    excel_engine = "xlsxwriter"
elif importlib.util.find_spec("openpyxl") is not None:
    excel_engine = "openpyxl"
else:
    excel_engine = None

excel_tables = {
    "R Handoff Integrity": r_audit,
    "Sector Audit": sector_audit,
    "Target Summary": target_summary,
    "HalfLife Selection": half_life_selection,
    "Temporal CV Folds": fold_table,
    "L1 CV Summary": cv_summary,
    "L1 Selection": lambda_selection,
    "Edge Stability": stability_all,
    "Stable Edges": stable_edges,
    "HalfLife Summary": half_life_summary,
    "HalfLife Overlap": half_life_overlap,
    "Network Diagnostics": network_diagnostics,
    "Forecast Metrics": forecast_metrics,
    "Forecast Skill": forecast_skill,
    "Graph Decision": graph_decision_table,
    "Hawkes 1SE Robustness": robustness_stability_1se,
    "Tail Quantile CV": tail_cv_summary,
    "Tail Edge Stability": tail_stability_table,
}

if excel_engine is not None:
    try:
        with pd.ExcelWriter(
            excel_path,
            engine=excel_engine,
        ) as writer:

            for sheet_name, dataframe in excel_tables.items():
                dataframe.to_excel(
                    writer,
                    sheet_name=sheet_name[:31],
                    index=False,
                )

            if excel_engine == "xlsxwriter":
                workbook = writer.book
                header_format = workbook.add_format({
                    "bold": True,
                    "text_wrap": True,
                    "valign": "top",
                    "border": 1,
                })

                for _, worksheet in writer.sheets.items():
                    worksheet.freeze_panes(1, 0)
                    worksheet.set_row(
                        0,
                        28,
                        header_format,
                    )
                    worksheet.set_column(
                        0,
                        25,
                        16,
                    )

            else:
                for _, worksheet in writer.sheets.items():
                    worksheet.freeze_panes = "A2"

        log(
            f"Excel workbook saved using "
            f"{excel_engine}: {excel_path}"
        )

    except Exception as exc:
        log(
            f"WARNING: Excel export failed: {repr(exc)}"
        )
else:
    log(
        "WARNING: neither xlsxwriter nor openpyxl "
        "is available. CSV outputs remain complete."
    )

# =============================================================================
# 26. METADATA FOR DEEP-LEARNING PHASE
# =============================================================================

python_metadata = {
    "ProjectTitle": (
        "Forecasting Sectoral Crash Risk and Contagion: "
        "Integrating Dynamic Volatility, Extreme-Value Modelling "
        "and Graph Deep Learning"
    ),
    "PythonPhase": "1",
    "ScriptVersion": "1.2",
    "GeneratedAt": datetime.now().isoformat(),
    "RandomSeed": SEED,
    "RScriptVersionRecorded": metadata.get("ScriptVersion"),
    "PrimarySectors": sectors,
    "ForecastHorizons": HORIZONS,
    "SelectedHawkesHalfLife": primary_half_life,
    "SelectedL1Lambda_Min_Primary": primary_lambda,
    "SelectedL1Lambda_1SE_Robustness": primary_lambda_1se,
    "StabilityThreshold": STABILITY_THRESHOLD,
    "StableHawkesCrossEdges": n_stable_cross_edges,
    "StableTailQuantileCrossEdges": tail_stable_cross_edges,
    "FinalGraphSource": final_graph_source,
    "GraphDecision": graph_decision,
    "TestUsedForStructureSelection": False,
    "TestCoefficientRefitSample": "Train+Validation only",
}

with open(
    DATA_DIR / "python_phase1_metadata.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        python_metadata,
        f,
        indent=2,
        default=str,
    )

# =============================================================================
# 27. FINAL INTEGRITY CHECKS
# =============================================================================

integrity_checks = [
    (
        "Corrected R EVT parameter variation passed",
        shape_sd >= 1e-6 and scale_sd >= 1e-6,
    ),
    (
        "R Crash_Main equals 2.5% EVT label",
        main_mismatch == 0,
    ),
    (
        "R EVT thresholds correctly ordered",
        threshold_order_ok,
    ),
    (
        "No Test dates used in temporal CV",
        not any(
            np.any(
                fold["FitMask"]
                & test_date_mask
            )
            or np.any(
                fold["ValidationMask"]
                & test_date_mask
            )
            for fold in folds
        ),
    ),
    (
        "Final stable Hawkes parameters finite",
        bool(
            np.all(
                np.isfinite(
                    final_stable_fit["mu"]
                )
            )
            and np.all(
                np.isfinite(
                    final_stable_fit["alpha"]
                )
            )
        ),
    ),
    (
        "Final stable Hawkes parameters non-negative",
        bool(
            np.all(
                final_stable_fit["mu"] > 0
            )
            and np.all(
                final_stable_fit["alpha"] >= -1e-12
            )
        ),
    ),
    (
        "Final stable network below branching guard",
        final_stable_radius
        < MAX_BRANCHING_SPECTRAL_RADIUS,
    ),
    (
        "Final graph source resolved without Test selection",
        final_graph_source in {
            "HAWKES_LAMBDA_MIN_STABILITY",
            "LOWER_TAIL_QUANTILE_STABILITY",
            "NO_STABLE_ECONOMETRIC_GRAPH",
        },
    ),
    (
        "Validation targets contain both classes",
        all(
            len(
                np.unique(
                    target_within[h][
                        validation_date_mask
                    ][
                        np.isfinite(
                            target_within[h][
                                validation_date_mask
                            ]
                        )
                    ]
                )
            ) == 2
            for h in HORIZONS
        ),
    ),
    (
        "Test targets contain both classes",
        all(
            len(
                np.unique(
                    target_within[h][
                        test_date_mask
                    ][
                        np.isfinite(
                            target_within[h][
                                test_date_mask
                            ]
                        )
                    ]
                )
            ) == 2
            for h in HORIZONS
        ),
    ),
]

integrity_table = pd.DataFrame(
    integrity_checks,
    columns=["Check", "Passed"],
)

integrity_table.to_csv(
    TABLE_DIR / "Table_P18_Final_Integrity_Checks.csv",
    index=False
)

if not integrity_table["Passed"].all():
    failed = integrity_table.loc[
        ~integrity_table["Passed"],
        "Check",
    ].tolist()

    raise RuntimeError(
        f"Final Python Phase 1 integrity checks failed: {failed}"
    )

# =============================================================================
# 28. ZIP OUTPUTS
# =============================================================================

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

zip_output = (
    ZIP_DIR
    / "Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip"
)

if zip_output.exists():
    zip_output.unlink()

with zipfile.ZipFile(
    zip_output,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:

    for path in OUTPUT_ROOT.rglob("*"):

        if (
            path.is_file()
            and path != zip_output
        ):
            zf.write(
                path,
                arcname=path.relative_to(
                    OUTPUT_ROOT
                ),
            )

log(
    f"All Python Phase 1 outputs zipped to: "
    f"{zip_output}"
)

# =============================================================================
# 29. CONSOLE SUMMARY
# =============================================================================

print("\n" + "=" * 96)
print("PYTHON PHASE 1 COMPLETED SUCCESSFULLY")
print("=" * 96)

print(
    f"Input: final R handoff "
    f"({INPUT_ZIP_PATH.name})"
)

print(
    f"Primary sectors: {S}"
)

for sector in sectors:
    print(f"  - {sector}")

print(
    f"\nSelected Hawkes half-life: "
    f"{primary_half_life:g} trading days"
)

print(
    f"Selected L1 lambda_min (primary): "
    f"{primary_lambda:g}"
)

print(
    f"Stable Hawkes cross-sector edges: "
    f"{n_stable_cross_edges}"
)

print(
    f"Final graph source: "
    f"{final_graph_source}"
)

print(
    f"Graph decision: "
    f"{graph_decision}"
)

print("\nStable cross-sector edges:")

stable_cross_final = stable_edges[
    stable_edges["SelectedCrossEdge"] == 1
].sort_values(
    [
        "SelectionFrequency",
        "TrainValidationRefitAlpha",
    ],
    ascending=False,
)

if len(stable_cross_final):
    for row in stable_cross_final.itertuples(index=False):
        print(
            f"  {row.FromSector} -> {row.ToSector}: "
            f"stability={row.SelectionFrequency:.3f}, "
            f"alpha_final={row.TrainValidationRefitAlpha:.6f}"
        )
else:
    print("  None survived the stability threshold.")

if final_graph_source == "LOWER_TAIL_QUANTILE_STABILITY":
    print("\nStable lower-tail quantile cross-sector edges:")
    tail_selected_console = tail_stability_table[
        tail_stability_table["StableCrossEdge"] == 1
    ].sort_values(
        ["SelectionFrequency", "MeanBetaAcrossReps"],
        ascending=False,
    )
    for row in tail_selected_console.itertuples(index=False):
        print(
            f"  {row.FromSector} -> {row.ToSector}: "
            f"stability={row.SelectionFrequency:.3f}, "
            f"mean_beta={row.MeanBetaAcrossReps:.6f}"
        )

print("\nDeep-learning handoff files:")
print(
    f"  {DATA_DIR / 'phase2_deep_learning_master.csv.gz'}"
)
print(
    f"  {DATA_DIR / 'econometric_dynamic_graph.csv.gz'}"
)
print(
    f"  {DATA_DIR / 'econometric_dynamic_graph_arrays.npz'}"
)
print(
    f"  {MODEL_DIR / 'sparse_stable_hawkes_bundle.pkl'}"
)

print("\nZIP to send back for review:")
print(f"  {zip_output}")

print("=" * 96)

# =============================================================================
# 30. AUTOMATIC COLAB DOWNLOAD
# =============================================================================

try:
    from google.colab import files

    print(
        "\nStarting browser download of "
        "Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip ..."
    )

    files.download(
        str(zip_output)
    )

except ImportError:
    print(
        "\nNot running in Google Colab. "
        f"Retrieve the ZIP manually from: {zip_output}"
    )

# =============================================================================
# END
# =============================================================================


[2026-09-01 12:19:18] ================================================================================================
[2026-09-01 12:19:18] PYTHON PHASE 1 START — DIRECT R HANDOFF
[2026-09-01 12:19:18] Python: 3.13.15
[2026-09-01 12:19:18] Platform: Linux-6.6.122+-x86_64-with-glibc2.35
[2026-09-01 12:19:18] Seed: 20260901

Please upload the FINAL R handoff file:
    04_Python_Handoff.zip



Saving 04_Python_Handoff.zip to 04_Python_Handoff (4).zip
[2026-09-01 12:20:39] Validated uploaded R handoff: /content/04_Python_Handoff (4).zip
[2026-09-01 12:20:39] Accepted R handoff: /content/04_Python_Handoff (4).zip
[2026-09-01 12:20:40] Primary sectors (6): ['Banking', 'Commercial and services', 'Energy and Petroleum', 'Insurance', 'Investment', 'Manufacturing and Allied']
[2026-09-01 12:20:40] R master: 27,368 rows, 2,488 dates, 11 total sectors.
[2026-09-01 12:20:40] Event panel: 2488 dates x 6 sectors. Train crashes=141; Validation crashes=73; Test crashes=66.
[2026-09-01 12:20:40] Selecting Hawkes half-life by TRAIN-only BIC...
[2026-09-01 12:20:40]   Fitting full Hawkes at half-life=1 days.
[2026-09-01 12:20:42]   Fitting full Hawkes at half-life=2 days.
[2026-09-01 12:20:43]   Fitting full Hawkes at half-life=5 days.
[2026-09-01 12:20:44]   Fitting full Hawkes at half-life=10 days.
[2026-09-01 12:20:45]   Fitting full Hawkes at half-life=22 days.
[2026-09-01 12:20:46] TRAI

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
# =============================================================================
# PYTHON PHASE 2.1 — CORRECTED ML / DEEP LEARNING / GRAPH SURVIVAL FORECASTING
#
# Forecasting Sectoral Crash Risk and Contagion:
# Integrating Dynamic Volatility, Extreme-Value Modelling and Graph Deep Learning
#
# INPUT
# -----
# Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip
#
# PURPOSE
# -------
# This script consumes the frozen Phase-1 econometric/contagion output and
# estimates the forecasting models used in the main paper:
#
#   1. Historical probability baselines.
#   2. Stability-selected Hawkes probability baseline.
#   3. XGBoost benchmark (four horizon classifiers; monotone rearrangement).
#   4. LSTM discrete-time survival model.
#   5. Temporal Transformer discrete-time survival benchmark.
#   6. Graph Survival Transformer — no graph prior.
#   7. Graph Survival Transformer — random static graph placebo.
#   8. Graph Survival Transformer — static stability-selected Hawkes graph.
#   9. PROPOSED: Graph Survival Transformer with the dynamic Hawkes graph
#      supplied as a SOFT attention prior.
#
# KEY DESIGN PRINCIPLES
# ---------------------
# * Only the uploaded NSE/R-derived data are used. No external predictors.
# * Train / Validation / Test are chronological; no random data split.
# * Forecast-origin features contain information available through day t only.
# * Time-to-crash outcomes are RECONSTRUCTED HERE with split-aware censoring.
#   Therefore Validation targets never use Test outcomes and Train targets
#   never use Validation outcomes.
# * 1/5/10/22-day neural probabilities are derived from one 22-day daily
#   hazard path:
#
#       P(T <= H) = 1 - product_{k=1}^H [1 - h_k].
#
#   Hence neural probabilities are coherent by construction.
# * XGBoost is a conventional benchmark and uses separate horizon classifiers;
#   its four probabilities are monotonically rearranged after calibration.
# * Deep models use the SAME economic predictors. Graph models differ only in
#   their graph prior, permitting clean ablation tests.
# * The Hawkes graph is a SOFT prior: it biases graph attention but does not
#   hard-mask other sector interactions.
# * Validation is used for early stopping / calibration only.
# * Test is evaluated once after all model choices are fixed.
# * Neural rare-event weighting uses sqrt imbalance capped at 3; all neural
#   models receive post-hoc hazard temperature calibration on Validation.
# * Main probability metrics: Brier Score, Log Score, PR-AUC, ROC-AUC and
#   calibration intercept/slope. Accuracy is deliberately not a headline metric.
# * Paired moving-block bootstrap inference is reported for the proposed model
#   versus every benchmark on the untouched Test sample.
#
# VERSION: 2.1
# DATE: 2026-09-01
# =============================================================================

from __future__ import annotations

import os
import sys
import gc
import json
import math
import time
import random
import shutil
import pickle
import zipfile
import warnings
import platform
import subprocess
import importlib.util
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =============================================================================
# 0. REPRODUCIBILITY / CONFIGURATION
# =============================================================================

SEED = 20260901

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

INPUT_ZIP = os.getenv(
    "NSE_PHASE1_ZIP",
    "/content/Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip"
)

OUTPUT_ROOT = Path(os.getenv(
    "NSE_PHASE2_OUTPUT_DIR",
    "/content/Sectoral_Crash_Risk_Contagion_Python_Phase2_1"
))

HORIZONS = [1, 5, 10, 22]
MAX_HORIZON = 22
LOOKBACK = int(os.getenv("NSE_LOOKBACK", "60"))

# Deep-learning hyperparameters. These are intentionally compact because the
# dataset contains six sectors and ~2,500 trading dates.
BATCH_SIZE = int(os.getenv("NSE_BATCH_SIZE", "128"))
GRAPH_BATCH_SIZE = int(os.getenv("NSE_GRAPH_BATCH_SIZE", "32"))
MAX_EPOCHS = int(os.getenv("NSE_MAX_EPOCHS", "60"))
PATIENCE = int(os.getenv("NSE_PATIENCE", "8"))
LEARNING_RATE = float(os.getenv("NSE_LEARNING_RATE", "0.001"))
WEIGHT_DECAY = float(os.getenv("NSE_WEIGHT_DECAY", "0.0001"))
D_MODEL = int(os.getenv("NSE_D_MODEL", "48"))
N_HEADS = int(os.getenv("NSE_N_HEADS", "4"))
N_TRANSFORMER_LAYERS = int(os.getenv("NSE_TRANSFORMER_LAYERS", "2"))
DROPOUT = float(os.getenv("NSE_DROPOUT", "0.10"))
SECTOR_EMBED_DIM = int(os.getenv("NSE_SECTOR_EMBED_DIM", "8"))

# Positive hazard weighting.
#
# CORRECTION 2.1:
# The Phase-2 neural cap of 10 produced a hazard weight near 6.7 and required
# very strong post-hoc temperature flattening. Neural survival models now use a
# conservative cap of 3. XGBoost retains the previous cap of 10 so the strong
# tabular benchmark is otherwise unchanged.
NEURAL_MAX_POS_WEIGHT = float(
    os.getenv("NSE_NEURAL_MAX_POS_WEIGHT", "3.0")
)
XGB_MAX_POS_WEIGHT = float(
    os.getenv("NSE_XGB_MAX_POS_WEIGHT", "10.0")
)

# Magnitude-preserving Hawkes graph transformation.
#
# A positive Train-only q99 reference is mapped to 1:
#
#     scaled(A) = log(1 + A / q99_train) / log(2)
#
# Values are globally clipped only at 3 to limit extreme numerical leverage.
# Crucially, there is NO row normalization, so 0.001 remains much weaker than
# 0.10 after transformation.
GRAPH_SCALE_QUANTILE = float(
    os.getenv("NSE_GRAPH_SCALE_QUANTILE", "0.99")
)
GRAPH_SCALE_CLIP = float(
    os.getenv("NSE_GRAPH_SCALE_CLIP", "3.0")
)

# XGBoost.
XGB_MAX_ROUNDS = int(os.getenv("NSE_XGB_MAX_ROUNDS", "1500"))
XGB_EARLY_STOP = int(os.getenv("NSE_XGB_EARLY_STOP", "75"))

# Moving-block bootstrap.
BOOTSTRAP_REPS = int(os.getenv("NSE_BOOTSTRAP_REPS", "500"))
BOOTSTRAP_BLOCK = int(os.getenv("NSE_BOOTSTRAP_BLOCK", "22"))

EPS = 1e-8

PROPOSED_MODEL = "DynamicHawkesGraphTransformer"

# =============================================================================
# 1. OUTPUT DIRECTORIES / LOGGING
# =============================================================================

TABLE_DIR = OUTPUT_ROOT / "01_Tables"
FIG_DIR = OUTPUT_ROOT / "02_Figures"
EXCEL_DIR = OUTPUT_ROOT / "03_Excel"
MODEL_DIR = OUTPUT_ROOT / "04_Model_Objects"
DATA_DIR = OUTPUT_ROOT / "05_Processed_Data"
LOG_DIR = OUTPUT_ROOT / "06_Logs"
ZIP_DIR = OUTPUT_ROOT / "07_Zip"
EXTRACT_DIR = OUTPUT_ROOT / "_Phase1_Extracted"

for directory in [
    OUTPUT_ROOT, TABLE_DIR, FIG_DIR, EXCEL_DIR, MODEL_DIR,
    DATA_DIR, LOG_DIR, ZIP_DIR, EXTRACT_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / "Python_Phase2_run_log.txt"

def log(message: str):
    stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{stamp}] {message}"
    print(line)
    with open(LOG_FILE, "a", encoding="utf-8") as handle:
        handle.write(line + "\n")

log("=" * 100)
log("PYTHON PHASE 2.1 START — CORRECTED ML / DL / GRAPH SURVIVAL FORECASTING")
log(f"Python={sys.version.split()[0]}; platform={platform.platform()}; seed={SEED}")

# =============================================================================
# 2. PACKAGE CHECKS
# =============================================================================

def ensure_package(import_name: str, pip_name: str | None = None):
    if importlib.util.find_spec(import_name) is not None:
        return
    pip_name = pip_name or import_name
    log(f"Installing missing package: {pip_name}")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", pip_name]
    )

ensure_package("torch")
ensure_package("xgboost")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import xgboost as xgb

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log(f"PyTorch={torch.__version__}; device={DEVICE}")

# =============================================================================
# 3. ALWAYS ASK USER TO UPLOAD PHASE-1 ZIP IN COLAB
# =============================================================================

REQUIRED_PHASE1_MEMBERS = {
    "phase2_deep_learning_master.csv.gz",
    "econometric_dynamic_graph_arrays.npz",
    "python_phase1_metadata.json",
    "Table_P12_Final_Stable_Edge_Parameters.csv",
}

def zip_basenames(path: Path) -> set[str]:
    try:
        with zipfile.ZipFile(path, "r") as archive:
            return {
                Path(name).name
                for name in archive.namelist()
                if not name.endswith("/")
            }
    except Exception:
        return set()

def is_valid_phase1_zip(path: Path) -> bool:
    return (
        path.exists()
        and path.suffix.lower() == ".zip"
        and REQUIRED_PHASE1_MEMBERS.issubset(zip_basenames(path))
    )

def resolve_phase1_zip(configured: str) -> Path:
    """
    Colab behaviour: ALWAYS open a file picker. This avoids accidentally using
    a stale R handoff or previous Python ZIP already present in /content.
    """
    try:
        from google.colab import files

        while True:
            print(
                "\nPlease upload the completed Phase-1 ZIP:\n"
                "    Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip\n"
            )
            uploaded = files.upload()

            candidates = [
                Path("/content") / name
                for name in uploaded
                if name.lower().endswith(".zip")
            ]

            valid = [p for p in candidates if is_valid_phase1_zip(p)]

            if len(valid) == 1:
                log(f"Validated uploaded Phase-1 ZIP: {valid[0]}")
                return valid[0]

            if len(valid) > 1:
                print(
                    "\nMore than one valid Phase-1 ZIP was uploaded. "
                    "Please upload exactly one ZIP.\n"
                )
                continue

            for p in candidates:
                missing = sorted(
                    REQUIRED_PHASE1_MEMBERS - zip_basenames(p)
                )
                log(
                    f"Rejected '{p.name}'. Missing Phase-1 members: {missing}"
                )

            print(
                "\nThe selected ZIP is not the required completed Phase-1 ZIP. "
                "Please choose:\n"
                "    Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip\n"
            )

    except ImportError:
        configured_path = Path(configured)

        if is_valid_phase1_zip(configured_path):
            return configured_path

        # Local / notebook fallback: locate a validated ZIP by contents.
        for folder in [Path.cwd(), Path("/mnt/data")]:
            if folder.exists():
                for candidate in folder.glob("*.zip"):
                    if is_valid_phase1_zip(candidate):
                        return candidate

        raise FileNotFoundError(
            "Could not locate a valid Phase-1 ZIP. "
            "Set NSE_PHASE1_ZIP to the correct path."
        )

INPUT_ZIP_PATH = resolve_phase1_zip(INPUT_ZIP)
log(f"Accepted Phase-1 ZIP: {INPUT_ZIP_PATH}")

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(INPUT_ZIP_PATH, "r") as archive:
    archive.extractall(EXTRACT_DIR)

def find_one(filename: str) -> Path:
    matches = list(EXTRACT_DIR.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one '{filename}' in Phase-1 ZIP; "
            f"found {len(matches)}."
        )
    return matches[0]

MASTER_FILE = find_one("phase2_deep_learning_master.csv.gz")
GRAPH_FILE = find_one("econometric_dynamic_graph_arrays.npz")
META_FILE = find_one("python_phase1_metadata.json")
EDGE_FILE = find_one("Table_P12_Final_Stable_Edge_Parameters.csv")

# =============================================================================
# 4. LOAD / AUDIT PHASE-1 OUTPUT
# =============================================================================

master = pd.read_csv(MASTER_FILE, parse_dates=["Date"])
master = master.sort_values(["Date", "Sector"]).reset_index(drop=True)

with open(META_FILE, "r", encoding="utf-8") as handle:
    phase1_meta = json.load(handle)

graph_npz = np.load(GRAPH_FILE, allow_pickle=True)
stable_edges = pd.read_csv(EDGE_FILE)

sectors = graph_npz["sectors"].astype(str).tolist()
S = len(sectors)
sector_to_idx = {sector: i for i, sector in enumerate(sectors)}

dates = pd.DatetimeIndex(graph_npz["dates"].astype(str))
T = len(dates)
date_to_idx = {pd.Timestamp(date): i for i, date in enumerate(dates)}

graph_source = str(graph_npz["graph_source"].reshape(-1)[0])
dynamic_graph_all = graph_npz["adjacency"].astype(np.float32)
dynamic_graph_train_fit = graph_npz["train_fit_adjacency"].astype(np.float32)
dynamic_graph_final_fit = graph_npz["final_fit_adjacency"].astype(np.float32)
hawkes_train_alpha = graph_npz["hawkes_train_alpha"].astype(np.float32)
hawkes_final_alpha = graph_npz["hawkes_final_alpha"].astype(np.float32)

if dynamic_graph_all.shape != (T, S, S):
    raise RuntimeError(
        f"Unexpected graph shape: {dynamic_graph_all.shape}; "
        f"expected {(T, S, S)}."
    )

if set(master["Sector"].unique()) != set(sectors):
    raise RuntimeError("Master sectors do not match graph-array sectors.")

if master["Date"].nunique() != T:
    raise RuntimeError("Master dates do not match graph-array dates.")

if graph_source != "HAWKES_LAMBDA_MIN_STABILITY":
    log(
        f"WARNING: graph source is '{graph_source}', not the expected Hawkes "
        "lambda-min stability graph. The code will still proceed."
    )

# Ensure every Date-Sector combination is unique and complete.
if master.duplicated(["Date", "Sector"]).any():
    raise RuntimeError("Duplicate Date-Sector rows found in DL master.")

counts = master.groupby("Date")["Sector"].nunique()
if not (counts == S).all():
    raise RuntimeError("The DL master is not a complete six-sector date panel.")

# Date-level split.
split_by_date = (
    master[["Date", "Split"]]
    .drop_duplicates()
    .set_index("Date")["Split"]
    .reindex(dates)
)
splits = split_by_date.astype(str).to_numpy()

if set(np.unique(splits)) != {"Train", "Validation", "Test"}:
    raise RuntimeError(f"Unexpected split labels: {np.unique(splits)}")

train_date_mask = splits == "Train"
validation_date_mask = splits == "Validation"
test_date_mask = splits == "Test"

log(
    f"Loaded {len(master):,} sector-days; {T:,} dates; {S} sectors; "
    f"graph source={graph_source}."
)

# =============================================================================
# 5. RECONSTRUCT CURRENT CRASH PANEL
# =============================================================================

C = np.zeros((T, S), dtype=np.float32)
M = np.zeros((T, S), dtype=bool)

for row in master[["Date", "Sector", "Crash_Main"]].itertuples(index=False):
    t = date_to_idx[pd.Timestamp(row.Date)]
    s = sector_to_idx[row.Sector]

    if pd.notna(row.Crash_Main):
        C[t, s] = float(row.Crash_Main)
        M[t, s] = True

if C[train_date_mask].sum() <= 0:
    raise RuntimeError("No Train crash events found.")

# =============================================================================
# 6. SPLIT-AWARE SURVIVAL TARGETS — CRITICAL ANTI-LEAKAGE CORRECTION
# =============================================================================
#
# Earlier Phase-1 target construction was censoring-aware but did not explicitly
# stop future-outcome searches at Train/Validation boundaries. For the ML/DL
# experiment we rebuild targets so that:
#
#   * Train origins never use Validation events as future outcomes.
#   * Validation origins never use Test events as future outcomes.
#   * Test origins stop at the dataset end.
#
# A positive event observed before the split boundary remains a valid outcome.
# A no-event origin is right-censored at the split boundary.

daily_hazard_target = np.zeros((T, S, MAX_HORIZON), dtype=np.float32)
daily_hazard_mask = np.zeros((T, S, MAX_HORIZON), dtype=np.float32)

split_event_time = np.full((T, S), np.nan, dtype=np.float32)
split_censor_time = np.full((T, S), np.nan, dtype=np.float32)
split_event_observed = np.full((T, S), np.nan, dtype=np.float32)

horizon_target = {
    h: np.full((T, S), np.nan, dtype=np.float32)
    for h in HORIZONS
}

for t in range(T):
    split_t = splits[t]

    for s in range(S):

        if not M[t, s]:
            continue

        observed_until = 0
        event_k = None

        for k in range(1, MAX_HORIZON + 1):
            future_t = t + k

            if future_t >= T:
                break

            # Hard stop at split boundary.
            if splits[future_t] != split_t:
                break

            # Hard stop at missing sector-day crash state.
            if not M[future_t, s]:
                break

            observed_until = k

            if C[future_t, s] == 1:
                event_k = k
                break

        if event_k is not None:
            split_event_observed[t, s] = 1.0
            split_event_time[t, s] = float(event_k)
            split_censor_time[t, s] = float(event_k)

            # At-risk through the event day.
            daily_hazard_mask[t, s, :event_k] = 1.0
            daily_hazard_target[t, s, event_k - 1] = 1.0

        else:
            split_event_observed[t, s] = 0.0
            split_censor_time[t, s] = float(observed_until)

            if observed_until > 0:
                daily_hazard_mask[t, s, :observed_until] = 1.0

        for h in HORIZONS:
            if event_k is not None and event_k <= h:
                horizon_target[h][t, s] = 1.0
            elif observed_until >= h:
                horizon_target[h][t, s] = 0.0
            else:
                horizon_target[h][t, s] = np.nan

# Save corrected target audit.
target_audit_rows = []

for split_name, split_mask in [
    ("Train", train_date_mask),
    ("Validation", validation_date_mask),
    ("Test", test_date_mask),
]:
    for h in HORIZONS:
        y = horizon_target[h][split_mask].reshape(-1)
        ok = np.isfinite(y)

        target_audit_rows.append({
            "Split": split_name,
            "Sector": "POOLED",
            "Horizon": h,
            "ValidTargets": int(ok.sum()),
            "Events": int(np.nansum(y)),
            "EventRate": float(np.nanmean(y)) if ok.any() else np.nan,
        })

        for s, sector in enumerate(sectors):
            ys = horizon_target[h][split_mask, s]
            oks = np.isfinite(ys)

            target_audit_rows.append({
                "Split": split_name,
                "Sector": sector,
                "Horizon": h,
                "ValidTargets": int(oks.sum()),
                "Events": int(np.nansum(ys)),
                "EventRate": float(np.nanmean(ys)) if oks.any() else np.nan,
            })

target_audit = pd.DataFrame(target_audit_rows)
target_audit.to_csv(
    TABLE_DIR / "Table_D01_Split_Aware_Target_Audit.csv",
    index=False
)

log("Split-aware censoring targets reconstructed successfully.")

# =============================================================================
# 7. FEATURE GOVERNANCE — EXPLICITLY REMOVE FUTURE / TARGET VARIABLES
# =============================================================================

# Predictors intentionally excluded because they are outcomes, future-event
# summaries, benchmark predictions, IDs/text, or direct target thresholds.
EXCLUDE_COLUMNS = {
    "Date",
    "Sector",
    "Split",
    "PrimarySector",
    "ExclusionReason",
    "EconometricGraphSource",

    # Survival / future labels.
    "EventObservedWithin22",
    "EventTime",
    "CensorTime",
    "CrashWithin_1",
    "CrashWithin_5",
    "CrashWithin_10",
    "CrashWithin_22",

    # Baseline/model predictions.
    "FixedHistoricalProb_1",
    "FixedHistoricalProb_5",
    "FixedHistoricalProb_10",
    "FixedHistoricalProb_22",
    "ExpandingHistoricalProb_1",
    "ExpandingHistoricalProb_5",
    "ExpandingHistoricalProb_10",
    "ExpandingHistoricalProb_22",
    "StableHawkesProb_1",
    "StableHawkesProb_5",
    "StableHawkesProb_10",
    "StableHawkesProb_22",

    # Redundant current-label copy.
    "CurrentCrash",

    # Direct crash thresholds / alternative crash outcomes are excluded to keep
    # the feature set economically interpretable and avoid near-target proxies.
    "CrashThreshold_001",
    "CrashThreshold_0025",
    "CrashThreshold_005",
    "Crash_001",
    "Crash_0025",
    "Crash_005",

    # Optimizer diagnostics are not economic predictors.
    "EVT_fit_method",
    "EVT_fit_convergence",
    "EVT_loglik",
    "EVT_nll_improvement",
    "EVT_at_boundary",
    "EVT_refit_id",
}

# Crash_Main at forecast origin t IS permitted: a crash observed today is valid
# information when forecasting t+1 onward and is central to excitation dynamics.
candidate_features = []

for column in master.columns:
    if column in EXCLUDE_COLUMNS:
        continue

    if pd.api.types.is_numeric_dtype(master[column]):
        candidate_features.append(column)

# Explicitly retain current crash state and observation indicator.
for mandatory in ["Crash_Main", "CurrentCrashObserved"]:
    if mandatory in master.columns and mandatory not in candidate_features:
        candidate_features.append(mandatory)

# Remove accidental future-like names defensively.
forbidden_name_fragments = [
    "CrashWithin_",
    "EventTime",
    "CensorTime",
    "Prob_",
]

feature_columns = [
    c for c in candidate_features
    if not any(fragment in c for fragment in forbidden_name_fragments)
]

if "Crash_Main" not in feature_columns:
    raise RuntimeError("Crash_Main should be available as current-state input.")

log(f"Selected {len(feature_columns)} leakage-screened numeric predictors.")

pd.DataFrame({
    "Feature": feature_columns
}).to_csv(
    TABLE_DIR / "Table_D02_Model_Features.csv",
    index=False
)

# =============================================================================
# 8. PANELIZE, EVENT-SAFE IMPUTE, TRAIN-ONLY STANDARDIZE
# =============================================================================

# Reindex each sector to the graph date order.
feature_panel_raw = np.full(
    (T, S, len(feature_columns)),
    np.nan,
    dtype=np.float64,
)

for s, sector in enumerate(sectors):
    sector_df = (
        master[master["Sector"] == sector]
        .set_index("Date")
        .reindex(dates)
    )

    feature_panel_raw[:, s, :] = (
        sector_df[feature_columns]
        .astype(float)
        .to_numpy()
    )

# -------------------------------------------------------------------------
# CORRECTION 2.1A — Crash_Main is an EVENT indicator and must never be
# forward-filled. Missing crash states are represented as Crash_Main = 0
# together with CurrentCrashObserved = 0. Thus the model can distinguish
# "observed non-crash" from "crash state unavailable" without propagating a
# previous crash across later ineligible sector-days.
# -------------------------------------------------------------------------

feature_panel_preimpute = feature_panel_raw.copy()

crash_feature_index = feature_columns.index("Crash_Main")
feature_panel_preimpute[:, :, crash_feature_index] = C.astype(np.float64)

if "CurrentCrashObserved" in feature_columns:
    crash_observed_feature_index = feature_columns.index(
        "CurrentCrashObserved"
    )
    feature_panel_preimpute[
        :, :, crash_observed_feature_index
    ] = M.astype(np.float64)
else:
    crash_observed_feature_index = None

# Past-only forward fill for CONTINUOUS / STATE predictors.
# Crash_Main and CurrentCrashObserved are already complete after the explicit
# event-safe assignment above, so they cannot be propagated by ffill.
feature_panel_ffill = feature_panel_preimpute.copy()

for s in range(S):
    frame = pd.DataFrame(
        feature_panel_ffill[:, s, :],
        columns=feature_columns,
    )
    feature_panel_ffill[:, s, :] = frame.ffill().to_numpy()

# Train-only medians for remaining gaps.
train_values = feature_panel_ffill[train_date_mask]
train_median = np.nanmedian(
    train_values,
    axis=(0, 1),
)

# If a feature is entirely missing in Train, drop it.
valid_feature_mask = np.isfinite(train_median)

if not valid_feature_mask.all():
    dropped = [
        feature_columns[i]
        for i in np.where(~valid_feature_mask)[0]
    ]
    log(
        f"Dropping all-missing Train features: {dropped}"
    )

    feature_columns = [
        feature_columns[i]
        for i in np.where(valid_feature_mask)[0]
    ]
    feature_panel_ffill = (
        feature_panel_ffill[:, :, valid_feature_mask]
    )
    train_median = train_median[valid_feature_mask]

# Missingness indicators use the ORIGINAL R/Python handoff missingness pattern.
# Crash_Main itself does not receive a generic MISS__ indicator because
# CurrentCrashObserved already carries precisely that information.
raw_after_feature_filter = (
    feature_panel_raw[:, :, valid_feature_mask]
)

raw_train = raw_after_feature_filter[
    train_date_mask
]

missing_rate = np.mean(
    ~np.isfinite(raw_train),
    axis=(0, 1),
)

missing_indicator_candidate = (
    missing_rate >= 0.02
)

for special_name in [
    "Crash_Main",
    "CurrentCrashObserved",
]:
    if special_name in feature_columns:
        missing_indicator_candidate[
            feature_columns.index(special_name)
        ] = False

missing_indicator_indices = np.where(
    missing_indicator_candidate
)[0]

missing_indicators = (
    ~np.isfinite(
        raw_after_feature_filter[
            :, :, missing_indicator_indices
        ]
    )
).astype(np.float64)

missing_indicator_names = [
    f"MISS__{feature_columns[i]}"
    for i in missing_indicator_indices
]

# Fill remaining missing values with Train medians.
X_numeric = feature_panel_ffill.copy()

for j in range(X_numeric.shape[2]):
    bad = ~np.isfinite(
        X_numeric[:, :, j]
    )
    X_numeric[:, :, j][bad] = train_median[j]

if len(missing_indicator_indices):
    X_numeric = np.concatenate(
        [
            X_numeric,
            missing_indicators,
        ],
        axis=2,
    )
    model_feature_names = (
        feature_columns
        + missing_indicator_names
    )
else:
    model_feature_names = (
        feature_columns.copy()
    )

# Train-only standardization.
train_block = X_numeric[train_date_mask]

train_mean = train_block.mean(
    axis=(0, 1)
)
train_std = train_block.std(
    axis=(0, 1)
)
train_std[
    train_std < 1e-8
] = 1.0

X_panel = (
    (
        X_numeric
        - train_mean[
            None, None, :
        ]
    )
    / train_std[
        None, None, :
    ]
).astype(np.float32)

F_DIM = X_panel.shape[2]

# Audit the special event imputation before scaling.
crash_after_special = feature_panel_preimpute[
    :, :, crash_feature_index
]
original_crash_missing = ~np.isfinite(
    feature_panel_raw[
        :, :, crash_feature_index
    ]
)

crash_missing_set_to_zero = bool(
    np.all(
        crash_after_special[
            original_crash_missing
        ] == 0
    )
)

if not crash_missing_set_to_zero:
    raise RuntimeError(
        "Crash_Main event-safe imputation failed."
    )

preprocess_bundle = {
    "features": model_feature_names,
    "base_features": feature_columns,
    "missing_indicator_features": missing_indicator_names,
    "train_median": train_median,
    "train_mean": train_mean,
    "train_std": train_std,
    "lookback": LOOKBACK,
    "sectors": sectors,
    "CrashMainImputation": (
        "Missing Crash_Main -> 0; "
        "CurrentCrashObserved -> 0; "
        "Crash_Main is never forward-filled"
    ),
}

with open(
    MODEL_DIR
    / "feature_preprocessing_bundle.pkl",
    "wb",
) as handle:
    pickle.dump(
        preprocess_bundle,
        handle,
    )

pd.DataFrame({
    "Feature": model_feature_names,
    "TrainMeanBeforeScaling": train_mean,
    "TrainStdBeforeScaling": train_std,
}).to_csv(
    TABLE_DIR
    / "Table_D03_Feature_Scaling.csv",
    index=False,
)

event_imputation_audit = pd.DataFrame({
    "Item": [
        "Original missing Crash_Main cells",
        "Missing Crash_Main cells set to zero",
        "Crash_Main forward-filled",
        "CurrentCrashObserved used",
    ],
    "Value": [
        int(original_crash_missing.sum()),
        int(
            (
                crash_after_special[
                    original_crash_missing
                ] == 0
            ).sum()
        ),
        "NO",
        "YES",
    ],
})

event_imputation_audit.to_csv(
    TABLE_DIR
    / "Table_D03A_Crash_Event_Imputation_Audit.csv",
    index=False,
)

log(
    f"Final neural feature dimension={F_DIM}; "
    f"missingness indicators="
    f"{len(missing_indicator_names)}."
)
log(
    "Crash_Main event-safe imputation applied: "
    "missing events set to zero and never forward-filled."
)

# =============================================================================
# 9. MAGNITUDE-PRESERVING GRAPH SCALING / ABLATION PRIORS
# =============================================================================
#
# CORRECTION 2.1B
# ----------------
# Phase 2 row-normalized each receiver row at each date. For a sparse Hawkes
# network, that operation can turn the only active incoming edge into weight 1
# irrespective of whether its raw excitation contribution is tiny or large.
#
# Here we preserve ABSOLUTE dynamic excitation magnitude. A single Train-only
# global scale is estimated from positive dynamic Hawkes weights:
#
#     q = Q_0.99(A_train | A_train > 0)
#
#     scaled(A) = log(1 + A/q) / log(2)
#
# Therefore q maps to 1, weaker states remain weak, stronger states remain
# stronger, and the exact same transformation is used for Train, Validation,
# Test, static Hawkes and random-placebo priors. No Test information enters q.

def clean_nonnegative_graph(
    A: np.ndarray,
) -> np.ndarray:
    A = np.asarray(
        A,
        dtype=np.float32,
    )

    return np.where(
        np.isfinite(A)
        & (A > 0),
        A,
        0.0,
    ).astype(np.float32)

raw_dynamic_train = clean_nonnegative_graph(
    dynamic_graph_train_fit[
        train_date_mask
    ]
)

positive_train_dynamic = raw_dynamic_train[
    raw_dynamic_train > 0
]

if len(positive_train_dynamic) == 0:
    raise RuntimeError(
        "No positive Train Hawkes graph weights available "
        "for magnitude-preserving scaling."
    )

GRAPH_TRAIN_Q = float(
    np.quantile(
        positive_train_dynamic,
        GRAPH_SCALE_QUANTILE,
    )
)

if not np.isfinite(GRAPH_TRAIN_Q) or GRAPH_TRAIN_Q <= 0:
    raise RuntimeError(
        "Invalid Train-only Hawkes graph scale reference."
    )

def scale_graph_magnitude(
    A: np.ndarray,
    reference: float = GRAPH_TRAIN_Q,
) -> np.ndarray:
    """
    Train-reference global log scaling. There is NO row normalization.
    """
    A = clean_nonnegative_graph(A)

    scaled = (
        np.log1p(
            A / max(reference, EPS)
        )
        / np.log(2.0)
    )

    scaled = np.clip(
        scaled,
        0.0,
        GRAPH_SCALE_CLIP,
    )

    return scaled.astype(np.float32)

# Dynamic split-aware Hawkes prior. The Phase-1 array already uses the
# Train-fitted process for Train/Validation and the frozen-structure
# Train+Validation coefficient refit for Test.
dynamic_graph_scaled = scale_graph_magnitude(
    dynamic_graph_all
)

# Static split-aware Hawkes graph, transformed with exactly the SAME q99
# reference estimated from Train dynamic excitation contributions.
static_graph_by_date = np.zeros(
    (T, S, S),
    dtype=np.float32,
)

static_train = scale_graph_magnitude(
    hawkes_train_alpha
)
static_final = scale_graph_magnitude(
    hawkes_final_alpha
)

static_graph_by_date[
    train_date_mask
] = static_train
static_graph_by_date[
    validation_date_mask
] = static_train
static_graph_by_date[
    test_date_mask
] = static_final

# No econometric graph prior.
zero_graph_by_date = np.zeros(
    (T, S, S),
    dtype=np.float32,
)

# Random static placebo:
# preserve the number of directed CROSS-sector edges in the stable Hawkes
# support and use the mean raw positive Hawkes alpha as the placebo edge
# magnitude before applying the SAME Train-derived graph transformation.
rng_graph = np.random.default_rng(
    SEED
)

cross_support = (
    (hawkes_train_alpha > 0)
    & (~np.eye(S, dtype=bool))
)

n_cross_edges = int(
    cross_support.sum()
)

all_cross_positions = [
    (i, j)
    for i in range(S)
    for j in range(S)
    if i != j
]

chosen_positions = rng_graph.choice(
    len(all_cross_positions),
    size=max(1, n_cross_edges),
    replace=False,
)

random_static_raw = np.zeros(
    (S, S),
    dtype=np.float32,
)

positive_cross_values = (
    hawkes_train_alpha[
        cross_support
    ]
)

random_raw_weight = (
    float(
        np.mean(
            positive_cross_values
        )
    )
    if len(positive_cross_values)
    else GRAPH_TRAIN_Q
)

for idx in np.atleast_1d(
    chosen_positions
):
    receiver, source = (
        all_cross_positions[
            int(idx)
        ]
    )
    random_static_raw[
        receiver,
        source,
    ] = random_raw_weight

random_static = scale_graph_magnitude(
    random_static_raw
)

random_graph_by_date = np.repeat(
    random_static[
        None, :, :
    ],
    T,
    axis=0,
)

# -------------------------------------------------------------------------
# Explicit node-level contagion-state summaries.
#
# Matrix convention is A[receiver, source].
#
# IncomingRisk_i   = sum_{j != i} A[i,j]
# OutgoingRisk_i   = sum_{r != i} A[r,i]
# MaxIncomingRisk_i= max_{j != i} A[i,j]
#
# These are derived INSIDE the graph model from whichever prior that ablation
# receives. NoGraph therefore receives exact zeros; random/static receive their
# corresponding controls; dynamic receives time-varying Hawkes state.
# -------------------------------------------------------------------------

GRAPH_STATE_FEATURE_NAMES = [
    "HawkesIncomingRisk",
    "HawkesOutgoingRisk",
    "HawkesMaxIncomingRisk",
]

def graph_state_numpy(
    graph_array: np.ndarray,
) -> np.ndarray:
    graph_array = np.asarray(
        graph_array,
        dtype=np.float32,
    )

    cross = graph_array.copy()

    diag = np.arange(S)
    cross[..., diag, diag] = 0.0

    incoming = cross.sum(
        axis=-1
    )
    outgoing = cross.sum(
        axis=-2
    )
    max_incoming = cross.max(
        axis=-1
    )

    return np.stack(
        [
            incoming,
            outgoing,
            max_incoming,
        ],
        axis=-1,
    ).astype(np.float32)

dynamic_graph_state = graph_state_numpy(
    dynamic_graph_scaled
)
static_graph_state = graph_state_numpy(
    static_graph_by_date
)
random_graph_state = graph_state_numpy(
    random_graph_by_date
)
zero_graph_state = graph_state_numpy(
    zero_graph_by_date
)

# Graph scaling audit.
graph_scaling_audit = pd.DataFrame({
    "Item": [
        "Scaling method",
        "Train-only graph quantile",
        "Train-only q reference",
        "Global clip",
        "Row normalization used",
        "Positive Train raw minimum",
        "Positive Train raw median",
        "Positive Train raw q90",
        "Positive Train raw q99",
        "Positive Train raw maximum",
        "Scaled Train positive median",
        "Scaled Train positive q90",
        "Scaled Train positive q99",
        "Scaled Train positive maximum",
    ],
    "Value": [
        "log1p(A/q_train)/log(2)",
        GRAPH_SCALE_QUANTILE,
        GRAPH_TRAIN_Q,
        GRAPH_SCALE_CLIP,
        "NO",
        float(np.min(positive_train_dynamic)),
        float(np.median(positive_train_dynamic)),
        float(np.quantile(positive_train_dynamic, 0.90)),
        float(np.quantile(positive_train_dynamic, 0.99)),
        float(np.max(positive_train_dynamic)),
        float(np.median(
            dynamic_graph_scaled[
                train_date_mask
            ][
                dynamic_graph_scaled[
                    train_date_mask
                ] > 0
            ]
        )),
        float(np.quantile(
            dynamic_graph_scaled[
                train_date_mask
            ][
                dynamic_graph_scaled[
                    train_date_mask
                ] > 0
            ],
            0.90,
        )),
        float(np.quantile(
            dynamic_graph_scaled[
                train_date_mask
            ][
                dynamic_graph_scaled[
                    train_date_mask
                ] > 0
            ],
            0.99,
        )),
        float(np.max(
            dynamic_graph_scaled[
                train_date_mask
            ]
        )),
    ],
})

graph_scaling_audit.to_csv(
    TABLE_DIR
    / "Table_D04A_Graph_Magnitude_Scaling_Audit.csv",
    index=False,
)

# Per-edge magnitude preservation diagnostics for stable cross-sector edges.
edge_scaling_rows = []

for receiver in range(S):
    for source in range(S):

        if receiver == source:
            continue

        raw_series = (
            dynamic_graph_train_fit[
                train_date_mask,
                receiver,
                source,
            ].astype(float)
        )

        scaled_series = (
            dynamic_graph_scaled[
                train_date_mask,
                receiver,
                source,
            ].astype(float)
        )

        if np.any(raw_series > 0):

            if (
                np.std(raw_series) > 0
                and np.std(scaled_series) > 0
            ):
                correlation = float(
                    np.corrcoef(
                        raw_series,
                        scaled_series,
                    )[0, 1]
                )
            else:
                correlation = np.nan

            edge_scaling_rows.append({
                "FromSector": sectors[source],
                "ToSector": sectors[receiver],
                "RawMinimum": float(np.min(raw_series)),
                "RawMedian": float(np.median(raw_series)),
                "RawMaximum": float(np.max(raw_series)),
                "ScaledMinimum": float(np.min(scaled_series)),
                "ScaledMedian": float(np.median(scaled_series)),
                "ScaledMaximum": float(np.max(scaled_series)),
                "RawScaledPearsonCorrelation": correlation,
                "UniqueScaledValues": int(
                    len(
                        np.unique(
                            np.round(
                                scaled_series,
                                8,
                            )
                        )
                    )
                ),
            })

edge_scaling_diagnostics = pd.DataFrame(
    edge_scaling_rows
)

edge_scaling_diagnostics.to_csv(
    TABLE_DIR
    / "Table_D04B_Edge_Magnitude_Preservation.csv",
    index=False,
)

graph_state_summary = pd.DataFrame({
    "Feature": GRAPH_STATE_FEATURE_NAMES,
    "TrainMean_Dynamic": [
        float(
            dynamic_graph_state[
                train_date_mask,
                :,
                k,
            ].mean()
        )
        for k in range(3)
    ],
    "TrainSD_Dynamic": [
        float(
            dynamic_graph_state[
                train_date_mask,
                :,
                k,
            ].std()
        )
        for k in range(3)
    ],
    "TrainMax_Dynamic": [
        float(
            dynamic_graph_state[
                train_date_mask,
                :,
                k,
            ].max()
        )
        for k in range(3)
    ],
})

graph_state_summary.to_csv(
    TABLE_DIR
    / "Table_D04C_Graph_State_Features.csv",
    index=False,
)

graph_ablation_summary = pd.DataFrame({
    "GraphVariant": [
        "NoGraph",
        "RandomStaticGraph",
        "StaticHawkesGraph",
        "DynamicHawkesGraph",
    ],
    "Description": [
        (
            "Zero econometric prior; graph attention remains data-learned; "
            "graph-state summaries are zero."
        ),
        (
            "Random directed prior with Hawkes cross-edge density; "
            "same Train-derived magnitude scale."
        ),
        (
            "Time-invariant stability-selected Hawkes coefficient prior; "
            "same Train-derived magnitude scale."
        ),
        (
            "Time-varying stability-selected Hawkes excitation prior; "
            "absolute excitation magnitude preserved."
        ),
    ],
    "CrossEdgesTrainPrior": [
        0,
        int(
            (
                random_static
                * (~np.eye(S, dtype=bool))
            > 0
            ).sum()
        ),
        int(
            (
                static_train
                * (~np.eye(S, dtype=bool))
            > 0
            ).sum()
        ),
        int(
            (
                np.any(
                    dynamic_graph_scaled[
                        train_date_mask
                    ] > 0,
                    axis=0,
                )
                & (~np.eye(S, dtype=bool))
            ).sum()
        ),
    ],
    "RowNormalized": [
        "NO",
        "NO",
        "NO",
        "NO",
    ],
    "ExplicitNodeGraphState": [
        "ZERO",
        "YES",
        "YES",
        "YES",
    ],
})

graph_ablation_summary.to_csv(
    TABLE_DIR
    / "Table_D04_Graph_Ablation_Design.csv",
    index=False,
)

log(
    "Magnitude-preserving graph scaling applied. "
    f"Train q{GRAPH_SCALE_QUANTILE:.2f}="
    f"{GRAPH_TRAIN_Q:.8f}; row normalization disabled."
)

# =============================================================================
# 10. SAMPLE INDEX CONSTRUCTION
# =============================================================================

# Sector-specific samples for XGBoost/LSTM/Transformer.
sector_sample_rows = []

for t in range(LOOKBACK - 1, T):
    split_name = splits[t]

    for s, sector in enumerate(sectors):

        if not M[t, s]:
            continue

        # Require at least one observed future risk day.
        if daily_hazard_mask[t, s].sum() <= 0:
            continue

        sector_sample_rows.append({
            "t": t,
            "s": s,
            "Date": dates[t],
            "Sector": sector,
            "Split": split_name,
        })

sector_samples = pd.DataFrame(sector_sample_rows)

# Graph samples are date-level and carry all sectors.
graph_sample_rows = []

for t in range(LOOKBACK - 1, T):
    if daily_hazard_mask[t].sum() <= 0:
        continue

    graph_sample_rows.append({
        "t": t,
        "Date": dates[t],
        "Split": splits[t],
    })

graph_samples = pd.DataFrame(graph_sample_rows)

sector_samples.to_csv(
    DATA_DIR / "sector_sample_index.csv.gz",
    index=False,
    compression="gzip",
)

graph_samples.to_csv(
    DATA_DIR / "graph_sample_index.csv.gz",
    index=False,
    compression="gzip",
)

log(
    f"Sector samples={len(sector_samples):,}; "
    f"graph-date samples={len(graph_samples):,}."
)

# =============================================================================
# 11. DATASET CLASSES
# =============================================================================

class SectorSequenceDataset(Dataset):
    def __init__(self, sample_frame: pd.DataFrame):
        self.rows = sample_frame.reset_index(drop=True)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        t = int(row["t"])
        s = int(row["s"])

        x = X_panel[t - LOOKBACK + 1:t + 1, s, :]
        y = daily_hazard_target[t, s, :]
        mask = daily_hazard_mask[t, s, :]

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(s, dtype=torch.long),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32),
            torch.tensor(t, dtype=torch.long),
        )

class GraphSequenceDataset(Dataset):
    def __init__(
        self,
        sample_frame: pd.DataFrame,
        graph_by_date: np.ndarray,
    ):
        self.rows = sample_frame.reset_index(drop=True)
        self.graph_by_date = graph_by_date

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        t = int(row["t"])

        # L x S x F
        x = X_panel[t - LOOKBACK + 1:t + 1, :, :]

        # L x S x S, aligned with each feature date.
        g = self.graph_by_date[
            t - LOOKBACK + 1:t + 1
        ]

        y = daily_hazard_target[t, :, :]
        mask = daily_hazard_mask[t, :, :]

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(g, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32),
            torch.tensor(t, dtype=torch.long),
        )

# =============================================================================
# 12. SURVIVAL LOSS / PROBABILITY UTILITIES
# =============================================================================

def compute_hazard_pos_weight() -> float:
    train_mask = train_date_mask[:, None, None]
    valid = (
        daily_hazard_mask > 0
    ) & train_mask

    y = daily_hazard_target[valid]

    positives = float(y.sum())
    negatives = float(len(y) - positives)

    if positives <= 0:
        return 1.0

    raw = math.sqrt(max(negatives / positives, 1.0))
    return float(np.clip(raw, 1.0, NEURAL_MAX_POS_WEIGHT))

HAZARD_POS_WEIGHT = compute_hazard_pos_weight()
log(f"Neural hazard positive-class weight={HAZARD_POS_WEIGHT:.4f}")

def masked_survival_bce(
    logits: torch.Tensor,
    targets: torch.Tensor,
    mask: torch.Tensor,
    pos_weight: float,
) -> torch.Tensor:
    base = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none",
    )

    weight = torch.where(
        targets > 0.5,
        torch.tensor(
            pos_weight,
            dtype=base.dtype,
            device=base.device,
        ),
        torch.tensor(
            1.0,
            dtype=base.dtype,
            device=base.device,
        ),
    )

    weighted = base * weight * mask
    denom = torch.clamp(mask.sum(), min=1.0)

    return weighted.sum() / denom

def masked_unweighted_survival_nll(
    logits: torch.Tensor,
    targets: torch.Tensor,
    mask: torch.Tensor,
) -> torch.Tensor:
    base = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none",
    )

    return (base * mask).sum() / torch.clamp(mask.sum(), min=1.0)

def hazard_logits_to_horizon_probs(
    logits: np.ndarray,
    temperature: float = 1.0,
    bias: float = 0.0,
) -> dict[int, np.ndarray]:
    calibrated_logits = logits / max(temperature, 1e-4) + bias
    hazard = 1.0 / (1.0 + np.exp(-np.clip(calibrated_logits, -30, 30)))

    survival = np.cumprod(1.0 - hazard, axis=-1)
    cumulative_event = 1.0 - survival

    return {
        h: cumulative_event[..., h - 1]
        for h in HORIZONS
    }

# =============================================================================
# 13. NEURAL MODEL DEFINITIONS
# =============================================================================

class LSTMSurvival(nn.Module):
    def __init__(
        self,
        feature_dim: int,
        n_sectors: int,
        hidden_dim: int = D_MODEL,
    ):
        super().__init__()

        self.sector_embedding = nn.Embedding(
            n_sectors,
            SECTOR_EMBED_DIM,
        )

        self.lstm = nn.LSTM(
            input_size=feature_dim + SECTOR_EMBED_DIM,
            hidden_size=hidden_dim,
            num_layers=2,
            dropout=DROPOUT,
            batch_first=True,
        )

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden_dim, MAX_HORIZON),
        )

    def forward(self, x, sector_idx):
        embedding = self.sector_embedding(sector_idx)
        embedding_seq = embedding[:, None, :].expand(
            -1,
            x.size(1),
            -1,
        )

        z = torch.cat([x, embedding_seq], dim=-1)
        output, _ = self.lstm(z)

        return self.head(output[:, -1, :])


class TemporalTransformerSurvival(nn.Module):
    def __init__(
        self,
        feature_dim: int,
        n_sectors: int,
        d_model: int = D_MODEL,
    ):
        super().__init__()

        self.input_projection = nn.Linear(
            feature_dim,
            d_model,
        )

        self.sector_embedding = nn.Embedding(
            n_sectors,
            d_model,
        )

        self.position_embedding = nn.Parameter(
            torch.zeros(1, LOOKBACK, d_model)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=N_HEADS,
            dim_feedforward=4 * d_model,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=N_TRANSFORMER_LAYERS,
        )

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(d_model, MAX_HORIZON),
        )

    def forward(self, x, sector_idx):
        z = self.input_projection(x)

        sector_emb = self.sector_embedding(
            sector_idx
        )[:, None, :]

        z = (
            z
            + sector_emb
            + self.position_embedding[:, :x.size(1), :]
        )

        z = self.encoder(z)

        return self.head(z[:, -1, :])


class SoftGraphAttention(nn.Module):
    """
    Multi-head node attention with an additive econometric graph bias.

    The prior is SOFT: non-edge pairs remain available to learned attention.
    A learnable positive eta controls how strongly the econometric graph
    influences attention scores.
    """
    def __init__(
        self,
        d_model: int,
        n_heads: int,
        dropout: float,
    ):
        super().__init__()

        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads.")

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

        # softplus(raw_eta) ensures eta >= 0.
        self.raw_eta = nn.Parameter(
            torch.tensor(0.0)
        )

    def forward(self, h, graph_prior):
        # h: B x L x S x D
        # graph_prior: B x L x S(receiver) x S(source)

        B, L, S_, D = h.shape

        q = self.q_proj(h).view(
            B, L, S_, self.n_heads, self.head_dim
        ).permute(0, 1, 3, 2, 4)

        k = self.k_proj(h).view(
            B, L, S_, self.n_heads, self.head_dim
        ).permute(0, 1, 3, 2, 4)

        v = self.v_proj(h).view(
            B, L, S_, self.n_heads, self.head_dim
        ).permute(0, 1, 3, 2, 4)

        # receiver i queries source j.
        scores = torch.einsum(
            "blhid,blhjd->blhij",
            q,
            k,
        ) / math.sqrt(self.head_dim)

        eta = F.softplus(self.raw_eta)

        graph_bias = graph_prior[:, :, None, :, :]
        scores = scores + eta * graph_bias

        attention = torch.softmax(scores, dim=-1)
        attention = self.dropout(attention)

        message = torch.einsum(
            "blhij,blhjd->blhid",
            attention,
            v,
        )

        message = message.permute(
            0, 1, 3, 2, 4
        ).contiguous().view(B, L, S_, D)

        return self.norm(
            h + self.out_proj(message)
        )


class GraphSurvivalTransformer(nn.Module):
    """
    Temporal graph survival Transformer with TWO econometric graph channels:

      1. a soft additive edge prior in cross-sector attention;
      2. explicit node-level contagion-state summaries:
           incoming risk,
           outgoing risk,
           maximum incoming risk.

    The same architecture is used for NoGraph / Random / Static / Dynamic
    ablations. Only graph_prior changes. For NoGraph all three state summaries
    are identically zero.
    """

    def __init__(
        self,
        feature_dim: int,
        n_sectors: int,
        d_model: int = D_MODEL,
    ):
        super().__init__()

        self.input_projection = nn.Linear(
            feature_dim,
            d_model,
        )

        self.node_embedding = nn.Embedding(
            n_sectors,
            d_model,
        )

        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                LOOKBACK,
                1,
                d_model,
            )
        )

        self.graph_attention = SoftGraphAttention(
            d_model=d_model,
            n_heads=N_HEADS,
            dropout=DROPOUT,
        )

        # Bias=False is deliberate: a zero graph prior must inject exactly zero
        # graph-state signal in the NoGraph ablation.
        self.graph_state_projection = nn.Linear(
            3,
            d_model,
            bias=False,
        )

        # Non-negative learnable gate for explicit graph-state summaries.
        self.raw_graph_state_eta = nn.Parameter(
            torch.tensor(0.0)
        )

        temporal_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=N_HEADS,
            dim_feedforward=4 * d_model,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.temporal_encoder = nn.TransformerEncoder(
            temporal_layer,
            num_layers=N_TRANSFORMER_LAYERS,
        )

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(
                d_model,
                MAX_HORIZON,
            ),
        )

    @staticmethod
    def graph_state_features(
        graph_prior: torch.Tensor,
    ) -> torch.Tensor:
        """
        graph_prior:
            B x L x S(receiver) x S(source)

        Returns:
            B x L x S x 3
            [incoming, outgoing, max incoming], excluding diagonal edges.
        """

        S_ = graph_prior.size(-1)

        eye = torch.eye(
            S_,
            dtype=graph_prior.dtype,
            device=graph_prior.device,
        )

        cross = (
            graph_prior
            * (
                1.0
                - eye[
                    None,
                    None,
                    :, :,
                ]
            )
        )

        incoming = cross.sum(
            dim=-1
        )

        outgoing = cross.sum(
            dim=-2
        )

        max_incoming = cross.max(
            dim=-1
        ).values

        return torch.stack(
            [
                incoming,
                outgoing,
                max_incoming,
            ],
            dim=-1,
        )

    def forward(
        self,
        x,
        graph_prior,
    ):
        # x: B x L x S x F
        # graph_prior: B x L x S(receiver) x S(source)

        B, L, S_, _ = x.shape

        z = self.input_projection(x)

        node_ids = torch.arange(
            S_,
            device=x.device,
        )

        node_emb = self.node_embedding(
            node_ids
        )[
            None,
            None,
            :,
            :,
        ]

        z = (
            z
            + node_emb
            + self.position_embedding[
                :, :L, :, :
            ]
        )

        # Explicit node-level econometric contagion state.
        graph_state = self.graph_state_features(
            graph_prior
        )

        graph_state_embedding = (
            self.graph_state_projection(
                graph_state
            )
        )

        graph_state_eta = F.softplus(
            self.raw_graph_state_eta
        )

        z = (
            z
            + graph_state_eta
            * graph_state_embedding
        )

        # Cross-sector attention with magnitude-preserving soft graph bias.
        z = self.graph_attention(
            z,
            graph_prior,
        )

        # Temporal Transformer independently for each sector after graph mixing.
        z = z.permute(
            0,
            2,
            1,
            3,
        ).contiguous().view(
            B * S_,
            L,
            -1,
        )

        z = self.temporal_encoder(
            z
        )

        last = z[:, -1, :]

        logits = self.head(
            last
        ).view(
            B,
            S_,
            MAX_HORIZON,
        )

        return logits


# =============================================================================
# 14. TRAIN / VALIDATION HELPERS
# =============================================================================

@dataclass
class TrainResult:
    best_state: dict
    best_epoch: int
    best_val_nll: float
    history: pd.DataFrame

def make_sector_loader(
    frame: pd.DataFrame,
    shuffle: bool,
) -> DataLoader:
    return DataLoader(
        SectorSequenceDataset(frame),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )

def make_graph_loader(
    frame: pd.DataFrame,
    graph_by_date: np.ndarray,
    shuffle: bool,
) -> DataLoader:
    return DataLoader(
        GraphSequenceDataset(frame, graph_by_date),
        batch_size=GRAPH_BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )

def train_sector_neural_model(
    model: nn.Module,
    model_name: str,
) -> TrainResult:

    train_frame = sector_samples[
        sector_samples["Split"] == "Train"
    ]
    val_frame = sector_samples[
        sector_samples["Split"] == "Validation"
    ]

    train_loader = make_sector_loader(
        train_frame,
        shuffle=True,
    )
    val_loader = make_sector_loader(
        val_frame,
        shuffle=False,
    )

    model = model.to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_state = None
    best_epoch = 0
    best_val = np.inf
    patience_counter = 0
    history_rows = []

    for epoch in range(1, MAX_EPOCHS + 1):

        model.train()
        train_loss_num = 0.0
        train_mask_num = 0.0

        for x, sector_idx, y, mask, _ in train_loader:

            x = x.to(DEVICE)
            sector_idx = sector_idx.to(DEVICE)
            y = y.to(DEVICE)
            mask = mask.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            logits = model(x, sector_idx)

            loss = masked_survival_bce(
                logits,
                y,
                mask,
                HAZARD_POS_WEIGHT,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

            nmask = float(mask.sum().item())
            train_loss_num += float(loss.item()) * nmask
            train_mask_num += nmask

        model.eval()
        val_loss_num = 0.0
        val_mask_num = 0.0

        with torch.no_grad():
            for x, sector_idx, y, mask, _ in val_loader:

                x = x.to(DEVICE)
                sector_idx = sector_idx.to(DEVICE)
                y = y.to(DEVICE)
                mask = mask.to(DEVICE)

                logits = model(x, sector_idx)

                val_loss = masked_unweighted_survival_nll(
                    logits,
                    y,
                    mask,
                )

                nmask = float(mask.sum().item())
                val_loss_num += float(val_loss.item()) * nmask
                val_mask_num += nmask

        train_loss_epoch = (
            train_loss_num / max(train_mask_num, 1.0)
        )
        val_nll_epoch = (
            val_loss_num / max(val_mask_num, 1.0)
        )

        history_rows.append({
            "Model": model_name,
            "Epoch": epoch,
            "TrainWeightedLoss": train_loss_epoch,
            "ValidationNLL": val_nll_epoch,
        })

        log(
            f"{model_name}: epoch={epoch:02d}, "
            f"train={train_loss_epoch:.5f}, "
            f"valNLL={val_nll_epoch:.5f}"
        )

        if val_nll_epoch < best_val - 1e-5:
            best_val = val_nll_epoch
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            break

    if best_state is None:
        raise RuntimeError(
            f"No best state captured for {model_name}."
        )

    model.load_state_dict(best_state)

    return TrainResult(
        best_state=best_state,
        best_epoch=best_epoch,
        best_val_nll=best_val,
        history=pd.DataFrame(history_rows),
    )

def train_graph_neural_model(
    model: nn.Module,
    model_name: str,
    graph_by_date: np.ndarray,
) -> TrainResult:

    train_frame = graph_samples[
        graph_samples["Split"] == "Train"
    ]
    val_frame = graph_samples[
        graph_samples["Split"] == "Validation"
    ]

    train_loader = make_graph_loader(
        train_frame,
        graph_by_date,
        shuffle=True,
    )
    val_loader = make_graph_loader(
        val_frame,
        graph_by_date,
        shuffle=False,
    )

    model = model.to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_state = None
    best_epoch = 0
    best_val = np.inf
    patience_counter = 0
    history_rows = []

    for epoch in range(1, MAX_EPOCHS + 1):

        model.train()
        train_loss_num = 0.0
        train_mask_num = 0.0

        for x, graph_prior, y, mask, _ in train_loader:

            x = x.to(DEVICE)
            graph_prior = graph_prior.to(DEVICE)
            y = y.to(DEVICE)
            mask = mask.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            logits = model(
                x,
                graph_prior,
            )

            loss = masked_survival_bce(
                logits,
                y,
                mask,
                HAZARD_POS_WEIGHT,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

            nmask = float(mask.sum().item())
            train_loss_num += float(loss.item()) * nmask
            train_mask_num += nmask

        model.eval()
        val_loss_num = 0.0
        val_mask_num = 0.0

        with torch.no_grad():
            for x, graph_prior, y, mask, _ in val_loader:

                x = x.to(DEVICE)
                graph_prior = graph_prior.to(DEVICE)
                y = y.to(DEVICE)
                mask = mask.to(DEVICE)

                logits = model(
                    x,
                    graph_prior,
                )

                val_loss = masked_unweighted_survival_nll(
                    logits,
                    y,
                    mask,
                )

                nmask = float(mask.sum().item())
                val_loss_num += float(val_loss.item()) * nmask
                val_mask_num += nmask

        train_loss_epoch = (
            train_loss_num / max(train_mask_num, 1.0)
        )
        val_nll_epoch = (
            val_loss_num / max(val_mask_num, 1.0)
        )

        history_rows.append({
            "Model": model_name,
            "Epoch": epoch,
            "TrainWeightedLoss": train_loss_epoch,
            "ValidationNLL": val_nll_epoch,
        })

        log(
            f"{model_name}: epoch={epoch:02d}, "
            f"train={train_loss_epoch:.5f}, "
            f"valNLL={val_nll_epoch:.5f}"
        )

        if val_nll_epoch < best_val - 1e-5:
            best_val = val_nll_epoch
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            break

    if best_state is None:
        raise RuntimeError(
            f"No best state captured for {model_name}."
        )

    model.load_state_dict(best_state)

    return TrainResult(
        best_state=best_state,
        best_epoch=best_epoch,
        best_val_nll=best_val,
        history=pd.DataFrame(history_rows),
    )

# =============================================================================
# 15. NEURAL PREDICTION / CALIBRATION
# =============================================================================

def predict_sector_logits(
    model: nn.Module,
    split_name: str,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:

    frame = sector_samples[
        sector_samples["Split"] == split_name
    ].copy()

    loader = make_sector_loader(
        frame,
        shuffle=False,
    )

    model = model.to(DEVICE)
    model.eval()

    logits_list = []
    y_list = []
    mask_list = []
    t_list = []
    s_list = []

    cursor = 0

    with torch.no_grad():
        for x, sector_idx, y, mask, t in loader:

            x = x.to(DEVICE)
            sector_idx_device = sector_idx.to(DEVICE)

            logits = model(
                x,
                sector_idx_device,
            ).cpu().numpy()

            logits_list.append(logits)
            y_list.append(y.numpy())
            mask_list.append(mask.numpy())
            t_list.append(t.numpy())
            s_list.append(sector_idx.numpy())
            cursor += len(t)

    return (
        np.concatenate(logits_list, axis=0),
        np.concatenate(y_list, axis=0),
        np.concatenate(mask_list, axis=0),
        np.column_stack([
            np.concatenate(t_list),
            np.concatenate(s_list),
        ]),
    )

def predict_graph_logits(
    model: nn.Module,
    split_name: str,
    graph_by_date: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:

    frame = graph_samples[
        graph_samples["Split"] == split_name
    ].copy()

    loader = make_graph_loader(
        frame,
        graph_by_date,
        shuffle=False,
    )

    model = model.to(DEVICE)
    model.eval()

    logits_list = []
    y_list = []
    mask_list = []
    t_list = []

    with torch.no_grad():
        for x, graph_prior, y, mask, t in loader:

            x = x.to(DEVICE)
            graph_prior = graph_prior.to(DEVICE)

            logits = model(
                x,
                graph_prior,
            ).cpu().numpy()

            logits_list.append(logits)
            y_list.append(y.numpy())
            mask_list.append(mask.numpy())
            t_list.append(t.numpy())

    return (
        np.concatenate(logits_list, axis=0),
        np.concatenate(y_list, axis=0),
        np.concatenate(mask_list, axis=0),
        np.concatenate(t_list),
    )

def fit_hazard_temperature(
    logits: np.ndarray,
    targets: np.ndarray,
    mask: np.ndarray,
) -> tuple[float, float]:

    valid = mask > 0

    z = logits[valid].astype(float)
    y = targets[valid].astype(float)

    if len(y) == 0:
        return 1.0, 0.0

    def objective(theta):
        log_temperature, bias = theta
        temperature = np.exp(log_temperature)

        zc = z / temperature + bias
        p = 1.0 / (
            1.0 + np.exp(-np.clip(zc, -30, 30))
        )
        p = np.clip(p, EPS, 1.0 - EPS)

        return float(
            -np.mean(
                y * np.log(p)
                + (1.0 - y) * np.log1p(-p)
            )
        )

    result = minimize(
        objective,
        x0=np.array([0.0, 0.0]),
        method="L-BFGS-B",
        bounds=[
            (-3.0, 3.0),
            (-5.0, 5.0),
        ],
    )

    if not result.success:
        log(
            f"WARNING: hazard calibration optimizer: {result.message}"
        )

    temperature = float(
        np.exp(result.x[0])
    )
    bias = float(result.x[1])

    return temperature, bias

# =============================================================================
# 16. XGBOOST FEATURE ENGINEERING
# =============================================================================

# Summary windows reduce the 60-day sequence to robust tabular statistics.
XGB_WINDOWS = [5, 22, 60]

def xgb_feature_vector(t: int, s: int) -> np.ndarray:
    sequence = X_panel[
        t - LOOKBACK + 1:t + 1,
        s,
        :,
    ]

    pieces = [
        sequence[-1],  # current state
    ]

    for window in XGB_WINDOWS:
        block = sequence[-window:]
        pieces.extend([
            block.mean(axis=0),
            block.std(axis=0),
            block.min(axis=0),
            block.max(axis=0),
        ])

    # Sector one-hot.
    one_hot = np.zeros(S, dtype=np.float32)
    one_hot[s] = 1.0
    pieces.append(one_hot)

    return np.concatenate(pieces).astype(np.float32)

log("Building XGBoost summary-feature matrix...")

X_xgb = np.vstack([
    xgb_feature_vector(
        int(row.t),
        int(row.s),
    )
    for row in sector_samples.itertuples(index=False)
])

xgb_split = sector_samples["Split"].to_numpy()
xgb_t = sector_samples["t"].to_numpy(dtype=int)
xgb_s = sector_samples["s"].to_numpy(dtype=int)

# =============================================================================
# 17. XGBOOST TRAINING + HORIZON CALIBRATION
# =============================================================================

def fit_binary_platt(
    p_validation: np.ndarray,
    y_validation: np.ndarray,
) -> tuple[float, float]:

    p_validation = np.clip(
        p_validation,
        EPS,
        1.0 - EPS,
    )

    x = np.log(
        p_validation / (1.0 - p_validation)
    )
    y = y_validation.astype(float)

    def objective(theta):
        intercept, slope = theta
        z = intercept + slope * x
        q = 1.0 / (
            1.0 + np.exp(-np.clip(z, -30, 30))
        )
        q = np.clip(q, EPS, 1.0 - EPS)

        return float(
            -np.mean(
                y * np.log(q)
                + (1.0 - y) * np.log1p(-q)
            )
        )

    result = minimize(
        objective,
        x0=np.array([0.0, 1.0]),
        method="L-BFGS-B",
        bounds=[
            (-10.0, 10.0),
            (0.0, 10.0),
        ],
    )

    return float(result.x[0]), float(result.x[1])

def apply_binary_platt(
    p: np.ndarray,
    intercept: float,
    slope: float,
) -> np.ndarray:

    p = np.clip(p, EPS, 1.0 - EPS)
    x = np.log(p / (1.0 - p))
    z = intercept + slope * x

    return 1.0 / (
        1.0 + np.exp(-np.clip(z, -30, 30))
    )

xgb_models = {}
xgb_calibration = {}
xgb_predictions = {
    "Validation": {},
    "Test": {},
}

for h in HORIZONS:
    log(f"Training XGBoost horizon={h}...")

    # Build split-aware target per sector sample.
    y_all = np.array([
        horizon_target[h][t, s]
        for t, s in zip(xgb_t, xgb_s)
    ], dtype=float)

    train_idx = (
        (xgb_split == "Train")
        & np.isfinite(y_all)
    )
    val_idx = (
        (xgb_split == "Validation")
        & np.isfinite(y_all)
    )
    test_idx = (
        (xgb_split == "Test")
        & np.isfinite(y_all)
    )

    y_train = y_all[train_idx].astype(int)
    y_val = y_all[val_idx].astype(int)

    positive = max(int(y_train.sum()), 1)
    negative = max(len(y_train) - positive, 1)
    scale_pos_weight = min(
        math.sqrt(negative / positive),
        XGB_MAX_POS_WEIGHT,
    )

    dtrain = xgb.DMatrix(
        X_xgb[train_idx],
        label=y_train,
    )
    dval = xgb.DMatrix(
        X_xgb[val_idx],
        label=y_val,
    )

    params = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "eta": 0.03,
        "max_depth": 4,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "lambda": 1.0,
        "alpha": 0.0,
        "scale_pos_weight": scale_pos_weight,
        "seed": SEED,
        "nthread": max(1, os.cpu_count() or 1),
        "tree_method": "hist",
    }

    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=XGB_MAX_ROUNDS,
        evals=[(dval, "validation")],
        early_stopping_rounds=XGB_EARLY_STOP,
        verbose_eval=False,
    )

    best_iteration = int(
        booster.best_iteration
        if booster.best_iteration is not None
        else XGB_MAX_ROUNDS - 1
    )

    p_val_raw = booster.predict(
        dval,
        iteration_range=(0, best_iteration + 1),
    )

    intercept, slope = fit_binary_platt(
        p_val_raw,
        y_val,
    )

    p_val = apply_binary_platt(
        p_val_raw,
        intercept,
        slope,
    )

    dtest = xgb.DMatrix(
        X_xgb[test_idx]
    )

    p_test_raw = booster.predict(
        dtest,
        iteration_range=(0, best_iteration + 1),
    )

    p_test = apply_binary_platt(
        p_test_raw,
        intercept,
        slope,
    )

    xgb_models[h] = booster
    xgb_calibration[h] = {
        "intercept": intercept,
        "slope": slope,
        "best_iteration": best_iteration,
        "scale_pos_weight": scale_pos_weight,
    }

    # Store by sample coordinate for later panel assembly.
    xgb_predictions["Validation"][h] = {
        "t": xgb_t[val_idx],
        "s": xgb_s[val_idx],
        "p": p_val,
    }
    xgb_predictions["Test"][h] = {
        "t": xgb_t[test_idx],
        "s": xgb_s[test_idx],
        "p": p_test,
    }

    booster.save_model(
        MODEL_DIR / f"xgboost_h{h}.json"
    )

with open(
    MODEL_DIR / "xgboost_calibration.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        xgb_calibration,
        handle,
        indent=2,
    )

# Assemble monotone XGBoost probabilities into T x S matrices.
xgb_panel_probs = {
    split_name: {
        h: np.full((T, S), np.nan, dtype=np.float32)
        for h in HORIZONS
    }
    for split_name in ["Validation", "Test"]
}

for split_name in ["Validation", "Test"]:

    # Coordinate union for this split.
    coordinate_set = set()

    for h in HORIZONS:
        record = xgb_predictions[split_name][h]

        for t, s, p in zip(
            record["t"],
            record["s"],
            record["p"],
        ):
            xgb_panel_probs[split_name][h][t, s] = p
            coordinate_set.add((int(t), int(s)))

    # Monotone rearrangement across horizons at common coordinates.
    for t, s in coordinate_set:
        values = np.array([
            xgb_panel_probs[split_name][h][t, s]
            for h in HORIZONS
        ])

        if np.all(np.isfinite(values)):
            values = np.maximum.accumulate(values)

            for h, value in zip(HORIZONS, values):
                xgb_panel_probs[split_name][h][t, s] = value

log("XGBoost benchmark training complete.")

# =============================================================================
# 18. TRAIN LSTM / TEMPORAL TRANSFORMER
# =============================================================================

neural_histories = []
neural_models = {}
neural_calibration = {}

# LSTM
lstm_model = LSTMSurvival(
    feature_dim=F_DIM,
    n_sectors=S,
)

lstm_result = train_sector_neural_model(
    lstm_model,
    "LSTM_Survival",
)

lstm_model.load_state_dict(
    lstm_result.best_state
)

torch.save(
    lstm_result.best_state,
    MODEL_DIR / "LSTM_Survival_best.pt",
)

neural_histories.append(
    lstm_result.history
)

neural_models["LSTM_Survival"] = lstm_model

# Temporal Transformer
transformer_model = TemporalTransformerSurvival(
    feature_dim=F_DIM,
    n_sectors=S,
)

transformer_result = train_sector_neural_model(
    transformer_model,
    "TemporalTransformer_Survival",
)

transformer_model.load_state_dict(
    transformer_result.best_state
)

torch.save(
    transformer_result.best_state,
    MODEL_DIR / "TemporalTransformer_Survival_best.pt",
)

neural_histories.append(
    transformer_result.history
)

neural_models[
    "TemporalTransformer_Survival"
] = transformer_model

# =============================================================================
# 19. TRAIN GRAPH ABLATIONS
# =============================================================================

graph_variants = {
    "NoGraphTransformer": zero_graph_by_date,
    "RandomGraphTransformer": random_graph_by_date,
    "StaticHawkesGraphTransformer": static_graph_by_date,
    "DynamicHawkesGraphTransformer": dynamic_graph_scaled,
}

graph_train_results = {}

for model_name, graph_array in graph_variants.items():

    log(f"Training graph ablation: {model_name}")

    model = GraphSurvivalTransformer(
        feature_dim=F_DIM,
        n_sectors=S,
    )

    result = train_graph_neural_model(
        model,
        model_name,
        graph_array,
    )

    model.load_state_dict(
        result.best_state
    )

    torch.save(
        result.best_state,
        MODEL_DIR / f"{model_name}_best.pt",
    )

    neural_histories.append(
        result.history
    )

    neural_models[model_name] = model
    graph_train_results[model_name] = result

# Save all training histories.
training_history = pd.concat(
    neural_histories,
    ignore_index=True,
)

training_history.to_csv(
    TABLE_DIR / "Table_D05_Neural_Training_History.csv",
    index=False
)

best_epoch_rows = []

for model_name, result in [
    ("LSTM_Survival", lstm_result),
    ("TemporalTransformer_Survival", transformer_result),
    *[
        (name, graph_train_results[name])
        for name in graph_variants
    ],
]:
    best_epoch_rows.append({
        "Model": model_name,
        "BestEpoch": result.best_epoch,
        "BestValidationSurvivalNLL": result.best_val_nll,
    })

best_epochs = pd.DataFrame(best_epoch_rows)

best_epochs.to_csv(
    TABLE_DIR / "Table_D06_Neural_Best_Epochs.csv",
    index=False
)

# =============================================================================
# 20. CALIBRATE NEURAL HAZARDS ON VALIDATION
# =============================================================================

# Final panel probability storage by model/split/horizon.
model_probabilities = {}

def initialize_probability_dict():
    return {
        split_name: {
            h: np.full((T, S), np.nan, dtype=np.float32)
            for h in HORIZONS
        }
        for split_name in ["Validation", "Test"]
    }

# LSTM + temporal transformer.
for model_name in [
    "LSTM_Survival",
    "TemporalTransformer_Survival",
]:

    model = neural_models[model_name]

    val_logits, val_y, val_mask, val_coords = predict_sector_logits(
        model,
        "Validation",
    )

    temperature, bias = fit_hazard_temperature(
        val_logits,
        val_y,
        val_mask,
    )

    neural_calibration[model_name] = {
        "temperature": temperature,
        "bias": bias,
    }

    model_probabilities[model_name] = initialize_probability_dict()

    for split_name in ["Validation", "Test"]:

        logits, y, mask, coords = predict_sector_logits(
            model,
            split_name,
        )

        probs = hazard_logits_to_horizon_probs(
            logits,
            temperature,
            bias,
        )

        for row_idx, (t, s) in enumerate(coords.astype(int)):
            for h in HORIZONS:
                model_probabilities[
                    model_name
                ][split_name][h][t, s] = probs[h][row_idx]

# Graph models.
for model_name, graph_array in graph_variants.items():

    model = neural_models[model_name]

    val_logits, val_y, val_mask, val_t = predict_graph_logits(
        model,
        "Validation",
        graph_array,
    )

    temperature, bias = fit_hazard_temperature(
        val_logits,
        val_y,
        val_mask,
    )

    eta = float(
        F.softplus(
            model.graph_attention.raw_eta.detach().cpu()
        ).item()
    )

    graph_state_eta = float(
        F.softplus(
            model.raw_graph_state_eta.detach().cpu()
        ).item()
    )

    neural_calibration[model_name] = {
        "temperature": temperature,
        "bias": bias,
        "learned_graph_eta": eta,
        "learned_graph_state_eta": graph_state_eta,
    }

    model_probabilities[model_name] = initialize_probability_dict()

    for split_name in ["Validation", "Test"]:

        logits, y, mask, t_values = predict_graph_logits(
            model,
            split_name,
            graph_array,
        )

        probs = hazard_logits_to_horizon_probs(
            logits,
            temperature,
            bias,
        )

        for row_idx, t in enumerate(t_values.astype(int)):
            for s in range(S):
                for h in HORIZONS:
                    model_probabilities[
                        model_name
                    ][split_name][h][t, s] = probs[h][row_idx, s]

with open(
    MODEL_DIR / "neural_hazard_calibration.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        neural_calibration,
        handle,
        indent=2,
    )

# =============================================================================
# 21. ADD XGBOOST / ECONOMETRIC BASELINES TO COMMON FORECAST STORE
# =============================================================================

model_probabilities["XGBoost"] = xgb_panel_probs

# Baselines from Phase-1 master are reused only as PREDICTIONS. Their evaluation
# targets are the corrected split-aware outcomes rebuilt in this script.
def master_probability_panel(column: str) -> np.ndarray:
    out = np.full((T, S), np.nan, dtype=np.float32)

    for row in master[["Date", "Sector", column]].itertuples(index=False):
        t = date_to_idx[pd.Timestamp(row.Date)]
        s = sector_to_idx[row.Sector]

        if pd.notna(getattr(row, column)):
            out[t, s] = float(getattr(row, column))

    return out

for baseline_name, prefix in [
    ("FixedHistorical", "FixedHistoricalProb"),
    ("ExpandingHistorical", "ExpandingHistoricalProb"),
    ("StableHawkes", "StableHawkesProb"),
]:
    model_probabilities[baseline_name] = initialize_probability_dict()

    for h in HORIZONS:
        panel = master_probability_panel(
            f"{prefix}_{h}"
        )

        model_probabilities[baseline_name]["Validation"][h] = panel
        model_probabilities[baseline_name]["Test"][h] = panel

# =============================================================================
# 22. METRICS
# =============================================================================

def log_score(y: np.ndarray, p: np.ndarray) -> float:
    p = np.clip(p, EPS, 1.0 - EPS)

    return float(
        -np.mean(
            y * np.log(p)
            + (1.0 - y) * np.log1p(-p)
        )
    )

def fit_calibration_intercept_slope(
    y: np.ndarray,
    p: np.ndarray,
) -> tuple[float, float]:

    p = np.clip(p, EPS, 1.0 - EPS)
    x = np.log(p / (1.0 - p))

    if len(np.unique(y)) < 2:
        return np.nan, np.nan

    def objective(theta):
        intercept, slope = theta
        z = intercept + slope * x
        q = 1.0 / (
            1.0 + np.exp(-np.clip(z, -30, 30))
        )
        q = np.clip(q, EPS, 1.0 - EPS)

        return float(
            -np.mean(
                y * np.log(q)
                + (1.0 - y) * np.log1p(-q)
            )
        )

    result = minimize(
        objective,
        x0=np.array([0.0, 1.0]),
        method="L-BFGS-B",
        bounds=[
            (-20.0, 20.0),
            (-10.0, 10.0),
        ],
    )

    return float(result.x[0]), float(result.x[1])

def probability_metrics(
    y_true: np.ndarray,
    probability: np.ndarray,
) -> dict:

    y_true = np.asarray(y_true, dtype=float)
    probability = np.asarray(probability, dtype=float)

    ok = np.isfinite(y_true) & np.isfinite(probability)

    y = y_true[ok]
    p = np.clip(
        probability[ok],
        EPS,
        1.0 - EPS,
    )

    result = {
        "N": int(len(y)),
        "Events": int(y.sum()) if len(y) else 0,
        "EventRate": float(y.mean()) if len(y) else np.nan,
        "Brier": np.nan,
        "LogScore": np.nan,
        "PR_AUC": np.nan,
        "ROC_AUC": np.nan,
        "CalibrationIntercept": np.nan,
        "CalibrationSlope": np.nan,
    }

    if len(y) == 0:
        return result

    result["Brier"] = float(
        np.mean((p - y) ** 2)
    )
    result["LogScore"] = log_score(y, p)

    if len(np.unique(y)) == 2:
        result["PR_AUC"] = float(
            average_precision_score(y, p)
        )
        result["ROC_AUC"] = float(
            roc_auc_score(y, p)
        )

        intercept, slope = fit_calibration_intercept_slope(
            y,
            p,
        )

        result["CalibrationIntercept"] = intercept
        result["CalibrationSlope"] = slope

    return result

metric_rows = []

for split_name, split_mask in [
    ("Validation", validation_date_mask),
    ("Test", test_date_mask),
]:

    for model_name, split_dict in model_probabilities.items():

        for h in HORIZONS:
            p_panel = split_dict[split_name][h]
            y_panel = horizon_target[h]

            # Pooled.
            pooled = probability_metrics(
                y_panel[split_mask].reshape(-1),
                p_panel[split_mask].reshape(-1),
            )

            metric_rows.append({
                "Split": split_name,
                "Sector": "POOLED",
                "Horizon": h,
                "Model": model_name,
                **pooled,
            })

            # Sector specific.
            for s, sector in enumerate(sectors):
                sector_metrics = probability_metrics(
                    y_panel[split_mask, s],
                    p_panel[split_mask, s],
                )

                metric_rows.append({
                    "Split": split_name,
                    "Sector": sector,
                    "Horizon": h,
                    "Model": model_name,
                    **sector_metrics,
                })

forecast_metrics = pd.DataFrame(metric_rows)

forecast_metrics.to_csv(
    TABLE_DIR / "Table_D07_All_Forecast_Metrics.csv",
    index=False
)

# Main pooled test table.
main_test = (
    forecast_metrics[
        (forecast_metrics["Split"] == "Test")
        & (forecast_metrics["Sector"] == "POOLED")
    ]
    .sort_values(["Horizon", "Brier"])
    .reset_index(drop=True)
)

main_test.to_csv(
    TABLE_DIR / "Table_D08_Main_Pooled_Test_Results.csv",
    index=False
)

# Proposed-model Brier skill vs each benchmark.
proposed_test = (
    main_test[
        main_test["Model"] == PROPOSED_MODEL
    ][["Horizon", "Brier", "LogScore", "PR_AUC"]]
    .rename(columns={
        "Brier": "ProposedBrier",
        "LogScore": "ProposedLogScore",
        "PR_AUC": "ProposedPR_AUC",
    })
)

skill_table = (
    main_test.merge(
        proposed_test,
        on="Horizon",
        how="left",
    )
)

skill_table["ProposedBrierSkill_vs_Model"] = (
    1.0
    - skill_table["ProposedBrier"]
    / skill_table["Brier"]
)

skill_table["ProposedLogScoreImprovement"] = (
    skill_table["LogScore"]
    - skill_table["ProposedLogScore"]
)

skill_table["ProposedPR_AUCGain"] = (
    skill_table["ProposedPR_AUC"]
    - skill_table["PR_AUC"]
)

skill_table.to_csv(
    TABLE_DIR / "Table_D09_Proposed_Model_Skill.csv",
    index=False
)

# =============================================================================
# 23. COHERENCE CHECKS
# =============================================================================

coherence_rows = []

for model_name, split_dict in model_probabilities.items():

    for split_name in ["Validation", "Test"]:

        stack = np.stack(
            [
                split_dict[split_name][h]
                for h in HORIZONS
            ],
            axis=-1,
        )

        finite_all = np.all(
            np.isfinite(stack),
            axis=-1,
        )

        diffs = np.diff(
            stack,
            axis=-1,
        )

        violations = (
            np.any(
                diffs < -1e-8,
                axis=-1,
            )
            & finite_all
        )

        coherence_rows.append({
            "Model": model_name,
            "Split": split_name,
            "CompleteProbabilityRows": int(finite_all.sum()),
            "CoherenceViolations": int(violations.sum()),
            "ViolationRate": (
                float(violations.sum() / finite_all.sum())
                if finite_all.sum()
                else np.nan
            ),
        })

coherence_table = pd.DataFrame(coherence_rows)

coherence_table.to_csv(
    TABLE_DIR / "Table_D10_Horizon_Coherence.csv",
    index=False
)

neural_names = [
    "LSTM_Survival",
    "TemporalTransformer_Survival",
    "NoGraphTransformer",
    "RandomGraphTransformer",
    "StaticHawkesGraphTransformer",
    "DynamicHawkesGraphTransformer",
]

neural_coherence_bad = coherence_table[
    coherence_table["Model"].isin(neural_names)
]["CoherenceViolations"].sum()

if neural_coherence_bad != 0:
    raise RuntimeError(
        "A survival neural model violated horizon probability coherence."
    )

# =============================================================================
# 24. PAIRED MOVING-BLOCK BOOTSTRAP ON TEST DATES
# =============================================================================

def moving_block_sample_indices(
    n_dates: int,
    block_length: int,
    rng: np.random.Generator,
) -> np.ndarray:

    selected = []

    while len(selected) < n_dates:

        if n_dates <= block_length:
            start = 0
        else:
            start = int(
                rng.integers(
                    0,
                    n_dates - block_length + 1,
                )
            )

        selected.extend(
            range(
                start,
                min(n_dates, start + block_length),
            )
        )

    return np.array(
        selected[:n_dates],
        dtype=int,
    )

test_indices = np.where(test_date_mask)[0]

bootstrap_rows = []

benchmark_models = [
    model_name
    for model_name in model_probabilities
    if model_name != PROPOSED_MODEL
]

log(
    f"Starting paired moving-block bootstrap: reps={BOOTSTRAP_REPS}, "
    f"block={BOOTSTRAP_BLOCK} dates."
)

for h in HORIZONS:

    y = horizon_target[h][test_indices]
    p_proposed = model_probabilities[
        PROPOSED_MODEL
    ]["Test"][h][test_indices]

    for benchmark in benchmark_models:

        p_benchmark = model_probabilities[
            benchmark
        ]["Test"][h][test_indices]

        # Require common valid sector-date cells.
        common = (
            np.isfinite(y)
            & np.isfinite(p_proposed)
            & np.isfinite(p_benchmark)
        )

        # Date-level loss means preserve the six-sector cross-section.
        proposed_brier_date = np.full(
            len(test_indices),
            np.nan,
        )
        benchmark_brier_date = np.full(
            len(test_indices),
            np.nan,
        )
        proposed_log_date = np.full(
            len(test_indices),
            np.nan,
        )
        benchmark_log_date = np.full(
            len(test_indices),
            np.nan,
        )

        for local_t in range(len(test_indices)):

            ok = common[local_t]

            if not ok.any():
                continue

            yt = y[local_t, ok]
            pp = np.clip(
                p_proposed[local_t, ok],
                EPS,
                1.0 - EPS,
            )
            pb = np.clip(
                p_benchmark[local_t, ok],
                EPS,
                1.0 - EPS,
            )

            proposed_brier_date[local_t] = np.mean(
                (pp - yt) ** 2
            )
            benchmark_brier_date[local_t] = np.mean(
                (pb - yt) ** 2
            )

            proposed_log_date[local_t] = -np.mean(
                yt * np.log(pp)
                + (1.0 - yt) * np.log1p(-pp)
            )

            benchmark_log_date[local_t] = -np.mean(
                yt * np.log(pb)
                + (1.0 - yt) * np.log1p(-pb)
            )

        valid_dates = (
            np.isfinite(proposed_brier_date)
            & np.isfinite(benchmark_brier_date)
            & np.isfinite(proposed_log_date)
            & np.isfinite(benchmark_log_date)
        )

        pbd = proposed_brier_date[valid_dates]
        bbd = benchmark_brier_date[valid_dates]
        pld = proposed_log_date[valid_dates]
        bld = benchmark_log_date[valid_dates]

        observed_brier_diff = float(
            np.mean(pbd - bbd)
        )
        observed_log_diff = float(
            np.mean(pld - bld)
        )

        rng = np.random.default_rng(
            SEED
            + 1000 * h
            + sum(ord(c) for c in benchmark)
        )

        brier_diffs = []
        log_diffs = []

        for _ in range(BOOTSTRAP_REPS):

            idx = moving_block_sample_indices(
                len(pbd),
                min(BOOTSTRAP_BLOCK, len(pbd)),
                rng,
            )

            brier_diffs.append(
                float(
                    np.mean(
                        pbd[idx] - bbd[idx]
                    )
                )
            )

            log_diffs.append(
                float(
                    np.mean(
                        pld[idx] - bld[idx]
                    )
                )
            )

        brier_diffs = np.asarray(brier_diffs)
        log_diffs = np.asarray(log_diffs)

        bootstrap_rows.append({
            "Horizon": h,
            "ProposedModel": PROPOSED_MODEL,
            "Benchmark": benchmark,
            "BrierDiff_ProposedMinusBenchmark": observed_brier_diff,
            "BrierDiff_CI2.5": float(np.quantile(brier_diffs, 0.025)),
            "BrierDiff_CI97.5": float(np.quantile(brier_diffs, 0.975)),
            "BrierProb_ProposedBetter": float(np.mean(brier_diffs < 0)),
            "LogScoreDiff_ProposedMinusBenchmark": observed_log_diff,
            "LogScoreDiff_CI2.5": float(np.quantile(log_diffs, 0.025)),
            "LogScoreDiff_CI97.5": float(np.quantile(log_diffs, 0.975)),
            "LogScoreProb_ProposedBetter": float(np.mean(log_diffs < 0)),
            "BootstrapReps": BOOTSTRAP_REPS,
            "BlockLength": BOOTSTRAP_BLOCK,
        })

bootstrap_results = pd.DataFrame(bootstrap_rows)

bootstrap_results.to_csv(
    TABLE_DIR / "Table_D11_Paired_Block_Bootstrap.csv",
    index=False
)

# =============================================================================
# 25. SECTOR / HORIZON HETEROGENEITY
# =============================================================================

sector_proposed = forecast_metrics[
    (forecast_metrics["Split"] == "Test")
    & (forecast_metrics["Model"] == PROPOSED_MODEL)
    & (forecast_metrics["Sector"] != "POOLED")
].copy()

sector_nograph = forecast_metrics[
    (forecast_metrics["Split"] == "Test")
    & (forecast_metrics["Model"] == "NoGraphTransformer")
    & (forecast_metrics["Sector"] != "POOLED")
][["Sector", "Horizon", "Brier", "LogScore"]].rename(
    columns={
        "Brier": "NoGraphBrier",
        "LogScore": "NoGraphLogScore",
    }
)

heterogeneity = sector_proposed.merge(
    sector_nograph,
    on=["Sector", "Horizon"],
    how="left",
)

heterogeneity["DynamicGraph_BrierSkill_vs_NoGraph"] = (
    1.0
    - heterogeneity["Brier"]
    / heterogeneity["NoGraphBrier"]
)

heterogeneity["DynamicGraph_LogScoreImprovement_vs_NoGraph"] = (
    heterogeneity["NoGraphLogScore"]
    - heterogeneity["LogScore"]
)

heterogeneity.to_csv(
    TABLE_DIR / "Table_D12_Sector_Horizon_Heterogeneity.csv",
    index=False
)

# =============================================================================
# 26. SAVE PREDICTION PANEL
# =============================================================================

prediction_rows = []

for split_name, split_mask in [
    ("Validation", validation_date_mask),
    ("Test", test_date_mask),
]:

    for t in np.where(split_mask)[0]:
        for s, sector in enumerate(sectors):

            row = {
                "Date": dates[t],
                "Split": split_name,
                "Sector": sector,
            }

            for h in HORIZONS:
                row[f"Target_{h}"] = horizon_target[h][t, s]

                for model_name in model_probabilities:
                    row[
                        f"{model_name}__P{h}"
                    ] = model_probabilities[
                        model_name
                    ][split_name][h][t, s]

            prediction_rows.append(row)

prediction_panel = pd.DataFrame(
    prediction_rows
)

prediction_panel.to_csv(
    DATA_DIR / "phase2_validation_test_predictions.csv.gz",
    index=False,
    compression="gzip",
)

# =============================================================================
# 27. FIGURES
# =============================================================================

# Figure D01: pooled Test Brier score.
fig, ax = plt.subplots(figsize=(11, 6))

for model_name, group in main_test.groupby("Model"):
    group = group.sort_values("Horizon")

    ax.plot(
        group["Horizon"],
        group["Brier"],
        marker="o",
        label=model_name,
    )

ax.set_xlabel("Forecast horizon (trading days)")
ax.set_ylabel("Brier score (lower is better)")
ax.set_xticks(HORIZONS)
ax.set_title("Out-of-Sample Sector Crash Probability Forecasting")
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_D01_Test_Brier_All_Models.png",
    dpi=300,
)
plt.close(fig)

# Figure D02: PR-AUC.
fig, ax = plt.subplots(figsize=(11, 6))

for model_name, group in main_test.groupby("Model"):
    group = group.sort_values("Horizon")

    ax.plot(
        group["Horizon"],
        group["PR_AUC"],
        marker="o",
        label=model_name,
    )

ax.set_xlabel("Forecast horizon (trading days)")
ax.set_ylabel("PR-AUC")
ax.set_xticks(HORIZONS)
ax.set_title("Rare-Event Discrimination on the Test Sample")
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_D02_Test_PR_AUC_All_Models.png",
    dpi=300,
)
plt.close(fig)

# Figure D03: graph ablation Brier.
graph_model_names = [
    "NoGraphTransformer",
    "RandomGraphTransformer",
    "StaticHawkesGraphTransformer",
    "DynamicHawkesGraphTransformer",
]

graph_test = main_test[
    main_test["Model"].isin(graph_model_names)
]

fig, ax = plt.subplots(figsize=(9, 6))

for model_name, group in graph_test.groupby("Model"):
    group = group.sort_values("Horizon")

    ax.plot(
        group["Horizon"],
        group["Brier"],
        marker="o",
        label=model_name,
    )

ax.set_xlabel("Forecast horizon (trading days)")
ax.set_ylabel("Brier score (lower is better)")
ax.set_xticks(HORIZONS)
ax.set_title("Graph-Prior Ablation")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_D03_Graph_Ablation_Brier.png",
    dpi=300,
)
plt.close(fig)

# Figure D04: sector-specific dynamic graph skill vs no graph.
plot_heterogeneity = heterogeneity.pivot(
    index="Sector",
    columns="Horizon",
    values="DynamicGraph_BrierSkill_vs_NoGraph",
)

fig, ax = plt.subplots(figsize=(10, 7))
image = ax.imshow(
    plot_heterogeneity.to_numpy(),
    aspect="auto",
)
ax.set_xticks(range(len(plot_heterogeneity.columns)))
ax.set_xticklabels(
    [str(int(h)) for h in plot_heterogeneity.columns]
)
ax.set_yticks(range(len(plot_heterogeneity.index)))
ax.set_yticklabels(plot_heterogeneity.index)
ax.set_xlabel("Forecast horizon")
ax.set_ylabel("Sector")
ax.set_title("Dynamic Hawkes Graph Brier Skill Relative to No-Graph Transformer")
fig.colorbar(
    image,
    ax=ax,
    label="Brier skill",
)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_D04_Sector_Graph_Skill_Heatmap.png",
    dpi=300,
)
plt.close(fig)

# Figure D05: training curves.
fig, ax = plt.subplots(figsize=(10, 6))

for model_name, group in training_history.groupby("Model"):
    ax.plot(
        group["Epoch"],
        group["ValidationNLL"],
        label=model_name,
    )

ax.set_xlabel("Epoch")
ax.set_ylabel("Validation survival NLL")
ax.set_title("Neural Model Early-Stopping Curves")
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_D05_Neural_Validation_Curves.png",
    dpi=300,
)
plt.close(fig)

# =============================================================================
# 28. EXCEL WORKBOOK
# =============================================================================

excel_path = (
    EXCEL_DIR
    / "Python_Phase2_ML_DL_Graph_Survival_Results.xlsx"
)

if importlib.util.find_spec("xlsxwriter") is not None:
    excel_engine = "xlsxwriter"
elif importlib.util.find_spec("openpyxl") is not None:
    excel_engine = "openpyxl"
else:
    excel_engine = None

excel_tables = {
    "Target Audit": target_audit,
    "Features": pd.DataFrame({"Feature": model_feature_names}),
    "Crash Imputation": event_imputation_audit,
    "Graph Design": graph_ablation_summary,
    "Graph Scaling": graph_scaling_audit,
    "Edge Magnitudes": edge_scaling_diagnostics,
    "Graph State": graph_state_summary,
    "Training History": training_history,
    "Best Epochs": best_epochs,
    "All Metrics": forecast_metrics,
    "Main Test": main_test,
    "Proposed Skill": skill_table,
    "Coherence": coherence_table,
    "Bootstrap": bootstrap_results,
    "Sector Heterogeneity": heterogeneity,
}

if excel_engine is not None:
    try:
        with pd.ExcelWriter(
            excel_path,
            engine=excel_engine,
        ) as writer:

            for sheet_name, dataframe in excel_tables.items():
                dataframe.to_excel(
                    writer,
                    sheet_name=sheet_name[:31],
                    index=False,
                )

            if excel_engine == "xlsxwriter":
                workbook = writer.book
                header_format = workbook.add_format({
                    "bold": True,
                    "text_wrap": True,
                    "valign": "top",
                    "border": 1,
                })

                for _, worksheet in writer.sheets.items():
                    worksheet.freeze_panes(1, 0)
                    worksheet.set_row(
                        0,
                        28,
                        header_format,
                    )
                    worksheet.set_column(
                        0,
                        30,
                        16,
                    )

            else:
                for _, worksheet in writer.sheets.items():
                    worksheet.freeze_panes = "A2"

        log(
            f"Excel workbook saved with engine={excel_engine}: "
            f"{excel_path}"
        )

    except Exception as exc:
        log(
            f"WARNING: Excel export failed: {repr(exc)}"
        )
else:
    log(
        "WARNING: xlsxwriter/openpyxl unavailable; CSV tables remain complete."
    )

# =============================================================================
# 29. FINAL INTEGRITY CHECKS
# =============================================================================

integrity_checks = []

# Split-aware targets cannot cross boundaries by construction.
integrity_checks.append((
    "Train/Validation/Test split labels present",
    set(np.unique(splits)) == {"Train", "Validation", "Test"},
))

integrity_checks.append((
    "No future-label columns in predictors",
    not any(
        c in model_feature_names
        for c in [
            "CrashWithin_1",
            "CrashWithin_5",
            "CrashWithin_10",
            "CrashWithin_22",
            "EventTime",
            "CensorTime",
        ]
    ),
))

integrity_checks.append((
    "No Hawkes/historical forecast probabilities in generic predictors",
    not any(
        ("Prob_" in c or "Prob" in c)
        for c in model_feature_names
    ),
))

integrity_checks.append((
    "Feature matrix finite after Train-only imputation/scaling",
    bool(np.all(np.isfinite(X_panel))),
))

integrity_checks.append((
    "Crash_Main missing states set to zero and never forward-filled",
    bool(crash_missing_set_to_zero),
))

integrity_checks.append((
    "Neural hazard class-weight cap is at most 3",
    bool(
        NEURAL_MAX_POS_WEIGHT <= 3.0
        and HAZARD_POS_WEIGHT <= 3.0 + 1e-12
    ),
))

integrity_checks.append((
    "Graph uses Train-only positive magnitude scale",
    bool(
        np.isfinite(GRAPH_TRAIN_Q)
        and GRAPH_TRAIN_Q > 0
    ),
))

integrity_checks.append((
    "Dynamic Hawkes graph is not row-normalized",
    bool(
        np.any(
            np.abs(
                dynamic_graph_scaled[
                    train_date_mask
                ].sum(axis=-1)
                - 1.0
            ) > 1e-4
        )
    ),
))

integrity_checks.append((
    "All survival neural models horizon coherent",
    int(neural_coherence_bad) == 0,
))

integrity_checks.append((
    "Proposed model present in Test metrics",
    bool(
        (
            (main_test["Model"] == PROPOSED_MODEL)
        ).any()
    ),
))

integrity_checks.append((
    "Both classes present in Test at all horizons",
    all(
        len(
            np.unique(
                horizon_target[h][test_date_mask][
                    np.isfinite(
                        horizon_target[h][test_date_mask]
                    )
                ]
            )
        ) == 2
        for h in HORIZONS
    ),
))

integrity_checks.append((
    "Graph source recorded",
    isinstance(graph_source, str)
    and len(graph_source) > 0,
))

integrity_table = pd.DataFrame(
    integrity_checks,
    columns=["Check", "Passed"],
)

integrity_table.to_csv(
    TABLE_DIR / "Table_D13_Final_Integrity_Checks.csv",
    index=False
)

if not integrity_table["Passed"].all():
    failed = integrity_table.loc[
        ~integrity_table["Passed"],
        "Check",
    ].tolist()

    raise RuntimeError(
        f"Phase 2 integrity checks failed: {failed}"
    )

# =============================================================================
# 30. METADATA / MODEL SUMMARY
# =============================================================================

metadata = {
    "ProjectTitle": (
        "Forecasting Sectoral Crash Risk and Contagion: "
        "Integrating Dynamic Volatility, Extreme-Value Modelling "
        "and Graph Deep Learning"
    ),
    "PythonPhase": "2.1",
    "ScriptVersion": "2.1",
    "GeneratedAt": datetime.now().isoformat(),
    "Seed": SEED,
    "InputPhase1Zip": INPUT_ZIP_PATH.name,
    "GraphSource": graph_source,
    "Sectors": sectors,
    "ForecastHorizons": HORIZONS,
    "LookbackTradingDays": LOOKBACK,
    "FeatureCount": F_DIM,
    "BaseFeatureCount": len(feature_columns),
    "MissingIndicatorCount": len(missing_indicator_names),
    "HazardPositiveWeight": HAZARD_POS_WEIGHT,
    "NeuralMaxPositiveWeight": NEURAL_MAX_POS_WEIGHT,
    "XGBoostMaxPositiveWeight": XGB_MAX_POS_WEIGHT,
    "GraphScalingMethod": "log1p(A/q99_train)/log(2); no row normalization",
    "GraphScaleQuantile": GRAPH_SCALE_QUANTILE,
    "GraphTrainScaleReference": GRAPH_TRAIN_Q,
    "GraphScaleClip": GRAPH_SCALE_CLIP,
    "GraphStateFeatures": GRAPH_STATE_FEATURE_NAMES,
    "CrashMainImputation": (
        "Missing Crash_Main set to zero; CurrentCrashObserved marks availability; "
        "Crash_Main never forward-filled"
    ),
    "ProposedModel": PROPOSED_MODEL,
    "Device": str(DEVICE),
    "BootstrapReps": BOOTSTRAP_REPS,
    "BootstrapBlockLength": BOOTSTRAP_BLOCK,
    "AntiLeakageTargetConstruction": (
        "Split-aware right censoring; future search stops at split boundary"
    ),
}

with open(
    DATA_DIR / "python_phase2_metadata.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        metadata,
        handle,
        indent=2,
        default=str,
    )

# Save compact model-selection/calibration bundle.
phase2_bundle = {
    "metadata": metadata,
    "neural_calibration": neural_calibration,
    "xgboost_calibration": xgb_calibration,
    "best_epochs": best_epochs.to_dict(orient="records"),
    "model_feature_names": model_feature_names,
    "sectors": sectors,
}

with open(
    MODEL_DIR / "phase2_model_selection_bundle.pkl",
    "wb",
) as handle:
    pickle.dump(
        phase2_bundle,
        handle,
    )

# =============================================================================
# 31. CLEAN EXTRACTED INPUT / ZIP ALL OUTPUTS
# =============================================================================

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

zip_output = (
    ZIP_DIR
    / "Python_Phase2_1_Corrected_Graph_Survival_All_Outputs.zip"
)

if zip_output.exists():
    zip_output.unlink()

with zipfile.ZipFile(
    zip_output,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    for path in OUTPUT_ROOT.rglob("*"):

        if path.is_file() and path != zip_output:
            archive.write(
                path,
                arcname=path.relative_to(OUTPUT_ROOT),
            )

log(f"All Phase-2 outputs zipped to: {zip_output}")

# =============================================================================
# 32. CONSOLE SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print("PYTHON PHASE 2.1 COMPLETED SUCCESSFULLY")
print("=" * 100)

print(f"Device: {DEVICE}")
print(f"Graph source: {graph_source}")
print(f"Lookback: {LOOKBACK} trading days")
print(f"Predictor dimension after missing indicators: {F_DIM}")
print(f"Hazard positive-class weight: {HAZARD_POS_WEIGHT:.4f}")
print(
    f"Graph scaling: Train q{GRAPH_SCALE_QUANTILE:.2f}="
    f"{GRAPH_TRAIN_Q:.8f}; NO row normalization"
)

print("\nPooled TEST results sorted by horizon and Brier:")
display_columns = [
    "Horizon",
    "Model",
    "Brier",
    "LogScore",
    "PR_AUC",
    "ROC_AUC",
    "CalibrationIntercept",
    "CalibrationSlope",
]

print(
    main_test[
        display_columns
    ].to_string(index=False)
)

print("\nGraph prior strengths:")
for model_name in graph_variants:
    eta = neural_calibration[
        model_name
    ].get(
        "learned_graph_eta",
        np.nan,
    )

    state_eta = neural_calibration[
        model_name
    ].get(
        "learned_graph_state_eta",
        np.nan,
    )

    print(
        f"  {model_name}: "
        f"attention_eta={eta:.4f}, "
        f"state_eta={state_eta:.4f}"
    )

print("\nProposed model:")
print(f"  {PROPOSED_MODEL}")

print("\nFinal ZIP to send back for review:")
print(f"  {zip_output}")

print("=" * 100)

# =============================================================================
# 33. AUTOMATIC COLAB DOWNLOAD
# =============================================================================

try:
    from google.colab import files

    print(
        "\nStarting browser download of "
        "Python_Phase2_ML_DL_Graph_Survival_All_Outputs.zip ..."
    )

    files.download(
        str(zip_output)
    )

except ImportError:
    print(
        "\nNot running in Google Colab. "
        f"Retrieve the ZIP manually from: {zip_output}"
    )

# =============================================================================
# END OF PYTHON PHASE 2
# =============================================================================


[2026-09-01 14:34:53] ====================================================================================================
[2026-09-01 14:34:53] PYTHON PHASE 2.1 START — CORRECTED ML / DL / GRAPH SURVIVAL FORECASTING
[2026-09-01 14:34:53] Python=3.13.15; platform=Linux-6.6.122+-x86_64-with-glibc2.35; seed=20260901
[2026-09-01 14:34:53] PyTorch=2.11.0+cpu; device=cpu

Please upload the completed Phase-1 ZIP:
    Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip



Saving Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip to Python_Phase1_v1_2_From_R_Handoff_All_Outputs (1).zip
[2026-09-01 14:36:02] Validated uploaded Phase-1 ZIP: /content/Python_Phase1_v1_2_From_R_Handoff_All_Outputs (1).zip
[2026-09-01 14:36:02] Accepted Phase-1 ZIP: /content/Python_Phase1_v1_2_From_R_Handoff_All_Outputs (1).zip
[2026-09-01 14:36:03] Loaded 14,928 sector-days; 2,488 dates; 6 sectors; graph source=HAWKES_LAMBDA_MIN_STABILITY.
[2026-09-01 14:36:03] Split-aware censoring targets reconstructed successfully.
[2026-09-01 14:36:03] Selected 49 leakage-screened numeric predictors.
[2026-09-01 14:36:03] Final neural feature dimension=59; missingness indicators=10.
[2026-09-01 14:36:03] Crash_Main event-safe imputation applied: missing events set to zero and never forward-filled.
[2026-09-01 14:36:03] Magnitude-preserving graph scaling applied. Train q0.99=0.06118932; row normalization disabled.
[2026-09-01 14:36:03] Sector samples=11,501; graph-date samples=1,984.
[2026-

/tmp/ipykernel_1929/3093604367.py:1775: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


[2026-09-01 14:38:23] TemporalTransformer_Survival: epoch=01, train=0.48029, valNLL=0.14955
[2026-09-01 14:38:40] TemporalTransformer_Survival: epoch=02, train=0.25052, valNLL=0.13605
[2026-09-01 14:38:57] TemporalTransformer_Survival: epoch=03, train=0.24357, valNLL=0.18170
[2026-09-01 14:39:14] TemporalTransformer_Survival: epoch=04, train=0.23793, valNLL=0.33770
[2026-09-01 14:39:31] TemporalTransformer_Survival: epoch=05, train=0.23044, valNLL=0.34087
[2026-09-01 14:39:48] TemporalTransformer_Survival: epoch=06, train=0.21879, valNLL=0.31922
[2026-09-01 14:40:04] TemporalTransformer_Survival: epoch=07, train=0.20860, valNLL=0.37163
[2026-09-01 14:40:22] TemporalTransformer_Survival: epoch=08, train=0.19501, valNLL=0.47677
[2026-09-01 14:40:39] TemporalTransformer_Survival: epoch=09, train=0.18776, valNLL=0.46811
[2026-09-01 14:40:55] TemporalTransformer_Survival: epoch=10, train=0.18048, valNLL=0.44786
[2026-09-01 14:40:55] Training graph ablation: NoGraphTransformer


/tmp/ipykernel_1929/3093604367.py:1961: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal_encoder = nn.TransformerEncoder(


[2026-09-01 14:41:15] NoGraphTransformer: epoch=01, train=0.50871, valNLL=0.17796
[2026-09-01 14:41:34] NoGraphTransformer: epoch=02, train=0.25323, valNLL=0.13670
[2026-09-01 14:41:53] NoGraphTransformer: epoch=03, train=0.24868, valNLL=0.13502
[2026-09-01 14:42:13] NoGraphTransformer: epoch=04, train=0.24405, valNLL=0.13877
[2026-09-01 14:42:32] NoGraphTransformer: epoch=05, train=0.23934, valNLL=0.17108
[2026-09-01 14:42:51] NoGraphTransformer: epoch=06, train=0.23092, valNLL=0.14334
[2026-09-01 14:43:11] NoGraphTransformer: epoch=07, train=0.22604, valNLL=0.17777
[2026-09-01 14:43:29] NoGraphTransformer: epoch=08, train=0.22488, valNLL=0.17824
[2026-09-01 14:43:49] NoGraphTransformer: epoch=09, train=0.21823, valNLL=0.18212
[2026-09-01 14:44:08] NoGraphTransformer: epoch=10, train=0.21214, valNLL=0.15275
[2026-09-01 14:44:28] NoGraphTransformer: epoch=11, train=0.20539, valNLL=0.16763
[2026-09-01 14:44:28] Training graph ablation: RandomGraphTransformer
[2026-09-01 14:44:47] Random

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
# =============================================================================
# PYTHON PHASE 2.2 — FIVE-SEED ROBUSTNESS / GRAPH PLACEBO REPLICATION
#
# Forecasting Sectoral Crash Risk and Contagion:
# Integrating Dynamic Volatility, Extreme-Value Modelling and Graph Deep Learning
#
# INPUT
# -----
# Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip
#
# PURPOSE
# -------
# This script consumes the frozen Phase-1 econometric/contagion output and
# estimates the forecasting models used in the main paper:
#
#   1. Historical probability baselines.
#   2. Stability-selected Hawkes probability baseline.
#   3. XGBoost benchmark (four horizon classifiers; monotone rearrangement).
#   4. LSTM discrete-time survival model.
#   5. Temporal Transformer discrete-time survival benchmark.
#   6. Graph Survival Transformer — no graph prior.
#   7. Graph Survival Transformer — random static graph placebo.
#   8. Graph Survival Transformer — static stability-selected Hawkes graph.
#   9. PROPOSED: Graph Survival Transformer with the dynamic Hawkes graph
#      supplied as a SOFT attention prior.
#
# KEY DESIGN PRINCIPLES
# ---------------------
# * Only the uploaded NSE/R-derived data are used. No external predictors.
# * Train / Validation / Test are chronological; no random data split.
# * Forecast-origin features contain information available through day t only.
# * Time-to-crash outcomes are RECONSTRUCTED HERE with split-aware censoring.
#   Therefore Validation targets never use Test outcomes and Train targets
#   never use Validation outcomes.
# * 1/5/10/22-day neural probabilities are derived from one 22-day daily
#   hazard path:
#
#       P(T <= H) = 1 - product_{k=1}^H [1 - h_k].
#
#   Hence neural probabilities are coherent by construction.
# * XGBoost is a conventional benchmark and uses separate horizon classifiers;
#   its four probabilities are monotonically rearranged after calibration.
# * Deep models use the SAME economic predictors. Graph models differ only in
#   their graph prior, permitting clean ablation tests.
# * The Hawkes graph is a SOFT prior: it biases graph attention but does not
#   hard-mask other sector interactions.
# * Validation is used for early stopping / calibration only.
# * Test is evaluated once after all model choices are fixed.
# * Neural rare-event weighting uses sqrt imbalance capped at 3; all neural
#   models receive post-hoc hazard temperature calibration on Validation.
# * Main probability metrics: Brier Score, Log Score, PR-AUC, ROC-AUC and
#   calibration intercept/slope. Accuracy is deliberately not a headline metric.
# * Paired moving-block bootstrap inference is reported for the proposed model
#   versus every benchmark on the untouched Test sample.
#
# VERSION: 2.2
# DATE: 2026-09-01
# =============================================================================

from __future__ import annotations

import os
import sys
import gc
import json
import math
import time
import random
import shutil
import pickle
import zipfile
import warnings
import platform
import subprocess
import importlib.util
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =============================================================================
# 0. REPRODUCIBILITY / CONFIGURATION
# =============================================================================

SEED = 20260901

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

INPUT_ZIP = os.getenv(
    "NSE_PHASE1_ZIP",
    "/content/Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip"
)

OUTPUT_ROOT = Path(os.getenv(
    "NSE_PHASE2_OUTPUT_DIR",
    "/content/Sectoral_Crash_Risk_Contagion_Python_Phase2_2"
))

HORIZONS = [1, 5, 10, 22]
MAX_HORIZON = 22
LOOKBACK = int(os.getenv("NSE_LOOKBACK", "60"))

# Deep-learning hyperparameters. These are intentionally compact because the
# dataset contains six sectors and ~2,500 trading dates.
BATCH_SIZE = int(os.getenv("NSE_BATCH_SIZE", "128"))
GRAPH_BATCH_SIZE = int(os.getenv("NSE_GRAPH_BATCH_SIZE", "32"))
MAX_EPOCHS = int(os.getenv("NSE_MAX_EPOCHS", "60"))
PATIENCE = int(os.getenv("NSE_PATIENCE", "8"))
LEARNING_RATE = float(os.getenv("NSE_LEARNING_RATE", "0.001"))
WEIGHT_DECAY = float(os.getenv("NSE_WEIGHT_DECAY", "0.0001"))
D_MODEL = int(os.getenv("NSE_D_MODEL", "48"))
N_HEADS = int(os.getenv("NSE_N_HEADS", "4"))
N_TRANSFORMER_LAYERS = int(os.getenv("NSE_TRANSFORMER_LAYERS", "2"))
DROPOUT = float(os.getenv("NSE_DROPOUT", "0.10"))
SECTOR_EMBED_DIM = int(os.getenv("NSE_SECTOR_EMBED_DIM", "8"))

# Positive hazard weighting.
#
# CORRECTION 2.1:
# The Phase-2 neural cap of 10 produced a hazard weight near 6.7 and required
# very strong post-hoc temperature flattening. Neural survival models now use a
# conservative cap of 3. XGBoost retains the previous cap of 10 so the strong
# tabular benchmark is otherwise unchanged.
NEURAL_MAX_POS_WEIGHT = float(
    os.getenv("NSE_NEURAL_MAX_POS_WEIGHT", "3.0")
)
XGB_MAX_POS_WEIGHT = float(
    os.getenv("NSE_XGB_MAX_POS_WEIGHT", "10.0")
)

# Magnitude-preserving Hawkes graph transformation.
#
# A positive Train-only q99 reference is mapped to 1:
#
#     scaled(A) = log(1 + A / q99_train) / log(2)
#
# Values are globally clipped only at 3 to limit extreme numerical leverage.
# Crucially, there is NO row normalization, so 0.001 remains much weaker than
# 0.10 after transformation.
GRAPH_SCALE_QUANTILE = float(
    os.getenv("NSE_GRAPH_SCALE_QUANTILE", "0.99")
)
GRAPH_SCALE_CLIP = float(
    os.getenv("NSE_GRAPH_SCALE_CLIP", "3.0")
)

# XGBoost.
XGB_MAX_ROUNDS = int(os.getenv("NSE_XGB_MAX_ROUNDS", "1500"))
XGB_EARLY_STOP = int(os.getenv("NSE_XGB_EARLY_STOP", "75"))

# Moving-block bootstrap.
BOOTSTRAP_REPS = int(os.getenv("NSE_BOOTSTRAP_REPS", "500"))
BOOTSTRAP_BLOCK = int(os.getenv("NSE_BOOTSTRAP_BLOCK", "22"))

EPS = 1e-8

PROPOSED_MODEL = "DynamicHawkesGraphTransformer"

# =============================================================================
# 1. OUTPUT DIRECTORIES / LOGGING
# =============================================================================

TABLE_DIR = OUTPUT_ROOT / "01_Tables"
FIG_DIR = OUTPUT_ROOT / "02_Figures"
EXCEL_DIR = OUTPUT_ROOT / "03_Excel"
MODEL_DIR = OUTPUT_ROOT / "04_Model_Objects"
DATA_DIR = OUTPUT_ROOT / "05_Processed_Data"
LOG_DIR = OUTPUT_ROOT / "06_Logs"
ZIP_DIR = OUTPUT_ROOT / "07_Zip"
EXTRACT_DIR = OUTPUT_ROOT / "_Phase1_Extracted"

for directory in [
    OUTPUT_ROOT, TABLE_DIR, FIG_DIR, EXCEL_DIR, MODEL_DIR,
    DATA_DIR, LOG_DIR, ZIP_DIR, EXTRACT_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / "Python_Phase2_2_run_log.txt"

def log(message: str):
    stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{stamp}] {message}"
    print(line)
    with open(LOG_FILE, "a", encoding="utf-8") as handle:
        handle.write(line + "\n")

log("=" * 100)
log("PYTHON PHASE 2.2 START — FIVE-SEED ROBUSTNESS / GRAPH PLACEBO REPLICATION")
log(f"Python={sys.version.split()[0]}; platform={platform.platform()}; seed={SEED}")

# =============================================================================
# 2. PACKAGE CHECKS
# =============================================================================

def ensure_package(import_name: str, pip_name: str | None = None):
    if importlib.util.find_spec(import_name) is not None:
        return
    pip_name = pip_name or import_name
    log(f"Installing missing package: {pip_name}")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", pip_name]
    )

ensure_package("torch")
ensure_package("xgboost")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import xgboost as xgb

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log(f"PyTorch={torch.__version__}; device={DEVICE}")

# =============================================================================
# 3. ALWAYS ASK USER TO UPLOAD PHASE-1 ZIP IN COLAB
# =============================================================================

REQUIRED_PHASE1_MEMBERS = {
    "phase2_deep_learning_master.csv.gz",
    "econometric_dynamic_graph_arrays.npz",
    "python_phase1_metadata.json",
    "Table_P12_Final_Stable_Edge_Parameters.csv",
}

def zip_basenames(path: Path) -> set[str]:
    try:
        with zipfile.ZipFile(path, "r") as archive:
            return {
                Path(name).name
                for name in archive.namelist()
                if not name.endswith("/")
            }
    except Exception:
        return set()

def is_valid_phase1_zip(path: Path) -> bool:
    return (
        path.exists()
        and path.suffix.lower() == ".zip"
        and REQUIRED_PHASE1_MEMBERS.issubset(zip_basenames(path))
    )

def resolve_phase1_zip(configured: str) -> Path:
    """
    Colab behaviour: ALWAYS open a file picker. This avoids accidentally using
    a stale R handoff or previous Python ZIP already present in /content.
    """
    try:
        from google.colab import files

        while True:
            print(
                "\nPlease upload the completed Phase-1 ZIP:\n"
                "    Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip\n"
            )
            uploaded = files.upload()

            candidates = [
                Path("/content") / name
                for name in uploaded
                if name.lower().endswith(".zip")
            ]

            valid = [p for p in candidates if is_valid_phase1_zip(p)]

            if len(valid) == 1:
                log(f"Validated uploaded Phase-1 ZIP: {valid[0]}")
                return valid[0]

            if len(valid) > 1:
                print(
                    "\nMore than one valid Phase-1 ZIP was uploaded. "
                    "Please upload exactly one ZIP.\n"
                )
                continue

            for p in candidates:
                missing = sorted(
                    REQUIRED_PHASE1_MEMBERS - zip_basenames(p)
                )
                log(
                    f"Rejected '{p.name}'. Missing Phase-1 members: {missing}"
                )

            print(
                "\nThe selected ZIP is not the required completed Phase-1 ZIP. "
                "Please choose:\n"
                "    Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip\n"
            )

    except ImportError:
        configured_path = Path(configured)

        if is_valid_phase1_zip(configured_path):
            return configured_path

        # Local / notebook fallback: locate a validated ZIP by contents.
        for folder in [Path.cwd(), Path("/mnt/data")]:
            if folder.exists():
                for candidate in folder.glob("*.zip"):
                    if is_valid_phase1_zip(candidate):
                        return candidate

        raise FileNotFoundError(
            "Could not locate a valid Phase-1 ZIP. "
            "Set NSE_PHASE1_ZIP to the correct path."
        )

INPUT_ZIP_PATH = resolve_phase1_zip(INPUT_ZIP)
log(f"Accepted Phase-1 ZIP: {INPUT_ZIP_PATH}")

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(INPUT_ZIP_PATH, "r") as archive:
    archive.extractall(EXTRACT_DIR)

def find_one(filename: str) -> Path:
    matches = list(EXTRACT_DIR.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one '{filename}' in Phase-1 ZIP; "
            f"found {len(matches)}."
        )
    return matches[0]

MASTER_FILE = find_one("phase2_deep_learning_master.csv.gz")
GRAPH_FILE = find_one("econometric_dynamic_graph_arrays.npz")
META_FILE = find_one("python_phase1_metadata.json")
EDGE_FILE = find_one("Table_P12_Final_Stable_Edge_Parameters.csv")

# =============================================================================
# 4. LOAD / AUDIT PHASE-1 OUTPUT
# =============================================================================

master = pd.read_csv(MASTER_FILE, parse_dates=["Date"])
master = master.sort_values(["Date", "Sector"]).reset_index(drop=True)

with open(META_FILE, "r", encoding="utf-8") as handle:
    phase1_meta = json.load(handle)

graph_npz = np.load(GRAPH_FILE, allow_pickle=True)
stable_edges = pd.read_csv(EDGE_FILE)

sectors = graph_npz["sectors"].astype(str).tolist()
S = len(sectors)
sector_to_idx = {sector: i for i, sector in enumerate(sectors)}

dates = pd.DatetimeIndex(graph_npz["dates"].astype(str))
T = len(dates)
date_to_idx = {pd.Timestamp(date): i for i, date in enumerate(dates)}

graph_source = str(graph_npz["graph_source"].reshape(-1)[0])
dynamic_graph_all = graph_npz["adjacency"].astype(np.float32)
dynamic_graph_train_fit = graph_npz["train_fit_adjacency"].astype(np.float32)
dynamic_graph_final_fit = graph_npz["final_fit_adjacency"].astype(np.float32)
hawkes_train_alpha = graph_npz["hawkes_train_alpha"].astype(np.float32)
hawkes_final_alpha = graph_npz["hawkes_final_alpha"].astype(np.float32)

if dynamic_graph_all.shape != (T, S, S):
    raise RuntimeError(
        f"Unexpected graph shape: {dynamic_graph_all.shape}; "
        f"expected {(T, S, S)}."
    )

if set(master["Sector"].unique()) != set(sectors):
    raise RuntimeError("Master sectors do not match graph-array sectors.")

if master["Date"].nunique() != T:
    raise RuntimeError("Master dates do not match graph-array dates.")

if graph_source != "HAWKES_LAMBDA_MIN_STABILITY":
    log(
        f"WARNING: graph source is '{graph_source}', not the expected Hawkes "
        "lambda-min stability graph. The code will still proceed."
    )

# Ensure every Date-Sector combination is unique and complete.
if master.duplicated(["Date", "Sector"]).any():
    raise RuntimeError("Duplicate Date-Sector rows found in DL master.")

counts = master.groupby("Date")["Sector"].nunique()
if not (counts == S).all():
    raise RuntimeError("The DL master is not a complete six-sector date panel.")

# Date-level split.
split_by_date = (
    master[["Date", "Split"]]
    .drop_duplicates()
    .set_index("Date")["Split"]
    .reindex(dates)
)
splits = split_by_date.astype(str).to_numpy()

if set(np.unique(splits)) != {"Train", "Validation", "Test"}:
    raise RuntimeError(f"Unexpected split labels: {np.unique(splits)}")

train_date_mask = splits == "Train"
validation_date_mask = splits == "Validation"
test_date_mask = splits == "Test"

log(
    f"Loaded {len(master):,} sector-days; {T:,} dates; {S} sectors; "
    f"graph source={graph_source}."
)

# =============================================================================
# 5. RECONSTRUCT CURRENT CRASH PANEL
# =============================================================================

C = np.zeros((T, S), dtype=np.float32)
M = np.zeros((T, S), dtype=bool)

for row in master[["Date", "Sector", "Crash_Main"]].itertuples(index=False):
    t = date_to_idx[pd.Timestamp(row.Date)]
    s = sector_to_idx[row.Sector]

    if pd.notna(row.Crash_Main):
        C[t, s] = float(row.Crash_Main)
        M[t, s] = True

if C[train_date_mask].sum() <= 0:
    raise RuntimeError("No Train crash events found.")

# =============================================================================
# 6. SPLIT-AWARE SURVIVAL TARGETS — CRITICAL ANTI-LEAKAGE CORRECTION
# =============================================================================
#
# Earlier Phase-1 target construction was censoring-aware but did not explicitly
# stop future-outcome searches at Train/Validation boundaries. For the ML/DL
# experiment we rebuild targets so that:
#
#   * Train origins never use Validation events as future outcomes.
#   * Validation origins never use Test events as future outcomes.
#   * Test origins stop at the dataset end.
#
# A positive event observed before the split boundary remains a valid outcome.
# A no-event origin is right-censored at the split boundary.

daily_hazard_target = np.zeros((T, S, MAX_HORIZON), dtype=np.float32)
daily_hazard_mask = np.zeros((T, S, MAX_HORIZON), dtype=np.float32)

split_event_time = np.full((T, S), np.nan, dtype=np.float32)
split_censor_time = np.full((T, S), np.nan, dtype=np.float32)
split_event_observed = np.full((T, S), np.nan, dtype=np.float32)

horizon_target = {
    h: np.full((T, S), np.nan, dtype=np.float32)
    for h in HORIZONS
}

for t in range(T):
    split_t = splits[t]

    for s in range(S):

        if not M[t, s]:
            continue

        observed_until = 0
        event_k = None

        for k in range(1, MAX_HORIZON + 1):
            future_t = t + k

            if future_t >= T:
                break

            # Hard stop at split boundary.
            if splits[future_t] != split_t:
                break

            # Hard stop at missing sector-day crash state.
            if not M[future_t, s]:
                break

            observed_until = k

            if C[future_t, s] == 1:
                event_k = k
                break

        if event_k is not None:
            split_event_observed[t, s] = 1.0
            split_event_time[t, s] = float(event_k)
            split_censor_time[t, s] = float(event_k)

            # At-risk through the event day.
            daily_hazard_mask[t, s, :event_k] = 1.0
            daily_hazard_target[t, s, event_k - 1] = 1.0

        else:
            split_event_observed[t, s] = 0.0
            split_censor_time[t, s] = float(observed_until)

            if observed_until > 0:
                daily_hazard_mask[t, s, :observed_until] = 1.0

        for h in HORIZONS:
            if event_k is not None and event_k <= h:
                horizon_target[h][t, s] = 1.0
            elif observed_until >= h:
                horizon_target[h][t, s] = 0.0
            else:
                horizon_target[h][t, s] = np.nan

# Save corrected target audit.
target_audit_rows = []

for split_name, split_mask in [
    ("Train", train_date_mask),
    ("Validation", validation_date_mask),
    ("Test", test_date_mask),
]:
    for h in HORIZONS:
        y = horizon_target[h][split_mask].reshape(-1)
        ok = np.isfinite(y)

        target_audit_rows.append({
            "Split": split_name,
            "Sector": "POOLED",
            "Horizon": h,
            "ValidTargets": int(ok.sum()),
            "Events": int(np.nansum(y)),
            "EventRate": float(np.nanmean(y)) if ok.any() else np.nan,
        })

        for s, sector in enumerate(sectors):
            ys = horizon_target[h][split_mask, s]
            oks = np.isfinite(ys)

            target_audit_rows.append({
                "Split": split_name,
                "Sector": sector,
                "Horizon": h,
                "ValidTargets": int(oks.sum()),
                "Events": int(np.nansum(ys)),
                "EventRate": float(np.nanmean(ys)) if oks.any() else np.nan,
            })

target_audit = pd.DataFrame(target_audit_rows)
target_audit.to_csv(
    TABLE_DIR / "Table_D01_Split_Aware_Target_Audit.csv",
    index=False
)

log("Split-aware censoring targets reconstructed successfully.")

# =============================================================================
# 7. FEATURE GOVERNANCE — EXPLICITLY REMOVE FUTURE / TARGET VARIABLES
# =============================================================================

# Predictors intentionally excluded because they are outcomes, future-event
# summaries, benchmark predictions, IDs/text, or direct target thresholds.
EXCLUDE_COLUMNS = {
    "Date",
    "Sector",
    "Split",
    "PrimarySector",
    "ExclusionReason",
    "EconometricGraphSource",

    # Survival / future labels.
    "EventObservedWithin22",
    "EventTime",
    "CensorTime",
    "CrashWithin_1",
    "CrashWithin_5",
    "CrashWithin_10",
    "CrashWithin_22",

    # Baseline/model predictions.
    "FixedHistoricalProb_1",
    "FixedHistoricalProb_5",
    "FixedHistoricalProb_10",
    "FixedHistoricalProb_22",
    "ExpandingHistoricalProb_1",
    "ExpandingHistoricalProb_5",
    "ExpandingHistoricalProb_10",
    "ExpandingHistoricalProb_22",
    "StableHawkesProb_1",
    "StableHawkesProb_5",
    "StableHawkesProb_10",
    "StableHawkesProb_22",

    # Redundant current-label copy.
    "CurrentCrash",

    # Direct crash thresholds / alternative crash outcomes are excluded to keep
    # the feature set economically interpretable and avoid near-target proxies.
    "CrashThreshold_001",
    "CrashThreshold_0025",
    "CrashThreshold_005",
    "Crash_001",
    "Crash_0025",
    "Crash_005",

    # Optimizer diagnostics are not economic predictors.
    "EVT_fit_method",
    "EVT_fit_convergence",
    "EVT_loglik",
    "EVT_nll_improvement",
    "EVT_at_boundary",
    "EVT_refit_id",
}

# Crash_Main at forecast origin t IS permitted: a crash observed today is valid
# information when forecasting t+1 onward and is central to excitation dynamics.
candidate_features = []

for column in master.columns:
    if column in EXCLUDE_COLUMNS:
        continue

    if pd.api.types.is_numeric_dtype(master[column]):
        candidate_features.append(column)

# Explicitly retain current crash state and observation indicator.
for mandatory in ["Crash_Main", "CurrentCrashObserved"]:
    if mandatory in master.columns and mandatory not in candidate_features:
        candidate_features.append(mandatory)

# Remove accidental future-like names defensively.
forbidden_name_fragments = [
    "CrashWithin_",
    "EventTime",
    "CensorTime",
    "Prob_",
]

feature_columns = [
    c for c in candidate_features
    if not any(fragment in c for fragment in forbidden_name_fragments)
]

if "Crash_Main" not in feature_columns:
    raise RuntimeError("Crash_Main should be available as current-state input.")

log(f"Selected {len(feature_columns)} leakage-screened numeric predictors.")

pd.DataFrame({
    "Feature": feature_columns
}).to_csv(
    TABLE_DIR / "Table_D02_Model_Features.csv",
    index=False
)

# =============================================================================
# 8. PANELIZE, EVENT-SAFE IMPUTE, TRAIN-ONLY STANDARDIZE
# =============================================================================

# Reindex each sector to the graph date order.
feature_panel_raw = np.full(
    (T, S, len(feature_columns)),
    np.nan,
    dtype=np.float64,
)

for s, sector in enumerate(sectors):
    sector_df = (
        master[master["Sector"] == sector]
        .set_index("Date")
        .reindex(dates)
    )

    feature_panel_raw[:, s, :] = (
        sector_df[feature_columns]
        .astype(float)
        .to_numpy()
    )

# -------------------------------------------------------------------------
# CORRECTION 2.1A — Crash_Main is an EVENT indicator and must never be
# forward-filled. Missing crash states are represented as Crash_Main = 0
# together with CurrentCrashObserved = 0. Thus the model can distinguish
# "observed non-crash" from "crash state unavailable" without propagating a
# previous crash across later ineligible sector-days.
# -------------------------------------------------------------------------

feature_panel_preimpute = feature_panel_raw.copy()

crash_feature_index = feature_columns.index("Crash_Main")
feature_panel_preimpute[:, :, crash_feature_index] = C.astype(np.float64)

if "CurrentCrashObserved" in feature_columns:
    crash_observed_feature_index = feature_columns.index(
        "CurrentCrashObserved"
    )
    feature_panel_preimpute[
        :, :, crash_observed_feature_index
    ] = M.astype(np.float64)
else:
    crash_observed_feature_index = None

# Past-only forward fill for CONTINUOUS / STATE predictors.
# Crash_Main and CurrentCrashObserved are already complete after the explicit
# event-safe assignment above, so they cannot be propagated by ffill.
feature_panel_ffill = feature_panel_preimpute.copy()

for s in range(S):
    frame = pd.DataFrame(
        feature_panel_ffill[:, s, :],
        columns=feature_columns,
    )
    feature_panel_ffill[:, s, :] = frame.ffill().to_numpy()

# Train-only medians for remaining gaps.
train_values = feature_panel_ffill[train_date_mask]
train_median = np.nanmedian(
    train_values,
    axis=(0, 1),
)

# If a feature is entirely missing in Train, drop it.
valid_feature_mask = np.isfinite(train_median)

if not valid_feature_mask.all():
    dropped = [
        feature_columns[i]
        for i in np.where(~valid_feature_mask)[0]
    ]
    log(
        f"Dropping all-missing Train features: {dropped}"
    )

    feature_columns = [
        feature_columns[i]
        for i in np.where(valid_feature_mask)[0]
    ]
    feature_panel_ffill = (
        feature_panel_ffill[:, :, valid_feature_mask]
    )
    train_median = train_median[valid_feature_mask]

# Missingness indicators use the ORIGINAL R/Python handoff missingness pattern.
# Crash_Main itself does not receive a generic MISS__ indicator because
# CurrentCrashObserved already carries precisely that information.
raw_after_feature_filter = (
    feature_panel_raw[:, :, valid_feature_mask]
)

raw_train = raw_after_feature_filter[
    train_date_mask
]

missing_rate = np.mean(
    ~np.isfinite(raw_train),
    axis=(0, 1),
)

missing_indicator_candidate = (
    missing_rate >= 0.02
)

for special_name in [
    "Crash_Main",
    "CurrentCrashObserved",
]:
    if special_name in feature_columns:
        missing_indicator_candidate[
            feature_columns.index(special_name)
        ] = False

missing_indicator_indices = np.where(
    missing_indicator_candidate
)[0]

missing_indicators = (
    ~np.isfinite(
        raw_after_feature_filter[
            :, :, missing_indicator_indices
        ]
    )
).astype(np.float64)

missing_indicator_names = [
    f"MISS__{feature_columns[i]}"
    for i in missing_indicator_indices
]

# Fill remaining missing values with Train medians.
X_numeric = feature_panel_ffill.copy()

for j in range(X_numeric.shape[2]):
    bad = ~np.isfinite(
        X_numeric[:, :, j]
    )
    X_numeric[:, :, j][bad] = train_median[j]

if len(missing_indicator_indices):
    X_numeric = np.concatenate(
        [
            X_numeric,
            missing_indicators,
        ],
        axis=2,
    )
    model_feature_names = (
        feature_columns
        + missing_indicator_names
    )
else:
    model_feature_names = (
        feature_columns.copy()
    )

# Train-only standardization.
train_block = X_numeric[train_date_mask]

train_mean = train_block.mean(
    axis=(0, 1)
)
train_std = train_block.std(
    axis=(0, 1)
)
train_std[
    train_std < 1e-8
] = 1.0

X_panel = (
    (
        X_numeric
        - train_mean[
            None, None, :
        ]
    )
    / train_std[
        None, None, :
    ]
).astype(np.float32)

F_DIM = X_panel.shape[2]

# Audit the special event imputation before scaling.
crash_after_special = feature_panel_preimpute[
    :, :, crash_feature_index
]
original_crash_missing = ~np.isfinite(
    feature_panel_raw[
        :, :, crash_feature_index
    ]
)

crash_missing_set_to_zero = bool(
    np.all(
        crash_after_special[
            original_crash_missing
        ] == 0
    )
)

if not crash_missing_set_to_zero:
    raise RuntimeError(
        "Crash_Main event-safe imputation failed."
    )

preprocess_bundle = {
    "features": model_feature_names,
    "base_features": feature_columns,
    "missing_indicator_features": missing_indicator_names,
    "train_median": train_median,
    "train_mean": train_mean,
    "train_std": train_std,
    "lookback": LOOKBACK,
    "sectors": sectors,
    "CrashMainImputation": (
        "Missing Crash_Main -> 0; "
        "CurrentCrashObserved -> 0; "
        "Crash_Main is never forward-filled"
    ),
}

with open(
    MODEL_DIR
    / "feature_preprocessing_bundle.pkl",
    "wb",
) as handle:
    pickle.dump(
        preprocess_bundle,
        handle,
    )

pd.DataFrame({
    "Feature": model_feature_names,
    "TrainMeanBeforeScaling": train_mean,
    "TrainStdBeforeScaling": train_std,
}).to_csv(
    TABLE_DIR
    / "Table_D03_Feature_Scaling.csv",
    index=False,
)

event_imputation_audit = pd.DataFrame({
    "Item": [
        "Original missing Crash_Main cells",
        "Missing Crash_Main cells set to zero",
        "Crash_Main forward-filled",
        "CurrentCrashObserved used",
    ],
    "Value": [
        int(original_crash_missing.sum()),
        int(
            (
                crash_after_special[
                    original_crash_missing
                ] == 0
            ).sum()
        ),
        "NO",
        "YES",
    ],
})

event_imputation_audit.to_csv(
    TABLE_DIR
    / "Table_D03A_Crash_Event_Imputation_Audit.csv",
    index=False,
)

log(
    f"Final neural feature dimension={F_DIM}; "
    f"missingness indicators="
    f"{len(missing_indicator_names)}."
)
log(
    "Crash_Main event-safe imputation applied: "
    "missing events set to zero and never forward-filled."
)

# =============================================================================
# 9. MAGNITUDE-PRESERVING GRAPH SCALING / ABLATION PRIORS
# =============================================================================
#
# CORRECTION 2.1B
# ----------------
# Phase 2 row-normalized each receiver row at each date. For a sparse Hawkes
# network, that operation can turn the only active incoming edge into weight 1
# irrespective of whether its raw excitation contribution is tiny or large.
#
# Here we preserve ABSOLUTE dynamic excitation magnitude. A single Train-only
# global scale is estimated from positive dynamic Hawkes weights:
#
#     q = Q_0.99(A_train | A_train > 0)
#
#     scaled(A) = log(1 + A/q) / log(2)
#
# Therefore q maps to 1, weaker states remain weak, stronger states remain
# stronger, and the exact same transformation is used for Train, Validation,
# Test, static Hawkes and random-placebo priors. No Test information enters q.

def clean_nonnegative_graph(
    A: np.ndarray,
) -> np.ndarray:
    A = np.asarray(
        A,
        dtype=np.float32,
    )

    return np.where(
        np.isfinite(A)
        & (A > 0),
        A,
        0.0,
    ).astype(np.float32)

raw_dynamic_train = clean_nonnegative_graph(
    dynamic_graph_train_fit[
        train_date_mask
    ]
)

positive_train_dynamic = raw_dynamic_train[
    raw_dynamic_train > 0
]

if len(positive_train_dynamic) == 0:
    raise RuntimeError(
        "No positive Train Hawkes graph weights available "
        "for magnitude-preserving scaling."
    )

GRAPH_TRAIN_Q = float(
    np.quantile(
        positive_train_dynamic,
        GRAPH_SCALE_QUANTILE,
    )
)

if not np.isfinite(GRAPH_TRAIN_Q) or GRAPH_TRAIN_Q <= 0:
    raise RuntimeError(
        "Invalid Train-only Hawkes graph scale reference."
    )

def scale_graph_magnitude(
    A: np.ndarray,
    reference: float = GRAPH_TRAIN_Q,
) -> np.ndarray:
    """
    Train-reference global log scaling. There is NO row normalization.
    """
    A = clean_nonnegative_graph(A)

    scaled = (
        np.log1p(
            A / max(reference, EPS)
        )
        / np.log(2.0)
    )

    scaled = np.clip(
        scaled,
        0.0,
        GRAPH_SCALE_CLIP,
    )

    return scaled.astype(np.float32)

# Dynamic split-aware Hawkes prior. The Phase-1 array already uses the
# Train-fitted process for Train/Validation and the frozen-structure
# Train+Validation coefficient refit for Test.
dynamic_graph_scaled = scale_graph_magnitude(
    dynamic_graph_all
)

# Static split-aware Hawkes graph, transformed with exactly the SAME q99
# reference estimated from Train dynamic excitation contributions.
static_graph_by_date = np.zeros(
    (T, S, S),
    dtype=np.float32,
)

static_train = scale_graph_magnitude(
    hawkes_train_alpha
)
static_final = scale_graph_magnitude(
    hawkes_final_alpha
)

static_graph_by_date[
    train_date_mask
] = static_train
static_graph_by_date[
    validation_date_mask
] = static_train
static_graph_by_date[
    test_date_mask
] = static_final

# No econometric graph prior.
zero_graph_by_date = np.zeros(
    (T, S, S),
    dtype=np.float32,
)

# Random static placebo:
# preserve the number of directed CROSS-sector edges in the stable Hawkes
# support and use the mean raw positive Hawkes alpha as the placebo edge
# magnitude before applying the SAME Train-derived graph transformation.
rng_graph = np.random.default_rng(
    SEED
)

cross_support = (
    (hawkes_train_alpha > 0)
    & (~np.eye(S, dtype=bool))
)

n_cross_edges = int(
    cross_support.sum()
)

all_cross_positions = [
    (i, j)
    for i in range(S)
    for j in range(S)
    if i != j
]

chosen_positions = rng_graph.choice(
    len(all_cross_positions),
    size=max(1, n_cross_edges),
    replace=False,
)

random_static_raw = np.zeros(
    (S, S),
    dtype=np.float32,
)

positive_cross_values = (
    hawkes_train_alpha[
        cross_support
    ]
)

random_raw_weight = (
    float(
        np.mean(
            positive_cross_values
        )
    )
    if len(positive_cross_values)
    else GRAPH_TRAIN_Q
)

for idx in np.atleast_1d(
    chosen_positions
):
    receiver, source = (
        all_cross_positions[
            int(idx)
        ]
    )
    random_static_raw[
        receiver,
        source,
    ] = random_raw_weight

random_static = scale_graph_magnitude(
    random_static_raw
)

random_graph_by_date = np.repeat(
    random_static[
        None, :, :
    ],
    T,
    axis=0,
)

# -------------------------------------------------------------------------
# Explicit node-level contagion-state summaries.
#
# Matrix convention is A[receiver, source].
#
# IncomingRisk_i   = sum_{j != i} A[i,j]
# OutgoingRisk_i   = sum_{r != i} A[r,i]
# MaxIncomingRisk_i= max_{j != i} A[i,j]
#
# These are derived INSIDE the graph model from whichever prior that ablation
# receives. NoGraph therefore receives exact zeros; random/static receive their
# corresponding controls; dynamic receives time-varying Hawkes state.
# -------------------------------------------------------------------------

GRAPH_STATE_FEATURE_NAMES = [
    "HawkesIncomingRisk",
    "HawkesOutgoingRisk",
    "HawkesMaxIncomingRisk",
]

def graph_state_numpy(
    graph_array: np.ndarray,
) -> np.ndarray:
    graph_array = np.asarray(
        graph_array,
        dtype=np.float32,
    )

    cross = graph_array.copy()

    diag = np.arange(S)
    cross[..., diag, diag] = 0.0

    incoming = cross.sum(
        axis=-1
    )
    outgoing = cross.sum(
        axis=-2
    )
    max_incoming = cross.max(
        axis=-1
    )

    return np.stack(
        [
            incoming,
            outgoing,
            max_incoming,
        ],
        axis=-1,
    ).astype(np.float32)

dynamic_graph_state = graph_state_numpy(
    dynamic_graph_scaled
)
static_graph_state = graph_state_numpy(
    static_graph_by_date
)
random_graph_state = graph_state_numpy(
    random_graph_by_date
)
zero_graph_state = graph_state_numpy(
    zero_graph_by_date
)

# Graph scaling audit.
graph_scaling_audit = pd.DataFrame({
    "Item": [
        "Scaling method",
        "Train-only graph quantile",
        "Train-only q reference",
        "Global clip",
        "Row normalization used",
        "Positive Train raw minimum",
        "Positive Train raw median",
        "Positive Train raw q90",
        "Positive Train raw q99",
        "Positive Train raw maximum",
        "Scaled Train positive median",
        "Scaled Train positive q90",
        "Scaled Train positive q99",
        "Scaled Train positive maximum",
    ],
    "Value": [
        "log1p(A/q_train)/log(2)",
        GRAPH_SCALE_QUANTILE,
        GRAPH_TRAIN_Q,
        GRAPH_SCALE_CLIP,
        "NO",
        float(np.min(positive_train_dynamic)),
        float(np.median(positive_train_dynamic)),
        float(np.quantile(positive_train_dynamic, 0.90)),
        float(np.quantile(positive_train_dynamic, 0.99)),
        float(np.max(positive_train_dynamic)),
        float(np.median(
            dynamic_graph_scaled[
                train_date_mask
            ][
                dynamic_graph_scaled[
                    train_date_mask
                ] > 0
            ]
        )),
        float(np.quantile(
            dynamic_graph_scaled[
                train_date_mask
            ][
                dynamic_graph_scaled[
                    train_date_mask
                ] > 0
            ],
            0.90,
        )),
        float(np.quantile(
            dynamic_graph_scaled[
                train_date_mask
            ][
                dynamic_graph_scaled[
                    train_date_mask
                ] > 0
            ],
            0.99,
        )),
        float(np.max(
            dynamic_graph_scaled[
                train_date_mask
            ]
        )),
    ],
})

graph_scaling_audit.to_csv(
    TABLE_DIR
    / "Table_D04A_Graph_Magnitude_Scaling_Audit.csv",
    index=False,
)

# Per-edge magnitude preservation diagnostics for stable cross-sector edges.
edge_scaling_rows = []

for receiver in range(S):
    for source in range(S):

        if receiver == source:
            continue

        raw_series = (
            dynamic_graph_train_fit[
                train_date_mask,
                receiver,
                source,
            ].astype(float)
        )

        scaled_series = (
            dynamic_graph_scaled[
                train_date_mask,
                receiver,
                source,
            ].astype(float)
        )

        if np.any(raw_series > 0):

            if (
                np.std(raw_series) > 0
                and np.std(scaled_series) > 0
            ):
                correlation = float(
                    np.corrcoef(
                        raw_series,
                        scaled_series,
                    )[0, 1]
                )
            else:
                correlation = np.nan

            edge_scaling_rows.append({
                "FromSector": sectors[source],
                "ToSector": sectors[receiver],
                "RawMinimum": float(np.min(raw_series)),
                "RawMedian": float(np.median(raw_series)),
                "RawMaximum": float(np.max(raw_series)),
                "ScaledMinimum": float(np.min(scaled_series)),
                "ScaledMedian": float(np.median(scaled_series)),
                "ScaledMaximum": float(np.max(scaled_series)),
                "RawScaledPearsonCorrelation": correlation,
                "UniqueScaledValues": int(
                    len(
                        np.unique(
                            np.round(
                                scaled_series,
                                8,
                            )
                        )
                    )
                ),
            })

edge_scaling_diagnostics = pd.DataFrame(
    edge_scaling_rows
)

edge_scaling_diagnostics.to_csv(
    TABLE_DIR
    / "Table_D04B_Edge_Magnitude_Preservation.csv",
    index=False,
)

graph_state_summary = pd.DataFrame({
    "Feature": GRAPH_STATE_FEATURE_NAMES,
    "TrainMean_Dynamic": [
        float(
            dynamic_graph_state[
                train_date_mask,
                :,
                k,
            ].mean()
        )
        for k in range(3)
    ],
    "TrainSD_Dynamic": [
        float(
            dynamic_graph_state[
                train_date_mask,
                :,
                k,
            ].std()
        )
        for k in range(3)
    ],
    "TrainMax_Dynamic": [
        float(
            dynamic_graph_state[
                train_date_mask,
                :,
                k,
            ].max()
        )
        for k in range(3)
    ],
})

graph_state_summary.to_csv(
    TABLE_DIR
    / "Table_D04C_Graph_State_Features.csv",
    index=False,
)

graph_ablation_summary = pd.DataFrame({
    "GraphVariant": [
        "NoGraph",
        "RandomStaticGraph",
        "StaticHawkesGraph",
        "DynamicHawkesGraph",
    ],
    "Description": [
        (
            "Zero econometric prior; graph attention remains data-learned; "
            "graph-state summaries are zero."
        ),
        (
            "Random directed prior with Hawkes cross-edge density; "
            "same Train-derived magnitude scale."
        ),
        (
            "Time-invariant stability-selected Hawkes coefficient prior; "
            "same Train-derived magnitude scale."
        ),
        (
            "Time-varying stability-selected Hawkes excitation prior; "
            "absolute excitation magnitude preserved."
        ),
    ],
    "CrossEdgesTrainPrior": [
        0,
        int(
            (
                random_static
                * (~np.eye(S, dtype=bool))
            > 0
            ).sum()
        ),
        int(
            (
                static_train
                * (~np.eye(S, dtype=bool))
            > 0
            ).sum()
        ),
        int(
            (
                np.any(
                    dynamic_graph_scaled[
                        train_date_mask
                    ] > 0,
                    axis=0,
                )
                & (~np.eye(S, dtype=bool))
            ).sum()
        ),
    ],
    "RowNormalized": [
        "NO",
        "NO",
        "NO",
        "NO",
    ],
    "ExplicitNodeGraphState": [
        "ZERO",
        "YES",
        "YES",
        "YES",
    ],
})

graph_ablation_summary.to_csv(
    TABLE_DIR
    / "Table_D04_Graph_Ablation_Design.csv",
    index=False,
)

log(
    "Magnitude-preserving graph scaling applied. "
    f"Train q{GRAPH_SCALE_QUANTILE:.2f}="
    f"{GRAPH_TRAIN_Q:.8f}; row normalization disabled."
)

# =============================================================================
# 10. SAMPLE INDEX CONSTRUCTION
# =============================================================================

# Sector-specific samples for XGBoost/LSTM/Transformer.
sector_sample_rows = []

for t in range(LOOKBACK - 1, T):
    split_name = splits[t]

    for s, sector in enumerate(sectors):

        if not M[t, s]:
            continue

        # Require at least one observed future risk day.
        if daily_hazard_mask[t, s].sum() <= 0:
            continue

        sector_sample_rows.append({
            "t": t,
            "s": s,
            "Date": dates[t],
            "Sector": sector,
            "Split": split_name,
        })

sector_samples = pd.DataFrame(sector_sample_rows)

# Graph samples are date-level and carry all sectors.
graph_sample_rows = []

for t in range(LOOKBACK - 1, T):
    if daily_hazard_mask[t].sum() <= 0:
        continue

    graph_sample_rows.append({
        "t": t,
        "Date": dates[t],
        "Split": splits[t],
    })

graph_samples = pd.DataFrame(graph_sample_rows)

sector_samples.to_csv(
    DATA_DIR / "sector_sample_index.csv.gz",
    index=False,
    compression="gzip",
)

graph_samples.to_csv(
    DATA_DIR / "graph_sample_index.csv.gz",
    index=False,
    compression="gzip",
)

log(
    f"Sector samples={len(sector_samples):,}; "
    f"graph-date samples={len(graph_samples):,}."
)

# =============================================================================
# 11. DATASET CLASSES
# =============================================================================

class SectorSequenceDataset(Dataset):
    def __init__(self, sample_frame: pd.DataFrame):
        self.rows = sample_frame.reset_index(drop=True)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        t = int(row["t"])
        s = int(row["s"])

        x = X_panel[t - LOOKBACK + 1:t + 1, s, :]
        y = daily_hazard_target[t, s, :]
        mask = daily_hazard_mask[t, s, :]

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(s, dtype=torch.long),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32),
            torch.tensor(t, dtype=torch.long),
        )

class GraphSequenceDataset(Dataset):
    def __init__(
        self,
        sample_frame: pd.DataFrame,
        graph_by_date: np.ndarray,
    ):
        self.rows = sample_frame.reset_index(drop=True)
        self.graph_by_date = graph_by_date

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        t = int(row["t"])

        # L x S x F
        x = X_panel[t - LOOKBACK + 1:t + 1, :, :]

        # L x S x S, aligned with each feature date.
        g = self.graph_by_date[
            t - LOOKBACK + 1:t + 1
        ]

        y = daily_hazard_target[t, :, :]
        mask = daily_hazard_mask[t, :, :]

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(g, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32),
            torch.tensor(t, dtype=torch.long),
        )

# =============================================================================
# 12. SURVIVAL LOSS / PROBABILITY UTILITIES
# =============================================================================

def compute_hazard_pos_weight() -> float:
    train_mask = train_date_mask[:, None, None]
    valid = (
        daily_hazard_mask > 0
    ) & train_mask

    y = daily_hazard_target[valid]

    positives = float(y.sum())
    negatives = float(len(y) - positives)

    if positives <= 0:
        return 1.0

    raw = math.sqrt(max(negatives / positives, 1.0))
    return float(np.clip(raw, 1.0, NEURAL_MAX_POS_WEIGHT))

HAZARD_POS_WEIGHT = compute_hazard_pos_weight()
log(f"Neural hazard positive-class weight={HAZARD_POS_WEIGHT:.4f}")

def masked_survival_bce(
    logits: torch.Tensor,
    targets: torch.Tensor,
    mask: torch.Tensor,
    pos_weight: float,
) -> torch.Tensor:
    base = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none",
    )

    weight = torch.where(
        targets > 0.5,
        torch.tensor(
            pos_weight,
            dtype=base.dtype,
            device=base.device,
        ),
        torch.tensor(
            1.0,
            dtype=base.dtype,
            device=base.device,
        ),
    )

    weighted = base * weight * mask
    denom = torch.clamp(mask.sum(), min=1.0)

    return weighted.sum() / denom

def masked_unweighted_survival_nll(
    logits: torch.Tensor,
    targets: torch.Tensor,
    mask: torch.Tensor,
) -> torch.Tensor:
    base = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none",
    )

    return (base * mask).sum() / torch.clamp(mask.sum(), min=1.0)

def hazard_logits_to_horizon_probs(
    logits: np.ndarray,
    temperature: float = 1.0,
    bias: float = 0.0,
) -> dict[int, np.ndarray]:
    calibrated_logits = logits / max(temperature, 1e-4) + bias
    hazard = 1.0 / (1.0 + np.exp(-np.clip(calibrated_logits, -30, 30)))

    survival = np.cumprod(1.0 - hazard, axis=-1)
    cumulative_event = 1.0 - survival

    return {
        h: cumulative_event[..., h - 1]
        for h in HORIZONS
    }

# =============================================================================
# 13. NEURAL MODEL DEFINITIONS
# =============================================================================

class LSTMSurvival(nn.Module):
    def __init__(
        self,
        feature_dim: int,
        n_sectors: int,
        hidden_dim: int = D_MODEL,
    ):
        super().__init__()

        self.sector_embedding = nn.Embedding(
            n_sectors,
            SECTOR_EMBED_DIM,
        )

        self.lstm = nn.LSTM(
            input_size=feature_dim + SECTOR_EMBED_DIM,
            hidden_size=hidden_dim,
            num_layers=2,
            dropout=DROPOUT,
            batch_first=True,
        )

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden_dim, MAX_HORIZON),
        )

    def forward(self, x, sector_idx):
        embedding = self.sector_embedding(sector_idx)
        embedding_seq = embedding[:, None, :].expand(
            -1,
            x.size(1),
            -1,
        )

        z = torch.cat([x, embedding_seq], dim=-1)
        output, _ = self.lstm(z)

        return self.head(output[:, -1, :])


class TemporalTransformerSurvival(nn.Module):
    def __init__(
        self,
        feature_dim: int,
        n_sectors: int,
        d_model: int = D_MODEL,
    ):
        super().__init__()

        self.input_projection = nn.Linear(
            feature_dim,
            d_model,
        )

        self.sector_embedding = nn.Embedding(
            n_sectors,
            d_model,
        )

        self.position_embedding = nn.Parameter(
            torch.zeros(1, LOOKBACK, d_model)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=N_HEADS,
            dim_feedforward=4 * d_model,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=N_TRANSFORMER_LAYERS,
        )

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(d_model, MAX_HORIZON),
        )

    def forward(self, x, sector_idx):
        z = self.input_projection(x)

        sector_emb = self.sector_embedding(
            sector_idx
        )[:, None, :]

        z = (
            z
            + sector_emb
            + self.position_embedding[:, :x.size(1), :]
        )

        z = self.encoder(z)

        return self.head(z[:, -1, :])


class SoftGraphAttention(nn.Module):
    """
    Multi-head node attention with an additive econometric graph bias.

    The prior is SOFT: non-edge pairs remain available to learned attention.
    A learnable positive eta controls how strongly the econometric graph
    influences attention scores.
    """
    def __init__(
        self,
        d_model: int,
        n_heads: int,
        dropout: float,
    ):
        super().__init__()

        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads.")

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

        # softplus(raw_eta) ensures eta >= 0.
        self.raw_eta = nn.Parameter(
            torch.tensor(0.0)
        )

    def forward(self, h, graph_prior):
        # h: B x L x S x D
        # graph_prior: B x L x S(receiver) x S(source)

        B, L, S_, D = h.shape

        q = self.q_proj(h).view(
            B, L, S_, self.n_heads, self.head_dim
        ).permute(0, 1, 3, 2, 4)

        k = self.k_proj(h).view(
            B, L, S_, self.n_heads, self.head_dim
        ).permute(0, 1, 3, 2, 4)

        v = self.v_proj(h).view(
            B, L, S_, self.n_heads, self.head_dim
        ).permute(0, 1, 3, 2, 4)

        # receiver i queries source j.
        scores = torch.einsum(
            "blhid,blhjd->blhij",
            q,
            k,
        ) / math.sqrt(self.head_dim)

        eta = F.softplus(self.raw_eta)

        graph_bias = graph_prior[:, :, None, :, :]
        scores = scores + eta * graph_bias

        attention = torch.softmax(scores, dim=-1)
        attention = self.dropout(attention)

        message = torch.einsum(
            "blhij,blhjd->blhid",
            attention,
            v,
        )

        message = message.permute(
            0, 1, 3, 2, 4
        ).contiguous().view(B, L, S_, D)

        return self.norm(
            h + self.out_proj(message)
        )


class GraphSurvivalTransformer(nn.Module):
    """
    Temporal graph survival Transformer with TWO econometric graph channels:

      1. a soft additive edge prior in cross-sector attention;
      2. explicit node-level contagion-state summaries:
           incoming risk,
           outgoing risk,
           maximum incoming risk.

    The same architecture is used for NoGraph / Random / Static / Dynamic
    ablations. Only graph_prior changes. For NoGraph all three state summaries
    are identically zero.
    """

    def __init__(
        self,
        feature_dim: int,
        n_sectors: int,
        d_model: int = D_MODEL,
    ):
        super().__init__()

        self.input_projection = nn.Linear(
            feature_dim,
            d_model,
        )

        self.node_embedding = nn.Embedding(
            n_sectors,
            d_model,
        )

        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                LOOKBACK,
                1,
                d_model,
            )
        )

        self.graph_attention = SoftGraphAttention(
            d_model=d_model,
            n_heads=N_HEADS,
            dropout=DROPOUT,
        )

        # Bias=False is deliberate: a zero graph prior must inject exactly zero
        # graph-state signal in the NoGraph ablation.
        self.graph_state_projection = nn.Linear(
            3,
            d_model,
            bias=False,
        )

        # Non-negative learnable gate for explicit graph-state summaries.
        self.raw_graph_state_eta = nn.Parameter(
            torch.tensor(0.0)
        )

        temporal_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=N_HEADS,
            dim_feedforward=4 * d_model,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.temporal_encoder = nn.TransformerEncoder(
            temporal_layer,
            num_layers=N_TRANSFORMER_LAYERS,
        )

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(
                d_model,
                MAX_HORIZON,
            ),
        )

    @staticmethod
    def graph_state_features(
        graph_prior: torch.Tensor,
    ) -> torch.Tensor:
        """
        graph_prior:
            B x L x S(receiver) x S(source)

        Returns:
            B x L x S x 3
            [incoming, outgoing, max incoming], excluding diagonal edges.
        """

        S_ = graph_prior.size(-1)

        eye = torch.eye(
            S_,
            dtype=graph_prior.dtype,
            device=graph_prior.device,
        )

        cross = (
            graph_prior
            * (
                1.0
                - eye[
                    None,
                    None,
                    :, :,
                ]
            )
        )

        incoming = cross.sum(
            dim=-1
        )

        outgoing = cross.sum(
            dim=-2
        )

        max_incoming = cross.max(
            dim=-1
        ).values

        return torch.stack(
            [
                incoming,
                outgoing,
                max_incoming,
            ],
            dim=-1,
        )

    def forward(
        self,
        x,
        graph_prior,
    ):
        # x: B x L x S x F
        # graph_prior: B x L x S(receiver) x S(source)

        B, L, S_, _ = x.shape

        z = self.input_projection(x)

        node_ids = torch.arange(
            S_,
            device=x.device,
        )

        node_emb = self.node_embedding(
            node_ids
        )[
            None,
            None,
            :,
            :,
        ]

        z = (
            z
            + node_emb
            + self.position_embedding[
                :, :L, :, :
            ]
        )

        # Explicit node-level econometric contagion state.
        graph_state = self.graph_state_features(
            graph_prior
        )

        graph_state_embedding = (
            self.graph_state_projection(
                graph_state
            )
        )

        graph_state_eta = F.softplus(
            self.raw_graph_state_eta
        )

        z = (
            z
            + graph_state_eta
            * graph_state_embedding
        )

        # Cross-sector attention with magnitude-preserving soft graph bias.
        z = self.graph_attention(
            z,
            graph_prior,
        )

        # Temporal Transformer independently for each sector after graph mixing.
        z = z.permute(
            0,
            2,
            1,
            3,
        ).contiguous().view(
            B * S_,
            L,
            -1,
        )

        z = self.temporal_encoder(
            z
        )

        last = z[:, -1, :]

        logits = self.head(
            last
        ).view(
            B,
            S_,
            MAX_HORIZON,
        )

        return logits


# =============================================================================
# 14. TRAIN / VALIDATION HELPERS
# =============================================================================

@dataclass
class TrainResult:
    best_state: dict
    best_epoch: int
    best_val_nll: float
    history: pd.DataFrame

def make_sector_loader(
    frame: pd.DataFrame,
    shuffle: bool,
) -> DataLoader:
    return DataLoader(
        SectorSequenceDataset(frame),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )

def make_graph_loader(
    frame: pd.DataFrame,
    graph_by_date: np.ndarray,
    shuffle: bool,
) -> DataLoader:
    return DataLoader(
        GraphSequenceDataset(frame, graph_by_date),
        batch_size=GRAPH_BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )

def train_sector_neural_model(
    model: nn.Module,
    model_name: str,
) -> TrainResult:

    train_frame = sector_samples[
        sector_samples["Split"] == "Train"
    ]
    val_frame = sector_samples[
        sector_samples["Split"] == "Validation"
    ]

    train_loader = make_sector_loader(
        train_frame,
        shuffle=True,
    )
    val_loader = make_sector_loader(
        val_frame,
        shuffle=False,
    )

    model = model.to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_state = None
    best_epoch = 0
    best_val = np.inf
    patience_counter = 0
    history_rows = []

    for epoch in range(1, MAX_EPOCHS + 1):

        model.train()
        train_loss_num = 0.0
        train_mask_num = 0.0

        for x, sector_idx, y, mask, _ in train_loader:

            x = x.to(DEVICE)
            sector_idx = sector_idx.to(DEVICE)
            y = y.to(DEVICE)
            mask = mask.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            logits = model(x, sector_idx)

            loss = masked_survival_bce(
                logits,
                y,
                mask,
                HAZARD_POS_WEIGHT,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

            nmask = float(mask.sum().item())
            train_loss_num += float(loss.item()) * nmask
            train_mask_num += nmask

        model.eval()
        val_loss_num = 0.0
        val_mask_num = 0.0

        with torch.no_grad():
            for x, sector_idx, y, mask, _ in val_loader:

                x = x.to(DEVICE)
                sector_idx = sector_idx.to(DEVICE)
                y = y.to(DEVICE)
                mask = mask.to(DEVICE)

                logits = model(x, sector_idx)

                val_loss = masked_unweighted_survival_nll(
                    logits,
                    y,
                    mask,
                )

                nmask = float(mask.sum().item())
                val_loss_num += float(val_loss.item()) * nmask
                val_mask_num += nmask

        train_loss_epoch = (
            train_loss_num / max(train_mask_num, 1.0)
        )
        val_nll_epoch = (
            val_loss_num / max(val_mask_num, 1.0)
        )

        history_rows.append({
            "Model": model_name,
            "Epoch": epoch,
            "TrainWeightedLoss": train_loss_epoch,
            "ValidationNLL": val_nll_epoch,
        })

        log(
            f"{model_name}: epoch={epoch:02d}, "
            f"train={train_loss_epoch:.5f}, "
            f"valNLL={val_nll_epoch:.5f}"
        )

        if val_nll_epoch < best_val - 1e-5:
            best_val = val_nll_epoch
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            break

    if best_state is None:
        raise RuntimeError(
            f"No best state captured for {model_name}."
        )

    model.load_state_dict(best_state)

    return TrainResult(
        best_state=best_state,
        best_epoch=best_epoch,
        best_val_nll=best_val,
        history=pd.DataFrame(history_rows),
    )

def train_graph_neural_model(
    model: nn.Module,
    model_name: str,
    graph_by_date: np.ndarray,
) -> TrainResult:

    train_frame = graph_samples[
        graph_samples["Split"] == "Train"
    ]
    val_frame = graph_samples[
        graph_samples["Split"] == "Validation"
    ]

    train_loader = make_graph_loader(
        train_frame,
        graph_by_date,
        shuffle=True,
    )
    val_loader = make_graph_loader(
        val_frame,
        graph_by_date,
        shuffle=False,
    )

    model = model.to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_state = None
    best_epoch = 0
    best_val = np.inf
    patience_counter = 0
    history_rows = []

    for epoch in range(1, MAX_EPOCHS + 1):

        model.train()
        train_loss_num = 0.0
        train_mask_num = 0.0

        for x, graph_prior, y, mask, _ in train_loader:

            x = x.to(DEVICE)
            graph_prior = graph_prior.to(DEVICE)
            y = y.to(DEVICE)
            mask = mask.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            logits = model(
                x,
                graph_prior,
            )

            loss = masked_survival_bce(
                logits,
                y,
                mask,
                HAZARD_POS_WEIGHT,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

            nmask = float(mask.sum().item())
            train_loss_num += float(loss.item()) * nmask
            train_mask_num += nmask

        model.eval()
        val_loss_num = 0.0
        val_mask_num = 0.0

        with torch.no_grad():
            for x, graph_prior, y, mask, _ in val_loader:

                x = x.to(DEVICE)
                graph_prior = graph_prior.to(DEVICE)
                y = y.to(DEVICE)
                mask = mask.to(DEVICE)

                logits = model(
                    x,
                    graph_prior,
                )

                val_loss = masked_unweighted_survival_nll(
                    logits,
                    y,
                    mask,
                )

                nmask = float(mask.sum().item())
                val_loss_num += float(val_loss.item()) * nmask
                val_mask_num += nmask

        train_loss_epoch = (
            train_loss_num / max(train_mask_num, 1.0)
        )
        val_nll_epoch = (
            val_loss_num / max(val_mask_num, 1.0)
        )

        history_rows.append({
            "Model": model_name,
            "Epoch": epoch,
            "TrainWeightedLoss": train_loss_epoch,
            "ValidationNLL": val_nll_epoch,
        })

        log(
            f"{model_name}: epoch={epoch:02d}, "
            f"train={train_loss_epoch:.5f}, "
            f"valNLL={val_nll_epoch:.5f}"
        )

        if val_nll_epoch < best_val - 1e-5:
            best_val = val_nll_epoch
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            break

    if best_state is None:
        raise RuntimeError(
            f"No best state captured for {model_name}."
        )

    model.load_state_dict(best_state)

    return TrainResult(
        best_state=best_state,
        best_epoch=best_epoch,
        best_val_nll=best_val,
        history=pd.DataFrame(history_rows),
    )

# =============================================================================
# 15. NEURAL PREDICTION / CALIBRATION
# =============================================================================

def predict_sector_logits(
    model: nn.Module,
    split_name: str,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:

    frame = sector_samples[
        sector_samples["Split"] == split_name
    ].copy()

    loader = make_sector_loader(
        frame,
        shuffle=False,
    )

    model = model.to(DEVICE)
    model.eval()

    logits_list = []
    y_list = []
    mask_list = []
    t_list = []
    s_list = []

    cursor = 0

    with torch.no_grad():
        for x, sector_idx, y, mask, t in loader:

            x = x.to(DEVICE)
            sector_idx_device = sector_idx.to(DEVICE)

            logits = model(
                x,
                sector_idx_device,
            ).cpu().numpy()

            logits_list.append(logits)
            y_list.append(y.numpy())
            mask_list.append(mask.numpy())
            t_list.append(t.numpy())
            s_list.append(sector_idx.numpy())
            cursor += len(t)

    return (
        np.concatenate(logits_list, axis=0),
        np.concatenate(y_list, axis=0),
        np.concatenate(mask_list, axis=0),
        np.column_stack([
            np.concatenate(t_list),
            np.concatenate(s_list),
        ]),
    )

def predict_graph_logits(
    model: nn.Module,
    split_name: str,
    graph_by_date: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:

    frame = graph_samples[
        graph_samples["Split"] == split_name
    ].copy()

    loader = make_graph_loader(
        frame,
        graph_by_date,
        shuffle=False,
    )

    model = model.to(DEVICE)
    model.eval()

    logits_list = []
    y_list = []
    mask_list = []
    t_list = []

    with torch.no_grad():
        for x, graph_prior, y, mask, t in loader:

            x = x.to(DEVICE)
            graph_prior = graph_prior.to(DEVICE)

            logits = model(
                x,
                graph_prior,
            ).cpu().numpy()

            logits_list.append(logits)
            y_list.append(y.numpy())
            mask_list.append(mask.numpy())
            t_list.append(t.numpy())

    return (
        np.concatenate(logits_list, axis=0),
        np.concatenate(y_list, axis=0),
        np.concatenate(mask_list, axis=0),
        np.concatenate(t_list),
    )

def fit_hazard_temperature(
    logits: np.ndarray,
    targets: np.ndarray,
    mask: np.ndarray,
) -> tuple[float, float]:

    valid = mask > 0

    z = logits[valid].astype(float)
    y = targets[valid].astype(float)

    if len(y) == 0:
        return 1.0, 0.0

    def objective(theta):
        log_temperature, bias = theta
        temperature = np.exp(log_temperature)

        zc = z / temperature + bias
        p = 1.0 / (
            1.0 + np.exp(-np.clip(zc, -30, 30))
        )
        p = np.clip(p, EPS, 1.0 - EPS)

        return float(
            -np.mean(
                y * np.log(p)
                + (1.0 - y) * np.log1p(-p)
            )
        )

    result = minimize(
        objective,
        x0=np.array([0.0, 0.0]),
        method="L-BFGS-B",
        bounds=[
            (-3.0, 3.0),
            (-5.0, 5.0),
        ],
    )

    if not result.success:
        log(
            f"WARNING: hazard calibration optimizer: {result.message}"
        )

    temperature = float(
        np.exp(result.x[0])
    )
    bias = float(result.x[1])

    return temperature, bias

# =============================================================================
# 16. XGBOOST FEATURE ENGINEERING
# =============================================================================

# Summary windows reduce the 60-day sequence to robust tabular statistics.
XGB_WINDOWS = [5, 22, 60]

def xgb_feature_vector(t: int, s: int) -> np.ndarray:
    sequence = X_panel[
        t - LOOKBACK + 1:t + 1,
        s,
        :,
    ]

    pieces = [
        sequence[-1],  # current state
    ]

    for window in XGB_WINDOWS:
        block = sequence[-window:]
        pieces.extend([
            block.mean(axis=0),
            block.std(axis=0),
            block.min(axis=0),
            block.max(axis=0),
        ])

    # Sector one-hot.
    one_hot = np.zeros(S, dtype=np.float32)
    one_hot[s] = 1.0
    pieces.append(one_hot)

    return np.concatenate(pieces).astype(np.float32)

log("Building XGBoost summary-feature matrix...")

X_xgb = np.vstack([
    xgb_feature_vector(
        int(row.t),
        int(row.s),
    )
    for row in sector_samples.itertuples(index=False)
])

xgb_split = sector_samples["Split"].to_numpy()
xgb_t = sector_samples["t"].to_numpy(dtype=int)
xgb_s = sector_samples["s"].to_numpy(dtype=int)

# =============================================================================
# 17. XGBOOST TRAINING + HORIZON CALIBRATION
# =============================================================================

def fit_binary_platt(
    p_validation: np.ndarray,
    y_validation: np.ndarray,
) -> tuple[float, float]:

    p_validation = np.clip(
        p_validation,
        EPS,
        1.0 - EPS,
    )

    x = np.log(
        p_validation / (1.0 - p_validation)
    )
    y = y_validation.astype(float)

    def objective(theta):
        intercept, slope = theta
        z = intercept + slope * x
        q = 1.0 / (
            1.0 + np.exp(-np.clip(z, -30, 30))
        )
        q = np.clip(q, EPS, 1.0 - EPS)

        return float(
            -np.mean(
                y * np.log(q)
                + (1.0 - y) * np.log1p(-q)
            )
        )

    result = minimize(
        objective,
        x0=np.array([0.0, 1.0]),
        method="L-BFGS-B",
        bounds=[
            (-10.0, 10.0),
            (0.0, 10.0),
        ],
    )

    return float(result.x[0]), float(result.x[1])

def apply_binary_platt(
    p: np.ndarray,
    intercept: float,
    slope: float,
) -> np.ndarray:

    p = np.clip(p, EPS, 1.0 - EPS)
    x = np.log(p / (1.0 - p))
    z = intercept + slope * x

    return 1.0 / (
        1.0 + np.exp(-np.clip(z, -30, 30))
    )

xgb_models = {}
xgb_calibration = {}
xgb_predictions = {
    "Validation": {},
    "Test": {},
}

for h in HORIZONS:
    log(f"Training XGBoost horizon={h}...")

    # Build split-aware target per sector sample.
    y_all = np.array([
        horizon_target[h][t, s]
        for t, s in zip(xgb_t, xgb_s)
    ], dtype=float)

    train_idx = (
        (xgb_split == "Train")
        & np.isfinite(y_all)
    )
    val_idx = (
        (xgb_split == "Validation")
        & np.isfinite(y_all)
    )
    test_idx = (
        (xgb_split == "Test")
        & np.isfinite(y_all)
    )

    y_train = y_all[train_idx].astype(int)
    y_val = y_all[val_idx].astype(int)

    positive = max(int(y_train.sum()), 1)
    negative = max(len(y_train) - positive, 1)
    scale_pos_weight = min(
        math.sqrt(negative / positive),
        XGB_MAX_POS_WEIGHT,
    )

    dtrain = xgb.DMatrix(
        X_xgb[train_idx],
        label=y_train,
    )
    dval = xgb.DMatrix(
        X_xgb[val_idx],
        label=y_val,
    )

    params = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "eta": 0.03,
        "max_depth": 4,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "lambda": 1.0,
        "alpha": 0.0,
        "scale_pos_weight": scale_pos_weight,
        "seed": SEED,
        "nthread": max(1, os.cpu_count() or 1),
        "tree_method": "hist",
    }

    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=XGB_MAX_ROUNDS,
        evals=[(dval, "validation")],
        early_stopping_rounds=XGB_EARLY_STOP,
        verbose_eval=False,
    )

    best_iteration = int(
        booster.best_iteration
        if booster.best_iteration is not None
        else XGB_MAX_ROUNDS - 1
    )

    p_val_raw = booster.predict(
        dval,
        iteration_range=(0, best_iteration + 1),
    )

    intercept, slope = fit_binary_platt(
        p_val_raw,
        y_val,
    )

    p_val = apply_binary_platt(
        p_val_raw,
        intercept,
        slope,
    )

    dtest = xgb.DMatrix(
        X_xgb[test_idx]
    )

    p_test_raw = booster.predict(
        dtest,
        iteration_range=(0, best_iteration + 1),
    )

    p_test = apply_binary_platt(
        p_test_raw,
        intercept,
        slope,
    )

    xgb_models[h] = booster
    xgb_calibration[h] = {
        "intercept": intercept,
        "slope": slope,
        "best_iteration": best_iteration,
        "scale_pos_weight": scale_pos_weight,
    }

    # Store by sample coordinate for later panel assembly.
    xgb_predictions["Validation"][h] = {
        "t": xgb_t[val_idx],
        "s": xgb_s[val_idx],
        "p": p_val,
    }
    xgb_predictions["Test"][h] = {
        "t": xgb_t[test_idx],
        "s": xgb_s[test_idx],
        "p": p_test,
    }

    booster.save_model(
        MODEL_DIR / f"xgboost_h{h}.json"
    )

with open(
    MODEL_DIR / "xgboost_calibration.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        xgb_calibration,
        handle,
        indent=2,
    )

# Assemble monotone XGBoost probabilities into T x S matrices.
xgb_panel_probs = {
    split_name: {
        h: np.full((T, S), np.nan, dtype=np.float32)
        for h in HORIZONS
    }
    for split_name in ["Validation", "Test"]
}

for split_name in ["Validation", "Test"]:

    # Coordinate union for this split.
    coordinate_set = set()

    for h in HORIZONS:
        record = xgb_predictions[split_name][h]

        for t, s, p in zip(
            record["t"],
            record["s"],
            record["p"],
        ):
            xgb_panel_probs[split_name][h][t, s] = p
            coordinate_set.add((int(t), int(s)))

    # Monotone rearrangement across horizons at common coordinates.
    for t, s in coordinate_set:
        values = np.array([
            xgb_panel_probs[split_name][h][t, s]
            for h in HORIZONS
        ])

        if np.all(np.isfinite(values)):
            values = np.maximum.accumulate(values)

            for h, value in zip(HORIZONS, values):
                xgb_panel_probs[split_name][h][t, s] = value

log("XGBoost benchmark training complete.")

# =============================================================================

# =============================================================================
# 18. PHASE 2.2 — MULTI-SEED ROBUSTNESS DESIGN
# =============================================================================
#
# Architecture is FROZEN at the accepted Phase-2.1 specification.
# This phase does NOT tune architecture using Test results.
#
# Five paired seeds are used by default. For every seed:
#   * the same seed is reset before each neural architecture is initialized;
#   * graph ablations therefore begin from paired random initializations;
#   * the random-graph placebo is independently redrawn for that seed while
#     preserving Hawkes cross-edge density and the empirical edge-weight multiset;
#   * early stopping and temperature calibration use Validation only;
#   * Test is evaluated once after the seed-specific model is fixed.
#
# The ensemble prediction is the arithmetic mean of the five separately
# calibrated probability forecasts. No Test-dependent ensemble weighting is used.

DEFAULT_MULTI_SEEDS = [
    20260901,
    20260902,
    20260903,
    20260904,
    20260905,
]

_seed_env = os.getenv("NSE_MULTI_SEEDS", "").strip()

if _seed_env:
    MULTI_SEEDS = [
        int(x.strip())
        for x in _seed_env.split(",")
        if x.strip()
    ]
else:
    MULTI_SEEDS = DEFAULT_MULTI_SEEDS.copy()

if len(MULTI_SEEDS) < 3:
    raise RuntimeError(
        "Phase 2.2 requires at least three independent seeds; "
        "five are recommended and used by default."
    )

if len(set(MULTI_SEEDS)) != len(MULTI_SEEDS):
    raise RuntimeError("MULTI_SEEDS contains duplicates.")

N_SEEDS = len(MULTI_SEEDS)

log(
    f"Phase 2.2 seeds ({N_SEEDS}): {MULTI_SEEDS}"
)

if DEVICE.type == "cpu":
    log(
        "NOTE: 30 neural fits will be estimated by default. "
        "Google Colab GPU is strongly recommended for speed."
    )

def set_all_seeds(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    try:
        torch.use_deterministic_algorithms(
            True,
            warn_only=True,
        )
    except Exception:
        pass

# =============================================================================
# 19. RANDOM-GRAPH PLACEBO GENERATOR — ONE PLACEBO PER SEED
# =============================================================================
#
# Stronger placebo than Phase 2.1:
#   * same number of directed cross-sector edges as stable Hawkes support;
#   * same empirical raw Hawkes cross-edge magnitudes (permuted across randomly
#     selected receiver/source pairs);
#   * same Train-derived magnitude transform;
#   * graph is static through time within a seed;
#   * support is independently redrawn across seeds.
#
# This tests whether the econometric topology matters beyond "having a sparse
# graph of approximately the same strength."

def build_random_graph_for_seed(
    seed: int,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:

    rng = np.random.default_rng(seed + 41011)

    all_positions = [
        (receiver, source)
        for receiver in range(S)
        for source in range(S)
        if receiver != source
    ]

    original_cross_positions = [
        (receiver, source)
        for receiver in range(S)
        for source in range(S)
        if (
            receiver != source
            and cross_support[
                receiver,
                source,
            ]
        )
    ]

    n_edges = max(
        1,
        len(original_cross_positions),
    )

    chosen = rng.choice(
        len(all_positions),
        size=n_edges,
        replace=False,
    )

    # Preserve the Hawkes self-excitation diagonal exactly. The placebo
    # randomizes only cross-sector topology, which makes Random vs Static a
    # cleaner test of whether the ECONOMETRIC cross-sector locations matter.
    train_raw = np.zeros(
        (S, S),
        dtype=np.float32,
    )
    final_raw = np.zeros(
        (S, S),
        dtype=np.float32,
    )

    diag = np.arange(S)

    train_raw[
        diag,
        diag,
    ] = np.diag(
        hawkes_train_alpha
    )

    final_raw[
        diag,
        diag,
    ] = np.diag(
        hawkes_final_alpha
    )

    train_cross_weights = np.asarray(
        [
            hawkes_train_alpha[
                receiver,
                source,
            ]
            for (
                receiver,
                source,
            ) in original_cross_positions
        ],
        dtype=np.float32,
    )

    final_cross_weights = np.asarray(
        [
            hawkes_final_alpha[
                receiver,
                source,
            ]
            for (
                receiver,
                source,
            ) in original_cross_positions
        ],
        dtype=np.float32,
    )

    if len(
        train_cross_weights
    ) == 0:

        train_cross_weights = np.array(
            [GRAPH_TRAIN_Q],
            dtype=np.float32,
        )

        final_cross_weights = np.array(
            [GRAPH_TRAIN_Q],
            dtype=np.float32,
        )

    permutation = rng.permutation(
        len(train_cross_weights)
    )

    rows = []

    for edge_number, position_index in enumerate(
        chosen
    ):

        receiver, source = all_positions[
            int(position_index)
        ]

        weight_index = int(
            permutation[
                edge_number
                % len(permutation)
            ]
        )

        train_weight = float(
            train_cross_weights[
                weight_index
            ]
        )

        final_weight = float(
            final_cross_weights[
                weight_index
            ]
        )

        train_raw[
            receiver,
            source,
        ] = train_weight

        final_raw[
            receiver,
            source,
        ] = final_weight

        rows.append({
            "Seed": seed,
            "EdgeNumber": edge_number + 1,
            "FromSector": sectors[source],
            "ToSector": sectors[receiver],
            "TrainRawPlaceboWeight": train_weight,
            "TestRefitRawPlaceboWeight": final_weight,
            "PreservesHawkesDiagonal": "YES",
        })

    train_scaled = scale_graph_magnitude(
        train_raw
    )

    final_scaled = scale_graph_magnitude(
        final_raw
    )

    by_date = np.zeros(
        (T, S, S),
        dtype=np.float32,
    )

    by_date[
        train_date_mask
    ] = train_scaled

    by_date[
        validation_date_mask
    ] = train_scaled

    by_date[
        test_date_mask
    ] = final_scaled

    return (
        by_date,
        train_scaled,
        pd.DataFrame(rows),
    )

# =============================================================================
# 20. COMMON FORECAST METRICS
# =============================================================================

def initialize_probability_dict():
    return {
        split_name: {
            h: np.full(
                (T, S),
                np.nan,
                dtype=np.float32,
            )
            for h in HORIZONS
        }
        for split_name in [
            "Validation",
            "Test",
        ]
    }

def log_score(
    y: np.ndarray,
    p: np.ndarray,
) -> float:

    p = np.clip(
        p,
        EPS,
        1.0 - EPS,
    )

    return float(
        -np.mean(
            y * np.log(p)
            + (1.0 - y) * np.log1p(-p)
        )
    )

def fit_calibration_intercept_slope(
    y: np.ndarray,
    p: np.ndarray,
) -> tuple[float, float]:

    p = np.clip(
        p,
        EPS,
        1.0 - EPS,
    )

    x = np.log(
        p / (1.0 - p)
    )

    if len(np.unique(y)) < 2:
        return np.nan, np.nan

    def objective(theta):
        intercept, slope = theta
        z = intercept + slope * x

        q = 1.0 / (
            1.0
            + np.exp(
                -np.clip(
                    z,
                    -30,
                    30,
                )
            )
        )

        q = np.clip(
            q,
            EPS,
            1.0 - EPS,
        )

        return float(
            -np.mean(
                y * np.log(q)
                + (1.0 - y)
                * np.log1p(-q)
            )
        )

    result = minimize(
        objective,
        x0=np.array(
            [0.0, 1.0]
        ),
        method="L-BFGS-B",
        bounds=[
            (-20.0, 20.0),
            (-10.0, 10.0),
        ],
    )

    return (
        float(result.x[0]),
        float(result.x[1]),
    )

def probability_metrics(
    y_true: np.ndarray,
    probability: np.ndarray,
) -> dict:

    y_true = np.asarray(
        y_true,
        dtype=float,
    )
    probability = np.asarray(
        probability,
        dtype=float,
    )

    ok = (
        np.isfinite(y_true)
        & np.isfinite(probability)
    )

    y = y_true[ok]
    p = np.clip(
        probability[ok],
        EPS,
        1.0 - EPS,
    )

    result = {
        "N": int(len(y)),
        "Events": int(y.sum()) if len(y) else 0,
        "EventRate": (
            float(y.mean())
            if len(y)
            else np.nan
        ),
        "Brier": np.nan,
        "LogScore": np.nan,
        "PR_AUC": np.nan,
        "ROC_AUC": np.nan,
        "CalibrationIntercept": np.nan,
        "CalibrationSlope": np.nan,
    }

    if len(y) == 0:
        return result

    result["Brier"] = float(
        np.mean(
            (p - y) ** 2
        )
    )

    result["LogScore"] = log_score(
        y,
        p,
    )

    if len(np.unique(y)) == 2:

        result["PR_AUC"] = float(
            average_precision_score(
                y,
                p,
            )
        )

        result["ROC_AUC"] = float(
            roc_auc_score(
                y,
                p,
            )
        )

        intercept, slope = (
            fit_calibration_intercept_slope(
                y,
                p,
            )
        )

        result[
            "CalibrationIntercept"
        ] = intercept

        result[
            "CalibrationSlope"
        ] = slope

    return result

# =============================================================================
# 21. PREDICTION HELPERS
# =============================================================================

def calibrated_sector_predictions(
    model: nn.Module,
) -> tuple[
    dict,
    dict,
]:

    val_logits, val_y, val_mask, _ = (
        predict_sector_logits(
            model,
            "Validation",
        )
    )

    temperature, bias = fit_hazard_temperature(
        val_logits,
        val_y,
        val_mask,
    )

    probabilities = initialize_probability_dict()

    for split_name in [
        "Validation",
        "Test",
    ]:

        logits, _, _, coordinates = (
            predict_sector_logits(
                model,
                split_name,
            )
        )

        horizon_probs = (
            hazard_logits_to_horizon_probs(
                logits,
                temperature,
                bias,
            )
        )

        for row_index, (
            t,
            s,
        ) in enumerate(
            coordinates.astype(int)
        ):

            for h in HORIZONS:
                probabilities[
                    split_name
                ][h][t, s] = (
                    horizon_probs[h][
                        row_index
                    ]
                )

    calibration = {
        "temperature": temperature,
        "bias": bias,
    }

    return probabilities, calibration

def calibrated_graph_predictions(
    model: nn.Module,
    graph_array: np.ndarray,
) -> tuple[
    dict,
    dict,
]:

    (
        val_logits,
        val_y,
        val_mask,
        _,
    ) = predict_graph_logits(
        model,
        "Validation",
        graph_array,
    )

    temperature, bias = fit_hazard_temperature(
        val_logits,
        val_y,
        val_mask,
    )

    eta = float(
        F.softplus(
            model.graph_attention.raw_eta
            .detach()
            .cpu()
        ).item()
    )

    probabilities = initialize_probability_dict()

    for split_name in [
        "Validation",
        "Test",
    ]:

        (
            logits,
            _,
            _,
            t_values,
        ) = predict_graph_logits(
            model,
            split_name,
            graph_array,
        )

        horizon_probs = (
            hazard_logits_to_horizon_probs(
                logits,
                temperature,
                bias,
            )
        )

        for row_index, t in enumerate(
            t_values.astype(int)
        ):

            for s in range(S):

                for h in HORIZONS:
                    probabilities[
                        split_name
                    ][h][t, s] = (
                        horizon_probs[
                            h
                        ][
                            row_index,
                            s,
                        ]
                    )

    calibration = {
        "temperature": temperature,
        "bias": bias,
        "learned_graph_eta": eta,
    }

    return probabilities, calibration

# =============================================================================
# 22. FIVE-SEED NEURAL ESTIMATION
# =============================================================================

NEURAL_MODEL_NAMES = [
    "LSTM_Survival",
    "TemporalTransformer_Survival",
    "NoGraphTransformer",
    "RandomGraphTransformer",
    "StaticHawkesGraphTransformer",
    "DynamicHawkesGraphTransformer",
]

seed_probability_store = {}
training_histories = []
best_epoch_rows = []
calibration_rows = []
random_placebo_rows = []

for seed_index, seed in enumerate(
    MULTI_SEEDS,
    start=1,
):

    log(
        "=" * 90
    )
    log(
        f"SEED {seed_index}/{N_SEEDS}: {seed}"
    )

    seed_probability_store[
        seed
    ] = {}

    seed_dir = (
        MODEL_DIR
        / f"Seed_{seed}"
    )
    seed_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # ---------------------------------------------------------------------
    # 22.1 LSTM
    # ---------------------------------------------------------------------
    set_all_seeds(seed)

    model = LSTMSurvival(
        feature_dim=F_DIM,
        n_sectors=S,
    )

    result = train_sector_neural_model(
        model,
        f"LSTM_Survival__Seed_{seed}",
    )

    model.load_state_dict(
        result.best_state
    )

    torch.save(
        result.best_state,
        seed_dir
        / "LSTM_Survival_best.pt",
    )

    probs, calibration = (
        calibrated_sector_predictions(
            model
        )
    )

    seed_probability_store[
        seed
    ][
        "LSTM_Survival"
    ] = probs

    history = result.history.copy()
    history["BaseModel"] = "LSTM_Survival"
    history["Seed"] = seed
    training_histories.append(history)

    best_epoch_rows.append({
        "Seed": seed,
        "Model": "LSTM_Survival",
        "BestEpoch": result.best_epoch,
        "BestValidationSurvivalNLL": result.best_val_nll,
    })

    calibration_rows.append({
        "Seed": seed,
        "Model": "LSTM_Survival",
        **calibration,
    })

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ---------------------------------------------------------------------
    # 22.2 TEMPORAL TRANSFORMER
    # ---------------------------------------------------------------------
    set_all_seeds(seed)

    model = TemporalTransformerSurvival(
        feature_dim=F_DIM,
        n_sectors=S,
    )

    result = train_sector_neural_model(
        model,
        f"TemporalTransformer_Survival__Seed_{seed}",
    )

    model.load_state_dict(
        result.best_state
    )

    torch.save(
        result.best_state,
        seed_dir
        / "TemporalTransformer_Survival_best.pt",
    )

    probs, calibration = (
        calibrated_sector_predictions(
            model
        )
    )

    seed_probability_store[
        seed
    ][
        "TemporalTransformer_Survival"
    ] = probs

    history = result.history.copy()
    history["BaseModel"] = (
        "TemporalTransformer_Survival"
    )
    history["Seed"] = seed
    training_histories.append(history)

    best_epoch_rows.append({
        "Seed": seed,
        "Model": "TemporalTransformer_Survival",
        "BestEpoch": result.best_epoch,
        "BestValidationSurvivalNLL": result.best_val_nll,
    })

    calibration_rows.append({
        "Seed": seed,
        "Model": "TemporalTransformer_Survival",
        **calibration,
    })

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ---------------------------------------------------------------------
    # 22.3 RANDOM PLACEBO GRAPH FOR THIS SEED
    # ---------------------------------------------------------------------
    (
        random_graph_seed,
        random_static_seed,
        placebo_edges_seed,
    ) = build_random_graph_for_seed(
        seed
    )

    random_placebo_rows.append(
        placebo_edges_seed
    )

    # The graph architectures below receive IDENTICAL initialization seed
    # within this run. This is a paired architectural ablation.
    graph_variants_seed = {
        "NoGraphTransformer": (
            zero_graph_by_date
        ),
        "RandomGraphTransformer": (
            random_graph_seed
        ),
        "StaticHawkesGraphTransformer": (
            static_graph_by_date
        ),
        "DynamicHawkesGraphTransformer": (
            dynamic_graph_scaled
        ),
    }

    # ---------------------------------------------------------------------
    # 22.4 GRAPH ABLATIONS
    # ---------------------------------------------------------------------
    for (
        model_name,
        graph_array,
    ) in graph_variants_seed.items():

        # Paired initialization / minibatch seed.
        set_all_seeds(seed)

        model = GraphSurvivalTransformer(
            feature_dim=F_DIM,
            n_sectors=S,
        )

        result = train_graph_neural_model(
            model,
            f"{model_name}__Seed_{seed}",
            graph_array,
        )

        model.load_state_dict(
            result.best_state
        )

        torch.save(
            result.best_state,
            seed_dir
            / f"{model_name}_best.pt",
        )

        probs, calibration = (
            calibrated_graph_predictions(
                model,
                graph_array,
            )
        )

        seed_probability_store[
            seed
        ][model_name] = probs

        history = result.history.copy()
        history[
            "BaseModel"
        ] = model_name
        history[
            "Seed"
        ] = seed
        training_histories.append(
            history
        )

        best_epoch_rows.append({
            "Seed": seed,
            "Model": model_name,
            "BestEpoch": result.best_epoch,
            "BestValidationSurvivalNLL": result.best_val_nll,
        })

        calibration_rows.append({
            "Seed": seed,
            "Model": model_name,
            **calibration,
        })

        del model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

training_history = pd.concat(
    training_histories,
    ignore_index=True,
)

best_epochs = pd.DataFrame(
    best_epoch_rows
)

neural_calibration_seeds = pd.DataFrame(
    calibration_rows
)

random_placebo_edges = pd.concat(
    random_placebo_rows,
    ignore_index=True,
)

training_history.to_csv(
    TABLE_DIR
    / "Table_R01_MultiSeed_Training_History.csv",
    index=False,
)

best_epochs.to_csv(
    TABLE_DIR
    / "Table_R02_MultiSeed_Best_Epochs.csv",
    index=False,
)

neural_calibration_seeds.to_csv(
    TABLE_DIR
    / "Table_R03_MultiSeed_Calibration.csv",
    index=False,
)

random_placebo_edges.to_csv(
    TABLE_DIR
    / "Table_R04_Random_Graph_Placebo_Edges.csv",
    index=False,
)

# =============================================================================
# 23. PER-SEED TEST / VALIDATION METRICS
# =============================================================================

seed_metric_rows = []

for seed in MULTI_SEEDS:

    for model_name in NEURAL_MODEL_NAMES:

        probabilities = (
            seed_probability_store[
                seed
            ][model_name]
        )

        for (
            split_name,
            split_mask,
        ) in [
            (
                "Validation",
                validation_date_mask,
            ),
            (
                "Test",
                test_date_mask,
            ),
        ]:

            for h in HORIZONS:

                y_panel = (
                    horizon_target[h]
                )

                p_panel = (
                    probabilities[
                        split_name
                    ][h]
                )

                pooled = probability_metrics(
                    y_panel[
                        split_mask
                    ].reshape(-1),
                    p_panel[
                        split_mask
                    ].reshape(-1),
                )

                seed_metric_rows.append({
                    "Seed": seed,
                    "Split": split_name,
                    "Sector": "POOLED",
                    "Horizon": h,
                    "Model": model_name,
                    **pooled,
                })

                for s, sector in enumerate(
                    sectors
                ):

                    sector_metrics = (
                        probability_metrics(
                            y_panel[
                                split_mask,
                                s,
                            ],
                            p_panel[
                                split_mask,
                                s,
                            ],
                        )
                    )

                    seed_metric_rows.append({
                        "Seed": seed,
                        "Split": split_name,
                        "Sector": sector,
                        "Horizon": h,
                        "Model": model_name,
                        **sector_metrics,
                    })

seed_metrics = pd.DataFrame(
    seed_metric_rows
)

seed_metrics.to_csv(
    TABLE_DIR
    / "Table_R05_All_PerSeed_Metrics.csv",
    index=False,
)

# =============================================================================
# 24. MULTI-SEED MEAN / SD / MEDIAN SUMMARIES
# =============================================================================

pooled_test_seed = seed_metrics[
    (seed_metrics["Split"] == "Test")
    & (
        seed_metrics["Sector"]
        == "POOLED"
    )
].copy()

metric_summary = (
    pooled_test_seed
    .groupby(
        [
            "Model",
            "Horizon",
        ],
        as_index=False,
    )
    .agg(
        Seeds=("Seed", "nunique"),
        BrierMean=("Brier", "mean"),
        BrierSD=("Brier", "std"),
        BrierMedian=("Brier", "median"),
        LogScoreMean=("LogScore", "mean"),
        LogScoreSD=("LogScore", "std"),
        LogScoreMedian=("LogScore", "median"),
        PR_AUCMean=("PR_AUC", "mean"),
        PR_AUCSD=("PR_AUC", "std"),
        ROC_AUCMean=("ROC_AUC", "mean"),
        ROC_AUCSD=("ROC_AUC", "std"),
        CalibrationInterceptMean=(
            "CalibrationIntercept",
            "mean",
        ),
        CalibrationInterceptSD=(
            "CalibrationIntercept",
            "std",
        ),
        CalibrationSlopeMean=(
            "CalibrationSlope",
            "mean",
        ),
        CalibrationSlopeSD=(
            "CalibrationSlope",
            "std",
        ),
    )
)

metric_summary.to_csv(
    TABLE_DIR
    / "Table_R06_MultiSeed_Pooled_Test_Mean_SD.csv",
    index=False,
)

# Sector x horizon mean/SD across seeds.
sector_test_seed = seed_metrics[
    (seed_metrics["Split"] == "Test")
    & (
        seed_metrics["Sector"]
        != "POOLED"
    )
].copy()

sector_seed_summary = (
    sector_test_seed
    .groupby(
        [
            "Model",
            "Sector",
            "Horizon",
        ],
        as_index=False,
    )
    .agg(
        BrierMean=("Brier", "mean"),
        BrierSD=("Brier", "std"),
        LogScoreMean=("LogScore", "mean"),
        LogScoreSD=("LogScore", "std"),
        PR_AUCMean=("PR_AUC", "mean"),
        PR_AUCSD=("PR_AUC", "std"),
        ROC_AUCMean=("ROC_AUC", "mean"),
        ROC_AUCSD=("ROC_AUC", "std"),
    )
)

sector_seed_summary.to_csv(
    TABLE_DIR
    / "Table_R07_Sector_Horizon_MultiSeed_Summary.csv",
    index=False,
)

# =============================================================================
# 25. PAIRED-SEED GRAPH ABLATION DIFFERENCES
# =============================================================================

paired_rows = []

comparison_models = [
    "NoGraphTransformer",
    "RandomGraphTransformer",
    "StaticHawkesGraphTransformer",
]

for h in HORIZONS:

    proposed = pooled_test_seed[
        (
            pooled_test_seed["Model"]
            == PROPOSED_MODEL
        )
        & (
            pooled_test_seed["Horizon"]
            == h
        )
    ][
        [
            "Seed",
            "Brier",
            "LogScore",
            "PR_AUC",
            "ROC_AUC",
        ]
    ].rename(
        columns={
            "Brier": "DynamicBrier",
            "LogScore": "DynamicLogScore",
            "PR_AUC": "DynamicPR_AUC",
            "ROC_AUC": "DynamicROC_AUC",
        }
    )

    for comparison in comparison_models:

        comp = pooled_test_seed[
            (
                pooled_test_seed["Model"]
                == comparison
            )
            & (
                pooled_test_seed["Horizon"]
                == h
            )
        ][
            [
                "Seed",
                "Brier",
                "LogScore",
                "PR_AUC",
                "ROC_AUC",
            ]
        ].rename(
            columns={
                "Brier": "ComparisonBrier",
                "LogScore": "ComparisonLogScore",
                "PR_AUC": "ComparisonPR_AUC",
                "ROC_AUC": "ComparisonROC_AUC",
            }
        )

        merged = proposed.merge(
            comp,
            on="Seed",
            how="inner",
            validate="one_to_one",
        )

        for row in merged.itertuples(
            index=False
        ):

            paired_rows.append({
                "Seed": row.Seed,
                "Horizon": h,
                "Comparison": comparison,
                "BrierDiff_DynamicMinusComparison": (
                    row.DynamicBrier
                    - row.ComparisonBrier
                ),
                "LogScoreDiff_DynamicMinusComparison": (
                    row.DynamicLogScore
                    - row.ComparisonLogScore
                ),
                "PR_AUCDiff_DynamicMinusComparison": (
                    row.DynamicPR_AUC
                    - row.ComparisonPR_AUC
                ),
                "ROC_AUCDiff_DynamicMinusComparison": (
                    row.DynamicROC_AUC
                    - row.ComparisonROC_AUC
                ),
            })

paired_seed_differences = pd.DataFrame(
    paired_rows
)

paired_seed_differences.to_csv(
    TABLE_DIR
    / "Table_R08_Paired_Seed_Graph_Differences.csv",
    index=False,
)

paired_seed_summary = (
    paired_seed_differences
    .groupby(
        [
            "Horizon",
            "Comparison",
        ],
        as_index=False,
    )
    .agg(
        Seeds=(
            "Seed",
            "nunique",
        ),
        MeanBrierDiff=(
            "BrierDiff_DynamicMinusComparison",
            "mean",
        ),
        SDBrierDiff=(
            "BrierDiff_DynamicMinusComparison",
            "std",
        ),
        DynamicBetter_Brier_SeedShare=(
            "BrierDiff_DynamicMinusComparison",
            lambda x: float(
                np.mean(
                    np.asarray(x)
                    < 0
                )
            ),
        ),
        MeanLogScoreDiff=(
            "LogScoreDiff_DynamicMinusComparison",
            "mean",
        ),
        SDLogScoreDiff=(
            "LogScoreDiff_DynamicMinusComparison",
            "std",
        ),
        DynamicBetter_LogScore_SeedShare=(
            "LogScoreDiff_DynamicMinusComparison",
            lambda x: float(
                np.mean(
                    np.asarray(x)
                    < 0
                )
            ),
        ),
        MeanPR_AUCDiff=(
            "PR_AUCDiff_DynamicMinusComparison",
            "mean",
        ),
        MeanROC_AUCDiff=(
            "ROC_AUCDiff_DynamicMinusComparison",
            "mean",
        ),
    )
)

paired_seed_summary.to_csv(
    TABLE_DIR
    / "Table_R09_Paired_Seed_Graph_Summary.csv",
    index=False,
)

# =============================================================================
# 26. EQUAL-WEIGHT FIVE-SEED ENSEMBLE PREDICTIONS
# =============================================================================

ensemble_probabilities = {}

for model_name in NEURAL_MODEL_NAMES:

    ensemble_probabilities[
        model_name
    ] = initialize_probability_dict()

    for split_name in [
        "Validation",
        "Test",
    ]:

        for h in HORIZONS:

            stack = np.stack(
                [
                    seed_probability_store[
                        seed
                    ][
                        model_name
                    ][
                        split_name
                    ][h]
                    for seed in MULTI_SEEDS
                ],
                axis=0,
            )

            ensemble_probabilities[
                model_name
            ][
                split_name
            ][h] = np.nanmean(
                stack,
                axis=0,
            ).astype(np.float32)

# Deterministic XGBoost from the frozen Phase-2.1 specification.
ensemble_probabilities[
    "XGBoost"
] = xgb_panel_probs

# Historical / Hawkes baselines from Phase-1 master.
def master_probability_panel(
    column: str,
) -> np.ndarray:

    out = np.full(
        (T, S),
        np.nan,
        dtype=np.float32,
    )

    for row in master[
        [
            "Date",
            "Sector",
            column,
        ]
    ].itertuples(index=False):

        t = date_to_idx[
            pd.Timestamp(
                row.Date
            )
        ]

        s = sector_to_idx[
            row.Sector
        ]

        value = getattr(
            row,
            column,
        )

        if pd.notna(value):
            out[
                t,
                s,
            ] = float(value)

    return out

for baseline_name, prefix_name in [
    (
        "FixedHistorical",
        "FixedHistoricalProb",
    ),
    (
        "ExpandingHistorical",
        "ExpandingHistoricalProb",
    ),
    (
        "StableHawkes",
        "StableHawkesProb",
    ),
]:

    ensemble_probabilities[
        baseline_name
    ] = initialize_probability_dict()

    for h in HORIZONS:

        panel = master_probability_panel(
            f"{prefix_name}_{h}"
        )

        ensemble_probabilities[
            baseline_name
        ][
            "Validation"
        ][h] = panel

        ensemble_probabilities[
            baseline_name
        ][
            "Test"
        ][h] = panel

# =============================================================================
# 27. ENSEMBLE FORECAST METRICS
# =============================================================================

ensemble_metric_rows = []

for (
    split_name,
    split_mask,
) in [
    (
        "Validation",
        validation_date_mask,
    ),
    (
        "Test",
        test_date_mask,
    ),
]:

    for (
        model_name,
        split_dict,
    ) in ensemble_probabilities.items():

        for h in HORIZONS:

            y_panel = horizon_target[h]
            p_panel = (
                split_dict[
                    split_name
                ][h]
            )

            pooled = probability_metrics(
                y_panel[
                    split_mask
                ].reshape(-1),
                p_panel[
                    split_mask
                ].reshape(-1),
            )

            ensemble_metric_rows.append({
                "Split": split_name,
                "Sector": "POOLED",
                "Horizon": h,
                "Model": model_name,
                "ForecastType": (
                    "FiveSeedMean"
                    if model_name
                    in NEURAL_MODEL_NAMES
                    else "SingleDeterministic"
                ),
                **pooled,
            })

            for s, sector in enumerate(
                sectors
            ):

                sector_metrics = probability_metrics(
                    y_panel[
                        split_mask,
                        s,
                    ],
                    p_panel[
                        split_mask,
                        s,
                    ],
                )

                ensemble_metric_rows.append({
                    "Split": split_name,
                    "Sector": sector,
                    "Horizon": h,
                    "Model": model_name,
                    "ForecastType": (
                        "FiveSeedMean"
                        if model_name
                        in NEURAL_MODEL_NAMES
                        else "SingleDeterministic"
                    ),
                    **sector_metrics,
                })

ensemble_metrics = pd.DataFrame(
    ensemble_metric_rows
)

ensemble_metrics.to_csv(
    TABLE_DIR
    / "Table_R10_Ensemble_All_Metrics.csv",
    index=False,
)

ensemble_test_pooled = (
    ensemble_metrics[
        (
            ensemble_metrics["Split"]
            == "Test"
        )
        & (
            ensemble_metrics["Sector"]
            == "POOLED"
        )
    ]
    .sort_values(
        [
            "Horizon",
            "Brier",
        ]
    )
    .reset_index(
        drop=True
    )
)

ensemble_test_pooled.to_csv(
    TABLE_DIR
    / "Table_R11_Ensemble_Main_Test_Results.csv",
    index=False,
)

# =============================================================================
# 28. HORIZON COHERENCE — EVERY SEED + ENSEMBLE
# =============================================================================

coherence_rows = []

for seed in MULTI_SEEDS:

    for model_name in NEURAL_MODEL_NAMES:

        for split_name in [
            "Validation",
            "Test",
        ]:

            stack = np.stack(
                [
                    seed_probability_store[
                        seed
                    ][model_name][
                        split_name
                    ][h]
                    for h in HORIZONS
                ],
                axis=-1,
            )

            complete = np.all(
                np.isfinite(stack),
                axis=-1,
            )

            violations = (
                np.any(
                    np.diff(
                        stack,
                        axis=-1,
                    ) < -1e-8,
                    axis=-1,
                )
                & complete
            )

            coherence_rows.append({
                "Level": "Seed",
                "Seed": seed,
                "Model": model_name,
                "Split": split_name,
                "CompleteRows": int(
                    complete.sum()
                ),
                "CoherenceViolations": int(
                    violations.sum()
                ),
            })

for model_name in NEURAL_MODEL_NAMES:

    for split_name in [
        "Validation",
        "Test",
    ]:

        stack = np.stack(
            [
                ensemble_probabilities[
                    model_name
                ][split_name][h]
                for h in HORIZONS
            ],
            axis=-1,
        )

        complete = np.all(
            np.isfinite(stack),
            axis=-1,
        )

        violations = (
            np.any(
                np.diff(
                    stack,
                    axis=-1,
                ) < -1e-8,
                axis=-1,
            )
            & complete
        )

        coherence_rows.append({
            "Level": "Ensemble",
            "Seed": np.nan,
            "Model": model_name,
            "Split": split_name,
            "CompleteRows": int(
                complete.sum()
            ),
            "CoherenceViolations": int(
                violations.sum()
            ),
        })

coherence_table = pd.DataFrame(
    coherence_rows
)

coherence_table.to_csv(
    TABLE_DIR
    / "Table_R12_MultiSeed_Horizon_Coherence.csv",
    index=False,
)

# =============================================================================
# 29. MOVING-BLOCK BOOTSTRAP — ENSEMBLE PROPOSED VS ALL BENCHMARKS
# =============================================================================

def moving_block_sample_indices(
    n_dates: int,
    block_length: int,
    rng: np.random.Generator,
) -> np.ndarray:

    selected = []

    while len(selected) < n_dates:

        if n_dates <= block_length:
            start = 0
        else:
            start = int(
                rng.integers(
                    0,
                    n_dates
                    - block_length
                    + 1,
                )
            )

        selected.extend(
            range(
                start,
                min(
                    n_dates,
                    start
                    + block_length,
                ),
            )
        )

    return np.asarray(
        selected[
            :n_dates
        ],
        dtype=int,
    )

test_indices = np.where(
    test_date_mask
)[0]

benchmark_models = [
    model_name
    for model_name
    in ensemble_probabilities
    if model_name
    != PROPOSED_MODEL
]

bootstrap_rows = []

log(
    f"Ensemble paired moving-block bootstrap: "
    f"reps={BOOTSTRAP_REPS}, block={BOOTSTRAP_BLOCK}."
)

for h in HORIZONS:

    y = horizon_target[h][
        test_indices
    ]

    p_proposed = (
        ensemble_probabilities[
            PROPOSED_MODEL
        ][
            "Test"
        ][h][
            test_indices
        ]
    )

    for benchmark in benchmark_models:

        p_benchmark = (
            ensemble_probabilities[
                benchmark
            ][
                "Test"
            ][h][
                test_indices
            ]
        )

        common = (
            np.isfinite(y)
            & np.isfinite(
                p_proposed
            )
            & np.isfinite(
                p_benchmark
            )
        )

        proposed_brier_date = np.full(
            len(test_indices),
            np.nan,
        )
        benchmark_brier_date = np.full(
            len(test_indices),
            np.nan,
        )
        proposed_log_date = np.full(
            len(test_indices),
            np.nan,
        )
        benchmark_log_date = np.full(
            len(test_indices),
            np.nan,
        )

        for local_t in range(
            len(test_indices)
        ):

            ok = common[
                local_t
            ]

            if not ok.any():
                continue

            yt = y[
                local_t,
                ok,
            ]

            pp = np.clip(
                p_proposed[
                    local_t,
                    ok,
                ],
                EPS,
                1.0 - EPS,
            )

            pb = np.clip(
                p_benchmark[
                    local_t,
                    ok,
                ],
                EPS,
                1.0 - EPS,
            )

            proposed_brier_date[
                local_t
            ] = np.mean(
                (pp - yt) ** 2
            )

            benchmark_brier_date[
                local_t
            ] = np.mean(
                (pb - yt) ** 2
            )

            proposed_log_date[
                local_t
            ] = -np.mean(
                yt * np.log(pp)
                + (1.0 - yt)
                * np.log1p(-pp)
            )

            benchmark_log_date[
                local_t
            ] = -np.mean(
                yt * np.log(pb)
                + (1.0 - yt)
                * np.log1p(-pb)
            )

        valid_dates = (
            np.isfinite(
                proposed_brier_date
            )
            & np.isfinite(
                benchmark_brier_date
            )
            & np.isfinite(
                proposed_log_date
            )
            & np.isfinite(
                benchmark_log_date
            )
        )

        pbd = proposed_brier_date[
            valid_dates
        ]
        bbd = benchmark_brier_date[
            valid_dates
        ]
        pld = proposed_log_date[
            valid_dates
        ]
        bld = benchmark_log_date[
            valid_dates
        ]

        observed_brier_diff = float(
            np.mean(
                pbd - bbd
            )
        )

        observed_log_diff = float(
            np.mean(
                pld - bld
            )
        )

        rng = np.random.default_rng(
            SEED
            + 1000 * h
            + sum(
                ord(c)
                for c in benchmark
            )
            + 92022
        )

        brier_diffs = []
        log_diffs = []

        for _ in range(
            BOOTSTRAP_REPS
        ):

            idx = (
                moving_block_sample_indices(
                    len(pbd),
                    min(
                        BOOTSTRAP_BLOCK,
                        len(pbd),
                    ),
                    rng,
                )
            )

            brier_diffs.append(
                float(
                    np.mean(
                        pbd[idx]
                        - bbd[idx]
                    )
                )
            )

            log_diffs.append(
                float(
                    np.mean(
                        pld[idx]
                        - bld[idx]
                    )
                )
            )

        brier_diffs = np.asarray(
            brier_diffs
        )
        log_diffs = np.asarray(
            log_diffs
        )

        bootstrap_rows.append({
            "Horizon": h,
            "ProposedModel": (
                PROPOSED_MODEL
            ),
            "Benchmark": benchmark,
            "BrierDiff_ProposedMinusBenchmark": (
                observed_brier_diff
            ),
            "BrierDiff_CI2.5": float(
                np.quantile(
                    brier_diffs,
                    0.025,
                )
            ),
            "BrierDiff_CI97.5": float(
                np.quantile(
                    brier_diffs,
                    0.975,
                )
            ),
            "BrierProb_ProposedBetter": float(
                np.mean(
                    brier_diffs
                    < 0
                )
            ),
            "LogScoreDiff_ProposedMinusBenchmark": (
                observed_log_diff
            ),
            "LogScoreDiff_CI2.5": float(
                np.quantile(
                    log_diffs,
                    0.025,
                )
            ),
            "LogScoreDiff_CI97.5": float(
                np.quantile(
                    log_diffs,
                    0.975,
                )
            ),
            "LogScoreProb_ProposedBetter": float(
                np.mean(
                    log_diffs
                    < 0
                )
            ),
            "BootstrapReps": BOOTSTRAP_REPS,
            "BlockLength": (
                BOOTSTRAP_BLOCK
            ),
        })

bootstrap_results = pd.DataFrame(
    bootstrap_rows
)

bootstrap_results.to_csv(
    TABLE_DIR
    / "Table_R13_Ensemble_Paired_Block_Bootstrap.csv",
    index=False,
)

# =============================================================================
# 30. SECTOR/HORIZON ENSEMBLE HETEROGENEITY
# =============================================================================

dynamic_sector = ensemble_metrics[
    (
        ensemble_metrics["Split"]
        == "Test"
    )
    & (
        ensemble_metrics["Model"]
        == PROPOSED_MODEL
    )
    & (
        ensemble_metrics["Sector"]
        != "POOLED"
    )
].copy()

nograph_sector = ensemble_metrics[
    (
        ensemble_metrics["Split"]
        == "Test"
    )
    & (
        ensemble_metrics["Model"]
        == "NoGraphTransformer"
    )
    & (
        ensemble_metrics["Sector"]
        != "POOLED"
    )
][
    [
        "Sector",
        "Horizon",
        "Brier",
        "LogScore",
    ]
].rename(
    columns={
        "Brier": "NoGraphBrier",
        "LogScore": "NoGraphLogScore",
    }
)

heterogeneity = dynamic_sector.merge(
    nograph_sector,
    on=[
        "Sector",
        "Horizon",
    ],
    how="left",
)

heterogeneity[
    "DynamicGraph_BrierSkill_vs_NoGraph"
] = (
    1.0
    - heterogeneity[
        "Brier"
    ]
    / heterogeneity[
        "NoGraphBrier"
    ]
)

heterogeneity[
    "DynamicGraph_LogScoreImprovement_vs_NoGraph"
] = (
    heterogeneity[
        "NoGraphLogScore"
    ]
    - heterogeneity[
        "LogScore"
    ]
)

heterogeneity.to_csv(
    TABLE_DIR
    / "Table_R14_Ensemble_Sector_Horizon_Heterogeneity.csv",
    index=False,
)

# =============================================================================
# 31. SAVE ENSEMBLE + PER-SEED PREDICTION PANELS
# =============================================================================

ensemble_prediction_rows = []

for (
    split_name,
    split_mask,
) in [
    (
        "Validation",
        validation_date_mask,
    ),
    (
        "Test",
        test_date_mask,
    ),
]:

    for t in np.where(
        split_mask
    )[0]:

        for s, sector in enumerate(
            sectors
        ):

            row = {
                "Date": dates[t],
                "Split": split_name,
                "Sector": sector,
            }

            for h in HORIZONS:

                row[
                    f"Target_{h}"
                ] = (
                    horizon_target[
                        h
                    ][t, s]
                )

                for model_name in (
                    ensemble_probabilities
                ):

                    row[
                        f"{model_name}__P{h}"
                    ] = (
                        ensemble_probabilities[
                            model_name
                        ][
                            split_name
                        ][h][t, s]
                    )

            ensemble_prediction_rows.append(
                row
            )

ensemble_prediction_panel = pd.DataFrame(
    ensemble_prediction_rows
)

ensemble_prediction_panel.to_csv(
    DATA_DIR
    / "phase2_2_ensemble_validation_test_predictions.csv.gz",
    index=False,
    compression="gzip",
)

# Per-seed neural predictions in long form for transparent replication.
per_seed_prediction_rows = []

for seed in MULTI_SEEDS:

    for split_name, split_mask in [
        ("Validation", validation_date_mask),
        ("Test", test_date_mask),
    ]:

        for t in np.where(split_mask)[0]:

            for s, sector in enumerate(sectors):

                for h in HORIZONS:

                    row = {
                        "Seed": seed,
                        "Date": dates[t],
                        "Split": split_name,
                        "Sector": sector,
                        "Horizon": h,
                        "Target": horizon_target[h][t, s],
                    }

                    for model_name in NEURAL_MODEL_NAMES:
                        row[model_name] = (
                            seed_probability_store[
                                seed
                            ][model_name][
                                split_name
                            ][h][t, s]
                        )

                    per_seed_prediction_rows.append(
                        row
                    )

per_seed_prediction_panel = pd.DataFrame(
    per_seed_prediction_rows
)

per_seed_prediction_panel.to_csv(
    DATA_DIR
    / "phase2_2_per_seed_predictions.csv.gz",
    index=False,
    compression="gzip",
)

# =============================================================================
# 32. FIGURES
# =============================================================================

# R01 — Test Brier mean +/- one SD across seeds.
fig, ax = plt.subplots(
    figsize=(11, 6)
)

for model_name, group in (
    metric_summary
    .groupby("Model")
):

    group = group.sort_values(
        "Horizon"
    )

    ax.errorbar(
        group["Horizon"],
        group["BrierMean"],
        yerr=group["BrierSD"],
        marker="o",
        capsize=3,
        label=model_name,
    )

ax.set_xlabel(
    "Forecast horizon (trading days)"
)
ax.set_ylabel(
    "Test Brier score: mean ± SD across seeds"
)
ax.set_xticks(
    HORIZONS
)
ax.set_title(
    "Neural Forecast Robustness Across Independent Random Seeds"
)
ax.legend(
    fontsize=7
)

fig.tight_layout()
fig.savefig(
    FIG_DIR
    / "Figure_R01_MultiSeed_Brier_Mean_SD.png",
    dpi=300,
)
plt.close(fig)

# R02 — Ensemble pooled Test Brier.
fig, ax = plt.subplots(
    figsize=(11, 6)
)

for model_name, group in (
    ensemble_test_pooled
    .groupby("Model")
):

    group = group.sort_values(
        "Horizon"
    )

    ax.plot(
        group["Horizon"],
        group["Brier"],
        marker="o",
        label=model_name,
    )

ax.set_xlabel(
    "Forecast horizon (trading days)"
)
ax.set_ylabel(
    "Brier score (lower is better)"
)
ax.set_xticks(
    HORIZONS
)
ax.set_title(
    "Five-Seed Ensemble and Benchmark Test Performance"
)
ax.legend(
    fontsize=7
)

fig.tight_layout()
fig.savefig(
    FIG_DIR
    / "Figure_R02_Ensemble_Test_Brier.png",
    dpi=300,
)
plt.close(fig)

# R03 — paired seed dynamic-vs-graph-control Brier differences.
fig, ax = plt.subplots(
    figsize=(10, 6)
)

for comparison, group in (
    paired_seed_differences
    .groupby("Comparison")
):

    summary = (
        group
        .groupby(
            "Horizon",
            as_index=False,
        )
        .agg(
            Mean=(
                "BrierDiff_DynamicMinusComparison",
                "mean",
            ),
            SD=(
                "BrierDiff_DynamicMinusComparison",
                "std",
            ),
        )
        .sort_values(
            "Horizon"
        )
    )

    ax.errorbar(
        summary["Horizon"],
        summary["Mean"],
        yerr=summary["SD"],
        marker="o",
        capsize=3,
        label=comparison,
    )

ax.axhline(
    0.0,
    linewidth=1.0,
)

ax.set_xlabel(
    "Forecast horizon (trading days)"
)
ax.set_ylabel(
    "Brier difference: Dynamic Hawkes minus comparison"
)
ax.set_xticks(
    HORIZONS
)
ax.set_title(
    "Paired-Seed Graph Ablation Robustness"
)
ax.legend(
    fontsize=8
)

fig.tight_layout()
fig.savefig(
    FIG_DIR
    / "Figure_R03_PairedSeed_Graph_Brier_Differences.png",
    dpi=300,
)
plt.close(fig)

# R04 — sector/horizon ensemble graph skill.
plot_heterogeneity = heterogeneity.pivot(
    index="Sector",
    columns="Horizon",
    values="DynamicGraph_BrierSkill_vs_NoGraph",
)

fig, ax = plt.subplots(
    figsize=(10, 7)
)

image = ax.imshow(
    plot_heterogeneity.to_numpy(),
    aspect="auto",
)

ax.set_xticks(
    range(
        len(
            plot_heterogeneity.columns
        )
    )
)
ax.set_xticklabels(
    [
        str(int(h))
        for h in
        plot_heterogeneity.columns
    ]
)
ax.set_yticks(
    range(
        len(
            plot_heterogeneity.index
        )
    )
)
ax.set_yticklabels(
    plot_heterogeneity.index
)
ax.set_xlabel(
    "Forecast horizon"
)
ax.set_ylabel(
    "Sector"
)
ax.set_title(
    "Five-Seed Ensemble Dynamic-Graph Brier Skill vs No Graph"
)

fig.colorbar(
    image,
    ax=ax,
    label="Brier skill",
)

fig.tight_layout()
fig.savefig(
    FIG_DIR
    / "Figure_R04_Ensemble_Sector_Graph_Skill.png",
    dpi=300,
)
plt.close(fig)

# =============================================================================
# 33. EXCEL WORKBOOK
# =============================================================================

excel_path = (
    EXCEL_DIR
    / "Python_Phase2_2_FiveSeed_Robustness.xlsx"
)

if (
    importlib.util.find_spec(
        "xlsxwriter"
    )
    is not None
):
    excel_engine = "xlsxwriter"
elif (
    importlib.util.find_spec(
        "openpyxl"
    )
    is not None
):
    excel_engine = "openpyxl"
else:
    excel_engine = None

excel_tables = {
    "Target Audit": target_audit,
    "Features": pd.DataFrame({
        "Feature": model_feature_names
    }),
    "Graph Scaling": graph_scaling_audit,
    "Edge Magnitude": edge_scaling_diagnostics,
    "Best Epochs": best_epochs,
    "Calibration": neural_calibration_seeds,
    "Random Placebos": random_placebo_edges,
    "PerSeed Metrics": seed_metrics,
    "Mean SD": metric_summary,
    "Sector Mean SD": sector_seed_summary,
    "Paired Seed Diff": paired_seed_differences,
    "Paired Seed Summary": paired_seed_summary,
    "Ensemble Metrics": ensemble_metrics,
    "Ensemble Test": ensemble_test_pooled,
    "Bootstrap": bootstrap_results,
    "Sector Heterogeneity": heterogeneity,
    "Coherence": coherence_table,
}

if excel_engine is not None:

    try:

        with pd.ExcelWriter(
            excel_path,
            engine=excel_engine,
        ) as writer:

            for (
                sheet_name,
                dataframe,
            ) in excel_tables.items():

                dataframe.to_excel(
                    writer,
                    sheet_name=(
                        sheet_name[
                            :31
                        ]
                    ),
                    index=False,
                )

            if (
                excel_engine
                == "xlsxwriter"
            ):

                workbook = writer.book

                header_format = workbook.add_format({
                    "bold": True,
                    "text_wrap": True,
                    "valign": "top",
                    "border": 1,
                })

                for (
                    _,
                    worksheet,
                ) in writer.sheets.items():

                    worksheet.freeze_panes(
                        1,
                        0,
                    )

                    worksheet.set_row(
                        0,
                        28,
                        header_format,
                    )

                    worksheet.set_column(
                        0,
                        30,
                        16,
                    )

            else:

                for (
                    _,
                    worksheet,
                ) in writer.sheets.items():

                    worksheet.freeze_panes = "A2"

        log(
            f"Excel workbook saved: "
            f"{excel_path}"
        )

    except Exception as exc:

        log(
            "WARNING: Excel export failed: "
            f"{repr(exc)}"
        )

# =============================================================================
# 34. FINAL INTEGRITY CHECKS
# =============================================================================

# Count distinct placebo support signatures across seeds.
placebo_signatures = []

for seed in MULTI_SEEDS:

    subset = (
        random_placebo_edges[
            random_placebo_edges[
                "Seed"
            ] == seed
        ][
            [
                "FromSector",
                "ToSector",
            ]
        ]
        .sort_values(
            [
                "FromSector",
                "ToSector",
            ]
        )
    )

    signature = tuple(
        map(
            tuple,
            subset.to_numpy(),
        )
    )

    placebo_signatures.append(
        signature
    )

integrity_checks = [
    (
        "Requested number of seeds completed",
        seed_metrics["Seed"].nunique()
        == N_SEEDS,
    ),
    (
        "All six neural architectures completed every seed",
        bool(
            (
                seed_metrics[
                    seed_metrics[
                        "Sector"
                    ] == "POOLED"
                ]
                .groupby(
                    [
                        "Seed",
                        "Model",
                    ]
                )
                .size()
                > 0
            ).all()
        ),
    ),
    (
        "Random placebo graph support changes across seeds",
        len(
            set(
                placebo_signatures
            )
        )
        >= min(
            2,
            N_SEEDS,
        ),
    ),
    (
        "No horizon incoherence in any seed or ensemble",
        int(
            coherence_table[
                "CoherenceViolations"
            ].sum()
        )
        == 0,
    ),
    (
        "Feature matrix finite",
        bool(
            np.all(
                np.isfinite(
                    X_panel
                )
            )
        ),
    ),
    (
        "Crash_Main event-safe imputation retained",
        crash_missing_set_to_zero,
    ),
    (
        "Neural positive weight remains capped at 3",
        HAZARD_POS_WEIGHT
        <= NEURAL_MAX_POS_WEIGHT
        + 1e-12,
    ),
    (
        "Magnitude-preserving graph scaling retained",
        bool(
            GRAPH_TRAIN_Q > 0
            and np.isfinite(
                GRAPH_TRAIN_Q
            )
        ),
    ),
    (
        "Dynamic ensemble present in final Test results",
        bool(
            (
                ensemble_test_pooled[
                    "Model"
                ]
                == PROPOSED_MODEL
            ).any()
        ),
    ),
    (
        "Both Test classes present at every horizon",
        all(
            len(
                np.unique(
                    horizon_target[h][
                        test_date_mask
                    ][
                        np.isfinite(
                            horizon_target[h][
                                test_date_mask
                            ]
                        )
                    ]
                )
            )
            == 2
            for h in HORIZONS
        ),
    ),
]

integrity_table = pd.DataFrame(
    integrity_checks,
    columns=[
        "Check",
        "Passed",
    ],
)

integrity_table.to_csv(
    TABLE_DIR
    / "Table_R15_Final_Integrity_Checks.csv",
    index=False,
)

if not integrity_table[
    "Passed"
].all():

    failed = (
        integrity_table.loc[
            ~integrity_table[
                "Passed"
            ],
            "Check",
        ]
        .tolist()
    )

    raise RuntimeError(
        "Phase 2.2 integrity checks failed: "
        f"{failed}"
    )

# =============================================================================
# 35. METADATA / ROBUSTNESS BUNDLE
# =============================================================================

metadata = {
    "ProjectTitle": (
        "Forecasting Sectoral Crash Risk and Contagion: "
        "Integrating Dynamic Volatility, Extreme-Value Modelling "
        "and Graph Deep Learning"
    ),
    "PythonPhase": "2.2",
    "ScriptVersion": "2.2",
    "GeneratedAt": datetime.now().isoformat(),
    "BaseSeed": SEED,
    "MultiSeeds": MULTI_SEEDS,
    "NumberOfSeeds": N_SEEDS,
    "InputPhase1Zip": INPUT_ZIP_PATH.name,
    "GraphSource": graph_source,
    "Sectors": sectors,
    "ForecastHorizons": HORIZONS,
    "LookbackTradingDays": LOOKBACK,
    "FeatureCount": F_DIM,
    "HazardPositiveWeight": HAZARD_POS_WEIGHT,
    "NeuralMaxPositiveWeight": NEURAL_MAX_POS_WEIGHT,
    "XGBoostMaxPositiveWeight": XGB_MAX_POS_WEIGHT,
    "GraphScalingMethod": (
        "log1p(A/q99_train)/log(2); no row normalization"
    ),
    "GraphTrainScaleReference": GRAPH_TRAIN_Q,
    "GraphScaleClip": GRAPH_SCALE_CLIP,
    "RandomGraphPlacebo": (
        "Independent support per seed; same cross-edge density "
        "and permuted Hawkes raw edge-weight multiset"
    ),
    "PairedGraphInitialization": (
        "Same seed reset before NoGraph/Random/Static/Dynamic "
        "model initialization within each seed"
    ),
    "EnsembleMethod": (
        "Unweighted arithmetic mean of individually "
        "Validation-calibrated seed probabilities"
    ),
    "ProposedModel": PROPOSED_MODEL,
    "BootstrapReps": BOOTSTRAP_REPS,
    "BootstrapBlockLength": BOOTSTRAP_BLOCK,
    "TestUsedForArchitectureTuning": False,
}

with open(
    DATA_DIR
    / "python_phase2_2_metadata.json",
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        metadata,
        handle,
        indent=2,
        default=str,
    )

robustness_bundle = {
    "metadata": metadata,
    "best_epochs": best_epochs.to_dict(
        orient="records"
    ),
    "calibration": neural_calibration_seeds.to_dict(
        orient="records"
    ),
    "metric_summary": metric_summary.to_dict(
        orient="records"
    ),
    "paired_seed_summary": paired_seed_summary.to_dict(
        orient="records"
    ),
    "model_feature_names": model_feature_names,
    "sectors": sectors,
}

with open(
    MODEL_DIR
    / "phase2_2_robustness_bundle.pkl",
    "wb",
) as handle:

    pickle.dump(
        robustness_bundle,
        handle,
    )

# =============================================================================
# 36. CLEAN / ZIP / CONSOLE SUMMARY
# =============================================================================

if EXTRACT_DIR.exists():
    shutil.rmtree(
        EXTRACT_DIR
    )

zip_output = (
    ZIP_DIR
    / "Python_Phase2_2_FiveSeed_Robustness_All_Outputs.zip"
)

if zip_output.exists():
    zip_output.unlink()

with zipfile.ZipFile(
    zip_output,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    for path in OUTPUT_ROOT.rglob("*"):

        if (
            path.is_file()
            and path
            != zip_output
        ):

            archive.write(
                path,
                arcname=path.relative_to(
                    OUTPUT_ROOT
                ),
            )

log(
    f"All Phase-2.2 outputs zipped to: "
    f"{zip_output}"
)

print(
    "\n"
    + "=" * 100
)

print(
    "PYTHON PHASE 2.2 COMPLETED SUCCESSFULLY"
)

print(
    "=" * 100
)

print(
    f"Seeds: {MULTI_SEEDS}"
)

print(
    f"Device: {DEVICE}"
)

print(
    "\nMulti-seed pooled TEST mean +/- SD:"
)

display_summary = metric_summary[
    [
        "Model",
        "Horizon",
        "BrierMean",
        "BrierSD",
        "LogScoreMean",
        "LogScoreSD",
        "PR_AUCMean",
        "PR_AUCSD",
    ]
].sort_values(
    [
        "Horizon",
        "BrierMean",
    ]
)

print(
    display_summary.to_string(
        index=False
    )
)

print(
    "\nFive-seed ENSEMBLE pooled TEST results:"
)

print(
    ensemble_test_pooled[
        [
            "Horizon",
            "Model",
            "Brier",
            "LogScore",
            "PR_AUC",
            "ROC_AUC",
            "CalibrationIntercept",
            "CalibrationSlope",
        ]
    ].to_string(
        index=False
    )
)

print(
    "\nPaired-seed graph robustness:"
)

print(
    paired_seed_summary.to_string(
        index=False
    )
)

print(
    "\nFinal ZIP to send back for review:"
)

print(
    f"  {zip_output}"
)

print(
    "=" * 100
)

# =============================================================================
# 37. AUTOMATIC COLAB DOWNLOAD
# =============================================================================

try:
    from google.colab import files

    print(
        "\nStarting browser download of "
        "Python_Phase2_2_FiveSeed_Robustness_All_Outputs.zip ..."
    )

    files.download(
        str(
            zip_output
        )
    )

except ImportError:

    print(
        "\nNot running in Google Colab. "
        f"Retrieve the ZIP manually from: "
        f"{zip_output}"
    )

# =============================================================================
# END OF PYTHON PHASE 2.2
# =============================================================================


[2026-09-01 17:13:33] ====================================================================================================
[2026-09-01 17:13:33] PYTHON PHASE 2.2 START — FIVE-SEED ROBUSTNESS / GRAPH PLACEBO REPLICATION
[2026-09-01 17:13:33] Python=3.13.15; platform=Linux-6.6.122+-x86_64-with-glibc2.35; seed=20260901
[2026-09-01 17:13:43] PyTorch=2.11.0+cu128; device=cuda

Please upload the completed Phase-1 ZIP:
    Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip



Saving Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip to Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip
[2026-09-01 17:15:00] Validated uploaded Phase-1 ZIP: /content/Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip
[2026-09-01 17:15:00] Accepted Phase-1 ZIP: /content/Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip
[2026-09-01 17:15:00] Loaded 14,928 sector-days; 2,488 dates; 6 sectors; graph source=HAWKES_LAMBDA_MIN_STABILITY.
[2026-09-01 17:15:01] Split-aware censoring targets reconstructed successfully.
[2026-09-01 17:15:01] Selected 49 leakage-screened numeric predictors.
[2026-09-01 17:15:01] Final neural feature dimension=59; missingness indicators=10.
[2026-09-01 17:15:01] Crash_Main event-safe imputation applied: missing events set to zero and never forward-filled.
[2026-09-01 17:15:01] Magnitude-preserving graph scaling applied. Train q0.99=0.06118932; row normalization disabled.
[2026-09-01 17:15:01] Sector samples=11,501; graph-date samples=1,984.
[2026-09-01 17:15:

/tmp/ipykernel_1257/2598098203.py:1775: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


[2026-09-01 17:15:37] TemporalTransformer_Survival__Seed_20260901: epoch=01, train=0.41120, valNLL=0.12994
[2026-09-01 17:15:38] TemporalTransformer_Survival__Seed_20260901: epoch=02, train=0.24942, valNLL=0.13218
[2026-09-01 17:15:40] TemporalTransformer_Survival__Seed_20260901: epoch=03, train=0.24367, valNLL=0.15687
[2026-09-01 17:15:42] TemporalTransformer_Survival__Seed_20260901: epoch=04, train=0.23702, valNLL=0.20137
[2026-09-01 17:15:43] TemporalTransformer_Survival__Seed_20260901: epoch=05, train=0.23088, valNLL=0.25460
[2026-09-01 17:15:45] TemporalTransformer_Survival__Seed_20260901: epoch=06, train=0.22277, valNLL=0.35077
[2026-09-01 17:15:46] TemporalTransformer_Survival__Seed_20260901: epoch=07, train=0.21721, valNLL=0.25614
[2026-09-01 17:15:48] TemporalTransformer_Survival__Seed_20260901: epoch=08, train=0.21098, valNLL=0.28454
[2026-09-01 17:15:50] TemporalTransformer_Survival__Seed_20260901: epoch=09, train=0.20264, valNLL=0.30910


/tmp/ipykernel_1257/2598098203.py:1961: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal_encoder = nn.TransformerEncoder(


...

[2026-09-01 17:20:52] StaticHawkesGraphTransformer__Seed_20260905: epoch=01, train=0.47219, valNLL=0.16950
[2026-09-01 17:20:53] StaticHawkesGraphTransformer__Seed_20260905: epoch=02, train=0.25582, valNLL=0.13710
[2026-09-01 17:20:54] StaticHawkesGraphTransformer__Seed_20260905: epoch=03, train=0.24893, valNLL=0.13617
[2026-09-01 17:20:54] StaticHawkesGraphTransformer__Seed_20260905: epoch=04, train=0.24632, valNLL=0.13350
[2026-09-01 17:20:55] StaticHawkesGraphTransformer__Seed_20260905: epoch=05, train=0.24233, valNLL=0.14536
[2026-09-01 17:20:56] StaticHawkesGraphTransformer__Seed_20260905: epoch=06, train=0.23381, valNLL=0.18418
[2026-09-01 17:20:57] StaticHawkesGraphTransformer__Seed_20260905: epoch=07, train=0.22605, valNLL=0.15940
[2026-09-01 17:20:57] StaticHawkesGraphTransformer__Seed_20260905: epoch=08, train=0.22140, valNLL=0.20844
[2026-09-01 17:20:58] StaticHawkesGraphTransformer__Seed_20260905: epoch=09, train=0.21716, valNLL=0.17421
[2026-09-01 17:20:59] StaticHawkesGra

/tmp/ipykernel_1257/2598098203.py:1961: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal_encoder = nn.TransformerEncoder(


[2026-09-01 17:21:02] DynamicHawkesGraphTransformer__Seed_20260905: epoch=01, train=0.46996, valNLL=0.16721
[2026-09-01 17:21:03] DynamicHawkesGraphTransformer__Seed_20260905: epoch=02, train=0.25559, valNLL=0.13743
[2026-09-01 17:21:04] DynamicHawkesGraphTransformer__Seed_20260905: epoch=03, train=0.24879, valNLL=0.13634
[2026-09-01 17:21:04] DynamicHawkesGraphTransformer__Seed_20260905: epoch=04, train=0.24538, valNLL=0.13000
[2026-09-01 17:21:05] DynamicHawkesGraphTransformer__Seed_20260905: epoch=05, train=0.24131, valNLL=0.16157
[2026-09-01 17:21:06] DynamicHawkesGraphTransformer__Seed_20260905: epoch=06, train=0.23339, valNLL=0.18466
[2026-09-01 17:21:07] DynamicHawkesGraphTransformer__Seed_20260905: epoch=07, train=0.22494, valNLL=0.16880
[2026-09-01 17:21:07] DynamicHawkesGraphTransformer__Seed_20260905: epoch=08, train=0.21971, valNLL=0.20513
[2026-09-01 17:21:08] DynamicHawkesGraphTransformer__Seed_20260905: epoch=09, train=0.21476, valNLL=0.19915
[2026-09-01 17:21:09] Dynami

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
# =============================================================================
# PYTHON PHASE 3 — COMPONENT ABLATIONS, DYNAMIC LOGIT, AND CRASH-DEFINITION
# ROBUSTNESS
#
# Forecasting Sectoral Crash Risk and Contagion:
# Integrating Dynamic Volatility, Extreme-Value Modelling and Graph Deep Learning
#
# INPUT
# -----
# Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip
#
# WHY PHASE 3 STARTS FROM PHASE 1
# ------------------------------
# Phase 2.2 is frozen as the final neural-architecture evidence. Phase 3 does
# NOT redesign or retune the neural models after observing Test performance.
# We return to the frozen Phase-1 econometric handoff because it contains:
#
#   * the original 1%, 2.5%, and 5% EVT crash labels,
#   * GJR-GARCH / EVT variables,
#   * the internally constructed volatility, liquidity, breadth, dispersion,
#     and market-state predictors,
#   * the original chronological Train / Validation / Test split.
#
# PURPOSE
# -------
# Phase 3 provides focused evidence for the remaining paper hypotheses:
#
# H1. Dynamic volatility adds incremental predictive information for sector
#     crash risk.
# H2. Volatility-adjusted EVT crash identification is more forecastable than a
#     simple fixed return-threshold definition.
# H3/H4. Contagion / graph evidence is already frozen in Phases 1–2.2.
# H5. Forecast contributions vary by sector and horizon.
#
# MODELS USED HERE
# ----------------
# 1. XGBoost — strongest nonlinear ML benchmark from Phase 2.2.
# 2. Dynamic ridge-logit — parsimonious econometric forecasting benchmark.
#
# We deliberately do NOT introduce another neural architecture in Phase 3.
#
# COMPONENT ABLATIONS FOR THE PRIMARY 2.5% EVT TARGET
# ---------------------------------------------------
#   Full
#   NoGARCH
#   NoVolatilityGroup
#   NoTailEVT
#   NoLiquidity
#   NoBreadthDispersion
#   NoMarketState
#
# CRASH-DEFINITION ROBUSTNESS
# ---------------------------
#   EVT_001     : final R 1% GJR-GARCH / EVT crash label
#   EVT_0025    : primary final R 2.5% crash label
#   EVT_005     : final R 5% crash label
#   Fixed_0025  : sector-specific unconditional 2.5% Train return quantile,
#                 applied out of sample on the SAME eligible sector-days as
#                 the primary EVT label.
#
# IMPORTANT COMPARISON RULE
# -------------------------
# Raw Brier scores from different event definitions are not directly treated as
# evidence that one label is "better", because event prevalence/difficulty can
# differ. EVT-vs-fixed comparisons therefore emphasize:
#
#     Brier Skill = 1 - BS_model / BS_expanding_historical
#
# relative to each definition's OWN strictly past-only historical benchmark.
#
# LEAKAGE CONTROLS
# ----------------
# * Train / Validation / Test are chronological.
# * Survival targets stop at split boundaries.
# * Fixed thresholds are estimated on Train only.
# * Imputation/scaling statistics are Train only.
# * Current-event histories are definition-specific and never forward-filled.
# * XGBoost early stopping / Platt calibration use Validation only.
# * Dynamic-logit ridge penalty is chosen on Validation only.
# * Test is used only for final evaluation and bootstrap comparison.
#
# VERSION: 3.0
# DATE: 2026-09-01
# =============================================================================

from __future__ import annotations

import os
import sys
import gc
import json
import math
import random
import shutil
import pickle
import zipfile
import warnings
import platform
import subprocess
import importlib.util
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.linear_model import LogisticRegression

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# =============================================================================
# 0. CONFIGURATION / REPRODUCIBILITY
# =============================================================================

SEED = 20260901
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

INPUT_ZIP = os.getenv(
    "NSE_PHASE1_ZIP",
    "/content/Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip",
)

OUTPUT_ROOT = Path(os.getenv(
    "NSE_PHASE3_OUTPUT_DIR",
    "/content/Sectoral_Crash_Risk_Contagion_Python_Phase3",
))

HORIZONS = [1, 5, 10, 22]
MAX_HORIZON = max(HORIZONS)
LOOKBACK = int(os.getenv("NSE_LOOKBACK", "60"))

# XGBoost settings deliberately match the established Phase-2 family closely.
XGB_MAX_ROUNDS = int(os.getenv("NSE_XGB_MAX_ROUNDS", "1200"))
XGB_EARLY_STOP = int(os.getenv("NSE_XGB_EARLY_STOP", "60"))
XGB_POS_WEIGHT_CAP = float(os.getenv("NSE_XGB_POS_WEIGHT_CAP", "10.0"))

# Dynamic ridge-logit candidate penalties. Selected on Validation only.
LOGIT_C_GRID = [0.01, 0.10, 1.0, 10.0]

# Paired moving-block bootstrap.
BOOTSTRAP_REPS = int(os.getenv("NSE_BOOTSTRAP_REPS", "500"))
BOOTSTRAP_BLOCK = int(os.getenv("NSE_BOOTSTRAP_BLOCK", "22"))

# Summary windows used by XGBoost. The dynamic logit uses a more parsimonious
# subset of the same summary representation.
XGB_WINDOWS = [5, 22, 60]
LOGIT_WINDOWS = [5, 22]

EPS = 1e-8

# =============================================================================
# 1. OUTPUT FOLDERS / LOGGING
# =============================================================================

TABLE_DIR = OUTPUT_ROOT / "01_Tables"
FIG_DIR = OUTPUT_ROOT / "02_Figures"
EXCEL_DIR = OUTPUT_ROOT / "03_Excel"
MODEL_DIR = OUTPUT_ROOT / "04_Model_Objects"
DATA_DIR = OUTPUT_ROOT / "05_Processed_Data"
LOG_DIR = OUTPUT_ROOT / "06_Logs"
ZIP_DIR = OUTPUT_ROOT / "07_Zip"
EXTRACT_DIR = OUTPUT_ROOT / "_Phase1_Extracted"

for directory in [
    OUTPUT_ROOT, TABLE_DIR, FIG_DIR, EXCEL_DIR,
    MODEL_DIR, DATA_DIR, LOG_DIR, ZIP_DIR, EXTRACT_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / "Python_Phase3_run_log.txt"

def log(message: str):
    stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{stamp}] {message}"
    print(line)
    with open(LOG_FILE, "a", encoding="utf-8") as handle:
        handle.write(line + "\n")

log("=" * 100)
log("PYTHON PHASE 3 START — ABLATIONS / DYNAMIC LOGIT / CRASH-DEFINITION ROBUSTNESS")
log(f"Python={sys.version.split()[0]}; platform={platform.platform()}; seed={SEED}")

# =============================================================================
# 2. PACKAGE CHECKS
# =============================================================================

def ensure_package(import_name: str, pip_name: str | None = None):
    if importlib.util.find_spec(import_name) is not None:
        return
    package = pip_name or import_name
    log(f"Installing missing package: {package}")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", package]
    )

ensure_package("xgboost")
import xgboost as xgb

log(f"xgboost={xgb.__version__}")

# =============================================================================
# 3. ALWAYS ASK FOR THE FROZEN PHASE-1 ZIP IN COLAB
# =============================================================================

REQUIRED_PHASE1_MEMBERS = {
    "phase2_deep_learning_master.csv.gz",
    "python_phase1_metadata.json",
    "Table_P12_Final_Stable_Edge_Parameters.csv",
}

def zip_basenames(path: Path) -> set[str]:
    try:
        with zipfile.ZipFile(path, "r") as archive:
            return {
                Path(name).name
                for name in archive.namelist()
                if not name.endswith("/")
            }
    except Exception:
        return set()

def is_valid_phase1_zip(path: Path) -> bool:
    return (
        path.exists()
        and path.suffix.lower() == ".zip"
        and REQUIRED_PHASE1_MEMBERS.issubset(zip_basenames(path))
    )

def resolve_phase1_zip(configured: str) -> Path:
    """
    In Google Colab the user is ALWAYS asked to upload the Phase-1 ZIP.
    The script never silently uses an older ZIP already sitting in /content.
    """
    try:
        from google.colab import files

        while True:
            print(
                "\nPlease upload the frozen Phase-1 file:\n"
                "    Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip\n"
            )
            uploaded = files.upload()

            candidates = [
                Path("/content") / name
                for name in uploaded
                if name.lower().endswith(".zip")
            ]

            valid = [p for p in candidates if is_valid_phase1_zip(p)]

            if len(valid) == 1:
                log(f"Validated uploaded Phase-1 ZIP: {valid[0]}")
                return valid[0]

            if len(valid) > 1:
                print(
                    "\nMore than one valid Phase-1 ZIP was uploaded. "
                    "Please upload exactly one.\n"
                )
                continue

            for candidate in candidates:
                missing = sorted(
                    REQUIRED_PHASE1_MEMBERS - zip_basenames(candidate)
                )
                log(
                    f"Rejected '{candidate.name}'. Missing Phase-1 files: {missing}"
                )

            print(
                "\nThe selected ZIP is not the frozen Phase-1 output. "
                "Please choose:\n"
                "    Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip\n"
            )

    except ImportError:
        configured_path = Path(configured)

        if is_valid_phase1_zip(configured_path):
            return configured_path

        for folder in [Path.cwd(), Path("/mnt/data")]:
            if not folder.exists():
                continue
            for candidate in folder.glob("*.zip"):
                if is_valid_phase1_zip(candidate):
                    return candidate

        raise FileNotFoundError(
            "Could not locate a valid Phase-1 ZIP. "
            "Set NSE_PHASE1_ZIP to the correct file path."
        )

INPUT_ZIP_PATH = resolve_phase1_zip(INPUT_ZIP)
log(f"Accepted Phase-1 ZIP: {INPUT_ZIP_PATH}")

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(INPUT_ZIP_PATH, "r") as archive:
    archive.extractall(EXTRACT_DIR)

def find_one(filename: str) -> Path:
    matches = list(EXTRACT_DIR.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one '{filename}' in Phase-1 ZIP; "
            f"found {len(matches)}."
        )
    return matches[0]

MASTER_FILE = find_one("phase2_deep_learning_master.csv.gz")
META_FILE = find_one("python_phase1_metadata.json")
EDGE_FILE = find_one("Table_P12_Final_Stable_Edge_Parameters.csv")

# =============================================================================
# 4. LOAD / AUDIT MASTER PANEL
# =============================================================================

master = pd.read_csv(MASTER_FILE, parse_dates=["Date"])
master = master.sort_values(["Date", "Sector"]).reset_index(drop=True)

with open(META_FILE, "r", encoding="utf-8") as handle:
    phase1_metadata = json.load(handle)

stable_edges = pd.read_csv(EDGE_FILE)

required_columns = {
    "Date", "Sector", "Split",
    "SectorReturn_Model",
    "GARCH_Sigma", "StdInnovation",
    "Crash_001", "Crash_0025", "Crash_005", "Crash_Main",
}
missing_required = sorted(required_columns - set(master.columns))

if missing_required:
    raise RuntimeError(
        f"Phase-1 modelling master lacks required columns: {missing_required}"
    )

sectors = sorted(master["Sector"].dropna().astype(str).unique().tolist())
S = len(sectors)
sector_to_idx = {sector: i for i, sector in enumerate(sectors)}

dates = pd.DatetimeIndex(sorted(master["Date"].unique()))
T = len(dates)
date_to_idx = {pd.Timestamp(date): i for i, date in enumerate(dates)}

if master.duplicated(["Date", "Sector"]).any():
    raise RuntimeError("Duplicate Date-Sector rows found.")

counts = master.groupby("Date")["Sector"].nunique()
if not (counts == S).all():
    raise RuntimeError("Master is not a complete sector-date panel.")

split_by_date = (
    master[["Date", "Split"]]
    .drop_duplicates()
    .set_index("Date")["Split"]
    .reindex(dates)
)

splits = split_by_date.astype(str).to_numpy()

if set(np.unique(splits)) != {"Train", "Validation", "Test"}:
    raise RuntimeError(f"Unexpected split labels: {np.unique(splits)}")

train_date_mask = splits == "Train"
validation_date_mask = splits == "Validation"
test_date_mask = splits == "Test"
trainval_date_mask = train_date_mask | validation_date_mask

log(
    f"Loaded {len(master):,} rows; {T:,} dates; {S} sectors; "
    f"Train dates={train_date_mask.sum()}, "
    f"Validation dates={validation_date_mask.sum()}, "
    f"Test dates={test_date_mask.sum()}."
)

# =============================================================================
# 5. BUILD EVENT PANELS FOR EVT DEFINITIONS
# =============================================================================

def event_panel_from_column(column: str) -> tuple[np.ndarray, np.ndarray]:
    C = np.zeros((T, S), dtype=np.float32)
    M = np.zeros((T, S), dtype=bool)

    for row in master[["Date", "Sector", column]].itertuples(index=False):
        t = date_to_idx[pd.Timestamp(row.Date)]
        s = sector_to_idx[str(row.Sector)]
        value = getattr(row, column)

        if pd.notna(value):
            C[t, s] = float(value)
            M[t, s] = True

    observed = C[M]
    if len(observed) == 0 or not np.isin(observed, [0.0, 1.0]).all():
        raise RuntimeError(f"Invalid binary event panel for {column}.")

    return C, M

event_definitions: dict[str, dict] = {}

for name, column in [
    ("EVT_001", "Crash_001"),
    ("EVT_0025", "Crash_0025"),
    ("EVT_005", "Crash_005"),
]:
    C, M = event_panel_from_column(column)

    event_definitions[name] = {
        "event": C,
        "observed": M,
        "source": column,
        "description": f"R GJR-GARCH / EVT event label from {column}",
    }

# Crash_Main must coincide with primary EVT_0025 where observed.
C_main, M_main = event_panel_from_column("Crash_Main")
primary_C = event_definitions["EVT_0025"]["event"]
primary_M = event_definitions["EVT_0025"]["observed"]

common_primary = M_main & primary_M
primary_mismatch = int(
    np.sum(
        C_main[common_primary]
        != primary_C[common_primary]
    )
)
if primary_mismatch != 0:
    raise RuntimeError(
        f"Crash_Main differs from Crash_0025 in {primary_mismatch} cells."
    )

# =============================================================================
# 6. TRAIN-ONLY FIXED 2.5% SECTOR RETURN THRESHOLD
# =============================================================================
#
# Fixed-threshold days deliberately use the SAME observation mask as EVT_0025.
# This isolates the event-definition rule from data availability.

return_panel = np.full((T, S), np.nan, dtype=np.float64)

for row in master[
    ["Date", "Sector", "SectorReturn_Model"]
].itertuples(index=False):
    t = date_to_idx[pd.Timestamp(row.Date)]
    s = sector_to_idx[str(row.Sector)]

    if pd.notna(row.SectorReturn_Model):
        return_panel[t, s] = float(row.SectorReturn_Model)

fixed_thresholds = np.full(S, np.nan, dtype=float)
fixed_C = np.zeros((T, S), dtype=np.float32)
fixed_M = primary_M.copy()

fixed_threshold_rows = []

for s, sector in enumerate(sectors):
    threshold_sample = (
        train_date_mask
        & primary_M[:, s]
        & np.isfinite(return_panel[:, s])
    )

    values = return_panel[threshold_sample, s]

    if len(values) < 100:
        raise RuntimeError(
            f"Too few Train returns to estimate fixed threshold for {sector}."
        )

    threshold = float(np.quantile(values, 0.025))
    fixed_thresholds[s] = threshold

    usable = (
        primary_M[:, s]
        & np.isfinite(return_panel[:, s])
    )

    fixed_C[usable, s] = (
        return_panel[usable, s] <= threshold
    ).astype(np.float32)

    # If the primary label exists but return is missing, fixed label cannot exist.
    fixed_M[:, s] = primary_M[:, s] & np.isfinite(return_panel[:, s])

    fixed_threshold_rows.append({
        "Sector": sector,
        "TrainFixedQuantile": 0.025,
        "FixedReturnThreshold": threshold,
        "TrainN": int(len(values)),
        "TrainFixedEventRate": float(
            fixed_C[train_date_mask & fixed_M[:, s], s].mean()
        ),
    })

fixed_threshold_table = pd.DataFrame(fixed_threshold_rows)

fixed_threshold_table.to_csv(
    TABLE_DIR / "Table_301_Fixed_Thresholds_Train_Only.csv",
    index=False,
)

event_definitions["Fixed_0025"] = {
    "event": fixed_C,
    "observed": fixed_M,
    "source": "Train-only sector return 2.5% quantile",
    "description": (
        "Sector-specific unconditional 2.5% Train return threshold, "
        "applied out of sample on primary-EVT eligible days"
    ),
}

# =============================================================================
# 7. SPLIT-AWARE MULTI-HORIZON TARGETS FOR EVERY EVENT DEFINITION
# =============================================================================

def build_split_aware_targets(
    C: np.ndarray,
    M: np.ndarray,
) -> dict[int, np.ndarray]:
    targets = {
        h: np.full((T, S), np.nan, dtype=np.float32)
        for h in HORIZONS
    }

    for t in range(T):
        split_t = splits[t]

        for s in range(S):
            if not M[t, s]:
                continue

            observed_until = 0
            event_k = None

            for k in range(1, MAX_HORIZON + 1):
                future_t = t + k

                if future_t >= T:
                    break

                # Critical anti-leakage boundary.
                if splits[future_t] != split_t:
                    break

                if not M[future_t, s]:
                    break

                observed_until = k

                if C[future_t, s] == 1:
                    event_k = k
                    break

            for h in HORIZONS:
                if event_k is not None and event_k <= h:
                    targets[h][t, s] = 1.0
                elif observed_until >= h:
                    targets[h][t, s] = 0.0
                else:
                    targets[h][t, s] = np.nan

    return targets

for definition in event_definitions.values():
    definition["targets"] = build_split_aware_targets(
        definition["event"],
        definition["observed"],
    )

# Target / prevalence audit.
target_audit_rows = []

for definition_name, definition in event_definitions.items():
    C = definition["event"]
    M = definition["observed"]

    for split_name, split_mask in [
        ("Train", train_date_mask),
        ("Validation", validation_date_mask),
        ("Test", test_date_mask),
    ]:
        same_day_mask = split_mask[:, None] & M

        target_audit_rows.append({
            "Definition": definition_name,
            "Split": split_name,
            "Horizon": 0,
            "ValidTargets": int(same_day_mask.sum()),
            "Events": int(C[same_day_mask].sum()),
            "EventRate": (
                float(C[same_day_mask].mean())
                if same_day_mask.sum()
                else np.nan
            ),
        })

        for h in HORIZONS:
            y = definition["targets"][h][split_mask].reshape(-1)
            ok = np.isfinite(y)

            target_audit_rows.append({
                "Definition": definition_name,
                "Split": split_name,
                "Horizon": h,
                "ValidTargets": int(ok.sum()),
                "Events": int(np.nansum(y)),
                "EventRate": (
                    float(np.nanmean(y))
                    if ok.any()
                    else np.nan
                ),
            })

target_audit = pd.DataFrame(target_audit_rows)

target_audit.to_csv(
    TABLE_DIR / "Table_302_Crash_Definition_Target_Audit.csv",
    index=False,
)

# =============================================================================
# 8. EVT-vs-FIXED EVENT OVERLAP
# =============================================================================

overlap_rows = []

evt_C = event_definitions["EVT_0025"]["event"]
evt_M = event_definitions["EVT_0025"]["observed"]
fix_C = event_definitions["Fixed_0025"]["event"]
fix_M = event_definitions["Fixed_0025"]["observed"]

for split_name, split_mask in [
    ("Train", train_date_mask),
    ("Validation", validation_date_mask),
    ("Test", test_date_mask),
]:
    for s, sector in enumerate(sectors):
        common = split_mask & evt_M[:, s] & fix_M[:, s]

        a = evt_C[common, s].astype(bool)
        b = fix_C[common, s].astype(bool)

        union = np.sum(a | b)
        intersection = np.sum(a & b)

        overlap_rows.append({
            "Split": split_name,
            "Sector": sector,
            "CommonDays": int(common.sum()),
            "EVTEvents": int(a.sum()),
            "FixedEvents": int(b.sum()),
            "CommonEvents": int(intersection),
            "Jaccard": (
                float(intersection / union)
                if union > 0
                else np.nan
            ),
            "AgreementRate": (
                float(np.mean(a == b))
                if len(a)
                else np.nan
            ),
        })

event_overlap = pd.DataFrame(overlap_rows)

event_overlap.to_csv(
    TABLE_DIR / "Table_303_EVT_vs_Fixed_Event_Overlap.csv",
    index=False,
)

# =============================================================================
# 9. FEATURE GOVERNANCE
# =============================================================================

EXCLUDE_COLUMNS = {
    "Date",
    "Sector",
    "Split",
    "PrimarySector",
    "ExclusionReason",
    "EconometricGraphSource",

    # Future survival labels / summaries.
    "EventObservedWithin22",
    "EventTime",
    "CensorTime",
    "CrashWithin_1",
    "CrashWithin_5",
    "CrashWithin_10",
    "CrashWithin_22",

    # Previously generated probability forecasts.
    "FixedHistoricalProb_1",
    "FixedHistoricalProb_5",
    "FixedHistoricalProb_10",
    "FixedHistoricalProb_22",
    "ExpandingHistoricalProb_1",
    "ExpandingHistoricalProb_5",
    "ExpandingHistoricalProb_10",
    "ExpandingHistoricalProb_22",
    "StableHawkesProb_1",
    "StableHawkesProb_5",
    "StableHawkesProb_10",
    "StableHawkesProb_22",

    # All crash labels are excluded. Current event history is injected
    # definition-by-definition below.
    "Crash_001",
    "Crash_0025",
    "Crash_005",
    "Crash_Main",
    "CurrentCrash",
    "CurrentCrashObserved",

    # Direct thresholds.
    "CrashThreshold_001",
    "CrashThreshold_0025",
    "CrashThreshold_005",

    # EVT optimizer diagnostics are not economic predictors.
    "EVT_fit_method",
    "EVT_fit_convergence",
    "EVT_loglik",
    "EVT_nll_improvement",
    "EVT_at_boundary",
    "EVT_refit_id",
}

base_feature_columns = []

for column in master.columns:
    if column in EXCLUDE_COLUMNS:
        continue
    if pd.api.types.is_numeric_dtype(master[column]):
        base_feature_columns.append(column)

forbidden_name_fragments = [
    "CrashWithin_",
    "EventTime",
    "CensorTime",
    "Prob_",
]

base_feature_columns = [
    c for c in base_feature_columns
    if not any(fragment in c for fragment in forbidden_name_fragments)
]

log(f"Leakage-screened base numeric features={len(base_feature_columns)}")

# =============================================================================
# 10. PANELIZE / PAST-ONLY IMPUTE / TRAIN-ONLY STANDARDIZE
# =============================================================================

feature_panel_raw = np.full(
    (T, S, len(base_feature_columns)),
    np.nan,
    dtype=np.float64,
)

for s, sector in enumerate(sectors):
    sector_df = (
        master[master["Sector"].astype(str) == sector]
        .set_index("Date")
        .reindex(dates)
    )

    feature_panel_raw[:, s, :] = (
        sector_df[base_feature_columns]
        .astype(float)
        .to_numpy()
    )

# Past-only forward filling for continuous predictors.
feature_panel_ffill = feature_panel_raw.copy()

for s in range(S):
    frame = pd.DataFrame(feature_panel_ffill[:, s, :])
    feature_panel_ffill[:, s, :] = frame.ffill().to_numpy()

train_values = feature_panel_ffill[train_date_mask]
train_median = np.nanmedian(train_values, axis=(0, 1))

valid_feature_mask = np.isfinite(train_median)

if not valid_feature_mask.all():
    dropped = [
        base_feature_columns[i]
        for i in np.where(~valid_feature_mask)[0]
    ]
    log(f"Dropping all-missing Train features: {dropped}")

    base_feature_columns = [
        base_feature_columns[i]
        for i in np.where(valid_feature_mask)[0]
    ]

    feature_panel_raw = feature_panel_raw[:, :, valid_feature_mask]
    feature_panel_ffill = feature_panel_ffill[:, :, valid_feature_mask]
    train_median = train_median[valid_feature_mask]

# Missingness indicators for variables with >=2% raw Train missingness.
raw_train = feature_panel_raw[train_date_mask]
missing_rate = np.mean(~np.isfinite(raw_train), axis=(0, 1))
missing_indicator_indices = np.where(missing_rate >= 0.02)[0]

missing_indicators = (
    ~np.isfinite(
        feature_panel_raw[:, :, missing_indicator_indices]
    )
).astype(np.float64)

missing_indicator_names = [
    f"MISS__{base_feature_columns[i]}"
    for i in missing_indicator_indices
]

X_numeric = feature_panel_ffill.copy()

for j in range(X_numeric.shape[2]):
    bad = ~np.isfinite(X_numeric[:, :, j])
    X_numeric[:, :, j][bad] = train_median[j]

if len(missing_indicator_indices):
    X_numeric = np.concatenate(
        [X_numeric, missing_indicators],
        axis=2,
    )
    model_feature_names = (
        base_feature_columns + missing_indicator_names
    )
else:
    model_feature_names = base_feature_columns.copy()

# Track every final feature back to its economic origin for ablations.
feature_origin = {}

for name in model_feature_names:
    if name.startswith("MISS__"):
        feature_origin[name] = name.replace("MISS__", "", 1)
    else:
        feature_origin[name] = name

train_block = X_numeric[train_date_mask]
train_mean = train_block.mean(axis=(0, 1))
train_std = train_block.std(axis=(0, 1))
train_std[train_std < 1e-8] = 1.0

X_panel = (
    (X_numeric - train_mean[None, None, :])
    / train_std[None, None, :]
).astype(np.float32)

F_BASE = X_panel.shape[2]

if not np.all(np.isfinite(X_panel)):
    raise RuntimeError("Feature panel contains non-finite values after preprocessing.")

pd.DataFrame({
    "Feature": model_feature_names,
    "OriginFeature": [feature_origin[x] for x in model_feature_names],
    "TrainMean": train_mean,
    "TrainSD": train_std,
}).to_csv(
    TABLE_DIR / "Table_304_Feature_Preprocessing.csv",
    index=False,
)

# =============================================================================
# 11. PRE-SPECIFIED FEATURE GROUPS / ABLATIONS
# =============================================================================

GARCH_GROUP = {
    "GARCH_Sigma",
}

VOLATILITY_GROUP = {
    "GARCH_Sigma",
    "ParkinsonVol",
    "GarmanKlassVol",
    "RogersSatchellVol",
}

TAIL_EVT_GROUP = {
    "StdInnovation",
    "EVT_u",
    "EVT_scale",
    "EVT_shape",
    "EVT_pu",
    "EVT_nhist",
    "EVT_nexc",
}

LIQUIDITY_GROUP = {
    "Turnover_MCW",
    "AmihudILLIQ_Median",
    "ZeroReturnShare",
    "ZeroVolumeShare",
    "MarketTurnover_MCW",
    "MarketAmihudILLIQ_Median",
    "MarketZeroReturnShare",
    "MarketZeroVolumeShare",
}

BREADTH_DISPERSION_GROUP = {
    "BreadthNegative",
    "ReturnDispersion",
    "MarketBreadthNegative",
    "MarketReturnDispersion",
    "NSectorsDown",
    "CrossSectorReturnDispersion",
}

MARKET_STATE_GROUP = {
    c for c in base_feature_columns
    if (
        c.startswith("Market")
        or c in {
            "NStocksObservedMarket",
            "NStocksReturnMarket",
            "NSectorsObserved",
            "NSectorsWithReturn",
            "NSectorsDown",
            "CrossSectorReturnDispersion",
        }
    )
}

ABLATION_DROP_GROUPS = {
    "Full": set(),
    "NoGARCH": GARCH_GROUP,
    "NoVolatilityGroup": VOLATILITY_GROUP,
    "NoTailEVT": TAIL_EVT_GROUP,
    "NoLiquidity": LIQUIDITY_GROUP,
    "NoBreadthDispersion": BREADTH_DISPERSION_GROUP,
    "NoMarketState": MARKET_STATE_GROUP,
}

ablation_design_rows = []

for variant, drop_group in ABLATION_DROP_GROUPS.items():
    present_drop = sorted(
        set(base_feature_columns) & set(drop_group)
    )

    ablation_design_rows.append({
        "Variant": variant,
        "DroppedFeatureCount": len(present_drop),
        "DroppedFeatures": " | ".join(present_drop),
    })

ablation_design = pd.DataFrame(ablation_design_rows)

ablation_design.to_csv(
    TABLE_DIR / "Table_305_Component_Ablation_Design.csv",
    index=False,
)

# =============================================================================
# 12. GENERIC FORECAST-ORIGIN COORDINATES
# =============================================================================

coordinate_rows = []

for t in range(LOOKBACK - 1, T):
    for s, sector in enumerate(sectors):
        coordinate_rows.append({
            "RowID": len(coordinate_rows),
            "t": t,
            "s": s,
            "Date": dates[t],
            "Sector": sector,
            "Split": splits[t],
        })

coordinates = pd.DataFrame(coordinate_rows)

coord_t = coordinates["t"].to_numpy(dtype=int)
coord_s = coordinates["s"].to_numpy(dtype=int)
coord_split = coordinates["Split"].to_numpy()

N_COORD = len(coordinates)

log(f"Generic forecast-origin coordinates={N_COORD:,}")

# =============================================================================
# 13. BASE SUMMARY FEATURE MATRIX
# =============================================================================
#
# Build once. Event-history features are appended separately for each event
# definition so EVT_1%, EVT_2.5%, EVT_5% and Fixed_2.5% remain comparable.

summary_vectors = []
summary_names = []
summary_origins = []

# Names/order for one feature are:
# current, then mean/std/min/max for windows 5,22,60.
for feature_name in model_feature_names:
    summary_names.append(f"CUR__{feature_name}")
    summary_origins.append(feature_origin[feature_name])

    for window in XGB_WINDOWS:
        for stat in ["MEAN", "SD", "MIN", "MAX"]:
            summary_names.append(
                f"{stat}{window}__{feature_name}"
            )
            summary_origins.append(feature_origin[feature_name])

summary_names += [
    f"SECTOR__{sector}"
    for sector in sectors
]
summary_origins += [
    "__SECTOR__"
    for _ in sectors
]

N_SUMMARY = len(summary_names)

log(
    f"Building reusable summary matrix: "
    f"{N_COORD:,} rows x {N_SUMMARY:,} columns."
)

X_summary = np.empty(
    (N_COORD, N_SUMMARY),
    dtype=np.float32,
)

for row_idx, (t, s) in enumerate(zip(coord_t, coord_s)):
    sequence = X_panel[
        t - LOOKBACK + 1:t + 1,
        s,
        :,
    ]

    pieces = [sequence[-1]]

    for window in XGB_WINDOWS:
        block = sequence[-window:]
        pieces.extend([
            block.mean(axis=0),
            block.std(axis=0),
            block.min(axis=0),
            block.max(axis=0),
        ])

    sector_one_hot = np.zeros(S, dtype=np.float32)
    sector_one_hot[s] = 1.0
    pieces.append(sector_one_hot)

    X_summary[row_idx] = np.concatenate(pieces).astype(np.float32)

if not np.all(np.isfinite(X_summary)):
    raise RuntimeError("Summary feature matrix contains non-finite values.")

# =============================================================================
# 14. DEFINITION-SPECIFIC CURRENT EVENT HISTORY FEATURES
# =============================================================================

EVENT_HISTORY_FEATURE_NAMES = [
    "CURRENT_EVENT",
    "EVENT_RATE_5",
    "EVENT_RATE_22",
    "EVENT_RATE_60",
    "EVENT_OBS_SHARE_5",
    "EVENT_OBS_SHARE_22",
    "EVENT_OBS_SHARE_60",
    "DAYS_SINCE_EVENT_CAP60",
]

def build_event_history_matrix(
    C: np.ndarray,
    M: np.ndarray,
) -> np.ndarray:

    out = np.zeros(
        (N_COORD, len(EVENT_HISTORY_FEATURE_NAMES)),
        dtype=np.float32,
    )

    for row_idx, (t, s) in enumerate(zip(coord_t, coord_s)):
        out[row_idx, 0] = C[t, s] if M[t, s] else 0.0

        col = 1

        for window in [5, 22, 60]:
            start = max(0, t - window + 1)
            events = C[start:t + 1, s]
            observed = M[start:t + 1, s]

            n_obs = int(observed.sum())

            out[row_idx, col] = (
                float(events[observed].mean())
                if n_obs > 0
                else 0.0
            )
            col += 1

        for window in [5, 22, 60]:
            start = max(0, t - window + 1)
            observed = M[start:t + 1, s]

            out[row_idx, col] = float(observed.mean())
            col += 1

        # Days since most recent observed crash, capped at 60.
        days_since = 60.0
        for lag in range(0, 60):
            tt = t - lag
            if tt < 0:
                break
            if M[tt, s] and C[tt, s] == 1:
                days_since = float(lag)
                break

        out[row_idx, col] = days_since / 60.0

    return out

event_history_matrices = {}

for definition_name, definition in event_definitions.items():
    event_history_matrices[definition_name] = build_event_history_matrix(
        definition["event"],
        definition["observed"],
    )

# =============================================================================
# 15. FEATURE SELECTION FOR EACH ABLATION
# =============================================================================

summary_origins_array = np.array(summary_origins, dtype=object)

def ablation_summary_indices(variant: str) -> np.ndarray:
    drop_group = ABLATION_DROP_GROUPS[variant]

    keep = np.array([
        (
            origin == "__SECTOR__"
            or origin not in drop_group
        )
        for origin in summary_origins_array
    ])

    return np.where(keep)[0]

ablation_indices = {
    variant: ablation_summary_indices(variant)
    for variant in ABLATION_DROP_GROUPS
}

# =============================================================================
# 16. TARGET VECTOR ACCESS
# =============================================================================

def target_vector(
    definition_name: str,
    horizon: int,
) -> np.ndarray:
    panel = event_definitions[
        definition_name
    ]["targets"][horizon]

    return np.array([
        panel[t, s]
        for t, s in zip(coord_t, coord_s)
    ], dtype=float)

def current_observed_vector(
    definition_name: str,
) -> np.ndarray:
    M = event_definitions[
        definition_name
    ]["observed"]

    return np.array([
        M[t, s]
        for t, s in zip(coord_t, coord_s)
    ], dtype=bool)

# =============================================================================
# 17. STRICT HISTORICAL BASELINES FOR EACH DEFINITION
# =============================================================================

historical_probabilities: dict[str, dict[str, dict[int, np.ndarray]]] = {}

for definition_name, definition in event_definitions.items():
    C = definition["event"]
    M = definition["observed"]

    fixed_prob = {
        h: np.full((T, S), np.nan, dtype=np.float32)
        for h in HORIZONS
    }
    expanding_prob = {
        h: np.full((T, S), np.nan, dtype=np.float32)
        for h in HORIZONS
    }

    train_rates = np.zeros(S, dtype=float)

    for s in range(S):
        valid = train_date_mask & M[:, s]
        train_rates[s] = (
            float(C[valid, s].mean())
            if valid.sum()
            else 0.025
        )

    for h in HORIZONS:
        fixed_prob[h][:] = (
            1.0
            - (1.0 - train_rates[None, :]) ** h
        )

    event_sum = np.zeros(S, dtype=float)
    event_n = np.zeros(S, dtype=float)

    for t in range(T):
        for s in range(S):
            if M[t, s]:
                event_sum[s] += C[t, s]
                event_n[s] += 1.0

        one_day_p = np.divide(
            event_sum,
            np.maximum(event_n, 1.0),
        )

        for h in HORIZONS:
            expanding_prob[h][t] = (
                1.0
                - (1.0 - one_day_p) ** h
            )

    historical_probabilities[definition_name] = {
        "FixedHistorical": fixed_prob,
        "ExpandingHistorical": expanding_prob,
    }

# =============================================================================
# 18. METRIC UTILITIES
# =============================================================================

def log_score(
    y: np.ndarray,
    p: np.ndarray,
) -> float:
    p = np.clip(p, EPS, 1.0 - EPS)

    return float(
        -np.mean(
            y * np.log(p)
            + (1.0 - y) * np.log1p(-p)
        )
    )

def fit_calibration_intercept_slope(
    y: np.ndarray,
    p: np.ndarray,
) -> tuple[float, float]:

    y = np.asarray(y, dtype=float)
    p = np.clip(
        np.asarray(p, dtype=float),
        EPS,
        1.0 - EPS,
    )

    if len(np.unique(y)) < 2:
        return np.nan, np.nan

    x = np.log(p / (1.0 - p))

    def objective(theta):
        intercept, slope = theta
        z = intercept + slope * x
        q = 1.0 / (
            1.0 + np.exp(-np.clip(z, -30, 30))
        )
        q = np.clip(q, EPS, 1.0 - EPS)

        return float(
            -np.mean(
                y * np.log(q)
                + (1.0 - y) * np.log1p(-q)
            )
        )

    result = minimize(
        objective,
        x0=np.array([0.0, 1.0]),
        method="L-BFGS-B",
        bounds=[
            (-20.0, 20.0),
            (-10.0, 10.0),
        ],
    )

    return float(result.x[0]), float(result.x[1])

def probability_metrics(
    y_true: np.ndarray,
    probability: np.ndarray,
) -> dict:

    y_true = np.asarray(y_true, dtype=float)
    probability = np.asarray(probability, dtype=float)

    ok = np.isfinite(y_true) & np.isfinite(probability)

    y = y_true[ok]
    p = np.clip(
        probability[ok],
        EPS,
        1.0 - EPS,
    )

    result = {
        "N": int(len(y)),
        "Events": int(y.sum()) if len(y) else 0,
        "EventRate": float(y.mean()) if len(y) else np.nan,
        "Brier": np.nan,
        "LogScore": np.nan,
        "PR_AUC": np.nan,
        "ROC_AUC": np.nan,
        "CalibrationIntercept": np.nan,
        "CalibrationSlope": np.nan,
    }

    if len(y) == 0:
        return result

    result["Brier"] = float(
        np.mean((p - y) ** 2)
    )
    result["LogScore"] = log_score(y, p)

    if len(np.unique(y)) == 2:
        result["PR_AUC"] = float(
            average_precision_score(y, p)
        )
        result["ROC_AUC"] = float(
            roc_auc_score(y, p)
        )

        intercept, slope = fit_calibration_intercept_slope(
            y,
            p,
        )

        result["CalibrationIntercept"] = intercept
        result["CalibrationSlope"] = slope

    return result

# =============================================================================
# 19. XGBOOST CALIBRATION
# =============================================================================

def fit_binary_platt(
    p_validation: np.ndarray,
    y_validation: np.ndarray,
) -> tuple[float, float]:

    p_validation = np.clip(
        p_validation,
        EPS,
        1.0 - EPS,
    )

    x = np.log(
        p_validation / (1.0 - p_validation)
    )
    y = y_validation.astype(float)

    def objective(theta):
        intercept, slope = theta
        z = intercept + slope * x
        q = 1.0 / (
            1.0 + np.exp(-np.clip(z, -30, 30))
        )
        q = np.clip(q, EPS, 1.0 - EPS)

        return float(
            -np.mean(
                y * np.log(q)
                + (1.0 - y) * np.log1p(-q)
            )
        )

    result = minimize(
        objective,
        x0=np.array([0.0, 1.0]),
        method="L-BFGS-B",
        bounds=[
            (-10.0, 10.0),
            (0.0, 10.0),
        ],
    )

    return float(result.x[0]), float(result.x[1])

def apply_binary_platt(
    p: np.ndarray,
    intercept: float,
    slope: float,
) -> np.ndarray:

    p = np.clip(p, EPS, 1.0 - EPS)
    x = np.log(p / (1.0 - p))
    z = intercept + slope * x

    return 1.0 / (
        1.0 + np.exp(-np.clip(z, -30, 30))
    )

# =============================================================================
# 20. MODEL-FITTING HELPERS
# =============================================================================

def assemble_model_matrix(
    definition_name: str,
    variant: str,
) -> tuple[np.ndarray, list[str]]:

    idx = ablation_indices[variant]

    X = np.concatenate(
        [
            X_summary[:, idx],
            event_history_matrices[definition_name],
        ],
        axis=1,
    ).astype(np.float32)

    names = (
        [summary_names[i] for i in idx]
        + EVENT_HISTORY_FEATURE_NAMES
    )

    return X, names

def fit_xgboost_horizon(
    X: np.ndarray,
    y: np.ndarray,
    horizon: int,
    model_tag: str,
) -> dict:

    train_idx = (
        (coord_split == "Train")
        & np.isfinite(y)
    )
    val_idx = (
        (coord_split == "Validation")
        & np.isfinite(y)
    )
    test_idx = (
        (coord_split == "Test")
        & np.isfinite(y)
    )

    y_train = y[train_idx].astype(int)
    y_val = y[val_idx].astype(int)

    if len(np.unique(y_train)) < 2:
        raise RuntimeError(
            f"XGBoost Train target has one class: {model_tag}, h={horizon}"
        )

    positive = max(int(y_train.sum()), 1)
    negative = max(len(y_train) - positive, 1)

    scale_pos_weight = min(
        math.sqrt(negative / positive),
        XGB_POS_WEIGHT_CAP,
    )

    dtrain = xgb.DMatrix(
        X[train_idx],
        label=y_train,
    )
    dval = xgb.DMatrix(
        X[val_idx],
        label=y_val,
    )
    dtest = xgb.DMatrix(
        X[test_idx],
    )

    params = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "eta": 0.03,
        "max_depth": 4,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "lambda": 1.0,
        "alpha": 0.0,
        "scale_pos_weight": scale_pos_weight,
        "seed": SEED,
        "nthread": max(1, os.cpu_count() or 1),
        "tree_method": "hist",
    }

    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=XGB_MAX_ROUNDS,
        evals=[(dval, "validation")],
        early_stopping_rounds=XGB_EARLY_STOP,
        verbose_eval=False,
    )

    best_iteration = int(
        booster.best_iteration
        if booster.best_iteration is not None
        else XGB_MAX_ROUNDS - 1
    )

    p_val_raw = booster.predict(
        dval,
        iteration_range=(0, best_iteration + 1),
    )

    intercept, slope = fit_binary_platt(
        p_val_raw,
        y_val,
    )

    p_val = apply_binary_platt(
        p_val_raw,
        intercept,
        slope,
    )

    p_test_raw = booster.predict(
        dtest,
        iteration_range=(0, best_iteration + 1),
    )

    p_test = apply_binary_platt(
        p_test_raw,
        intercept,
        slope,
    )

    return {
        "booster": booster,
        "best_iteration": best_iteration,
        "scale_pos_weight": float(scale_pos_weight),
        "platt_intercept": intercept,
        "platt_slope": slope,
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
        "p_val": p_val,
        "p_test": p_test,
    }

def fit_dynamic_logit_horizon(
    X: np.ndarray,
    y: np.ndarray,
    horizon: int,
    model_tag: str,
) -> dict:

    train_idx = (
        (coord_split == "Train")
        & np.isfinite(y)
    )
    val_idx = (
        (coord_split == "Validation")
        & np.isfinite(y)
    )
    test_idx = (
        (coord_split == "Test")
        & np.isfinite(y)
    )

    X_train = X[train_idx].astype(np.float64)
    X_val = X[val_idx].astype(np.float64)
    X_test = X[test_idx].astype(np.float64)

    y_train = y[train_idx].astype(int)
    y_val = y[val_idx].astype(int)

    # Train-only scaling at the final logit-design level.
    mean = X_train.mean(axis=0)
    sd = X_train.std(axis=0)

    keep = (
        np.isfinite(mean)
        & np.isfinite(sd)
        & (sd > 1e-8)
    )

    X_train = (X_train[:, keep] - mean[keep]) / sd[keep]
    X_val = (X_val[:, keep] - mean[keep]) / sd[keep]
    X_test = (X_test[:, keep] - mean[keep]) / sd[keep]

    best = None
    tuning_rows = []

    for C_value in LOGIT_C_GRID:
        model = LogisticRegression(
            penalty="l2",
            C=C_value,
            solver="lbfgs",
            max_iter=2000,
            fit_intercept=True,
            class_weight=None,
            random_state=SEED,
        )

        model.fit(
            X_train,
            y_train,
        )

        p_val = model.predict_proba(
            X_val
        )[:, 1]

        val_loss = log_score(
            y_val,
            p_val,
        )

        tuning_rows.append({
            "C": C_value,
            "ValidationLogScore": val_loss,
        })

        if best is None or val_loss < best["ValidationLogScore"]:
            best = {
                "C": C_value,
                "ValidationLogScore": val_loss,
                "model": model,
                "p_val": p_val,
            }

    if best is None:
        raise RuntimeError(
            f"Dynamic logit fitting failed: {model_tag}, h={horizon}"
        )

    p_test = best["model"].predict_proba(
        X_test
    )[:, 1]

    return {
        "model": best["model"],
        "selected_C": float(best["C"]),
        "validation_logscore": float(best["ValidationLogScore"]),
        "keep_columns": keep,
        "design_mean": mean,
        "design_sd": sd,
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
        "p_val": best["p_val"],
        "p_test": p_test,
        "tuning": pd.DataFrame(tuning_rows),
    }

# =============================================================================
# 21. PROBABILITY-PANEL HELPERS / HORIZON MONOTONICITY
# =============================================================================

def blank_probability_panels() -> dict[str, dict[int, np.ndarray]]:
    return {
        "Validation": {
            h: np.full((T, S), np.nan, dtype=np.float32)
            for h in HORIZONS
        },
        "Test": {
            h: np.full((T, S), np.nan, dtype=np.float32)
            for h in HORIZONS
        },
    }

def scatter_predictions_to_panel(
    panel_dict: dict[str, dict[int, np.ndarray]],
    split_name: str,
    horizon: int,
    row_mask: np.ndarray,
    probabilities: np.ndarray,
):
    selected_rows = np.where(row_mask)[0]

    if len(selected_rows) != len(probabilities):
        raise RuntimeError("Prediction scatter length mismatch.")

    for local_idx, row_idx in enumerate(selected_rows):
        t = coord_t[row_idx]
        s = coord_s[row_idx]
        panel_dict[split_name][horizon][t, s] = float(probabilities[local_idx])

def monotone_rearrange(
    panel_dict: dict[str, dict[int, np.ndarray]],
):
    for split_name in ["Validation", "Test"]:
        stack = np.stack(
            [panel_dict[split_name][h] for h in HORIZONS],
            axis=-1,
        )

        finite_all = np.all(np.isfinite(stack), axis=-1)

        rearranged = np.maximum.accumulate(stack, axis=-1)

        for k, h in enumerate(HORIZONS):
            target_panel = panel_dict[split_name][h]
            target_panel[finite_all] = rearranged[..., k][finite_all]

# =============================================================================
# 22. RUN ONE MODEL FAMILY ACROSS FOUR HORIZONS
# =============================================================================

def run_model_family(
    definition_name: str,
    variant: str,
    model_family: str,
    save_primary_objects: bool = False,
) -> dict:

    X, feature_names = assemble_model_matrix(
        definition_name,
        variant,
    )

    probability_panels = blank_probability_panels()
    fit_metadata_rows = []
    importance_rows = []
    logit_tuning_frames = []

    for h in HORIZONS:
        y = target_vector(
            definition_name,
            h,
        )

        tag = (
            f"{model_family}__{definition_name}__{variant}"
        )

        log(
            f"Fitting {model_family}: definition={definition_name}, "
            f"variant={variant}, h={h}"
        )

        if model_family == "XGBoost":
            fit = fit_xgboost_horizon(
                X=X,
                y=y,
                horizon=h,
                model_tag=tag,
            )

            scatter_predictions_to_panel(
                probability_panels,
                "Validation",
                h,
                fit["val_idx"],
                fit["p_val"],
            )
            scatter_predictions_to_panel(
                probability_panels,
                "Test",
                h,
                fit["test_idx"],
                fit["p_test"],
            )

            fit_metadata_rows.append({
                "ModelFamily": model_family,
                "Definition": definition_name,
                "Variant": variant,
                "Horizon": h,
                "BestIteration": fit["best_iteration"],
                "ScalePosWeight": fit["scale_pos_weight"],
                "PlattIntercept": fit["platt_intercept"],
                "PlattSlope": fit["platt_slope"],
                "SelectedC": np.nan,
            })

            # Gain importance.
            score = fit["booster"].get_score(
                importance_type="gain"
            )

            for raw_name, gain in score.items():
                # xgboost reports f0, f1, ...
                if raw_name.startswith("f"):
                    j = int(raw_name[1:])
                    if j < len(feature_names):
                        feature_name = feature_names[j]
                    else:
                        feature_name = raw_name
                else:
                    feature_name = raw_name

                importance_rows.append({
                    "Definition": definition_name,
                    "Variant": variant,
                    "Horizon": h,
                    "Feature": feature_name,
                    "Gain": float(gain),
                })

            if save_primary_objects:
                model_path = (
                    MODEL_DIR
                    / f"XGBoost_{definition_name}_{variant}_h{h}.json"
                )
                fit["booster"].save_model(model_path)

        elif model_family == "DynamicLogit":
            fit = fit_dynamic_logit_horizon(
                X=X,
                y=y,
                horizon=h,
                model_tag=tag,
            )

            scatter_predictions_to_panel(
                probability_panels,
                "Validation",
                h,
                fit["val_idx"],
                fit["p_val"],
            )
            scatter_predictions_to_panel(
                probability_panels,
                "Test",
                h,
                fit["test_idx"],
                fit["p_test"],
            )

            fit_metadata_rows.append({
                "ModelFamily": model_family,
                "Definition": definition_name,
                "Variant": variant,
                "Horizon": h,
                "BestIteration": np.nan,
                "ScalePosWeight": np.nan,
                "PlattIntercept": np.nan,
                "PlattSlope": np.nan,
                "SelectedC": fit["selected_C"],
            })

            tuning = fit["tuning"].copy()
            tuning["Definition"] = definition_name
            tuning["Variant"] = variant
            tuning["Horizon"] = h
            logit_tuning_frames.append(tuning)

            if save_primary_objects:
                with open(
                    MODEL_DIR
                    / f"DynamicLogit_{definition_name}_{variant}_h{h}.pkl",
                    "wb",
                ) as handle:
                    pickle.dump(
                        {
                            "model": fit["model"],
                            "keep_columns": fit["keep_columns"],
                            "design_mean": fit["design_mean"],
                            "design_sd": fit["design_sd"],
                            "feature_names": feature_names,
                            "selected_C": fit["selected_C"],
                        },
                        handle,
                    )

        else:
            raise ValueError(
                f"Unknown model family: {model_family}"
            )

        gc.collect()

    # Force coherent 1/5/10/22 horizon probabilities for both model families.
    monotone_rearrange(probability_panels)

    return {
        "probabilities": probability_panels,
        "fit_metadata": pd.DataFrame(fit_metadata_rows),
        "importance": (
            pd.DataFrame(importance_rows)
            if importance_rows
            else pd.DataFrame()
        ),
        "logit_tuning": (
            pd.concat(logit_tuning_frames, ignore_index=True)
            if logit_tuning_frames
            else pd.DataFrame()
        ),
    }

# =============================================================================
# 23. PHASE 3A — PRIMARY 2.5% COMPONENT ABLATIONS
# =============================================================================

component_results: dict[tuple[str, str], dict] = {}
all_fit_metadata = []
all_importance = []
all_logit_tuning = []

for model_family in [
    "XGBoost",
    "DynamicLogit",
]:
    for variant in ABLATION_DROP_GROUPS:

        save_objects = (
            variant == "Full"
            and model_family in {"XGBoost", "DynamicLogit"}
        )

        result = run_model_family(
            definition_name="EVT_0025",
            variant=variant,
            model_family=model_family,
            save_primary_objects=save_objects,
        )

        component_results[
            (model_family, variant)
        ] = result["probabilities"]

        all_fit_metadata.append(
            result["fit_metadata"]
        )

        if not result["importance"].empty:
            all_importance.append(
                result["importance"]
            )

        if not result["logit_tuning"].empty:
            all_logit_tuning.append(
                result["logit_tuning"]
            )

# =============================================================================
# 24. PHASE 3B — CRASH-DEFINITION ROBUSTNESS
# =============================================================================
#
# EVT_0025 Full already exists above and is reused. Only the other definitions
# require additional fits.

robustness_results: dict[tuple[str, str], dict] = {}

for model_family in [
    "XGBoost",
    "DynamicLogit",
]:
    robustness_results[
        (model_family, "EVT_0025")
    ] = component_results[
        (model_family, "Full")
    ]

    for definition_name in [
        "EVT_001",
        "EVT_005",
        "Fixed_0025",
    ]:
        result = run_model_family(
            definition_name=definition_name,
            variant="Full",
            model_family=model_family,
            save_primary_objects=False,
        )

        robustness_results[
            (model_family, definition_name)
        ] = result["probabilities"]

        all_fit_metadata.append(
            result["fit_metadata"]
        )

        if not result["importance"].empty:
            all_importance.append(
                result["importance"]
            )

        if not result["logit_tuning"].empty:
            all_logit_tuning.append(
                result["logit_tuning"]
            )

fit_metadata = pd.concat(
    all_fit_metadata,
    ignore_index=True,
)

fit_metadata.to_csv(
    TABLE_DIR / "Table_306_Model_Fitting_Metadata.csv",
    index=False,
)

if all_logit_tuning:
    logit_tuning = pd.concat(
        all_logit_tuning,
        ignore_index=True,
    )

    logit_tuning.to_csv(
        TABLE_DIR / "Table_307_Dynamic_Logit_Validation_Tuning.csv",
        index=False,
    )
else:
    logit_tuning = pd.DataFrame()

# =============================================================================
# 25. METRICS FOR COMPONENT ABLATIONS
# =============================================================================

component_metric_rows = []

primary_targets = event_definitions[
    "EVT_0025"
]["targets"]

for model_family in [
    "XGBoost",
    "DynamicLogit",
]:
    for variant in ABLATION_DROP_GROUPS:

        predictions = component_results[
            (model_family, variant)
        ]

        for split_name, split_mask in [
            ("Validation", validation_date_mask),
            ("Test", test_date_mask),
        ]:
            for h in HORIZONS:
                y_panel = primary_targets[h]
                p_panel = predictions[split_name][h]

                pooled = probability_metrics(
                    y_panel[split_mask].reshape(-1),
                    p_panel[split_mask].reshape(-1),
                )

                component_metric_rows.append({
                    "Split": split_name,
                    "ModelFamily": model_family,
                    "Variant": variant,
                    "Sector": "POOLED",
                    "Horizon": h,
                    **pooled,
                })

                if split_name == "Test":
                    for s, sector in enumerate(sectors):
                        sector_metrics = probability_metrics(
                            y_panel[split_mask, s],
                            p_panel[split_mask, s],
                        )

                        component_metric_rows.append({
                            "Split": split_name,
                            "ModelFamily": model_family,
                            "Variant": variant,
                            "Sector": sector,
                            "Horizon": h,
                            **sector_metrics,
                        })

component_metrics = pd.DataFrame(
    component_metric_rows
)

component_metrics.to_csv(
    TABLE_DIR / "Table_308_Component_Ablation_Metrics.csv",
    index=False,
)

# =============================================================================
# 26. COMPONENT BRIER SKILL / DEGRADATION RELATIVE TO FULL
# =============================================================================

component_pooled_test = component_metrics[
    (component_metrics["Split"] == "Test")
    & (component_metrics["Sector"] == "POOLED")
].copy()

full_reference = (
    component_pooled_test[
        component_pooled_test["Variant"] == "Full"
    ][
        ["ModelFamily", "Horizon", "Brier", "LogScore", "PR_AUC"]
    ]
    .rename(columns={
        "Brier": "FullBrier",
        "LogScore": "FullLogScore",
        "PR_AUC": "FullPR_AUC",
    })
)

component_skill = component_pooled_test.merge(
    full_reference,
    on=["ModelFamily", "Horizon"],
    how="left",
)

component_skill["BrierDegradation_vs_Full"] = (
    component_skill["Brier"]
    - component_skill["FullBrier"]
)

component_skill["BrierSkill_Full_vs_Ablation"] = (
    1.0
    - component_skill["FullBrier"]
    / component_skill["Brier"]
)

component_skill["LogScoreDegradation_vs_Full"] = (
    component_skill["LogScore"]
    - component_skill["FullLogScore"]
)

component_skill["PR_AUCLoss_vs_Full"] = (
    component_skill["FullPR_AUC"]
    - component_skill["PR_AUC"]
)

component_skill.to_csv(
    TABLE_DIR / "Table_309_Component_Contribution_Test.csv",
    index=False,
)

# =============================================================================
# 27. METRICS FOR CRASH-DEFINITION ROBUSTNESS
# =============================================================================

robustness_metric_rows = []

for definition_name, definition in event_definitions.items():
    targets = definition["targets"]

    # Definition-specific historical baselines.
    for baseline_name in [
        "FixedHistorical",
        "ExpandingHistorical",
    ]:
        baseline = historical_probabilities[
            definition_name
        ][baseline_name]

        for split_name, split_mask in [
            ("Validation", validation_date_mask),
            ("Test", test_date_mask),
        ]:
            for h in HORIZONS:
                metrics = probability_metrics(
                    targets[h][split_mask].reshape(-1),
                    baseline[h][split_mask].reshape(-1),
                )

                robustness_metric_rows.append({
                    "Definition": definition_name,
                    "Split": split_name,
                    "ModelFamily": baseline_name,
                    "Sector": "POOLED",
                    "Horizon": h,
                    **metrics,
                })

    # Fitted forecasting models.
    for model_family in [
        "XGBoost",
        "DynamicLogit",
    ]:
        predictions = robustness_results[
            (model_family, definition_name)
        ]

        for split_name, split_mask in [
            ("Validation", validation_date_mask),
            ("Test", test_date_mask),
        ]:
            for h in HORIZONS:
                pooled = probability_metrics(
                    targets[h][split_mask].reshape(-1),
                    predictions[split_name][h][split_mask].reshape(-1),
                )

                robustness_metric_rows.append({
                    "Definition": definition_name,
                    "Split": split_name,
                    "ModelFamily": model_family,
                    "Sector": "POOLED",
                    "Horizon": h,
                    **pooled,
                })

                if split_name == "Test":
                    for s, sector in enumerate(sectors):
                        sector_metrics = probability_metrics(
                            targets[h][split_mask, s],
                            predictions[split_name][h][split_mask, s],
                        )

                        robustness_metric_rows.append({
                            "Definition": definition_name,
                            "Split": split_name,
                            "ModelFamily": model_family,
                            "Sector": sector,
                            "Horizon": h,
                            **sector_metrics,
                        })

robustness_metrics = pd.DataFrame(
    robustness_metric_rows
)

robustness_metrics.to_csv(
    TABLE_DIR / "Table_310_Crash_Definition_Robustness_Metrics.csv",
    index=False,
)

# =============================================================================
# 28. DEFINITION-SPECIFIC BRIER SKILL VS OWN EXPANDING HISTORICAL BASELINE
# =============================================================================

robustness_pooled_test = robustness_metrics[
    (robustness_metrics["Split"] == "Test")
    & (robustness_metrics["Sector"] == "POOLED")
].copy()

historical_reference = (
    robustness_pooled_test[
        robustness_pooled_test["ModelFamily"] == "ExpandingHistorical"
    ][
        ["Definition", "Horizon", "Brier", "LogScore"]
    ]
    .rename(columns={
        "Brier": "HistoricalBrier",
        "LogScore": "HistoricalLogScore",
    })
)

definition_skill = robustness_pooled_test.merge(
    historical_reference,
    on=["Definition", "Horizon"],
    how="left",
)

definition_skill["BrierSkill_vs_OwnHistorical"] = (
    1.0
    - definition_skill["Brier"]
    / definition_skill["HistoricalBrier"]
)

definition_skill["LogScoreImprovement_vs_OwnHistorical"] = (
    definition_skill["HistoricalLogScore"]
    - definition_skill["LogScore"]
)

definition_skill.to_csv(
    TABLE_DIR / "Table_311_Crash_Definition_Brier_Skill.csv",
    index=False,
)

# =============================================================================
# 29. MOVING-BLOCK BOOTSTRAP UTILITIES
# =============================================================================

def moving_block_sample_indices(
    n_dates: int,
    block_length: int,
    rng: np.random.Generator,
) -> np.ndarray:

    selected = []

    while len(selected) < n_dates:
        if n_dates <= block_length:
            start = 0
        else:
            start = int(
                rng.integers(
                    0,
                    n_dates - block_length + 1,
                )
            )

        selected.extend(
            range(
                start,
                min(n_dates, start + block_length),
            )
        )

    return np.array(
        selected[:n_dates],
        dtype=int,
    )

def date_level_losses(
    y_panel: np.ndarray,
    p_panel: np.ndarray,
    date_indices: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:

    brier = np.full(len(date_indices), np.nan, dtype=float)
    logloss = np.full(len(date_indices), np.nan, dtype=float)

    for local_t, t in enumerate(date_indices):
        y = y_panel[t]
        p = p_panel[t]

        ok = np.isfinite(y) & np.isfinite(p)

        if not ok.any():
            continue

        yy = y[ok]
        pp = np.clip(
            p[ok],
            EPS,
            1.0 - EPS,
        )

        brier[local_t] = float(
            np.mean((pp - yy) ** 2)
        )

        logloss[local_t] = float(
            -np.mean(
                yy * np.log(pp)
                + (1.0 - yy) * np.log1p(-pp)
            )
        )

    return brier, logloss

test_date_indices = np.where(test_date_mask)[0]

# =============================================================================
# 30. PAIRED BOOTSTRAP — FULL MODEL VS COMPONENT ABLATIONS
# =============================================================================

component_bootstrap_rows = []

for model_family in [
    "XGBoost",
    "DynamicLogit",
]:
    full_predictions = component_results[
        (model_family, "Full")
    ]

    for variant in ABLATION_DROP_GROUPS:
        if variant == "Full":
            continue

        ablated_predictions = component_results[
            (model_family, variant)
        ]

        for h in HORIZONS:
            y_panel = primary_targets[h]

            full_brier, full_log = date_level_losses(
                y_panel,
                full_predictions["Test"][h],
                test_date_indices,
            )

            abl_brier, abl_log = date_level_losses(
                y_panel,
                ablated_predictions["Test"][h],
                test_date_indices,
            )

            valid = (
                np.isfinite(full_brier)
                & np.isfinite(abl_brier)
                & np.isfinite(full_log)
                & np.isfinite(abl_log)
            )

            fb = full_brier[valid]
            ab = abl_brier[valid]
            fl = full_log[valid]
            al = abl_log[valid]

            observed_brier_diff = float(
                np.mean(fb - ab)
            )
            observed_log_diff = float(
                np.mean(fl - al)
            )

            rng = np.random.default_rng(
                SEED
                + 1000 * h
                + sum(ord(c) for c in f"{model_family}{variant}")
            )

            brier_diffs = []
            log_diffs = []

            for _ in range(BOOTSTRAP_REPS):
                idx = moving_block_sample_indices(
                    len(fb),
                    min(BOOTSTRAP_BLOCK, len(fb)),
                    rng,
                )

                brier_diffs.append(
                    float(np.mean(fb[idx] - ab[idx]))
                )

                log_diffs.append(
                    float(np.mean(fl[idx] - al[idx]))
                )

            brier_diffs = np.asarray(brier_diffs)
            log_diffs = np.asarray(log_diffs)

            component_bootstrap_rows.append({
                "ModelFamily": model_family,
                "Ablation": variant,
                "Horizon": h,
                "BrierDiff_FullMinusAblation": observed_brier_diff,
                "BrierDiff_CI2.5": float(
                    np.quantile(brier_diffs, 0.025)
                ),
                "BrierDiff_CI97.5": float(
                    np.quantile(brier_diffs, 0.975)
                ),
                "Probability_Full_Better_Brier": float(
                    np.mean(brier_diffs < 0)
                ),
                "LogScoreDiff_FullMinusAblation": observed_log_diff,
                "LogScoreDiff_CI2.5": float(
                    np.quantile(log_diffs, 0.025)
                ),
                "LogScoreDiff_CI97.5": float(
                    np.quantile(log_diffs, 0.975)
                ),
                "Probability_Full_Better_LogScore": float(
                    np.mean(log_diffs < 0)
                ),
                "BootstrapReps": BOOTSTRAP_REPS,
                "BlockLength": BOOTSTRAP_BLOCK,
            })

component_bootstrap = pd.DataFrame(
    component_bootstrap_rows
)

component_bootstrap.to_csv(
    TABLE_DIR / "Table_312_Component_Ablation_Block_Bootstrap.csv",
    index=False,
)

# Focused H1 evidence.
h1_bootstrap = component_bootstrap[
    component_bootstrap["Ablation"].isin(
        ["NoGARCH", "NoVolatilityGroup"]
    )
].copy()

h1_bootstrap.to_csv(
    TABLE_DIR / "Table_313_H1_Volatility_Incremental_Evidence.csv",
    index=False,
)

# =============================================================================
# 31. EVT-vs-FIXED FORECASTABILITY BOOTSTRAP
# =============================================================================
#
# Different crash definitions produce different outcomes, so we do NOT compare
# their raw Brier scores as if they were the same target. Instead each fitted
# model is normalized by its OWN expanding-historical benchmark:
#
#     Skill = 1 - ModelBS / HistoricalBS.
#
# The bootstrap reports:
#
#     Skill_EVT_0025 - Skill_Fixed_0025
#
# Positive values favor the volatility-adjusted EVT definition.

definition_skill_bootstrap_rows = []

for model_family in [
    "XGBoost",
    "DynamicLogit",
]:
    for h in HORIZONS:

        evt_target = event_definitions[
            "EVT_0025"
        ]["targets"][h]

        fix_target = event_definitions[
            "Fixed_0025"
        ]["targets"][h]

        evt_model = robustness_results[
            (model_family, "EVT_0025")
        ]["Test"][h]

        fix_model = robustness_results[
            (model_family, "Fixed_0025")
        ]["Test"][h]

        evt_hist = historical_probabilities[
            "EVT_0025"
        ]["ExpandingHistorical"][h]

        fix_hist = historical_probabilities[
            "Fixed_0025"
        ]["ExpandingHistorical"][h]

        evt_model_b, _ = date_level_losses(
            evt_target,
            evt_model,
            test_date_indices,
        )
        evt_hist_b, _ = date_level_losses(
            evt_target,
            evt_hist,
            test_date_indices,
        )

        fix_model_b, _ = date_level_losses(
            fix_target,
            fix_model,
            test_date_indices,
        )
        fix_hist_b, _ = date_level_losses(
            fix_target,
            fix_hist,
            test_date_indices,
        )

        # Calendar-date sampling is paired, but each definition averages its own
        # finite date-level losses within a bootstrap draw.
        n_dates = len(test_date_indices)

        def skill(model_loss, hist_loss, idx):
            mm = model_loss[idx]
            hh = hist_loss[idx]
            ok = np.isfinite(mm) & np.isfinite(hh)

            if not ok.any():
                return np.nan

            denom = np.mean(hh[ok])

            if denom <= EPS:
                return np.nan

            return float(
                1.0 - np.mean(mm[ok]) / denom
            )

        full_idx = np.arange(n_dates)

        evt_skill_obs = skill(
            evt_model_b,
            evt_hist_b,
            full_idx,
        )
        fix_skill_obs = skill(
            fix_model_b,
            fix_hist_b,
            full_idx,
        )

        observed_difference = (
            evt_skill_obs - fix_skill_obs
        )

        rng = np.random.default_rng(
            SEED
            + 7000 * h
            + sum(ord(c) for c in model_family)
        )

        differences = []

        for _ in range(BOOTSTRAP_REPS):
            idx = moving_block_sample_indices(
                n_dates,
                min(BOOTSTRAP_BLOCK, n_dates),
                rng,
            )

            evt_skill = skill(
                evt_model_b,
                evt_hist_b,
                idx,
            )

            fix_skill = skill(
                fix_model_b,
                fix_hist_b,
                idx,
            )

            if np.isfinite(evt_skill) and np.isfinite(fix_skill):
                differences.append(
                    evt_skill - fix_skill
                )

        differences = np.asarray(differences)

        definition_skill_bootstrap_rows.append({
            "ModelFamily": model_family,
            "Horizon": h,
            "EVT_BrierSkill": evt_skill_obs,
            "Fixed_BrierSkill": fix_skill_obs,
            "SkillDifference_EVTminusFixed": observed_difference,
            "SkillDifference_CI2.5": float(
                np.quantile(differences, 0.025)
            ),
            "SkillDifference_CI97.5": float(
                np.quantile(differences, 0.975)
            ),
            "Probability_EVT_Skill_Higher": float(
                np.mean(differences > 0)
            ),
            "BootstrapReps": int(len(differences)),
            "BlockLength": BOOTSTRAP_BLOCK,
        })

definition_skill_bootstrap = pd.DataFrame(
    definition_skill_bootstrap_rows
)

definition_skill_bootstrap.to_csv(
    TABLE_DIR / "Table_314_H2_EVT_vs_Fixed_BrierSkill_Bootstrap.csv",
    index=False,
)

# =============================================================================
# 32. FEATURE IMPORTANCE FOR PRIMARY FULL XGBOOST
# =============================================================================

if all_importance:
    importance = pd.concat(
        all_importance,
        ignore_index=True,
    )
else:
    importance = pd.DataFrame()

if not importance.empty:
    importance.to_csv(
        TABLE_DIR / "Table_315_All_XGBoost_Gain_Importance.csv",
        index=False,
    )

    primary_importance = importance[
        (importance["Definition"] == "EVT_0025")
        & (importance["Variant"] == "Full")
    ].copy()

    def recover_origin(feature_name: str) -> str:
        if feature_name in EVENT_HISTORY_FEATURE_NAMES:
            return "__EVENT_HISTORY__"

        if "__" in feature_name:
            tail = feature_name.split("__", 1)[1]

            if tail.startswith("MISS__"):
                tail = tail.replace("MISS__", "", 1)

            if tail in sectors:
                return "__SECTOR__"

            return tail

        return feature_name

    primary_importance["OriginFeature"] = (
        primary_importance["Feature"]
        .astype(str)
        .map(recover_origin)
    )

    aggregated_importance = (
        primary_importance
        .groupby(
            ["Horizon", "OriginFeature"],
            as_index=False,
        )["Gain"]
        .sum()
    )

    aggregated_importance["GainShare"] = (
        aggregated_importance["Gain"]
        / aggregated_importance.groupby(
            "Horizon"
        )["Gain"].transform("sum")
    )

    aggregated_importance.to_csv(
        TABLE_DIR / "Table_316_Primary_XGBoost_Aggregated_Importance.csv",
        index=False,
    )
else:
    aggregated_importance = pd.DataFrame()

# =============================================================================
# 33. SECTOR/HORIZON HETEROGENEITY FOR FULL XGBOOST / DYNAMIC LOGIT
# =============================================================================

heterogeneity_rows = []

for model_family in [
    "XGBoost",
    "DynamicLogit",
]:
    predictions = component_results[
        (model_family, "Full")
    ]

    for h in HORIZONS:
        for s, sector in enumerate(sectors):
            metrics = probability_metrics(
                primary_targets[h][test_date_mask, s],
                predictions["Test"][h][test_date_mask, s],
            )

            no_garch_metrics = probability_metrics(
                primary_targets[h][test_date_mask, s],
                component_results[
                    (model_family, "NoGARCH")
                ]["Test"][h][test_date_mask, s],
            )

            no_vol_metrics = probability_metrics(
                primary_targets[h][test_date_mask, s],
                component_results[
                    (model_family, "NoVolatilityGroup")
                ]["Test"][h][test_date_mask, s],
            )

            heterogeneity_rows.append({
                "ModelFamily": model_family,
                "Sector": sector,
                "Horizon": h,
                "FullBrier": metrics["Brier"],
                "FullPR_AUC": metrics["PR_AUC"],
                "NoGARCHBrier": no_garch_metrics["Brier"],
                "NoVolatilityBrier": no_vol_metrics["Brier"],
                "GARCHIncrement_Brier": (
                    no_garch_metrics["Brier"]
                    - metrics["Brier"]
                ),
                "VolatilityGroupIncrement_Brier": (
                    no_vol_metrics["Brier"]
                    - metrics["Brier"]
                ),
            })

heterogeneity = pd.DataFrame(
    heterogeneity_rows
)

heterogeneity.to_csv(
    TABLE_DIR / "Table_317_Sector_Horizon_Volatility_Heterogeneity.csv",
    index=False,
)

# =============================================================================
# 34. PREDICTION EXPORT
# =============================================================================

prediction_rows = []

for split_name, split_mask in [
    ("Validation", validation_date_mask),
    ("Test", test_date_mask),
]:
    for t in np.where(split_mask)[0]:
        for s, sector in enumerate(sectors):
            row = {
                "Date": dates[t],
                "Split": split_name,
                "Sector": sector,
            }

            # Primary component-ablation predictions.
            for h in HORIZONS:
                row[f"EVT0025_Target_{h}"] = (
                    primary_targets[h][t, s]
                )

                for model_family in [
                    "XGBoost",
                    "DynamicLogit",
                ]:
                    for variant in ABLATION_DROP_GROUPS:
                        row[
                            f"{model_family}__{variant}__P{h}"
                        ] = component_results[
                            (model_family, variant)
                        ][split_name][h][t, s]

            prediction_rows.append(row)

component_prediction_panel = pd.DataFrame(
    prediction_rows
)

component_prediction_panel.to_csv(
    DATA_DIR / "phase3_component_ablation_predictions.csv.gz",
    index=False,
    compression="gzip",
)

# Crash-definition prediction panel.
definition_prediction_rows = []

for split_name, split_mask in [
    ("Validation", validation_date_mask),
    ("Test", test_date_mask),
]:
    for t in np.where(split_mask)[0]:
        for s, sector in enumerate(sectors):
            row = {
                "Date": dates[t],
                "Split": split_name,
                "Sector": sector,
            }

            for definition_name in event_definitions:
                for h in HORIZONS:
                    row[
                        f"{definition_name}__Target_{h}"
                    ] = event_definitions[
                        definition_name
                    ]["targets"][h][t, s]

                    row[
                        f"{definition_name}__Historical__P{h}"
                    ] = historical_probabilities[
                        definition_name
                    ]["ExpandingHistorical"][h][t, s]

                    for model_family in [
                        "XGBoost",
                        "DynamicLogit",
                    ]:
                        row[
                            f"{definition_name}__{model_family}__P{h}"
                        ] = robustness_results[
                            (model_family, definition_name)
                        ][split_name][h][t, s]

            definition_prediction_rows.append(row)

definition_prediction_panel = pd.DataFrame(
    definition_prediction_rows
)

definition_prediction_panel.to_csv(
    DATA_DIR / "phase3_crash_definition_predictions.csv.gz",
    index=False,
    compression="gzip",
)

# =============================================================================
# 35. FIGURES
# =============================================================================

# Figure 1: XGBoost component Brier degradation vs Full.
plot_component = component_skill[
    (component_skill["ModelFamily"] == "XGBoost")
    & (component_skill["Variant"] != "Full")
].copy()

fig, ax = plt.subplots(figsize=(10, 6))

for variant, group in plot_component.groupby("Variant"):
    group = group.sort_values("Horizon")

    ax.plot(
        group["Horizon"],
        group["BrierDegradation_vs_Full"],
        marker="o",
        label=variant,
    )

ax.axhline(0.0, linewidth=1)
ax.set_xlabel("Forecast horizon (trading days)")
ax.set_ylabel("Ablated Brier minus Full Brier")
ax.set_xticks(HORIZONS)
ax.set_title("Incremental Predictor-Group Contribution — XGBoost")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_301_XGBoost_Component_Ablation_Brier.png",
    dpi=300,
)
plt.close(fig)

# Figure 2: H1 volatility contribution by model.
plot_h1 = component_skill[
    component_skill["Variant"].isin(
        ["NoGARCH", "NoVolatilityGroup"]
    )
].copy()

fig, ax = plt.subplots(figsize=(10, 6))

for (model_family, variant), group in plot_h1.groupby(
    ["ModelFamily", "Variant"]
):
    group = group.sort_values("Horizon")

    ax.plot(
        group["Horizon"],
        group["BrierDegradation_vs_Full"],
        marker="o",
        label=f"{model_family}: {variant}",
    )

ax.axhline(0.0, linewidth=1)
ax.set_xlabel("Forecast horizon (trading days)")
ax.set_ylabel("Ablated Brier minus Full Brier")
ax.set_xticks(HORIZONS)
ax.set_title("Incremental Dynamic-Volatility Forecast Contribution")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_302_H1_Volatility_Contribution.png",
    dpi=300,
)
plt.close(fig)

# Figure 3: Crash-definition Brier skill vs own historical benchmark.
plot_def_skill = definition_skill[
    definition_skill["ModelFamily"].isin(
        ["XGBoost", "DynamicLogit"]
    )
].copy()

fig, ax = plt.subplots(figsize=(10, 6))

for (definition_name, model_family), group in plot_def_skill.groupby(
    ["Definition", "ModelFamily"]
):
    group = group.sort_values("Horizon")

    ax.plot(
        group["Horizon"],
        group["BrierSkill_vs_OwnHistorical"],
        marker="o",
        label=f"{definition_name}: {model_family}",
    )

ax.axhline(0.0, linewidth=1)
ax.set_xlabel("Forecast horizon (trading days)")
ax.set_ylabel("Brier skill vs definition-specific historical benchmark")
ax.set_xticks(HORIZONS)
ax.set_title("Crash-Definition Forecastability Robustness")
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_303_Crash_Definition_Brier_Skill.png",
    dpi=300,
)
plt.close(fig)

# Figure 4: EVT vs fixed Brier-skill difference.
fig, ax = plt.subplots(figsize=(9, 6))

for model_family, group in definition_skill_bootstrap.groupby("ModelFamily"):
    group = group.sort_values("Horizon")

    ax.plot(
        group["Horizon"],
        group["SkillDifference_EVTminusFixed"],
        marker="o",
        label=model_family,
    )

ax.axhline(0.0, linewidth=1)
ax.set_xlabel("Forecast horizon (trading days)")
ax.set_ylabel("Brier-skill difference: EVT minus Fixed")
ax.set_xticks(HORIZONS)
ax.set_title("EVT vs Fixed-Threshold Forecastability")
ax.legend()
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_304_H2_EVT_vs_Fixed_Skill.png",
    dpi=300,
)
plt.close(fig)

# Figure 5: Sector/horizon GARCH increment heatmap for XGBoost.
heat_data = heterogeneity[
    heterogeneity["ModelFamily"] == "XGBoost"
].pivot(
    index="Sector",
    columns="Horizon",
    values="GARCHIncrement_Brier",
)

fig, ax = plt.subplots(figsize=(9, 7))
image = ax.imshow(
    heat_data.to_numpy(),
    aspect="auto",
)
ax.set_xticks(range(len(heat_data.columns)))
ax.set_xticklabels(
    [str(int(h)) for h in heat_data.columns]
)
ax.set_yticks(range(len(heat_data.index)))
ax.set_yticklabels(heat_data.index)
ax.set_xlabel("Forecast horizon")
ax.set_ylabel("Sector")
ax.set_title("Sector/Horizon Increment from GJR-GARCH Volatility — XGBoost")
fig.colorbar(
    image,
    ax=ax,
    label="NoGARCH Brier minus Full Brier",
)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_305_H5_Sector_Horizon_GARCH_Contribution.png",
    dpi=300,
)
plt.close(fig)

# =============================================================================
# 36. EXCEL WORKBOOK
# =============================================================================

excel_path = (
    EXCEL_DIR
    / "Python_Phase3_Ablations_Logit_CrashRobustness.xlsx"
)

if importlib.util.find_spec("xlsxwriter") is not None:
    excel_engine = "xlsxwriter"
elif importlib.util.find_spec("openpyxl") is not None:
    excel_engine = "openpyxl"
else:
    excel_engine = None

excel_tables = {
    "Fixed Thresholds": fixed_threshold_table,
    "Target Audit": target_audit,
    "EVT Fixed Overlap": event_overlap,
    "Ablation Design": ablation_design,
    "Fit Metadata": fit_metadata,
    "Component Metrics": component_metrics,
    "Component Contribution": component_skill,
    "Component Bootstrap": component_bootstrap,
    "H1 Volatility": h1_bootstrap,
    "Definition Metrics": robustness_metrics,
    "Definition Skill": definition_skill,
    "H2 EVT vs Fixed": definition_skill_bootstrap,
    "Sector Heterogeneity": heterogeneity,
}

if not aggregated_importance.empty:
    excel_tables[
        "XGB Importance"
    ] = aggregated_importance

if excel_engine is not None:
    try:
        with pd.ExcelWriter(
            excel_path,
            engine=excel_engine,
        ) as writer:

            for sheet_name, dataframe in excel_tables.items():
                dataframe.to_excel(
                    writer,
                    sheet_name=sheet_name[:31],
                    index=False,
                )

            if excel_engine == "xlsxwriter":
                workbook = writer.book
                header_format = workbook.add_format({
                    "bold": True,
                    "text_wrap": True,
                    "valign": "top",
                    "border": 1,
                })

                for _, worksheet in writer.sheets.items():
                    worksheet.freeze_panes(1, 0)
                    worksheet.set_row(
                        0,
                        28,
                        header_format,
                    )
                    worksheet.set_column(
                        0,
                        30,
                        16,
                    )

            else:
                for _, worksheet in writer.sheets.items():
                    worksheet.freeze_panes = "A2"

        log(
            f"Excel workbook saved using engine={excel_engine}: {excel_path}"
        )

    except Exception as exc:
        log(f"WARNING: Excel export failed: {repr(exc)}")
else:
    log(
        "WARNING: xlsxwriter/openpyxl unavailable; CSV outputs remain complete."
    )

# =============================================================================
# 37. FINAL INTEGRITY CHECKS
# =============================================================================

# Monotonicity check for all fitted probability stores.
def count_coherence_violations(
    probability_store: dict,
) -> int:
    violations = 0

    for split_name in ["Validation", "Test"]:
        stack = np.stack(
            [
                probability_store[split_name][h]
                for h in HORIZONS
            ],
            axis=-1,
        )

        complete = np.all(
            np.isfinite(stack),
            axis=-1,
        )

        if complete.any():
            diff = np.diff(
                stack[complete],
                axis=-1,
            )
            violations += int(
                np.sum(
                    np.any(diff < -1e-8, axis=-1)
                )
            )

    return violations

coherence_violations = 0

for store in component_results.values():
    coherence_violations += count_coherence_violations(store)

for store in robustness_results.values():
    coherence_violations += count_coherence_violations(store)

integrity_checks = [
    (
        "Crash_Main equals primary 2.5% EVT label",
        primary_mismatch == 0,
    ),
    (
        "All four event definitions available",
        set(event_definitions.keys())
        == {"EVT_001", "EVT_0025", "EVT_005", "Fixed_0025"},
    ),
    (
        "Fixed thresholds estimated only from Train",
        bool(np.all(np.isfinite(fixed_thresholds))),
    ),
    (
        "No crash/future probability columns in generic base predictors",
        not any(
            (
                "Crash" in c
                or "EventTime" in c
                or "CensorTime" in c
                or "Prob_" in c
            )
            for c in base_feature_columns
        ),
    ),
    (
        "Preprocessed feature panel finite",
        bool(np.all(np.isfinite(X_panel))),
    ),
    (
        "All component variants completed",
        len(component_results)
        == 2 * len(ABLATION_DROP_GROUPS),
    ),
    (
        "All crash-definition robustness fits completed",
        len(robustness_results)
        == 2 * len(event_definitions),
    ),
    (
        "No fitted horizon-coherence violations",
        coherence_violations == 0,
    ),
    (
        "Test contains both classes for primary target at all horizons",
        all(
            len(
                np.unique(
                    primary_targets[h][test_date_mask][
                        np.isfinite(
                            primary_targets[h][test_date_mask]
                        )
                    ]
                )
            ) == 2
            for h in HORIZONS
        ),
    ),
    (
        "H1 volatility bootstrap table nonempty",
        len(h1_bootstrap) > 0,
    ),
    (
        "H2 EVT-vs-fixed skill bootstrap table complete",
        len(definition_skill_bootstrap)
        == 2 * len(HORIZONS),
    ),
]

integrity_table = pd.DataFrame(
    integrity_checks,
    columns=["Check", "Passed"],
)

integrity_table.to_csv(
    TABLE_DIR / "Table_318_Final_Integrity_Checks.csv",
    index=False,
)

if not integrity_table["Passed"].all():
    failed = integrity_table.loc[
        ~integrity_table["Passed"],
        "Check",
    ].tolist()

    raise RuntimeError(
        f"Phase 3 integrity checks failed: {failed}"
    )

# =============================================================================
# 38. METADATA
# =============================================================================

metadata = {
    "ProjectTitle": (
        "Forecasting Sectoral Crash Risk and Contagion: "
        "Integrating Dynamic Volatility, Extreme-Value Modelling "
        "and Graph Deep Learning"
    ),
    "PythonPhase": "3",
    "ScriptVersion": "3.0",
    "GeneratedAt": datetime.now().isoformat(),
    "Seed": SEED,
    "InputPhase1Zip": INPUT_ZIP_PATH.name,
    "PrimaryCrashDefinition": "EVT_0025",
    "RobustnessCrashDefinitions": [
        "EVT_001",
        "EVT_005",
        "Fixed_0025",
    ],
    "Models": [
        "XGBoost",
        "DynamicLogit",
    ],
    "ComponentAblations": list(ABLATION_DROP_GROUPS.keys()),
    "Horizons": HORIZONS,
    "Lookback": LOOKBACK,
    "BootstrapReps": BOOTSTRAP_REPS,
    "BootstrapBlockLength": BOOTSTRAP_BLOCK,
    "FixedThresholdRule": (
        "Sector-specific 2.5% unconditional Train return quantile; "
        "same observation mask as primary EVT definition"
    ),
    "TestUsedForModelSelection": False,
    "Phase2_2ArchitectureStatus": "Frozen; not redesigned in Phase 3",
}

with open(
    DATA_DIR / "python_phase3_metadata.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        metadata,
        handle,
        indent=2,
        default=str,
    )

with open(
    MODEL_DIR / "phase3_preprocessing_bundle.pkl",
    "wb",
) as handle:
    pickle.dump(
        {
            "base_feature_columns": base_feature_columns,
            "model_feature_names": model_feature_names,
            "feature_origin": feature_origin,
            "train_median": train_median,
            "train_mean": train_mean,
            "train_std": train_std,
            "fixed_thresholds": fixed_thresholds,
            "sectors": sectors,
            "lookback": LOOKBACK,
        },
        handle,
    )

# =============================================================================
# 39. CLEAN EXTRACTED INPUT / ZIP ALL OUTPUTS
# =============================================================================

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

zip_output = (
    ZIP_DIR
    / "Python_Phase3_Ablations_Logit_CrashRobustness_All_Outputs.zip"
)

if zip_output.exists():
    zip_output.unlink()

with zipfile.ZipFile(
    zip_output,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    for path in OUTPUT_ROOT.rglob("*"):
        if path.is_file() and path != zip_output:
            archive.write(
                path,
                arcname=path.relative_to(OUTPUT_ROOT),
            )

log(f"All Phase-3 outputs zipped to: {zip_output}")

# =============================================================================
# 40. CONSOLE SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print("PYTHON PHASE 3 COMPLETED SUCCESSFULLY")
print("=" * 100)

print(f"Primary definition: EVT_0025")
print(
    "Robustness definitions: EVT_001, EVT_005, Fixed_0025"
)
print(
    "Models: XGBoost + Dynamic ridge-logit"
)

print("\nH1 — volatility ablation summary:")
print(
    h1_bootstrap[
        [
            "ModelFamily",
            "Ablation",
            "Horizon",
            "BrierDiff_FullMinusAblation",
            "BrierDiff_CI2.5",
            "BrierDiff_CI97.5",
            "Probability_Full_Better_Brier",
        ]
    ].to_string(index=False)
)

print("\nH2 — EVT vs fixed-threshold Brier-skill summary:")
print(
    definition_skill_bootstrap[
        [
            "ModelFamily",
            "Horizon",
            "EVT_BrierSkill",
            "Fixed_BrierSkill",
            "SkillDifference_EVTminusFixed",
            "SkillDifference_CI2.5",
            "SkillDifference_CI97.5",
            "Probability_EVT_Skill_Higher",
        ]
    ].to_string(index=False)
)

print("\nFinal ZIP to send back for review:")
print(f"  {zip_output}")
print("=" * 100)

# =============================================================================
# 41. AUTOMATIC COLAB DOWNLOAD
# =============================================================================

try:
    from google.colab import files

    print(
        "\nStarting browser download of "
        "Python_Phase3_Ablations_Logit_CrashRobustness_All_Outputs.zip ..."
    )

    files.download(
        str(zip_output)
    )

except ImportError:
    print(
        "\nNot running in Google Colab. "
        f"Retrieve the ZIP manually from: {zip_output}"
    )

# =============================================================================
# END OF PYTHON PHASE 3
# =============================================================================


[2026-09-01 18:18:31] ====================================================================================================
[2026-09-01 18:18:31] PYTHON PHASE 3 START — ABLATIONS / DYNAMIC LOGIT / CRASH-DEFINITION ROBUSTNESS
[2026-09-01 18:18:31] Python=3.13.15; platform=Linux-6.6.122+-x86_64-with-glibc2.35; seed=20260901
[2026-09-01 18:18:31] xgboost=3.4.1

Please upload the frozen Phase-1 file:
    Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip



Saving Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip to Python_Phase1_v1_2_From_R_Handoff_All_Outputs (1).zip
[2026-09-01 18:19:40] Validated uploaded Phase-1 ZIP: /content/Python_Phase1_v1_2_From_R_Handoff_All_Outputs (1).zip
[2026-09-01 18:19:40] Accepted Phase-1 ZIP: /content/Python_Phase1_v1_2_From_R_Handoff_All_Outputs (1).zip
[2026-09-01 18:19:40] Loaded 14,928 rows; 2,488 dates; 6 sectors; Train dates=1497, Validation dates=496, Test dates=495.
[2026-09-01 18:19:41] Leakage-screened base numeric features=47
[2026-09-01 18:19:41] Generic forecast-origin coordinates=14,574
[2026-09-01 18:19:41] Building reusable summary matrix: 14,574 rows x 747 columns.
[2026-09-01 18:19:48] Fitting XGBoost: definition=EVT_0025, variant=Full, h=1
[2026-09-01 18:19:55] Fitting XGBoost: definition=EVT_0025, variant=Full, h=5
[2026-09-01 18:20:00] Fitting XGBoost: definition=EVT_0025, variant=Full, h=10
[2026-09-01 18:20:01] Fitting XGBoost: definition=EVT_0025, variant=Full, h=22
[2026-09-01 18

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>